# T7SL Traffic Counter — readable implementation V2
Actual source is visible below. Fresh GPU measurements from V1 are resumed when valid; only missing videos run inference. Role-context bug repaired at adapter boundary; counting rules unchanged. G2 provisional geometry. No accuracy claim.

Sections: configuration → dependencies → module source → input/geometry preview → model assets → smoke/full batch → output pivot.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess
ROOT=Path('/kaggle/working/t7sl_readable_v2_source')
ROOT.mkdir(exist_ok=False)
os.chdir(ROOT)


## 1. Explicit experiment configuration
Model, tracker and counting controls remain frozen. Timeout is60 source frames. Geometry is an engineering candidate; review is still required.

In [ ]:
CONFIG_FILES = {'results/t7sl_modular/config/fresh_request.json': {'artifacts': {'detector': {'path': '/workspace/scratch/modularization/assets/yolo26m.pt', 'sha256': '401cea9ab23ad19246ff7744859816bc599f350e93c9dd30367b6f0a0745d0b7'}, 'reid': {'path': '/workspace/scratch/modularization/assets/yolo26n-reid.onnx', 'sha256': '8529c383197ae4c468eda535d1b165f8b4162cf17bf5fbcff49c7cb6455bc0bb'}, 'video': {'path': '/workspace/scratch/modularization/assets/TGDI-WEEKDAY-SIANG.mp4', 'sha256': '5ffeec3d74e51eed43f45050c44fc019c989a8f2d5659c0bf6ccf3370284f4d1'}}, 'conf': 0.25, 'device': 'cuda:0', 'frame_limit': 32, 'half': False, 'imgsz': 640, 'input_root': '/workspace/scratch/modularization/assets', 'iou': 0.7, 'preprocessing': 'RAW', 'schema_version': 'fresh_inference_request_v1', 'seed': 0, 'start_frame': 0, 'threads': 4, 'tracker': {'appearance_thresh': 0.8, 'fuse_score': True, 'gmc_method': 'none', 'match_thresh': 0.8, 'new_track_thresh': 0.25, 'proximity_thresh': 0.5, 'track_buffer': 30, 'track_high_thresh': 0.25, 'track_low_thresh': 0.1}, 'ultralytics_version': '8.4.175'}, 'results/t7sl_modular/config/policy.json': {'base_b_v2_policy_sha256': '9cb4279c6e17f0f725f67e1a75fdaeb0d1b3fc9106378440be0de4fbc3b0522f', 'base_b_v3_policy_sha256': '59b8d34ceed6a7d4c910f891b6ab6464a2283b10a26c6050eb58d950dfb4b64a', 'base_b_v4_policy_sha256': '36950946d5de4e8588c1e596c16aa46df16ce2073b1395b18d2e66f9696cc898', 'base_b_v5_policy_sha256': '3defaf135c1a0fe0939e2454496327e8d977c7e42afa7d30f61022b6c309ea06', 'base_b_v6_policy_sha256': '88123a9441a6ea75aa23c033858327baf1f08c86b8390133bd3832df52b16384', 'category_policy': 'band_majority_ties_pending', 'component_policy': 'RETAINED_SUPPORTED_COMPONENT_OF_LOCAL_FRAMES_DISPLAY_UNION_RAW_VEHICLE_BC', 'decision_document_sha256': 'd90195ffa37790afda22a512cd39d4af83b9daba8b674b71586e4c86be5f9e54', 'display_category_prefixes': {'bus': 'B', 'car': 'C', 'motorcycle': 'M', 'pedestrian': 'P', 'truck': 'T'}, 'eligibility': {'current_observation_guard': 'NO_TIMEOUT_WHILE_CURRENT_MEASUREMENT_REMAINS_VISIBLE', 'finite_projection_required': True, 'min_contiguous_observations': 3, 'min_toward_progress_px': 25.0, 'numeric_provenance': '25px reuses current native640 margin; 3 observations is an explicit new research control, not a tuned default', 'progress_basis': 'NET_DISPLACEMENT_OVER_CONTIGUOUS_MEASURED_SOURCE_FRAMES', 'schema_version': 'phase8_b_v4_eligibility_policy_v1', 'status': 'EXPERIMENTAL_CANDIDATE_NOT_VALIDATED_PRODUCTION_DEFAULT'}, 'execution': 'OFFLINE_REPLAY_FROZEN_MODEL_CACHE_NO_INFERENCE', 'execution_branch': 'machine', 'identity': {'ambiguity_disposition': 'INVALID_PLUS_ZERO_BY_COUNTING_POLICY_NOT_IDENTITY_TRUTH', 'ambiguity_policy': 'SCOPED_HIGH_IOU_ALL_VEHICLE_CLASSES_NO_AUTOMATIC_ALIAS', 'appearance_features_used': False, 'automatic_handoff': 'DISABLED_WITHOUT_POSITIVE_SOURCE_BOUND_ASSOCIATION', 'category_is_identity_gate': False, 'component_bc_inside_fraction': 1.0, 'component_center_limit': 0.51, 'component_motion_limit': 0.15, 'component_motion_profile': 'median_temporal_v1', 'control_provenance': {'component_controls': 'FROZEN_PHASE8_COMPONENT_3_0.15_1.0_0.51_REUSED', 'component_motion_profile': 'NEW_PROPOSAL_MEDIAN_VS_RETAINED_MAX_ABLATION', 'max_center_jump_norm': 'NEW_PROPOSAL_CONTEXT_DISCONTINUITY_CONTROL_NOT_IDENTITY_PROOF', 'max_unobserved_frames': 'EXISTING_TIMEOUT_HORIZON_REUSED_AS_EXPERIMENT_NOT_VALIDATED_IDENTITY_LIFECYCLE', 'possible_handoff_controls': 'FROZEN_SAME_VEHICLE_CENTER_0.1_IOU_0.59_REUSED_FOR_HOLD_ONLY'}, 'cross_category_candidate_policy': 'EXACT_FRAME_HIGH_IOU_HOLDS_WEAK_COVARIATION_AUDIT_ONLY', 'cross_category_candidate_profile': 'p0_v2_high_iou_hold_weak_covariation_audit', 'direction_policy': 'OPPOSITE_OBSERVED_DIRECTION_NOT_DUPLICATE', 'historical_unit_policy': 'HYPOTHESIS_ONLY_ISOLATE_RAW_CONTEXT_UNTIL_SCOPED_POSITIVE_PROOF', 'max_center_jump_norm': 1.0, 'max_unobserved_frames': 60, 'min_component_run': 3, 'negative_relation_policy': 'DISJOINT_COPRESENCE_IS_A_CONSTRAINT_NOT_GLOBAL_PHYSICAL_TRUTH', 'numeric_provenance_v6': 'REUSE_BV5_POSSIBLE_HANDOFF_IOU_AND_COMPONENT_CENTER_CONTROLS_NO_NEW_THRESHOLD', 'possible_handoff_center_limit': 0.1, 'possible_handoff_min_iou': 0.59, 'role_propagation': 'EXACT_OBSERVED_CONSECUTIVE_SUPPORTED_RUN_ONLY', 'same_unit_scope': 'EXACT_FRAME_PROOF_OR_SOURCE_BOUND_REVIEW_NO_RETROACTIVE_ASSIGNMENT', 'schema_version': 'phase8_b_v6_identity_policy_v1', 'status': 'EXPERIMENTAL_CANDIDATE_NOT_VALIDATED_PRODUCTION_DEFAULT', 'unknown_person_is_pedestrian': False, 'unverified_handoff_policy': 'EXACT_OBSERVED_ENDPOINTS_NO_GAP_CROSSING_PROOF'}, 'identity_policy': 'FRESH_RAW_TRACK_CONTEXT_PLUS_EXISTING_BV7_GUARDS_NO_LEGACY_REVIEW', 'line_margin_half_width_px': 25.0, 'p0_counting': {'anchor_quality': 'WHOLE_BOX_TOWARD_SUPPORT_OVER_CONTIGUOUS_RUN', 'apply_inferred_anchor_guard': True, 'apply_measured_anchor_guard': False, 'band_born_route': 'UNCHANGED_OPEN_P1', 'category_conflict': 'PENDING_NOT_ARBITRARY_CATEGORY_WINNER', 'finalization': 'VERIFIED_EPISODE_SCOPED_RELATIONS_ONLY', 'minimum_corner_progress_px_for_measured': 0.0, 'relation_support': 'WITHIN_BOTH_CANDIDATE_WINDOWS_NO_FUTURE_BACKPROPAGATION', 'schema_version': 'phase8_b_v5_p0_counting_policy_v1', 'status': 'EXPERIMENTAL_RESEARCH_NOT_PRODUCTION_VALIDATED', 'timeout_corner_progress': 'REUSE_EXPLICIT_ELIGIBILITY_DISPLACEMENT'}, 'passage_display_id': {'bus_prefix': 'B', 'digits': 4, 'first_digit_A_TO_B': [2, 4, 6, 8], 'first_digit_B_TO_A': [1, 3, 5, 7, 9], 'leading_zero': False, 'prefixes': {'bus': 'B', 'car': 'C', 'motorcycle': 'M', 'pedestrian': 'P', 'truck': 'T'}, 'scope': 'PER_VIDEO_PASSAGE_NOT_PHYSICAL_IDENTITY'}, 'pedestrian_route_policy_status': 'CONSERVATIVE_RESEARCH_CONTROL_PENDING_USER_PREFERENCE_NOT_IMPLIED_APPROVAL', 'pedestrian_routes': 'measured_only', 'person_role_policy': 'POSITIVE_PEDESTRIAN_ONLY_RIDER_PLUS_ZERO_UNKNOWN_ROLE_QUEUE', 'policy_version': 'bounded_unverified_handoff_publication_v1', 'qc_adjudications_used_as_machine_predictions': False, 'return_policy': 'NEW_MEASURED_REVERSE_PASSAGE_ALLOWED', 'return_timeout_policy': 'require_new_passage_evidence', 'return_timeout_policy_status': 'CONSERVATIVE_RESEARCH_CONTROL_PENDING_USER_PREFERENCE_NOT_IMPLIED_APPROVAL', 'schema_version': 'phase9_b_v7_handoff_policy_v1', 'selected_profile': 'b_v7_offline_bounded_trajectory_conflict', 'selection_basis': 'Declared research controls; no model tuning or QC labels as predictions', 'source_template_sha256': '13f774f029c4d5e724d24ffb5fb6767b8cd62d96d3b1431e76bc0b849f9a064f', 'status': 'RESEARCH_SCOPED_CONFLICT_NOT_PHYSICAL_IDENTITY_VALIDATION', 'supported_categories': ['motorcycle', 'car', 'truck', 'bus', 'pedestrian'], 'timeout_clock': 'last_observed_origin', 'timeout_evidence_kind': 'ESTIMATED_TIMEOUT_NOT_PROVEN_ID_CHANGE', 'timeout_frames': 60, 'user_timer_clarification': 'Sejak terakhir terlihat di polygon asal', 'v6_counting': {'ambiguous_unit_decision': 'INVALID_PLUS_ZERO_PER_PASSAGE', 'appearing_away': 'NOT_ELIGIBLE_FROM_APPEARANCE_ALONE', 'bus_display_prefix': 'B', 'business_taxonomy': {'angkot': 'car', 'bus': 'bus', 'cargo_van': 'car', 'minivan': 'car', 'pickup': 'truck'}, 'opposite_direction': 'DISTINCT_VALID_PASSAGES_NOT_DUPLICATES', 'schema_version': 'phase8_b_v6_counting_policy_v1', 'status': 'USER_POLICY_IMPLEMENTED_RESEARCH_NOT_PRODUCTION_VALIDATED', 'subtype_recognition': 'POSITIVE_SOURCE_BOUND_EVIDENCE_REQUIRED', 'timeout': 'ELIGIBLE_LAST_ORIGIN_PLUS60_NO_RELEVANT_IDENTITY_CONFLICT', 'unknown_person': 'KEEP_PENDING_NOT_AUTOMATIC_PEDESTRIAN_OR_INVALID'}, 'v7_handoff': {'direction_gate': 'OBSERVED_LINE_NORMAL_MOTION_AND_EVENT_DIRECTIONS_AGREE', 'disposition': 'EXISTING_INVALID_PLUS_ZERO_FOR_EXPLICIT_PAIR_ONLY', 'fit': 'OLS_CENTER_CONSTANT_VELOCITY_BIDIRECTIONAL', 'gap_limit': 'INHERIT_EXISTING_TIMEOUT_SOURCE_FRAMES', 'identity_effect': 'UNVERIFIED_CONFLICT_NO_ALIAS_NO_MEASURED_CROSSING_UPGRADE', 'schema_version': 'phase8_b_v7_handoff_policy_v1', 'scope': 'ACTIVE_CANDIDATE_PAIR_AT_LEAST_ONE_ESTIMATE_NO_OVERLAPPING_CONTEXT_LIFETIMES', 'spatial_gate': 'BIDIRECTIONAL_TRANSLATED_ENDPOINT_BOX_POSITIVE_AREA_OVERLAP', 'spatial_rationale': 'BOTH_OBSERVED_BOX_EXTENTS_BOUND_SPATIAL_CONFLICT_NOT_VALIDATED_IDENTITY_THRESHOLD', 'window_observations': 5, 'window_rationale': 'FIVE_IS_DECLARED_RESEARCH_SAMPLE_SUPPORT_NOT_TUNED_ON_TARGET_OR_PRODUCTION_DEFAULT', 'window_requirement': 'CONSECUTIVE_OBSERVED_VEHICLE_MEASUREMENTS_BOTH_SIDES'}, 'namespace': 't7sl_fresh_machine_v1'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKEND-SORE.json': {'draft_id': 'T7SL-WEEKEND-SORE-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': '71a9bf7658cad48a098af52e0b07f0c3a9e5bdd4ea613df7305072c11763d17b', 'filename': 'T7SL-WEEKEND-SORE.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKEND-SIANG.json': {'draft_id': 'T7SL-WEEKEND-SIANG-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': 'de266518b4ce024c6a6058c9a839eac3cda9b9b4d7fd36cf7ca732fac8b158d0', 'filename': 'T7SL-WEEKEND-SIANG.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKEND-PAGI.json': {'draft_id': 'T7SL-WEEKEND-PAGI-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': '873130d1134599a037f705a23d26acc5eb0886f3694283823242b23d0a2b2298', 'filename': 'T7SL-WEEKEND-PAGI.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKDAY-PAGI.json': {'draft_id': 'T7SL-WEEKDAY-PAGI-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': '72faff73798d78ab5072647d68637dfd7464244096137e0989fb4a9e9bfc5d4d', 'filename': 'T7SL-WEEKDAY-PAGI.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKDAY-MALAM.json': {'draft_id': 'T7SL-WEEKDAY-MALAM-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': 'c8bfa2ff55d46a06659cb8aa9f168f2d1eb3e6a09d2023e3314797e9a69fe1c2', 'filename': 'T7SL-WEEKDAY-MALAM.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKDAY-SORE.json': {'draft_id': 'T7SL-WEEKDAY-SORE-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': '68e2bec900f9129684f5238a110bb8c617b3bcfee3d827c58b6ee4419329a591', 'filename': 'T7SL-WEEKDAY-SORE.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKEND-MALAM.json': {'draft_id': 'T7SL-WEEKEND-MALAM-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': '6774ae5e7851e8f2097701dddc5e4ce5d6201a3316ada5da2e6b68dbf397559c', 'filename': 'T7SL-WEEKEND-MALAM.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/geometry/T7SL-WEEKDAY-SIANG.json': {'draft_id': 'T7SL-WEEKDAY-SIANG-ENGINEERING-G2', 'status': 'ENGINEERING_PROVISIONAL_REQUIRES_LATER_UI_REVIEW', 'source_video': {'sha256': '2732601fe6f4bb8ccc0bce40c9ef0d8c3100c1808033b8f87489449a6543cc10', 'filename': 'T7SL-WEEKDAY-SIANG.mp4'}, 'normalized': {'line': {'P1': [0.08333333333333333, 0.7083333333333334], 'P2': [0.9930555555555556, 0.4791666666666667]}, 'side_a_polygon': [[0.041666666666666664, 0.4166666666666667], [0.9930555555555556, 0.4166666666666667], [0.9930555555555556, 0.4791666666666667], [0.08333333333333333, 0.7083333333333334]], 'side_b_polygon': [[0.08333333333333333, 0.7083333333333334], [0.9930555555555556, 0.4791666666666667], [0.9930555555555556, 0.9979166666666667], [0.3472222222222222, 0.9979166666666667]]}, 'direction_assumption': 'A_TO_B_APPROACHING_B_TO_A_AWAY_ENGINEERING_VISUAL_HYPOTHESIS', 'claim_type': 'EXPERIMENTAL_NOT_USER_APPROVED_GEOMETRY'}, 'results/t7sl_modular/config/experiment.json': {'schema_version': 't7sl_experiment_v1', 'experiment_id': 't7sl_readable_resume_v2', 'dataset': 'chrisbiran/t7sl-traffic-videos', 'preprocessing': 'RAW', 'filter_production_selection': False, 'geometry_quality': 'G2_ENGINEERING_PROVISIONAL', 'videos': [{'filename': 'T7SL-WEEKDAY-MALAM.mp4', 'sha256': 'c8bfa2ff55d46a06659cb8aa9f168f2d1eb3e6a09d2023e3314797e9a69fe1c2', 'bytes': 22029577, 'width': 720, 'height': 480, 'fps': 29.9460957088034, 'reported_frames': 9082, 'decoded_frames': 9082, 'duration_seconds': 303.27826666666664, 'sample_frames': [0, 4541, 9081]}, {'filename': 'T7SL-WEEKDAY-PAGI.mp4', 'sha256': '72faff73798d78ab5072647d68637dfd7464244096137e0989fb4a9e9bfc5d4d', 'bytes': 21971568, 'width': 720, 'height': 480, 'fps': 29.80365622372413, 'reported_frames': 9015, 'decoded_frames': 9015, 'duration_seconds': 302.4796666666667, 'sample_frames': [0, 4507, 9014]}, {'filename': 'T7SL-WEEKDAY-SIANG.mp4', 'sha256': '2732601fe6f4bb8ccc0bce40c9ef0d8c3100c1808033b8f87489449a6543cc10', 'bytes': 22067789, 'width': 720, 'height': 480, 'fps': 29.80256699007486, 'reported_frames': 9110, 'decoded_frames': 9110, 'duration_seconds': 305.67836666666665, 'sample_frames': [0, 4555, 9109]}, {'filename': 'T7SL-WEEKDAY-SORE.mp4', 'sha256': '68e2bec900f9129684f5238a110bb8c617b3bcfee3d827c58b6ee4419329a591', 'bytes': 21974811, 'width': 720, 'height': 480, 'fps': 29.32652578783342, 'reported_frames': 8899, 'decoded_frames': 8899, 'duration_seconds': 303.44542222222225, 'sample_frames': [0, 4449, 8898]}, {'filename': 'T7SL-WEEKEND-MALAM.mp4', 'sha256': '6774ae5e7851e8f2097701dddc5e4ce5d6201a3316ada5da2e6b68dbf397559c', 'bytes': 22027076, 'width': 720, 'height': 480, 'fps': 28.910926452325402, 'reported_frames': 8796, 'decoded_frames': 8796, 'duration_seconds': 304.24483333333336, 'sample_frames': [0, 4398, 8795]}, {'filename': 'T7SL-WEEKEND-PAGI.mp4', 'sha256': '873130d1134599a037f705a23d26acc5eb0886f3694283823242b23d0a2b2298', 'bytes': 21965595, 'width': 720, 'height': 480, 'fps': 28.314499136339162, 'reported_frames': 8574, 'decoded_frames': 8574, 'duration_seconds': 302.81305555555554, 'sample_frames': [0, 4287, 8573]}, {'filename': 'T7SL-WEEKEND-SIANG.mp4', 'sha256': 'de266518b4ce024c6a6058c9a839eac3cda9b9b4d7fd36cf7ca732fac8b158d0', 'bytes': 22007993, 'width': 720, 'height': 480, 'fps': 29.781679975106627, 'reported_frames': 9052, 'decoded_frames': 9052, 'duration_seconds': 303.9452444444444, 'sample_frames': [0, 4526, 9051]}, {'filename': 'T7SL-WEEKEND-SORE.mp4', 'sha256': '71a9bf7658cad48a098af52e0b07f0c3a9e5bdd4ea613df7305072c11763d17b', 'bytes': 21995195, 'width': 720, 'height': 480, 'fps': 29.139743335584004, 'reported_frames': 8852, 'decoded_frames': 8852, 'duration_seconds': 303.7775555555556, 'sample_frames': [0, 4426, 8851]}]}}
for name,value in CONFIG_FILES.items():
 p=ROOT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(json.dumps(value,indent=2)+"\n")
CONFIG=ROOT/"results/t7sl_modular/config"
print(json.dumps(CONFIG_FILES["results/t7sl_modular/config/experiment.json"],indent=2))


## 2. Runtime dependencies
No model execution in this cell. Asset bytes are verified before inference. The historical Docker image is pinned in Kaggle metadata.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','ultralytics==8.4.175','lap==0.5.13','onnx==1.20.1','onnxruntime-gpu==1.24.4'],check=True)


In [ ]:
for name in ['src/traffic_location', 'src/traffic_location', 'src/traffic_location', 'src/traffic_location/traffic', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic/_kernel', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'src/traffic_location/traffic', 'tools', 'tools']:
 (ROOT/name).mkdir(parents=True,exist_ok=True)


## 3. Executed implementation files
Each cell writes the exact Python source shown. These are the files imported by the final execution cell. They contain detection/tracking, source adaptation, identity, geometry, counting, rendering helpers and batch aggregation. There is no base64 archive or hidden download of implementation code.

### src/traffic_location/__init__.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/__init__.py
"""Versioned engine boundaries. No inference, network or filesystem work on import."""
__version__ = '0.1.0'


### src/traffic_location/artifacts.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/artifacts.py
"""Local immutable artifact boundaries; storage vendor adapters remain external."""
import gzip,hashlib,json
from pathlib import Path

def canonical(value):return json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False,allow_nan=False).encode()
def semantic_hash(value):return hashlib.sha256(canonical(value)).hexdigest()
def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
 return h.hexdigest()
def require(test,message):
 if not test:raise ValueError(message)
def artifact(ref,root):
 require(isinstance(ref,dict) and set(ref)>={'path','sha256'},'Explicit artifact path and SHA256 required')
 p=Path(ref['path']);p=Path(root)/p if not p.is_absolute() else p
 require(not p.is_symlink(),'Symlink artifact is not supported')
 p=p.resolve()
 require(p.is_file(),'Artifact missing/unsafe')
 require(isinstance(ref['sha256'],str) and len(ref['sha256'])==64 and sha(p)==ref['sha256'],'Artifact checksum differs: '+p.name)
 return p
def read(path):
 data=Path(path).read_bytes()
 return json.loads(gzip.decompress(data) if str(path).endswith('.gz') else data)
def write(path,value):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);data=canonical(value)
 path.write_bytes(gzip.compress(data,mtime=0) if path.suffix=='.gz' else data)


### src/traffic_location/persistence.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/persistence.py
"""Atomic local publication and read-only reopen, without rerunning an engine."""
import json,shutil,tempfile
from pathlib import Path
from .artifacts import require,sha,read,write,semantic_hash

def verify_manifest(root):
 root=Path(root).resolve();m=read(root/'manifest.json')
 paths=[r['path'] for r in m['files']]
 require(len(paths)==len(set(paths)),'Duplicate manifest path')
 actual={str(p.relative_to(root)) for p in root.rglob('*') if p.is_file() and p!=root/'manifest.json'}
 require(actual==set(paths),'Missing/unlisted output artifacts')
 for row in m['files']:
  p=root/row['path'];require(p.resolve().is_relative_to(root) and not p.is_symlink(),'Unsafe artifact path')
  require(p.stat().st_size==row['bytes'] and sha(p)==row['sha256'],'Artifact checksum mismatch')
 return m

def publish(output,files,*,metadata=None):
 output=Path(output).resolve();require(not output.exists(),'Output already exists; revisions immutable')
 output.parent.mkdir(parents=True,exist_ok=True)
 stage=Path(tempfile.mkdtemp(prefix='.'+output.name+'-',dir=output.parent))
 try:
  for name,value in files.items():
   p=stage/name;require(p.resolve().is_relative_to(stage.resolve()) and name!='manifest.json','Unsafe output name')
   write(p,value)
  rows=[{'path':str(p.relative_to(stage)),'bytes':p.stat().st_size,'sha256':sha(p)} for p in sorted(stage.rglob('*')) if p.is_file()]
  write(stage/'manifest.json',{'schema_version':'engine_artifacts_v1','files':rows,'metadata':metadata or {}})
  verify_manifest(stage);require(not output.exists(),'Output appeared during build');stage.rename(output)
 except BaseException:
  shutil.rmtree(stage,ignore_errors=True);raise
 return read(output/'manifest.json')

def reopen_traffic(root):
 verify_manifest(root)
 from .traffic.results import validate_native
 native=read(Path(root)/'native.json.gz');result=read(Path(root)/'result.json')
 validate_native(native)
 require(semantic_hash(native)==result['native_semantic_sha256'],'Native/result binding differs')
 return result


### src/traffic_location/traffic/__init__.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/__init__.py
"""Independent engine components; legacy algorithm semantics are preserved."""


### src/traffic_location/traffic/_kernel/__init__.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/__init__.py
"""Independent engine components; legacy algorithm semantics are preserved."""


### src/traffic_location/traffic/_kernel/eligibility.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/eligibility.py
"""User-authorized invalid/+0 policy on the immutable B v5 measured-track engine.

No detector, tracker, ReID or enhancement executes here. A rejected ambiguous
passage is an intentional exclusion, not proof that no traffic passed. Raw BC,
the 60-source-frame clock and genuine measured-return semantics stay in B v5.
"""
from __future__ import annotations
from collections import Counter
from copy import deepcopy
import hashlib
import json
from . import passage as previous
SUPPORTED_CATEGORIES = frozenset(('motorcycle', 'car', 'truck', 'bus', 'pedestrian'))
DISPLAY_PREFIXES = {**previous.DISPLAY_PREFIXES, 'bus': 'B'}
BUSINESS_TAXONOMY = {'pickup': 'truck', 'cargo_van': 'car', 'minivan': 'car', 'angkot': 'car', 'bus': 'bus'}
AMBIGUOUS_KINDS = frozenset(('POSSIBLE_CROSS_CATEGORY_UNIT_OR_COMPONENT', 'POSSIBLE_SCOPED_UNIT_OR_COMPONENT_AMBIGUITY', 'UNVERIFIED_ORIGIN_HANDOFF'))
IDENTITY_REASONS = frozenset(('UNRESOLVED_DUPLICATE_IDENTITY_IN_PASSAGE', 'IDENTITY_CONTINUITY_UNRESOLVED_FOR_TIMEOUT', 'RAW_TRACK_CHANGED_DURING_PASSAGE', 'TIMEOUT_RAW_TRACK_HANDOFF_UNRESOLVED', 'RAW_CONTEXT_CHANGED_WITHOUT_PHYSICAL_CONTINUITY_PROOF', 'ORIGIN_TIMER_CONTEXT_CONTINUITY_UNRESOLVED', 'CROSS_CATEGORY_UNIT_OR_COMPONENT_UNRESOLVED', 'SCOPED_UNIT_AMBIGUITY', 'UNVERIFIED_ORIGIN_HANDOFF'))

def experimental_counting_policy():
    return {'schema_version': 'phase8_b_v6_counting_policy_v1', 'status': 'USER_POLICY_IMPLEMENTED_RESEARCH_NOT_PRODUCTION_VALIDATED', 'ambiguous_unit_decision': 'INVALID_PLUS_ZERO_PER_PASSAGE', 'opposite_direction': 'DISTINCT_VALID_PASSAGES_NOT_DUPLICATES', 'timeout': 'ELIGIBLE_LAST_ORIGIN_PLUS60_NO_RELEVANT_IDENTITY_CONFLICT', 'appearing_away': 'NOT_ELIGIBLE_FROM_APPEARANCE_ALONE', 'unknown_person': 'KEEP_PENDING_NOT_AUTOMATIC_PEDESTRIAN_OR_INVALID', 'business_taxonomy': deepcopy(BUSINESS_TAXONOMY), 'subtype_recognition': 'POSITIVE_SOURCE_BOUND_EVIDENCE_REQUIRED', 'bus_display_prefix': 'B'}

def _controls(policy):
    value = policy.get('v6_counting')
    expected = experimental_counting_policy()
    previous._require(isinstance(value, dict) and all((value.get(key) == wanted for key, wanted in expected.items())), 'Explicit supported B v6 counting policy required')
    previous._require('bus' in policy.get('supported_categories', []) and policy.get('display_category_prefixes') == DISPLAY_PREFIXES, 'BUS category and B passage prefix must be explicitly configured')
    previous._require(policy.get('timeout_frames') == 60 and policy.get('timeout_clock') == 'last_observed_origin', 'B v6 must preserve the agreed 60 source-frame last-origin clock')
    return deepcopy(value)

def business_category(raw_class_name, *, vehicle_subtype=None, subtype_evidence=None):
    """Map only a known positive subtype; do not infer it from a raw car label."""
    if vehicle_subtype is None:
        return raw_class_name
    previous._require(vehicle_subtype in BUSINESS_TAXONOMY, 'Unknown business vehicle subtype')
    previous._require(isinstance(subtype_evidence, dict) and subtype_evidence.get('verified') is True and (subtype_evidence.get('claim_type') in ('USER_REVIEW', 'SOURCE_BOUND_SUBTYPE_REVIEW', 'ADJUDICATED_AI_PIXELS_REVIEW')) and bool(subtype_evidence.get('evidence_ref')), 'Business subtype requires positive, explicit source-bound evidence')
    previous._require(raw_class_name in previous.VEHICLE_CLASSES, 'Business vehicle subtype cannot relabel person or rider measurements')
    return BUSINESS_TAXONOMY[vehicle_subtype]

def _map_subtypes(frames, source_identity):
    audit = []
    for row in frames:
        for item in row.get('observations', []):
            subtype = item.get('vehicle_subtype')
            if subtype is None:
                continue
            evidence = item.get('vehicle_subtype_evidence')
            resolved = business_category(item.get('class_name'), vehicle_subtype=subtype, subtype_evidence=evidence)
            previous._require(evidence.get('video_sha256') == source_identity.get('video_sha256') and evidence.get('track_id') == item.get('track_id') and (evidence.get('frame_index') == row.get('frame_index')), 'Business subtype evidence belongs to another source/observation')
            previous._require(item.get('role', 'vehicle') == 'vehicle', 'Business subtype evidence must target a vehicle measurement')
            previous._require(item.get('resolved_class_name') in (None, resolved), 'Business subtype conflicts with an existing resolved category')
            item['resolved_class_name'] = resolved
            ref = {'claim_type': evidence['claim_type'], 'review_id': evidence['evidence_ref'], 'source_frame': row['frame_index'], 'track_id': item['track_id'], 'vehicle_subtype': subtype, 'business_category': resolved, 'source_video_sha256': source_identity['video_sha256']}
            item.setdefault('review_refs', []).append(ref)
            audit.append(ref)
    return audit

def passage_display_id(category, direction, ordinal, *, category_prefixes=None):
    return previous.passage_display_id(category, direction, ordinal, category_prefixes=DISPLAY_PREFIXES if category_prefixes is None else category_prefixes)

def _refs(value):
    result = []

    def walk(item):
        if isinstance(item, dict):
            if type(item.get('track_id')) is int and type(item.get('frame_index')) is int:
                result.append(item)
            for nested in item.values():
                walk(nested)
        elif isinstance(item, (list, tuple)):
            for nested in item:
                walk(nested)
    walk(value)
    return result

def _relation_matches(refs, relation, prefix):
    return [ref for ref in refs if ref.get('track_id') == relation.get(prefix + '_track_id') and ref.get('physical_episode_id') == relation.get(prefix + '_physical_episode_id')]

def _supported_candidate(value, relation, prefix):
    refs = _refs(value.get('evidence', value))
    matched = _relation_matches(refs, relation, prefix)
    if not matched or not refs:
        return False
    first, last = (min((ref['frame_index'] for ref in matched)), max((ref['frame_index'] for ref in matched)))
    support = relation.get('support_frames', [])
    return any((first <= frame <= last for frame in support))

def _key(row):
    return (row.get('physical_unit_id'), row.get('episode_id'), row.get('passage_id'))

def _direction(row):
    return row.get('direction')

def _proven_opposite(first, second):
    return first and second and all((a.get('direction') in ('A_TO_B', 'B_TO_A') and b.get('direction') in ('A_TO_B', 'B_TO_A') and (a['direction'] != b['direction']) and (not a.get('evidence', {}).get('direction_assumed', False)) and (not b.get('evidence', {}).get('direction_assumed', False)) for a in first for b in second))

def _refresh(result):
    active = sorted(result['events'], key=lambda e: (e['activation_frame'], e['event_id']))
    ordinals = Counter()
    for event in active:
        pair = (event['category'], event['direction'])
        ordinals[pair] += 1
        event['display_id'] = passage_display_id(*pair, ordinals[pair])
    displays = {e['event_id']: e['display_id'] for e in active}
    for row in result['passage_log'] + result['decisions']:
        row['display_id'] = displays.get(row.get('event_id'))
        if 'display_status' in row:
            row['display_status'] = 'final_active_passage' if row['display_id'] else 'no_final_active_passage_id'
    for event in result.get('cancelled_events', []):
        event['display_id'] = None
    result['events'] = active
    result['invalid_decisions'] = [deepcopy(row) for row in result['decisions'] if row['status'] == 'invalid']
    result['display_assignments'] = [{key: deepcopy(e[key]) for key in ('event_id', 'physical_unit_id', 'episode_id', 'passage_id', 'display_id', 'category', 'direction', 'activation_frame', 'decision_frame')} | {'source_frames': deepcopy(e['evidence']['source_frames'])} for e in active]
    categories = Counter((e['category'] for e in active))
    measured = sum((bool(e['evidence']['measured_crossing']) for e in active))
    result['aggregate'] = {'total': len(active), 'by_category': dict(sorted(categories.items())), 'by_direction': dict(sorted(Counter((e['direction'] for e in active)).items())), 'by_route': dict(sorted(Counter((e['route'] for e in active)).items())), 'measured_total': measured, 'inferred_total': len(active) - measured, 'pending': sum((row['status'] == 'pending' for row in result['decisions'])), 'invalid': len(result['invalid_decisions']), 'invalid_reasons': dict(Counter((row['invalid_reason'] for row in result['invalid_decisions']))), 'cancelled': len(result.get('cancelled_events', [])), 'vehicle_total': len(active) - categories['pedestrian'], 'pedestrian_total': categories['pedestrian']}

def _apply_invalid_policy(result, relations):
    output = deepcopy(result)
    events = {e['event_id']: e for e in output['events']}
    audits, implicated = ([], {})
    decisions = output['decisions']
    decision_by_key = {_key(row): row for row in decisions}

    def implicate(row, relation_id=None):
        key = _key(row)
        implicated.setdefault(key, {'row': row, 'relation_ids': set()})
        if relation_id:
            implicated[key]['relation_ids'].add(relation_id)
    for index, relation in enumerate(relations):
        kind = relation.get('candidate_kind')
        if relation.get('verified') is not False or kind not in AMBIGUOUS_KINDS:
            continue
        previous._require(relation.get('relation_type', relation.get('type')) == 'SAME_UNIT', 'An ambiguity hypothesis must use a scoped SAME_UNIT relation')
        support = relation.get('support_frames', [])
        previous._require(isinstance(support, list) and all((type(frame) is int and frame >= 0 for frame in support)), 'Ambiguity support requires actual source frames')
        rid = relation.get('relation_id', f'ambiguity:{index}')
        first = [e for e in events.values() if _supported_candidate(e, relation, 'first')]
        second = [e for e in events.values() if _supported_candidate(e, relation, 'second')]
        audit = {'relation_id': rid, 'candidate_kind': kind, 'support_frames': support, 'physical_episodes_merged': False, 'status': 'NO_ELIGIBLE_SCOPED_CANDIDATE'}
        audits.append(audit)
        if _proven_opposite(first, second):
            audit.update(status='OPPOSITE_DIRECTIONS_NOT_DUPLICATE', event_ids=sorted((e['event_id'] for e in first + second)))
            continue
        selected = first + second
        for event in selected:
            implicate(event, rid)
        for decision in decisions:
            if decision.get('status') != 'pending' or decision.get('role') == 'unknown_person':
                continue
            if any((_supported_candidate(decision, relation, prefix) for prefix in ('first', 'second'))):
                implicate(decision, rid)
        keys = sorted({str(_key(row)) for row in selected})
        if selected or any((rid in value['relation_ids'] for value in implicated.values())):
            audit.update(status='INVALID_PLUS_ZERO_SCOPED_AMBIGUITY', candidate_keys=keys)
    for decision in decisions:
        if decision.get('status') == 'pending' and decision.get('role') != 'unknown_person' and IDENTITY_REASONS.intersection(decision.get('reasons', [])):
            implicate(decision)
    for key, value in implicated.items():
        original = value['row']
        event = events.pop(original.get('event_id'), None)
        decision = decision_by_key.get(key)
        if decision is None:
            previous._require(event is not None, 'Invalid candidate lacks event or source decision')
            decision = {'frame_index': event['decision_frame'], 'track_id': event['track_id'], 'physical_unit_id': event['physical_unit_id'], 'episode_id': event.get('episode_id'), 'passage_id': event['passage_id'], 'role': event.get('role'), 'category': event['category'], 'direction': event['direction'], 'source_ref': deepcopy(event['evidence']['source_refs'][-1]), 'reasons': [], 'reason_evidence': []}
            decisions.append(decision)
            decision_by_key[key] = decision
        if event is not None:
            event.update(status='cancelled', accepted_count_event=False, display_id=None, cancellation_reason='INVALID_AMBIGUOUS_UNIT')
            output['cancelled_events'].append(event)
            decision.update(source_event_id=event['event_id'], cancelled_candidate_event_id=event['event_id'], category=event['category'], direction=event['direction'])
            output['passage_log'].append({'frame_index': output['report']['last_frame_index'], 'track_id': event['track_id'], 'physical_unit_id': event['physical_unit_id'], 'episode_id': event.get('episode_id'), 'passage_id': event['passage_id'], 'category': event['category'], 'direction': event['direction'], 'route': event['route'], 'event_id': event['event_id'], 'status': 'cancelled', 'count_delta': -1, 'message': 'INVALID_AMBIGUOUS_UNIT', 'measured_crossing': event['evidence']['measured_crossing']})
        decision.update(status='invalid', invalid_reason='INVALID_AMBIGUOUS_UNIT', count_delta=0, accepted_count_event=False, physical_episodes_merged=False, invalid_scope='THIS_CANDIDATE_PASSAGE_NOT_GLOBAL_RAW_ID', relation_ids=sorted(value['relation_ids']))
        decision.pop('event_id', None)
        decision['display_id'] = None
        if 'INVALID_AMBIGUOUS_UNIT' not in decision.setdefault('reasons', []):
            decision['reasons'].append('INVALID_AMBIGUOUS_UNIT')
        decision.setdefault('reason_evidence', []).append({'reason': 'INVALID_AMBIGUOUS_UNIT', 'relation_ids': sorted(value['relation_ids']), 'intentional_exclusion_not_zero_traffic': True})
        output['passage_log'].append({'frame_index': decision['frame_index'], **{field: decision.get(field) for field in ('track_id', 'physical_unit_id', 'episode_id', 'passage_id', 'category', 'direction')}, 'event_id': None, 'status': 'invalid', 'count_delta': 0, 'message': 'INVALID_AMBIGUOUS_UNIT', 'invalid_scope': 'THIS_CANDIDATE_PASSAGE_NOT_GLOBAL_RAW_ID'})
    output['events'] = list(events.values())
    for row in output.get('open_passage_diagnostics', []):
        if _key(row) in implicated:
            row.update(disposition='INVALID_AMBIGUOUS_UNIT', timeout_eligible=False)
    output['report']['open_passage_diagnostic_counts'] = dict(sorted(Counter((row['disposition'] for row in output.get('open_passage_diagnostics', []))).items()))
    output['ambiguity_invalid_audit'] = audits
    _refresh(output)
    return output

def _rename_event_ids(result):
    mapping = {row['event_id']: row['event_id'].replace('b_v5_', 'b_v6_', 1) for row in result['events'] + result.get('cancelled_events', [])}

    def walk(value):
        if isinstance(value, dict):
            return {key: walk(item) for key, item in value.items()}
        if isinstance(value, list):
            return [walk(item) for item in value]
        if isinstance(value, str):
            return mapping.get(value, value)
        return value
    return walk(result)

def replay_frames(frames, geometry, *, source_identity, policy):
    controls = _controls(policy)
    copied = deepcopy(list(frames))
    subtype_audit = _map_subtypes(copied, source_identity)
    projected = deepcopy(policy)
    relations = deepcopy(projected.get('episode_relations', []))
    projected['episode_relations'] = [r for r in relations if not (r.get('verified') is False and r.get('candidate_kind') in AMBIGUOUS_KINDS)]
    result = previous.replay_frames(copied, geometry, source_identity=source_identity, policy=projected)
    result = _apply_invalid_policy(result, relations)
    result['business_taxonomy_audit'] = subtype_audit
    result['report'].update(version='b_v6_user_invalid_policy_v1', previous_engine_version='b_v5_p0_eligibility_finalization_v1', invalid_means_intentional_exclusion_not_no_traffic=True, v6_counting_policy=controls, business_subtypes_not_automatically_recognized=True, model_or_tracking_rerun=False, source_observations_unchanged_except_explicit_subtype_resolution=True)
    result = _rename_event_ids(result)
    previous._require(len({_key(row) for row in result['invalid_decisions']}) == len(result['invalid_decisions']), 'One invalid record required per original candidate passage')
    previous._require(sum((row.get('count_delta', 0) for row in result['passage_log'])) == result['aggregate']['total'], 'Passage log does not reconcile after invalid finalization')
    return result


### src/traffic_location/traffic/_kernel/geometry.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/geometry.py
"""Phase 5 notebook geometry, independent of detector/tracker/model quality.

Outputs are geometric diagnostics, never accepted traffic-count events. All
noise/confirmation parameters are required experimental inputs. ``neutral_band_px``
is the distance on EACH side of the line, not the full width of that band.

Only floating-point roundoff is tolerated by primitives (1e-12 times coordinate
scale); that numerical tolerance is not a jitter/counting threshold.
"""
from __future__ import annotations
from collections import Counter
from dataclasses import dataclass
import gzip
import hashlib
import json
import math
from numbers import Real
from pathlib import Path
from typing import Any, Iterable
Point = tuple[float, float]

def _point(value: Iterable[float]) -> Point:
    try:
        values = tuple(value)
    except TypeError as exc:
        raise ValueError('A point requires two finite numeric coordinates') from exc
    if len(values) != 2 or any((isinstance(x, bool) or not isinstance(x, Real) for x in values)):
        raise ValueError('A point requires two finite numeric coordinates')
    try:
        result = (float(values[0]), float(values[1]))
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError('A point requires two finite numeric coordinates') from exc
    if not all((math.isfinite(x) for x in result)):
        raise ValueError('Non-finite geometry coordinate')
    return result

def _epsilon(*points: Point) -> float:
    return 1e-12 * max(1.0, *(abs(v) for p in points for v in p))

def _cross(a: Point, b: Point) -> float:
    return a[0] * b[1] - a[1] * b[0]

def _subtract(a: Point, b: Point) -> Point:
    return (a[0] - b[0], a[1] - b[1])

def _distance(a: Point, b: Point) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])

def _on_segment(point: Point, a: Point, b: Point) -> bool:
    tolerance = _epsilon(point, a, b)
    length = _distance(a, b)
    if length <= tolerance:
        return _distance(point, a) <= tolerance
    if abs(_cross(_subtract(b, a), _subtract(point, a))) / length > tolerance:
        return False
    return min(a[0], b[0]) - tolerance <= point[0] <= max(a[0], b[0]) + tolerance and min(a[1], b[1]) - tolerance <= point[1] <= max(a[1], b[1]) + tolerance

def point_in_polygon(point: Iterable[float], polygon: Iterable[Iterable[float]]) -> str:
    """Return inside/boundary/outside; no implicit pixel or jitter margin."""
    point = _point(point)
    vertices = tuple((_point(p) for p in polygon))
    if len(vertices) < 3:
        raise ValueError('A polygon needs at least three vertices')
    inside = False
    for a, b in zip(vertices, vertices[1:] + vertices[:1]):
        if _on_segment(point, a, b):
            return 'boundary'
        if (a[1] > point[1]) != (b[1] > point[1]):
            intersection_x = a[0] + (point[1] - a[1]) * (b[0] - a[0]) / (b[1] - a[1])
            if point[0] < intersection_x:
                inside = not inside
    return 'inside' if inside else 'outside'

def signed_distance(point: Iterable[float], p1: Iterable[float], p2: Iterable[float]) -> float:
    """Signed perpendicular distance in pixels; sign has no A/B semantics."""
    point, p1, p2 = (_point(point), _point(p1), _point(p2))
    length = _distance(p1, p2)
    if length <= _epsilon(p1, p2):
        raise ValueError('Counting-line endpoints must be distinct')
    return _cross(_subtract(p2, p1), _subtract(point, p1)) / length

def segment_intersection(a, b, c, d) -> dict[str, Any]:
    """Intersect finite path a-b with finite line c-d.

    ``proper``: interior of both segments. ``endpoint``: configured-line c/d
    contact. ``touch``: path endpoint at line interior. ``overlap``: positive
    collinear overlap. ``none`` includes intersections of infinite extensions.
    ``t`` and ``u`` are path and counting-line fractions, respectively.
    """
    a, b, c, d = (_point(a), _point(b), _point(c), _point(d))
    r, s = (_subtract(b, a), _subtract(d, c))
    r_length, s_length = (_distance(a, b), _distance(c, d))
    tolerance = _epsilon(a, b, c, d)
    empty = {'kind': 'none', 'point': None, 't': None, 'u': None}
    if s_length <= tolerance:
        raise ValueError('Configured line must have distinct endpoints')
    if r_length <= tolerance:
        if not _on_segment(a, c, d):
            return empty
        u = sum(((a[i] - c[i]) * s[i] for i in (0, 1))) / (s_length * s_length)
        endpoint = _distance(a, c) <= tolerance or _distance(a, d) <= tolerance
        return {'kind': 'endpoint' if endpoint else 'touch', 'point': list(a), 't': 0.0, 'u': u}
    denominator = _cross(r, s)
    delta = _subtract(c, a)
    if abs(denominator) <= tolerance * max(r_length, s_length):
        if abs(_cross(delta, r)) / r_length > tolerance:
            return empty
        t_c = sum((delta[i] * r[i] for i in (0, 1))) / (r_length * r_length)
        t_d = sum(((d[i] - a[i]) * r[i] for i in (0, 1))) / (r_length * r_length)
        lower, upper = (max(0.0, min(t_c, t_d)), min(1.0, max(t_c, t_d)))
        fraction_tolerance = tolerance / r_length
        if lower > upper + fraction_tolerance:
            return empty
        point = (a[0] + lower * r[0], a[1] + lower * r[1])
        if upper - lower > fraction_tolerance:
            end = [a[0] + upper * r[0], a[1] + upper * r[1]]
            return {'kind': 'overlap', 'point': list(point), 'end_point': end, 't': lower, 't_end': upper, 'u': None}
        endpoint = _distance(point, c) <= tolerance or _distance(point, d) <= tolerance
        u = sum(((point[i] - c[i]) * s[i] for i in (0, 1))) / (s_length * s_length)
        return {'kind': 'endpoint' if endpoint else 'touch', 'point': list(point), 't': lower, 'u': u}
    t, u = (_cross(delta, s) / denominator, _cross(delta, r) / denominator)
    t_tolerance, u_tolerance = (tolerance / r_length, tolerance / s_length)
    if not (-t_tolerance <= t <= 1 + t_tolerance and -u_tolerance <= u <= 1 + u_tolerance):
        return empty
    t, u = (min(1.0, max(0.0, t)), min(1.0, max(0.0, u)))
    point = [a[0] + t * r[0], a[1] + t * r[1]]
    endpoint = u <= u_tolerance or u >= 1 - u_tolerance
    touch = t <= t_tolerance or t >= 1 - t_tolerance
    return {'kind': 'endpoint' if endpoint else 'touch' if touch else 'proper', 'point': point, 't': t, 'u': u}

def _validate_polygon(vertices: tuple[Point, ...], name: str) -> None:
    if len(vertices) < 3:
        raise ValueError(f'{name} requires at least three vertices')
    edges = list(zip(vertices, vertices[1:] + vertices[:1]))
    if any((_distance(a, b) <= _epsilon(a, b) for a, b in edges)):
        raise ValueError(f'{name} has a zero-length edge')
    twice_area = sum((_cross(a, b) for a, b in edges))
    if abs(twice_area) <= _epsilon(*vertices) * sum((_distance(a, b) for a, b in edges)):
        raise ValueError(f'{name} has zero area')
    for i, (a, b) in enumerate(edges):
        for j, (c, d) in enumerate(edges):
            if j <= i or j == i + 1 or (i == 0 and j == len(edges) - 1):
                continue
            if segment_intersection(a, b, c, d)['kind'] != 'none':
                raise ValueError(f'{name} is not a simple polygon')

def _interior_samples(vertices: tuple[Point, ...]):
    """Interior points from horizontal scan intervals, including concave shapes.

    Used only for topology validation, not placement/noise thresholds. Samples
    are strict interior interval midpoints rather than a possibly exterior
    centroid. They detect coincident outlines with different vertex sampling.
    """
    levels = sorted(set((p[1] for p in vertices)))
    for lower, upper in zip(levels, levels[1:]):
        y = (lower + upper) / 2
        intersections = []
        for a, b in zip(vertices, vertices[1:] + vertices[:1]):
            if (a[1] > y) != (b[1] > y):
                intersections.append(a[0] + (y - a[1]) * (b[0] - a[0]) / (b[1] - a[1]))
        intersections.sort()
        for left, right in zip(intersections[::2], intersections[1::2]):
            point = ((left + right) / 2, y)
            if point_in_polygon(point, vertices) == 'inside':
                yield point

@dataclass(frozen=True)
class Geometry:
    p1: Point
    p2: Point
    side_a: tuple[Point, ...]
    side_b: tuple[Point, ...]
    width: int | None = None
    height: int | None = None
    geometry_id: str = 'explicit-fixture'
    video_sha256: str | None = None
    source_draft_status: str | None = None

    def __post_init__(self):
        for name in ('p1', 'p2'):
            object.__setattr__(self, name, _point(getattr(self, name)))
        for name in ('side_a', 'side_b'):
            vertices = tuple((_point(p) for p in getattr(self, name)))
            _validate_polygon(vertices, name)
            object.__setattr__(self, name, vertices)
        signed_distance(self.p1, self.p1, self.p2)
        if (self.width is None) != (self.height is None):
            raise ValueError('Both width and height must be provided, or neither')
        if self.width is not None:
            if any((isinstance(v, bool) or not isinstance(v, int) or v <= 0 for v in (self.width, self.height))):
                raise ValueError('Native dimensions must be positive integers')
            for x, y in (self.p1, self.p2, *self.side_a, *self.side_b):
                if not (0 <= x <= self.width and 0 <= y <= self.height):
                    raise ValueError('Geometry exceeds declared continuous image-edge coordinates')
        for first, second in ((self.side_a, self.side_b), (self.side_b, self.side_a)):
            if any((point_in_polygon(p, second) == 'inside' for p in _interior_samples(first))):
                raise ValueError('Side A/B polygon interiors overlap')
            for a, b in zip(first, first[1:] + first[:1]):
                midpoint = ((a[0] + b[0]) / 2, (a[1] + b[1]) / 2)
                if any((point_in_polygon(p, second) == 'inside' for p in (a, midpoint))):
                    raise ValueError('Side A/B polygon interiors overlap')
                for c, d in zip(second, second[1:] + second[:1]):
                    if segment_intersection(a, b, c, d)['kind'] == 'proper':
                        raise ValueError('Side A/B polygon interiors overlap')

    @classmethod
    def from_draft(cls, path, width: int, height: int) -> Geometry:
        """Resolve recorded normalized coordinates without snapping/cropping."""
        draft = json.loads(Path(path).read_text(encoding='utf-8'))
        normalized = draft['normalized']

        def resolve(p):
            u, v = _point(p)
            if not (0 <= u <= 1 and 0 <= v <= 1):
                raise ValueError('Normalized template coordinate outside [0,1]')
            return (u * width, v * height)
        return cls(resolve(normalized['line']['P1']), resolve(normalized['line']['P2']), tuple((resolve(p) for p in normalized['side_a_polygon'])), tuple((resolve(p) for p in normalized['side_b_polygon'])), width, height, draft.get('draft_id', 'explicit-template'), draft.get('source_video', {}).get('sha256'), draft.get('status'))

    def side(self, point: Iterable[float]) -> str | None:
        states = (point_in_polygon(point, self.side_a), point_in_polygon(point, self.side_b))
        if 'boundary' in states:
            return None
        if states == ('inside', 'outside'):
            return 'A'
        if states == ('outside', 'inside'):
            return 'B'
        return None

    def as_dict(self) -> dict[str, Any]:
        return {'geometry_id': self.geometry_id, 'width': self.width, 'height': self.height, 'coordinate_space': 'native decoded-frame continuous image-edge pixels', 'line': {'P1': list(self.p1), 'P2': list(self.p2)}, 'side_a_polygon': [list(p) for p in self.side_a], 'side_b_polygon': [list(p) for p in self.side_b], 'video_sha256': self.video_sha256, 'source_draft_status': self.source_draft_status}

class GeometryStateMachine:
    """One track's measured trajectory, with no category/counting policy.

    Origin and destination require polygon interior and distance outside the
    experimental neutral band. Missing/predicted observations never bridge a
    passage. A successful transition rebases geometric origin, so future return
    diagnostics remain available while their business counting policy stays open.
    """

    def __init__(self, geometry: Geometry, *, neutral_band_px: float, confirmation_observations: int):
        if isinstance(neutral_band_px, bool) or not isinstance(neutral_band_px, Real) or (not math.isfinite(neutral_band_px)) or (neutral_band_px < 0):
            raise ValueError('neutral_band_px must be an explicit finite nonnegative value')
        if isinstance(confirmation_observations, bool) or not isinstance(confirmation_observations, int) or confirmation_observations < 1:
            raise ValueError('confirmation_observations must be an explicit positive integer')
        self.geometry = geometry
        self.neutral_band_px = float(neutral_band_px)
        self.confirmation_observations = confirmation_observations
        self.last_update_frame = None
        self.last_observed = None
        self.origin = None
        self.pending = None
        self.touch_sequence = None
        self.continuity_broken = False
        self.transition_sequence = 0

    def _observation(self, frame_index, point, metadata):
        distance = signed_distance(point, self.geometry.p1, self.geometry.p2)
        zero = abs(distance) <= _epsilon(point, self.geometry.p1, self.geometry.p2)
        sign = 0 if zero else 1 if distance > 0 else -1
        side = self.geometry.side(point)
        stable_side = side if abs(distance) > self.neutral_band_px and (not zero) else None
        return {'frame_index': frame_index, 'point': list(point), 'metadata': dict(metadata or {}), 'side': side, 'stable_side': stable_side, 'signed_distance_px': distance, 'sign': sign}

    def _diagnostic(self, kind, **fields):
        return {'type': kind, 'geometry_id': self.geometry.geometry_id, 'accepted_count_event': False, 'repeat_business_policy_open': True, **fields}

    def _establish_origin(self, observation, diagnostics):
        if observation['stable_side'] is not None:
            self.origin = observation
            diagnostics.append(self._diagnostic('ORIGIN_ESTABLISHED', origin_side=observation['stable_side'], origin_frame=observation['frame_index'], origin_point=observation['point'], metadata=observation['metadata']))

    def update(self, frame_index: int, point, *, observed: bool=True, metadata=None) -> list[dict[str, Any]]:
        if isinstance(frame_index, bool) or not isinstance(frame_index, int) or frame_index < 0:
            raise ValueError('frame_index must be a nonnegative integer')
        if self.last_update_frame is not None and frame_index <= self.last_update_frame:
            raise ValueError('Track updates must have strictly increasing frame indices')
        if not isinstance(observed, bool):
            raise ValueError('observed must explicitly be boolean')
        self.last_update_frame = frame_index
        diagnostics = []
        if not observed:
            self.pending = None
            self.touch_sequence = None
            self.continuity_broken = True
            return [self._diagnostic('PREDICTION_IGNORED', frame_index=frame_index, metadata=dict(metadata or {}))]
        point = _point(point)
        current = self._observation(frame_index, point, metadata)
        previous = self.last_observed
        if previous is not None and (frame_index != previous['frame_index'] + 1 or self.continuity_broken):
            intersection = segment_intersection(previous['point'], point, self.geometry.p1, self.geometry.p2)
            possible_passage = self.pending is not None or self.touch_sequence is not None or (self.origin is not None and (previous['stable_side'] != current['stable_side'] or previous['sign'] != current['sign'] or intersection['kind'] != 'none'))
            if possible_passage:
                diagnostics.append(self._diagnostic('AMBIGUOUS_GAP', previous_observation=previous, current_observation=current, missing_observation_frames=frame_index - previous['frame_index'] - 1, interpolation_used=False))
            self.origin = self.pending = self.touch_sequence = None
            self.last_observed = current
            self.continuity_broken = False
            self._establish_origin(current, diagnostics)
            return diagnostics
        self.continuity_broken = False
        if self.origin is None:
            self._establish_origin(current, diagnostics)
        elif previous is not None:
            intersection = segment_intersection(previous['point'], point, self.geometry.p1, self.geometry.p2)
            overlap_reaches_endpoint = intersection['kind'] == 'overlap' and any((_on_segment(endpoint, _point(previous['point']), point) for endpoint in (self.geometry.p1, self.geometry.p2)))
            if intersection['kind'] == 'endpoint' or overlap_reaches_endpoint:
                diagnostics.append(self._diagnostic('AMBIGUOUS_ENDPOINT', previous_observation=previous, current_observation=current, intersection=intersection))
                self.pending = self.touch_sequence = None
            else:
                crossing = None
                if intersection['kind'] == 'proper' and previous['sign'] * current['sign'] == -1:
                    crossing = {'intersection': intersection, 'start_observation': previous, 'end_observation': current, 'method': 'consecutive_measured_segment', 'entry_sign': previous['sign'], 'exit_sign': current['sign']}
                elif current['sign'] == 0 and intersection['kind'] in ('touch', 'overlap'):
                    if self.touch_sequence is None and previous['sign'] != 0:
                        self.touch_sequence = {'entry_sign': previous['sign'], 'start_observation': previous, 'first_touch_observation': current, 'last_touch_observation': current, 'intersection': intersection}
                    elif self.touch_sequence is not None:
                        self.touch_sequence['last_touch_observation'] = current
                elif previous['sign'] == 0 and current['sign'] != 0:
                    if self.touch_sequence is not None and current['sign'] == -self.touch_sequence['entry_sign']:
                        crossing = {'intersection': self.touch_sequence['intersection'], 'start_observation': self.touch_sequence['start_observation'], 'end_observation': current, 'method': 'measured_touch_then_opposite_sign', 'entry_sign': self.touch_sequence['entry_sign'], 'exit_sign': current['sign'], 'first_touch_observation': self.touch_sequence['first_touch_observation'], 'last_touch_observation': self.touch_sequence['last_touch_observation']}
                    self.touch_sequence = None
                if self.pending is not None and current['sign'] == self.pending['crossing']['entry_sign']:
                    self.pending = None
                if crossing is not None:
                    self.pending = {'origin': self.origin, 'crossing': crossing, 'confirmations': [], 'left_origin_region': current['side'] != self.origin['stable_side']}
                elif self.pending is not None:
                    if current['side'] != self.origin['stable_side']:
                        self.pending['left_origin_region'] = True
                    elif current['stable_side'] == self.origin['stable_side'] and self.pending['left_origin_region']:
                        self.pending = None
                if self.pending is not None:
                    destination = 'B' if self.origin['stable_side'] == 'A' else 'A'
                    if current['stable_side'] == destination and current['sign'] == self.pending['crossing']['exit_sign']:
                        self.pending['confirmations'].append(current)
                    else:
                        self.pending['confirmations'] = []
                    if len(self.pending['confirmations']) >= self.confirmation_observations:
                        crossing = self.pending['crossing']
                        self.transition_sequence += 1
                        diagnostics.append(self._diagnostic('GEOMETRY_TRANSITION', software_transition_sequence=self.transition_sequence, origin_side=self.origin['stable_side'], destination_side=destination, direction=f"{self.origin['stable_side']}_TO_{destination}", origin_frame=self.origin['frame_index'], origin_point=self.origin['point'], crossing_frame_start=crossing['start_observation']['frame_index'], crossing_frame_end=crossing['end_observation']['frame_index'], crossing_point=crossing['intersection']['point'], confirmation_frame=frame_index, confirmation_point=current['point'], origin_observation=self.origin, crossing_evidence=crossing, confirmation_observations=self.pending['confirmations'], metadata=current['metadata']))
                        self.origin = current
                        self.pending = self.touch_sequence = None
        self.last_observed = current
        return diagnostics

def replay_cached_records(records_path, geometry: Geometry, *, neutral_band_px: float, confirmation_observations: int) -> dict[str, Any]:
    """Stream existing ReID emitted observations; no model inference or totals.

    Only raw detector-linked boxes of persistent emitted tracks provide BC.
    All raw classes remain diagnostic inputs, not accepted product categories.
    Source links, dimensions, video identity and measurement provenance are
    validated; predictions/lifecycle boxes are never substituted.
    """
    GeometryStateMachine(geometry, neutral_band_px=neutral_band_px, confirmation_observations=confirmation_observations)
    path = Path(records_path)
    source_hash = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            source_hash.update(block)
    opener = gzip.open if path.suffix == '.gz' else open
    machines, diagnostics = ({}, [])
    frame_count = observation_count = 0
    previous_frame = None
    source_video_sha256 = None
    source_parent_run_id = source_variant = None
    with opener(path, 'rt', encoding='utf-8') as source:
        for line_number, line in enumerate(source, 1):
            if not line.strip():
                raise ValueError(f'Empty source record on line {line_number}')
            record = json.loads(line)
            frame = record['frame_index']
            if isinstance(frame, bool) or not isinstance(frame, int) or frame < 0 or (previous_frame is not None and frame <= previous_frame):
                raise ValueError('Cached frame records must be strictly increasing nonnegative indices')
            previous_frame = frame
            frame_count += 1
            if (record['width'], record['height']) != (geometry.width, geometry.height):
                raise ValueError('Cached native dimensions do not match effective geometry')
            video_sha256 = record['video_sha256']
            if source_video_sha256 is None:
                source_video_sha256 = video_sha256
            if video_sha256 != source_video_sha256 or (geometry.video_sha256 is not None and video_sha256 != geometry.video_sha256):
                raise ValueError('Cached source video identity changed or does not match geometry')
            if frame_count == 1:
                source_parent_run_id, source_variant = (record['parent_run_id'], record['variant'])
            elif (record['parent_run_id'], record['variant']) != (source_parent_run_id, source_variant):
                raise ValueError('Cached parent run or variant changed within one replay input')
            raw_detections = record['raw_detections']
            frame_ids, frame_detection_indices = (set(), set())
            for observation in record['emitted_tracks']:
                track_id = observation['track_id']
                if isinstance(track_id, bool) or not isinstance(track_id, int) or track_id < 1 or (track_id in frame_ids):
                    raise ValueError('Emitted track IDs must be positive and unique within a frame')
                frame_ids.add(track_id)
                if observation['frame_index'] != frame or observation.get('emitted') is not True:
                    raise ValueError('Emitted measurement/frame contract mismatch')
                if observation.get('measurement_provenance') != 'retained_phase_2_detection':
                    raise ValueError('Geometry requires retained measured detection provenance')
                index = observation['original_detection_index']
                if isinstance(index, bool) or not isinstance(index, int) or (not 0 <= index < len(raw_detections)):
                    raise ValueError('Broken source-detection index')
                if index in frame_detection_indices:
                    raise ValueError('One source detection cannot supply two emitted track measurements in the same frame')
                frame_detection_indices.add(index)
                raw_bbox = observation['raw_bbox']
                if len(raw_bbox) != 4:
                    raise ValueError('Raw bbox must have four xyxy coordinates')
                a, b = (_point(raw_bbox[:2]), _point(raw_bbox[2:]))
                if a[0] >= b[0] or a[1] >= b[1]:
                    raise ValueError('Raw bbox must have positive width/height')
                detection = raw_detections[index]
                for confidence in (observation['confidence'], detection['confidence']):
                    if isinstance(confidence, bool) or not isinstance(confidence, Real) or (not math.isfinite(confidence)):
                        raise ValueError('Source and observation confidence must be finite numeric measurements')
                if list(raw_bbox) != detection['bbox_xyxy_pixels']:
                    raise ValueError('Raw bbox is not byte-value linked to source detection')
                if any((observation[key] != detection[key] for key in ('class_id', 'class_name', 'confidence'))):
                    raise ValueError('Track measurement class/confidence does not match linked source')
                metadata = {'track_id': track_id, 'video_sha256': video_sha256, 'parent_run_id': record['parent_run_id'], 'variant': record['variant'], 'original_detection_index': index, 'raw_bbox': list(raw_bbox), 'class_id': observation['class_id'], 'class_name': observation['class_name'], 'confidence': observation['confidence'], 'measurement_provenance': observation['measurement_provenance'], 'timestamp_estimate_from_index_fps_seconds': record.get('timestamp_estimate_from_index_fps_seconds'), 'timestamp_is_verified_source_pts': False}
                machine = machines.setdefault(track_id, GeometryStateMachine(geometry, neutral_band_px=neutral_band_px, confirmation_observations=confirmation_observations))
                anchor = ((a[0] + b[0]) / 2, b[1])
                for diagnostic in machine.update(frame, anchor, metadata=metadata):
                    diagnostics.append({'track_id': track_id, 'frame_index': frame, **diagnostic})
                observation_count += 1
    return {'diagnostics': diagnostics, 'report': {'claim_type': 'EXECUTED_SOFTWARE_GEOMETRY_DIAGNOSTICS_NOT_TRAFFIC_COUNTS', 'source_records_path': str(path), 'source_records_file_sha256': source_hash.hexdigest(), 'video_sha256': source_video_sha256, 'source_parent_run_id': source_parent_run_id, 'source_variant': source_variant, 'frames_processed': frame_count, 'emitted_measured_observations_processed': observation_count, 'software_track_ids': len(machines), 'software_diagnostic_counts': dict(Counter((d['type'] for d in diagnostics))), 'effective_geometry': geometry.as_dict(), 'anchor': 'raw_bbox_bottom_center', 'neutral_band_px': float(neutral_band_px), 'neutral_band_semantics': 'half-width per side', 'confirmation_observations': confirmation_observations, 'category_filter': 'none_all_raw_classes_are_diagnostic_inputs_only', 'detector_tracker_encoder_rerun': False, 'accepted_count_events_emitted': False, 'physical_identity_or_counting_accuracy_established': False, 'repeat_business_policy': 'OPEN', 'gap_policy': 'no interpolation; ambiguous passage; origin reset'}}


### src/traffic_location/traffic/_kernel/handoff.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/handoff.py
"""Bounded motion conflicts between provisional passages; never identity proof.

Five observed frames on each side support bidirectional linear extrapolation.
Predictions only propose uncertainty; raw boxes/BC and crossing evidence stay
unchanged. The 60-frame source horizon is inherited, not increased.
"""
from collections import defaultdict
from copy import deepcopy
import hashlib
import json
import math
from . import identity as identity
from .passage import _side_signs
KIND = 'UNVERIFIED_ORIGIN_HANDOFF'

def experimental_handoff_policy():
    return {'schema_version': 'phase8_b_v7_handoff_policy_v1', 'window_observations': 5, 'window_requirement': 'CONSECUTIVE_OBSERVED_VEHICLE_MEASUREMENTS_BOTH_SIDES', 'fit': 'OLS_CENTER_CONSTANT_VELOCITY_BIDIRECTIONAL', 'spatial_gate': 'BIDIRECTIONAL_TRANSLATED_ENDPOINT_BOX_POSITIVE_AREA_OVERLAP', 'gap_limit': 'INHERIT_EXISTING_TIMEOUT_SOURCE_FRAMES', 'direction_gate': 'OBSERVED_LINE_NORMAL_MOTION_AND_EVENT_DIRECTIONS_AGREE', 'scope': 'ACTIVE_CANDIDATE_PAIR_AT_LEAST_ONE_ESTIMATE_NO_OVERLAPPING_CONTEXT_LIFETIMES', 'identity_effect': 'UNVERIFIED_CONFLICT_NO_ALIAS_NO_MEASURED_CROSSING_UPGRADE', 'disposition': 'EXISTING_INVALID_PLUS_ZERO_FOR_EXPLICIT_PAIR_ONLY', 'window_rationale': 'FIVE_IS_DECLARED_RESEARCH_SAMPLE_SUPPORT_NOT_TUNED_ON_TARGET_OR_PRODUCTION_DEFAULT', 'spatial_rationale': 'BOTH_OBSERVED_BOX_EXTENTS_BOUND_SPATIAL_CONFLICT_NOT_VALIDATED_IDENTITY_THRESHOLD'}

def _require(value, message):
    if not value:
        raise ValueError(message)

def _controls(policy):
    controls = policy.get('v7_handoff')
    _require(controls == experimental_handoff_policy(), 'Explicit supported B v7 handoff policy required')
    _require(policy.get('timeout_frames') == 60, 'Existing 60 source-frame clock must stay fixed')
    return deepcopy(controls)

def _center(item):
    x1, y1, x2, y2 = item['raw_bbox']
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def _bc(item):
    x1, _, x2, y2 = item['raw_bbox']
    return ((x1 + x2) / 2, y2)

def _fit(items, point=_center):
    times = [i['frame_index'] for i in items]
    t0 = sum(times) / len(times)
    points = [point(i) for i in items]
    origin = [sum((p[k] for p in points)) / len(points) for k in (0, 1)]
    denominator = sum(((t - t0) ** 2 for t in times))
    _require(denominator > 0, 'Motion fit requires distinct observation times')
    velocity = [sum(((t - t0) * (p[k] - origin[k]) for t, p in zip(times, points))) / denominator for k in (0, 1)]
    return {'reference_frame': t0, 'center': origin, 'velocity_px_per_source_frame': velocity}

def _predict(fit, frame):
    return [fit['center'][k] + (frame - fit['reference_frame']) * fit['velocity_px_per_source_frame'][k] for k in (0, 1)]

def _translated_box(item, predicted_center):
    center = _center(item)
    dx, dy = [predicted_center[k] - center[k] for k in (0, 1)]
    x1, y1, x2, y2 = item['raw_bbox']
    return [x1 + dx, y1 + dy, x2 + dx, y2 + dy]

def _positive_overlap(first, second):
    return min(first[2], second[2]) > max(first[0], second[0]) and min(first[3], second[3]) > max(first[1], second[1])

def _context(item):
    return (item['track_id'], item.get('physical_episode_id'))

def _window(event):
    refs = event.get('evidence', {}).get('source_refs', [])
    times = [r['frame_index'] for r in refs if type(r.get('frame_index')) is int]
    return (min(times), max(times)) if times else None

def _event_context(event):
    refs = event.get('evidence', {}).get('source_refs', [])
    contexts = {_context(ref) for ref in refs if ref.get('role') == 'vehicle'}
    if len(contexts) != 1 or next(iter(contexts))[1] is None:
        return None
    return next(iter(contexts))

def _own_observations(event, context_items):
    window = _window(event)
    if window is None:
        return []
    return [item for item in context_items if window[0] <= item['frame_index'] <= window[1] and item.get('unit_id') == event['physical_unit_id'] and (item.get('episode_id') == event.get('episode_id')) and ((item.get('resolved_class_name') or item['class_name']) == event['category'])]

def _consecutive(items, count):
    return len(items) == count and all((b['frame_index'] == a['frame_index'] + 1 for a, b in zip(items, items[1:])))

def _distinct_constraint(first, second, relations, contexts):
    """A scoped verified identity constraint may survive within its context.

    It cannot migrate to another physical episode or be inferred from role or
    category review. Retained relation inputs were verified by the source adapter.
    """
    proof = identity._proven_distinct(first, second, relations)
    if proof:
        return proof
    for rel in relations:
        if rel.get('verified') is not True or rel.get('relation_type', rel.get('type')) != 'DIFFERENT_UNIT':
            continue
        support = rel.get('support_frames', [])
        if not support:
            continue
        for a, b in ((first, second), (second, first)):

            def matches(prefix, item):
                return rel.get(prefix + '_track_id') == item['track_id'] and rel.get(prefix + '_physical_episode_id') == item['physical_episode_id']
            if matches('first', a) and matches('second', b):
                frame_sets = [{item['frame_index'] for item in contexts[_context(x)]} for x in (a, b)]
                if all((set(support) & values for values in frame_sets)):
                    return {'basis': 'VERIFIED_DIFFERENT_UNIT_WITHIN_SAME_PHYSICAL_CONTEXTS', 'relation_id': rel.get('relation_id'), 'support_frames': support}
    return None

def propose_handoff_conflicts(provisional_result, frames, geometry, policy):
    controls = _controls(policy)
    count = controls['window_observations']
    contexts = defaultdict(list)
    for row in frames:
        for item in row.get('observations', []):
            if item.get('role') != 'vehicle':
                continue
            _require(item.get('observed') is True and item.get('emitted') is True, 'Handoff fitting requires observed emitted source measurements')
            _require(item.get('frame_index', row['frame_index']) == row['frame_index'], 'Frame/source mismatch')
            _require(item.get('physical_episode_id') is not None, 'Physical context must be explicit')
            _require(len(item['raw_bbox']) == 4 and all((math.isfinite(x) for x in item['raw_bbox'])), 'Invalid raw bbox')
            contexts[_context(item)].append(item)
    for items in contexts.values():
        items.sort(key=lambda i: i['frame_index'])
        _require(len({i['frame_index'] for i in items}) == len(items), 'Repeated source context/frame')
    events = sorted(provisional_result.get('events', []), key=lambda e: (min(e['evidence']['source_frames']), e['event_id']))
    signs = _side_signs(geometry)
    dx, dy = (geometry.p2[0] - geometry.p1[0], geometry.p2[1] - geometry.p1[1])
    length = math.hypot(dx, dy)
    normal = (-dy / length, dx / length)
    relations, audits = ([], [])
    negative = policy.get('episode_relations', [])
    for index, old in enumerate(events):
        for new in events[index + 1:]:
            if old.get('role') != 'vehicle' or new.get('role') != 'vehicle' or old['track_id'] == new['track_id'] or (old['category'] != new['category']) or (old['direction'] != new['direction']) or (old['direction'] not in ('A_TO_B', 'B_TO_A')) or all((e['evidence'].get('measured_crossing') for e in (old, new))):
                continue
            ca, cb = (_event_context(old), _event_context(new))
            if ca not in contexts or cb not in contexts or ca == cb:
                continue
            a, b = (contexts[ca], contexts[cb])
            if a[0]['frame_index'] > b[0]['frame_index']:
                continue
            audit = {'first_event_id': old['event_id'], 'second_event_id': new['event_id'], 'first_track_id': old['track_id'], 'second_track_id': new['track_id'], 'first_physical_episode_id': ca[1], 'second_physical_episode_id': cb[1], 'status': 'REJECTED', 'reason': None}
            gap = b[0]['frame_index'] - a[-1]['frame_index']
            if gap <= 0:
                continue
            if gap > policy['timeout_frames']:
                continue
            audits.append(audit)
            audit['gap_source_frames'] = gap
            first, second = (_own_observations(old, a), _own_observations(new, b))
            if not first or not second or first[-1] is not a[-1] or (second[0] is not b[0]):
                audit['reason'] = 'EVENT_WINDOW_DOES_NOT_CONTAIN_CONTEXT_ENDPOINTS'
                continue
            tail, head = (first[-count:], second[:count])
            if not _consecutive(tail, count) or not _consecutive(head, count):
                audit['reason'] = 'INSUFFICIENT_CONSECUTIVE_OBSERVATIONS'
                continue
            proof = _distinct_constraint(tail[-1], head[0], negative, contexts)
            if proof:
                audit.update(reason='VERIFIED_DISTINCT_VETO', distinct_proof=proof)
                continue
            fits = [_fit(tail), _fit(head)]
            velocities = [fit['velocity_px_per_source_frame'] for fit in fits]
            bc_velocities = [_fit(items, _bc)['velocity_px_per_source_frame'] for items in (tail, head)]
            origin_sign = signs['A' if old['direction'] == 'A_TO_B' else 'B']
            progress = [-origin_sign * sum((v[k] * normal[k] for k in (0, 1))) for v in bc_velocities]
            dot = sum((velocities[0][k] * velocities[1][k] for k in (0, 1)))
            audit.update(first_fit=fits[0], second_fit=fits[1], observed_normal_progress=progress, heading_dot=dot, first_support_frames=[i['frame_index'] for i in tail], second_support_frames=[i['frame_index'] for i in head])
            if dot <= 1e-12 or any((value <= 1e-12 for value in progress)):
                audit['reason'] = 'OBSERVED_MOTION_DIRECTION_NOT_SUPPORTED'
                continue
            forward, backward = (_predict(fits[0], head[0]['frame_index']), _predict(fits[1], tail[-1]['frame_index']))
            forward_box = _translated_box(tail[-1], forward)
            backward_box = _translated_box(head[0], backward)
            checks = [_positive_overlap(forward_box, head[0]['raw_bbox']), _positive_overlap(backward_box, tail[-1]['raw_bbox'])]
            audit.update(forward_prediction=forward, backward_prediction=backward, forward_translated_box=forward_box, backward_translated_box=backward_box, forward_positive_area_overlap=checks[0], backward_positive_area_overlap=checks[1])
            if not all(checks):
                audit['reason'] = 'BIDIRECTIONAL_OBSERVED_BOX_GATE_FAILED'
                continue
            material = [old['event_id'], new['event_id'], audit['first_support_frames'], audit['second_support_frames'], controls]
            rid = 'b7-motion-conflict:' + hashlib.sha256(json.dumps(material, sort_keys=True).encode()).hexdigest()[:24]
            relation = {'type': 'SAME_UNIT', 'relation_type': 'SAME_UNIT', 'candidate_kind': KIND, 'verified': False, 'relation_id': rid, 'event_ids': [old['event_id'], new['event_id']], 'first_event_id': old['event_id'], 'second_event_id': new['event_id'], 'first_track_id': old['track_id'], 'second_track_id': new['track_id'], 'first_physical_episode_id': ca[1], 'second_physical_episode_id': cb[1], 'first_unit_id': old['physical_unit_id'], 'second_unit_id': new['physical_unit_id'], 'first_episode_id': old.get('episode_id'), 'second_episode_id': new.get('episode_id'), 'first_support_frames': audit['first_support_frames'], 'second_support_frames': audit['second_support_frames'], 'support_frames': audit['first_support_frames'] + audit['second_support_frames'], 'gap_source_frames': gap, 'motion_provenance': 'BIDIRECTIONAL_OLS_OBSERVED_WINDOWS', 'first_source_refs': deepcopy(tail), 'second_source_refs': deepcopy(head), 'identity_merged': False, 'predicted_crossing_is_measured': False, 'claim_type': 'UNVERIFIED_MOTION_CONFLICT_NOT_PHYSICAL_IDENTITY'}
            relations.append(relation)
            audit.update(status='PROPOSED_UNVERIFIED_CONFLICT', reason='BIDIRECTIONAL_MOTION_SUPPORTED', relation_id=rid)
    return {'relations': relations, 'audit': audits, 'controls': controls}


### src/traffic_location/traffic/_kernel/identity.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/identity.py
"""Source-scoped ambiguity candidates for the user's conservative +0 policy.

This helper never merges geometric candidates, classifies rider from co-motion
alone, or creates gap measurements. It extends the existing high-overlap
conflict control to same-category tracks; candidate identity remains UNKNOWN.
"""
from __future__ import annotations
from collections import defaultdict
from copy import deepcopy
from . import identity_lifecycle as previous
from .geometry import signed_distance
VEHICLES = previous.VEHICLES
AMBIGUITY_KIND = 'POSSIBLE_SCOPED_UNIT_OR_COMPONENT_AMBIGUITY'
HANDOFF_KIND = 'UNVERIFIED_ORIGIN_HANDOFF'

def experimental_identity_policy(*, component_profile='median_temporal_v1'):
    controls = previous.experimental_identity_policy(component_profile=component_profile)
    controls.update(schema_version='phase8_b_v6_identity_policy_v1', ambiguity_policy='SCOPED_HIGH_IOU_ALL_VEHICLE_CLASSES_NO_AUTOMATIC_ALIAS', ambiguity_disposition='INVALID_PLUS_ZERO_BY_COUNTING_POLICY_NOT_IDENTITY_TRUTH', direction_policy='OPPOSITE_OBSERVED_DIRECTION_NOT_DUPLICATE', unverified_handoff_policy='EXACT_OBSERVED_ENDPOINTS_NO_GAP_CROSSING_PROOF', numeric_provenance_v6='REUSE_BV5_POSSIBLE_HANDOFF_IOU_AND_COMPONENT_CENTER_CONTROLS_NO_NEW_THRESHOLD')
    return controls

def _controls(policy):
    value = deepcopy(policy.get('identity', policy))
    expected = experimental_identity_policy()
    previous.previous._require(isinstance(value, dict) and value.get('schema_version') == expected['schema_version'], 'Explicit B v6 identity policy required')
    for key in ('ambiguity_policy', 'ambiguity_disposition', 'direction_policy', 'unverified_handoff_policy'):
        previous.previous._require(value.get(key) == expected[key], 'Unsupported B v6 identity control ' + key)
    inherited = deepcopy(value)
    inherited['schema_version'] = 'phase8_b_v5_identity_policy_v1'
    previous._controls(inherited)
    return (value, inherited)

def _point(item):
    box = item['raw_bbox']
    return ((box[0] + box[2]) / 2, box[3])

def _identity_review_proof(item):
    """Require typed identity provenance, never a role/category review alone.

    The retained adapter predates typed refs. Its SAME_VEHICLE assignment is
    identifiable by the exact matching verified association/review ID. A split
    needs an explicit review type; a custom unit name alone proves nothing.
    """
    if str(item['unit_id']).startswith(('retained-clique:', 'track:', 'person-track:')):
        return []
    refs = list(item.get('review_refs', []))
    if isinstance(item.get('review_ref'), dict):
        refs.append(item['review_ref'])
    evidence = []
    for ref in refs:
        kind = ref.get('review_type', ref.get('type'))
        exact = ref.get('source_frame') == item['frame_index'] and ref.get('track_id') == item['track_id'] and isinstance(ref.get('review_id'), str) and bool(ref['review_id']) and isinstance(ref.get('reviews_sha256'), str) and bool(ref['reviews_sha256'])
        typed = kind in ('SAME_VEHICLE', 'SPLIT_IDENTITY')
        legacy_same = kind is None and item.get('association_verified') is True and (item.get('association_id') == ref.get('review_id'))
        if exact and (typed or legacy_same):
            evidence.append({'review_id': ref['review_id'], 'source_frame': ref['source_frame'], 'track_id': item['track_id'], 'reviews_sha256': ref['reviews_sha256'], 'identity_review_type': kind if typed else 'LEGACY_VERIFIED_SAME_VEHICLE_ASSIGNMENT', 'scope_extent_is_engineering_assumption': ref.get('scope_extent_is_engineering_assumption')})
    return evidence

def _negative_matches(relation, first, second):
    if relation.get('verified') is not True or relation.get('relation_type', relation.get('type')) != 'DIFFERENT_UNIT' or (not {first['frame_index'], second['frame_index']}.issubset(set(relation.get('support_frames', [])))):
        return False

    def matches(prefix, item):
        if relation.get(prefix + '_track_id') != item['track_id']:
            return False
        context = relation.get(prefix + '_physical_episode_id')
        return context is not None and context == item['physical_episode_id'] or (relation.get(prefix + '_unit_id') == item['unit_id'] and relation.get(prefix + '_episode_id') == item['episode_id'])
    return matches('first', first) and matches('second', second) or (matches('first', second) and matches('second', first))

def _proven_distinct(first, second, negative_relations):
    for relation in negative_relations:
        if _negative_matches(relation, first, second):
            return {'basis': 'EXACT_VERIFIED_DIFFERENT_UNIT_RELATION', 'relation_id': relation.get('relation_id'), 'support_frames': [first['frame_index'], second['frame_index']]}
    a, b = (_identity_review_proof(first), _identity_review_proof(second))
    if first['unit_id'] != second['unit_id'] and a and b:
        return {'basis': 'DIFFERENT_SOURCE_BOUND_REVIEWED_PHYSICAL_UNITS', 'first_identity_review_proof': a, 'second_identity_review_proof': b}
    return None

def _distinct_audit(first, second, proof, kind):
    return {'first_track_id': first['track_id'], 'second_track_id': second['track_id'], 'first_physical_episode_id': first['physical_episode_id'], 'second_physical_episode_id': second['physical_episode_id'], 'support_frames': sorted({first['frame_index'], second['frame_index']}), 'vetoed_candidate_kind': kind, 'disposition': 'PROVEN_DISTINCT_AUDIT_ONLY', 'proof': proof, 'source_measurements_changed': False}

def _veto_inherited_cross_category(result, negative_relations, distinct_audit):
    """Trim only disproved exact support, retaining unrelated identity holds."""
    present = {(row['frame_index'], item['track_id']): item for row in result['frames'] for item in row['observations']}
    protected, active, kept = (set(), set(), [])
    for relation in result['unit_relations']:
        if relation.get('candidate_kind') != 'POSSIBLE_CROSS_CATEGORY_UNIT_OR_COMPONENT':
            kept.append(relation)
            continue
        support = []
        for frame in relation['support_frames']:
            first, second = (present[frame, relation[key + '_track_id']] for key in ('first', 'second'))
            proof = _proven_distinct(first, second, negative_relations)
            if proof:
                distinct_audit.append(_distinct_audit(first, second, proof, relation['candidate_kind']))
                protected.update(((frame, item['track_id']) for item in (first, second)))
            else:
                support.append(frame)
                active.update(((frame, item['track_id']) for item in (first, second)))
        if support:
            relation['support_frames'] = support
            relation['start_frame'], relation['end_frame'] = (support[0], support[-1])
            kept.append(relation)
    result['unit_relations'] = kept
    reason = 'CROSS_CATEGORY_UNIT_OR_COMPONENT_UNRESOLVED'
    for key in protected - active:
        item = present[key]
        item['identity_timeout_hold'] = [value for value in item['identity_timeout_hold'] if value != reason]

def _opposite(run, geometry, minimum):
    """Only observed monotone line-normal motion can exclude a candidate.

    This is an audit constraint, not independent proof of either count. Actual
    event directions are checked again by the counting finalizer.
    """
    if len(run) < minimum:
        return False
    paths = [[signed_distance(_point(pair[index]), geometry.p1, geometry.p2) for pair in run] for index in (0, 1)]
    deltas = [[b - a for a, b in zip(path, path[1:])] for path in paths]
    signs = [1 if all((step > 0 for step in delta)) else -1 if all((step < 0 for step in delta)) else 0 for delta in deltas]
    return signs[0] * signs[1] == -1

def _same_category_ambiguity(frames, geometry, controls, negative_relations, distinct_audit):
    """High overlap can invalidate an ambiguity, never establish same-unit.

    The numerical limit is the frozen possible-handoff IoU control. Merely
    nearby/disjoint boxes, weak overlap and parallel movement are not enough.
    """
    pairs = defaultdict(list)
    base = previous.previous
    for row in frames:
        vehicles = [item for item in row['observations'] if item['role'] == 'vehicle']
        for index, first in enumerate(vehicles):
            for second in vehicles[index + 1:]:
                if first['class_name'] != second['class_name']:
                    continue
                a, b = sorted((first, second), key=lambda item: item['track_id'])
                if (a['unit_id'], a['episode_id']) == (b['unit_id'], b['episode_id']) and a.get('association_verified') is True and (b.get('association_verified') is True) and (a.get('association_id') is not None) and (a.get('association_id') == b.get('association_id')):
                    continue
                scale = (base._diag(a['raw_bbox']) + base._diag(b['raw_bbox'])) / 2
                center_distance = base._distance(base._center(a['raw_bbox']), base._center(b['raw_bbox'])) / scale
                if base._iou(a['raw_bbox'], b['raw_bbox']) <= 0 or center_distance > controls['component_center_limit']:
                    continue
                proof = _proven_distinct(a, b, negative_relations)
                if proof:
                    if base._iou(a['raw_bbox'], b['raw_bbox']) >= controls['possible_handoff_min_iou']:
                        distinct_audit.append(_distinct_audit(a, b, proof, AMBIGUITY_KIND))
                    continue
                key = (a['track_id'], b['track_id'], a['physical_episode_id'], b['physical_episode_id'])
                pairs[key].append((a, b))
    relations, audits = ([], [])
    for key, observations in sorted(pairs.items()):
        runs, current = ([], [])
        for pair in observations:
            if current and pair[0]['frame_index'] != current[-1][0]['frame_index'] + 1:
                runs.append(current)
                current = []
            current.append(pair)
        if current:
            runs.append(current)
        for run in runs:
            opposite = _opposite(run, geometry, controls['min_component_run'])
            selected = [pair for pair in run if base._iou(pair[0]['raw_bbox'], pair[1]['raw_bbox']) >= controls['possible_handoff_min_iou']]
            support = [pair[0]['frame_index'] for pair in selected]
            relation_id = base._hash('b6-unit-ambiguity:', [*key, support, opposite])
            audits.append({'relation_id': relation_id, 'first_track_id': key[0], 'second_track_id': key[1], 'first_physical_episode_id': key[2], 'second_physical_episode_id': key[3], 'examined_source_frames': [pair[0]['frame_index'] for pair in run], 'support_frames': support, 'opposite_observed_direction': opposite, 'ambiguity_supported': bool(selected) and (not opposite), 'physical_identity_merged': False, 'numeric_control': 'FROZEN_POSSIBLE_HANDOFF_MIN_IOU', 'claim_type': 'SCOPED_AMBIGUITY_NOT_POSITIVE_IDENTITY_OR_TRAFFIC_GROUND_TRUTH'})
            if not selected or opposite:
                continue
            a, b = selected[0]
            relation = {'type': 'SAME_UNIT', 'relation_type': 'SAME_UNIT', 'relation_id': relation_id, 'candidate_kind': AMBIGUITY_KIND, 'verified': False, 'first_track_id': a['track_id'], 'second_track_id': b['track_id'], 'first_physical_episode_id': a['physical_episode_id'], 'second_physical_episode_id': b['physical_episode_id'], 'first_unit_id': a['unit_id'], 'second_unit_id': b['unit_id'], 'first_episode_id': a['episode_id'], 'second_episode_id': b['episode_id'], 'support_frames': support, 'start_frame': support[0], 'end_frame': support[-1], 'verification_basis': 'EXACT_OBSERVED_HIGH_OVERLAP_SAME_CATEGORY_CONFLICT_NOT_ALIAS', 'claim_type': 'USER_AUTHORIZED_INVALID_AMBIGUITY_NOT_SAME_PHYSICAL_UNIT_ASSERTION', 'earlier_frames_reassigned': False}
            relations.append(relation)
            for first, second in selected:
                for item in (first, second):
                    if 'SCOPED_UNIT_AMBIGUITY' not in item['identity_timeout_hold']:
                        item['identity_timeout_hold'].append('SCOPED_UNIT_AMBIGUITY')
                    item.setdefault('v6_ambiguity_relation_ids', []).append(relation_id)
    return (relations, audits)

def _handoff_relations(result, negative_relations, distinct_audit):
    """Address inherited possible handoffs without inventing gap observations."""
    present = {(row['frame_index'], item['track_id']): item for row in result['frames'] for item in row['observations']}
    relations, audits, protected_scopes, unresolved_scopes = ([], [], set(), set())
    for handoff in result['handoff_audit']:
        first_key = (handoff['last_first_frame'], handoff['first_track_id'])
        second_key = (handoff['first_second_frame'], handoff['second_track_id'])
        first, second = (present[first_key], present[second_key])
        if first['role'] != 'vehicle' or second['role'] != 'vehicle':
            continue
        scopes = {(item['track_id'], item['physical_episode_id']) for item in (first, second)}
        proof = _proven_distinct(first, second, negative_relations)
        if proof:
            protected_scopes.update(scopes)
            distinct_audit.append(_distinct_audit(first, second, proof, HANDOFF_KIND))
            handoff['v6_disposition'] = 'PROVEN_DISTINCT_NOT_A_HANDOFF'
            handoff['v6_distinct_proof'] = proof
            continue
        unresolved_scopes.update(scopes)
        support = [first_key[0], second_key[0]]
        rid = previous.previous._hash('b6-unverified-handoff:', [first_key, second_key, first['physical_episode_id'], second['physical_episode_id']])
        relation = {'type': 'SAME_UNIT', 'relation_type': 'SAME_UNIT', 'relation_id': rid, 'candidate_kind': HANDOFF_KIND, 'verified': False, 'first_track_id': first['track_id'], 'second_track_id': second['track_id'], 'first_physical_episode_id': first['physical_episode_id'], 'second_physical_episode_id': second['physical_episode_id'], 'first_unit_id': first['unit_id'], 'second_unit_id': second['unit_id'], 'first_episode_id': first['episode_id'], 'second_episode_id': second['episode_id'], 'first_endpoint_frame': support[0], 'second_endpoint_frame': support[1], 'support_frames': support, 'start_frame': support[0], 'end_frame': support[-1], 'gap_source_frames': handoff['gap_source_frames'], 'verification_basis': 'INHERITED_POSSIBLE_HANDOFF_ENDPOINT_CONTROL_NOT_IDENTITY_PROOF', 'claim_type': 'UNVERIFIED_EPISODE_HANDOFF_NOT_OBSERVED_CROSSING', 'gap_observations_created': False, 'earlier_frames_reassigned': False}
        relations.append(relation)
        audits.append({'relation_id': rid, 'support_frames': support, 'first_track_id': first['track_id'], 'second_track_id': second['track_id'], 'gap_observations_created': False, 'physical_identity_merged': False})
        for item in (first, second):
            if 'UNVERIFIED_ORIGIN_HANDOFF' not in item['identity_timeout_hold']:
                item['identity_timeout_hold'].append('UNVERIFIED_ORIGIN_HANDOFF')
            item.setdefault('v6_ambiguity_relation_ids', []).append(rid)
    reason = 'POSSIBLE_UNVERIFIED_RAW_ID_HANDOFF_TIMEOUT_HELD'
    for row in result['frames']:
        for item in row['observations']:
            key = (item['track_id'], item['physical_episode_id'])
            if key in protected_scopes - unresolved_scopes:
                item['identity_timeout_hold'] = [value for value in item['identity_timeout_hold'] if value != reason]
    return (relations, audits)

def prepare_identity_frames(frames, geometry, policy, relationships=None):
    controls, inherited = _controls(policy)
    result = previous.prepare_identity_frames(frames, geometry, {'identity': inherited}, relationships=relationships)
    negative_relations = [relation for relation in list(result['unit_relations']) + list((relationships or {}).get('unit_relations', [])) if relation.get('verified') is True and relation.get('relation_type', relation.get('type')) == 'DIFFERENT_UNIT']
    distinct_audit = []
    _veto_inherited_cross_category(result, negative_relations, distinct_audit)
    new_relations, ambiguity_audit = _same_category_ambiguity(result['frames'], geometry, controls, negative_relations, distinct_audit)
    handoff_relations, handoff_audit = _handoff_relations(result, negative_relations, distinct_audit)
    result['unit_relations'] += new_relations + handoff_relations
    result['ambiguity_audit'] = ambiguity_audit
    result['unverified_handoff_relation_audit'] = handoff_audit
    result['known_distinct_constraint_audit'] = distinct_audit
    result['assignments'] = previous._assignments(result['frames'])
    raw = {(row['frame_index'], item['track_id']): item for row in result['frames'] for item in row['observations']}
    for assignment in result['assignments']:
        refs = raw[assignment['frame_index'], assignment['track_id']].get('v6_ambiguity_relation_ids', [])
        if refs:
            assignment['v6_ambiguity_relation_ids'] = sorted(set(refs))
    result['policy'] = controls
    result['provenance'].update(v6_same_category_ambiguity_supported=True, geometric_ambiguity_is_not_verified_unit_identity=True, distinct_constraints_require_scoped_identity_proof_not_role_or_category_review=True, opposite_event_direction_must_be_checked_by_counting=True, unverified_handoff_does_not_create_gap_crossing=True)
    return result


### src/traffic_location/traffic/_kernel/identity_core.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/identity_core.py
"""Explicit B v4 identity/role experiment over retained measured observations.

This module performs no detection, tracking, embedding inference or interpolation.
It does not establish physical identity from raw IDs. Geometry-only component
association is an inspectable research hypothesis, not a trained role classifier.
All new numerical controls are returned with their experimental status. The
median-motion candidate deliberately keeps the frozen 0.15 limit; it does not
raise the limit to make development examples pass.
"""
from __future__ import annotations
from collections import Counter, defaultdict
from copy import deepcopy
import hashlib
import json
import math
from statistics import median
VEHICLES = frozenset(('motorcycle', 'car', 'truck', 'bus'))

def experimental_identity_policy(*, component_profile='median_temporal_v1'):
    """Return explicit research controls, never an implied production default."""
    return {'schema_version': 'phase8_b_v4_identity_policy_v1', 'status': 'EXPERIMENTAL_CANDIDATE_NOT_VALIDATED_PRODUCTION_DEFAULT', 'component_motion_profile': component_profile, 'max_unobserved_frames': 60, 'max_center_jump_norm': 1.0, 'min_component_run': 3, 'component_motion_limit': 0.15, 'component_bc_inside_fraction': 1.0, 'component_center_limit': 0.51, 'possible_handoff_center_limit': 0.1, 'possible_handoff_min_iou': 0.59, 'control_provenance': {'max_unobserved_frames': 'EXISTING_TIMEOUT_HORIZON_REUSED_AS_EXPERIMENT_NOT_VALIDATED_IDENTITY_LIFECYCLE', 'max_center_jump_norm': 'NEW_PROPOSAL_CONTEXT_DISCONTINUITY_CONTROL_NOT_IDENTITY_PROOF', 'component_motion_profile': 'NEW_PROPOSAL_MEDIAN_VS_RETAINED_MAX_ABLATION', 'component_controls': 'FROZEN_PHASE8_COMPONENT_3_0.15_1.0_0.51_REUSED', 'possible_handoff_controls': 'FROZEN_SAME_VEHICLE_CENTER_0.1_IOU_0.59_REUSED_FOR_HOLD_ONLY'}, 'automatic_handoff': 'DISABLED_WITHOUT_POSITIVE_SOURCE_BOUND_ASSOCIATION', 'appearance_features_used': False, 'role_propagation': 'EXACT_OBSERVED_CONSECUTIVE_SUPPORTED_RUN_ONLY', 'unknown_person_is_pedestrian': False}

def _require(condition, message):
    if not condition:
        raise ValueError(message)

def _hash(prefix, value):
    data = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return prefix + hashlib.sha256(data.encode()).hexdigest()[:24]

def _center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)

def _diag(box):
    return math.hypot(box[2] - box[0], box[3] - box[1])

def _distance(first, second):
    return math.hypot(first[0] - second[0], first[1] - second[1])

def _iou(first, second):
    intersection = max(0, min(first[2], second[2]) - max(first[0], second[0])) * max(0, min(first[3], second[3]) - max(first[1], second[1]))
    areas = (first[2] - first[0]) * (first[3] - first[1]) + (second[2] - second[0]) * (second[3] - second[1])
    return intersection / (areas - intersection)

def _jump(first, second):
    return _distance(_center(first['raw_bbox']), _center(second['raw_bbox'])) / ((_diag(first['raw_bbox']) + _diag(second['raw_bbox'])) / 2)

def _positive_review(observation):
    return bool(observation.get('review_refs') or observation.get('review_ref'))

def _verified_review_continuity(first, second):
    """Role/category review is not a positive physical-identity association."""

    def reviewed_association(item):
        association = item.get('association_id')
        refs = list(item.get('review_refs', []))
        if isinstance(item.get('review_ref'), dict):
            refs.append(item['review_ref'])
        return association if item.get('association_verified') is True and association is not None and any((ref.get('review_id') == association for ref in refs)) else None
    first_id, second_id = (reviewed_association(first), reviewed_association(second))
    return first_id is not None and first_id == second_id and (first.get('_source_unit_id', first['unit_id']) == second['unit_id']) and (first.get('_source_episode_id', first['episode_id']) == second['episode_id'])

def _validate_policy(policy):
    controls = deepcopy(policy.get('identity', policy))
    expected = experimental_identity_policy()
    _require(isinstance(controls, dict) and controls.get('schema_version') == expected['schema_version'], 'Explicit B v4 identity policy is required')
    _require(controls.get('component_motion_profile') in ('median_temporal_v1', 'retained_max_v1'), 'Unknown component motion experiment')
    for key in ('max_unobserved_frames', 'min_component_run'):
        _require(type(controls.get(key)) is int and controls[key] >= 1, f'Invalid {key}')
    for key in ('max_center_jump_norm', 'component_motion_limit', 'component_bc_inside_fraction', 'component_center_limit', 'possible_handoff_center_limit', 'possible_handoff_min_iou'):
        value = controls.get(key)
        _require(type(value) in (int, float) and math.isfinite(value) and (value >= 0), f'Invalid {key}')
    _require(controls['component_bc_inside_fraction'] <= 1 and controls['possible_handoff_min_iou'] <= 1, 'Fractions must be in [0,1]')
    _require(controls.get('appearance_features_used') is False, 'Appearance embeddings are not available')
    return controls

def _validated_frames(frames):
    result = deepcopy(list(frames))
    by_track = defaultdict(list)
    for index, row in enumerate(result):
        _require(row.get('frame_index') == index, 'Full measured frames must be continuous from zero')
        seen = set()
        for item in row.get('observations', []):
            track, box = (item.get('track_id'), item.get('raw_bbox'))
            _require(type(track) is int and track > 0 and (track not in seen), 'Invalid/repeated raw track')
            seen.add(track)
            _require(item.get('observed') is True and item.get('emitted') is True, 'Prediction/non-emitted observations cannot enter identity replay')
            _require(item.get('frame_index', index) == index, 'Measurement frame differs')
            _require(isinstance(box, (tuple, list)) and len(box) == 4 and all((type(x) in (int, float) and math.isfinite(x) for x in box)) and (box[0] < box[2]) and (box[1] < box[3]), 'Invalid measured bbox')
            _require(item.get('class_name') in VEHICLES | {'person'}, 'Unsupported raw observation family')
            _require(item.get('role') in ('vehicle', 'unknown_person', 'rider', 'pedestrian'), 'Source adapter must attach an explicit role')
            _require(item.get('role') not in ('rider', 'pedestrian') or item['class_name'] == 'person', 'Person roles require raw person measurements')
            item['frame_index'] = index
            item['_source_association_verified'] = item.get('association_verified') is True
            item['_source_association_id'] = item.get('association_id')
            item.setdefault('identity_timeout_hold', [])
            _require(isinstance(item['identity_timeout_hold'], list), 'Timeout hold must list explicit reasons')
            by_track[track].append(item)
    return (result, by_track)

def _role_candidates(frames, controls):
    """Candidate runs contain observed co-presence only, never gap interpolation."""
    pairs = defaultdict(list)
    for row in frames:
        people = [o for o in row['observations'] if o['class_name'] == 'person' and o['role'] != 'pedestrian' and (not _positive_review(o))]
        motors = [o for o in row['observations'] if o['class_name'] == 'motorcycle' and o['role'] == 'vehicle']
        for person in people:
            for motor in motors:
                distance = _distance(_center(person['raw_bbox']), _center(motor['raw_bbox'])) / _diag(motor['raw_bbox'])
                if distance <= controls['component_center_limit']:
                    pairs[person['track_id'], motor['track_id']].append((person, motor))
    candidates = []
    for (person_id, motor_id), observations in sorted(pairs.items()):
        runs, current = ([], [])
        for pair in observations:
            if current and (pair[0]['frame_index'] != current[-1][0]['frame_index'] + 1 or _jump(current[-1][0], pair[0]) > controls['max_center_jump_norm'] or pair[1]['_identity_parent_context'] != current[-1][1]['_identity_parent_context']):
                runs.append(current)
                current = []
            current.append(pair)
        if current:
            runs.append(current)
        for run in runs:
            residuals, inside = ([], [])
            offsets, lengths = ([], [])
            for person, motor in run:
                box, parent = (person['raw_bbox'], motor['raw_bbox'])
                center, parent_center = (_center(box), _center(parent))
                offsets.append((center[0] - parent_center[0], center[1] - parent_center[1]))
                lengths.append(_diag(parent))
                foot = ((box[0] + box[2]) / 2, box[3])
                inside.append(parent[0] <= foot[0] <= parent[2] and parent[1] <= foot[1] <= parent[3])
            for index in range(1, len(run)):
                residuals.append(_distance(offsets[index], offsets[index - 1]) / ((lengths[index] + lengths[index - 1]) / 2))
            score = (median(residuals) if controls['component_motion_profile'] == 'median_temporal_v1' else max(residuals)) if residuals else None
            reasons = []
            if len(run) < controls['min_component_run']:
                reasons.append('INSUFFICIENT_CONSECUTIVE_COMPONENT_OBSERVATIONS')
            if sum(inside) / len(inside) < controls['component_bc_inside_fraction']:
                reasons.append('PERSON_BC_NOT_CONSISTENTLY_INSIDE_PARENT')
            if score is None or score > controls['component_motion_limit']:
                reasons.append('COMPONENT_RELATIVE_MOTION_NOT_SUPPORTED')
            if any((p.get('identity_conflict') or m.get('identity_conflict') for p, m in run)):
                reasons.append('SOURCE_IDENTITY_CONFLICT_IN_COMPONENT_RUN')
            frames_in_run = [p['frame_index'] for p, _ in run]
            candidates.append({'relation_id': _hash('b4-component:', [person_id, motor_id, frames_in_run, controls['component_motion_profile']]), 'type': 'COMPONENT_OF', 'component_track_id': person_id, 'vehicle_track_id': motor_id, 'support_frames': frames_in_run, 'status': 'SUPPORTED' if not reasons else 'REJECTED', 'reasons': reasons, 'features': {'common_frames': len(run), 'first_frame': frames_in_run[0], 'last_frame': frames_in_run[-1], 'bc_inside_fraction': sum(inside) / len(inside), 'relative_motion_max': max(residuals) if residuals else None, 'relative_motion_median': median(residuals) if residuals else None, 'relative_motion_selected_score': score, 'appearance_features_used': False}, 'claim_type': 'EXPERIMENTAL_GEOMETRY_MOTION_ASSOCIATION_NOT_HUMAN_ROLE_PROOF', 'profile': controls['component_motion_profile'], '_pairs': run})
    return candidates

def _apply_roles(candidates, frames):
    active = defaultdict(list)
    for relation in candidates:
        if relation['status'] == 'SUPPORTED':
            for person, motor in relation['_pairs']:
                active[person['frame_index'], person['track_id']].append((relation, person, motor))
    for (frame, track), values in sorted(active.items()):
        parents = {motor['track_id'] for _, _, motor in values}
        person = values[0][1]
        if len(parents) != 1:
            person['identity_timeout_hold'].append('COMPETING_COMPONENT_PARENTS')
            person.update(role='unknown_person')
            person.pop('component_of_unit_id', None)
            person.pop('component_of_track_id', None)
            for relation, _, _ in values:
                relation.setdefault('ambiguous_frames', []).append(frame)
            continue
        relation, person, motor = values[0]
        person.update(role='rider', component_of_unit_id=motor['unit_id'], component_of_track_id=motor['track_id'], role_scope_id=relation['relation_id'], component_parent_context_id=motor['_identity_parent_context'], role_source='B_V4_EXPERIMENTAL_BOUNDED_COMPONENT_RUN')

def _parent_contexts(by_track, controls):
    """Parent raw ID reuse must not join component evidence across contexts."""
    for track, observations in by_track.items():
        previous, context = (None, None)
        for item in observations:
            if item['role'] != 'vehicle':
                continue
            source_key = (item['unit_id'], item['episode_id'])
            discontinuous = previous is None or source_key != (previous['unit_id'], previous['episode_id']) or (not _verified_review_continuity(previous, item) and (item['frame_index'] - previous['frame_index'] > controls['max_unobserved_frames'] or (_jump(previous, item) > controls['max_center_jump_norm'] and _iou(previous['raw_bbox'], item['raw_bbox']) == 0)))
            if discontinuous:
                context = _hash('b4-parent-context:', [track, source_key, item['frame_index']])
            item['_identity_parent_context'] = context
            previous = item

def _split_contexts(by_track, controls):
    audits = []
    for track, observations in sorted(by_track.items()):
        previous, segment_start, segment_reason, segment_items = (None, None, None, [])
        context_parent, boundary_episode, segment_identity = (None, None, None)
        seen_counting_keys = set()

        def finish():
            if not segment_items:
                return
            first, last = (segment_items[0], segment_items[-1])
            audits.append({'physical_episode_id': first['physical_episode_id'], 'track_id': track, 'physical_unit_id': first['unit_id'], 'episode_id': first['episode_id'], 'start_frame': first['frame_index'], 'end_frame': last['frame_index'], 'observed_frames': len(segment_items), 'boundary_reason': segment_reason, 'raw_category_observations': dict(Counter((o['class_name'] for o in segment_items))), 'source_measurements_unchanged': True, 'claim_type': 'DETERMINISTIC_CONTEXT_SEGMENT_NOT_VERIFIED_PHYSICAL_IDENTITY'})
        for item in observations:
            reason = None
            explicit = _positive_review(item)
            if previous is None:
                reason = 'INITIAL_OBSERVATION'
            elif (item['unit_id'], item['episode_id']) != (previous.get('_source_unit_id', previous['unit_id']), previous.get('_source_episode_id', previous['episode_id'])) and (explicit or _positive_review(previous)):
                reason = 'SOURCE_BOUND_ASSIGNMENT_CHANGE'
            else:
                delta = item['frame_index'] - previous['frame_index']
                jump = _jump(previous, item)
                continuity_review = _verified_review_continuity(previous, item)
                if delta > controls['max_unobserved_frames'] and (not continuity_review):
                    reason = 'LONG_MEASUREMENT_GAP_NO_IDENTITY_CONTINUITY_PROOF'
                elif jump > controls['max_center_jump_norm'] and _iou(previous['raw_bbox'], item['raw_bbox']) == 0 and (not continuity_review):
                    reason = 'SPATIAL_JUMP_WITH_DISJOINT_BOX_CONTEXT'
                elif previous['class_name'] in VEHICLES and item['class_name'] == 'person' and (item.get('component_of_track_id') not in (None, track)) and (item['role'] == 'rider'):
                    reason = 'VEHICLE_TO_PERSON_WITH_DIFFERENT_POSITIVE_PARENT_CONTEXT'
                elif item['class_name'] in VEHICLES and context_parent is not None:
                    reason = 'RAW_VEHICLE_AFTER_BOUNDED_COMPONENT_CONTEXT_NO_ORIGIN_INHERITANCE'
                elif item['role'] == 'rider' and context_parent is not None and ((item.get('component_of_track_id'), item.get('component_parent_context_id')) != context_parent):
                    reason = 'POSITIVE_COMPONENT_PARENT_CHANGED'
            if reason is not None:
                finish()
                segment_start, segment_reason, segment_items = (item['frame_index'], reason, [])
                context_parent = None
                segment_identity = [track, item['unit_id'], item['episode_id'], segment_start]
                boundary_episode = None if reason in ('INITIAL_OBSERVATION', 'SOURCE_BOUND_ASSIGNMENT_CHANGE') else f"{item['episode_id']}|b4-raw:{track}:start:{segment_start}"
                if reason == 'SOURCE_BOUND_ASSIGNMENT_CHANGE' and (item['unit_id'], item['episode_id']) in seen_counting_keys:
                    boundary_episode = f"{item['episode_id']}|b4-raw:{track}:start:{segment_start}"
            original_unit, original_episode = (item['unit_id'], item['episode_id'])
            item['_source_unit_id'], item['_source_episode_id'] = (original_unit, original_episode)
            if boundary_episode is not None:
                item['episode_id'] = boundary_episode
                item['source_bound_episode_id_before_context_split'] = original_episode
                item.update(association_verified=False)
                item.pop('association_id', None)
            item['identity_episode_boundary'] = reason is not None
            item['identity_episode_reason'] = segment_reason
            item['physical_episode_id'] = _hash('b4-physical-episode:', segment_identity)
            if reason and previous is not None:
                item['previous_physical_episode_id'] = previous['physical_episode_id']
            if item['role'] == 'rider' and item.get('component_of_track_id') is not None:
                context_parent = (item['component_of_track_id'], item.get('component_parent_context_id'))
            seen_counting_keys.add((item['unit_id'], item['episode_id']))
            segment_items.append(item)
            previous = item
        finish()
    return audits

def _bind_components(frames, candidates):
    links = defaultdict(list)
    for row in frames:
        present = {o['track_id']: o for o in row['observations']}
        for item in row['observations']:
            if item['role'] != 'rider':
                continue
            parent = present.get(item.get('component_of_track_id'))
            if parent is None:
                matches = [o for o in present.values() if o['role'] == 'vehicle' and o['class_name'] == 'motorcycle' and (o['unit_id'] == item.get('component_of_unit_id'))]
                parent = matches[0] if len(matches) == 1 else None
            if parent is None or parent['class_name'] != 'motorcycle':
                continue
            item.update(component_of_unit_id=parent['unit_id'], component_of_track_id=parent['track_id'], component_of_episode_id=parent['episode_id'], component_parent_context_id=parent.get('_identity_parent_context'))
            scope = item.get('role_scope_id', item.get('retained_component_relation_id', 'source-bound-review'))
            links[scope, item['track_id'], parent['track_id']].append(row['frame_index'])
    return sorted([{'type': 'COMPONENT_OF', 'relation_id': relation, 'component_track_id': child, 'vehicle_track_id': parent, 'support_frames': support, 'claim_type': 'EXACT_OBSERVED_ROLE_SCOPE_DISPLAY_ONLY_RAW_VEHICLE_BC_PRESERVED'} for (relation, child, parent), support in sorted(links.items())], key=lambda link: (link['support_frames'][0], link['component_track_id'], link['vehicle_track_id']))

def _align_local_alias_contexts(frames, episode_audit):
    """Align a new alias with the CURRENT positively co-observed context.

    A whole-ID retained clique must not reconnect a later alias to a closed
    earlier object that reused a raw ID. Actual exact-frame positive clique
    evidence may instead align the alias to the current local episode. This
    is never back-propagated to earlier frames. Subsequent observations in the
    same local episode retain that counting namespace as an operational
    continuity assumption; ``association_verified`` remains true only where
    the original source adapter supplied positive exact-frame evidence.
    """
    starts = {row['physical_episode_id']: row['start_frame'] for row in episode_audit}
    aliases, prior_keys, aligned = ({}, {}, [])
    by_physical = defaultdict(list)
    for row in frames:
        for item in row['observations']:
            mapped = aliases.get(item['physical_episode_id']) if item['role'] == 'vehicle' else None
            if mapped is not None:
                item['unit_id'], item['episode_id'] = mapped
        positive = defaultdict(list)
        for item in row['observations']:
            proof = item.get('source_ref', {}).get('retained_same_vehicle_association', {})
            if item['role'] == 'vehicle' and item.get('_source_association_verified') and (item.get('_source_association_id') is not None) and (proof.get('frame_index') == row['frame_index']) and (proof.get('whole_id_support_implied') is False):
                positive[item['_source_association_id']].append(item)
        for association, items in positive.items():
            tracks = {o['track_id'] for o in items}
            if len(items) < 2 or any((o.get('identity_conflict') or _positive_review(o) for o in items)) or any((set(o['source_ref']['retained_same_vehicle_association'].get('track_ids', [])) != tracks for o in items)):
                continue
            anchor = min(items, key=lambda item: (starts[item['physical_episode_id']], item['track_id']))
            canonical = (anchor['unit_id'], anchor['episode_id'])
            for item in items:
                prior = (item['unit_id'], item['episode_id'])
                if prior != canonical:
                    aligned.append({'frame_index': row['frame_index'], 'track_id': item['track_id'], 'physical_episode_id': item['physical_episode_id'], 'old_counting_key': list(prior), 'current_counting_key': list(canonical), 'anchor_track_id': anchor['track_id'], 'association_id': association, 'claim_type': 'EXACT_FRAME_SUPPORTED_ALIAS_CURRENT_CONTEXT_NOT_WHOLE_RAW_ID', 'earlier_frames_reassigned': False})
                item['unit_id'], item['episode_id'] = canonical
                item.update(association_verified=True, association_id=association)
                aliases[item['physical_episode_id']] = canonical
        for item in row['observations']:
            key = (item['unit_id'], item['episode_id'])
            prior = prior_keys.get(item['track_id'])
            if prior is not None and prior[0] != key and (prior[1] == item['physical_episode_id']) and (prior[2] == item['role'] == 'vehicle') and (not item['identity_episode_boundary']):
                item['identity_episode_boundary'] = True
                item['identity_episode_reason'] = 'SOURCE_FRAME_VERIFIED_ALIAS_CONTEXT_ALIGNMENT'
            prior_keys[item['track_id']] = (key, item['physical_episode_id'], item['role'])
            by_physical[item['physical_episode_id']].append(item)
    for audit in episode_audit:
        items = by_physical[audit['physical_episode_id']]
        audit['physical_unit_id'], audit['episode_id'] = (items[0]['unit_id'], items[0]['episode_id'])
        contexts = defaultdict(list)
        for item in items:
            contexts[item['unit_id'], item['episode_id']].append(item['frame_index'])
        audit['counting_contexts'] = [{'physical_unit_id': key[0], 'episode_id': key[1], 'first_frame': min(indices), 'last_frame': max(indices), 'observed_frames': len(indices)} for key, indices in sorted(contexts.items())]
    return aligned

def _possible_handoffs(by_track, controls):
    """Similar endpoint boxes only HOLD estimates; they never verify a merge."""
    episodes = defaultdict(list)
    for observations in by_track.values():
        for item in observations:
            if item['role'] == 'vehicle':
                episodes[item['physical_episode_id']].append(item)
    ordered = sorted(episodes.values(), key=lambda values: (values[0]['frame_index'], values[0]['track_id']))
    audits = []
    for index, first in enumerate(ordered):
        a = first[-1]
        for second in ordered[index + 1:]:
            b = second[0]
            delta = b['frame_index'] - a['frame_index']
            if b['frame_index'] - a['frame_index'] > controls['max_unobserved_frames']:
                break
            if delta <= 0 or a['track_id'] == b['track_id']:
                continue
            category_a = a.get('resolved_class_name', a['class_name'])
            category_b = b.get('resolved_class_name', b['class_name'])
            if category_a != category_b:
                continue
            if a.get('association_verified') is True and b.get('association_verified') is True and (a.get('association_id') == b.get('association_id')) and (a.get('association_id') is not None) and ((a['unit_id'], a['episode_id']) == (b['unit_id'], b['episode_id'])):
                continue
            distance = _jump(a, b)
            overlap = _iou(a['raw_bbox'], b['raw_bbox'])
            if distance > controls['possible_handoff_center_limit'] or overlap < controls['possible_handoff_min_iou']:
                continue
            reason = 'POSSIBLE_UNVERIFIED_RAW_ID_HANDOFF_TIMEOUT_HELD'
            for item in first + second:
                if reason not in item['identity_timeout_hold']:
                    item['identity_timeout_hold'].append(reason)
            audits.append({'first_track_id': a['track_id'], 'second_track_id': b['track_id'], 'first_episode_id': a['physical_episode_id'], 'second_episode_id': b['physical_episode_id'], 'last_first_frame': a['frame_index'], 'first_second_frame': b['frame_index'], 'gap_source_frames': delta, 'endpoint_center_distance_norm': distance, 'endpoint_iou': overlap, 'status': 'UNRESOLVED_HOLD_TIMEOUT_ONLY', 'automatic_alias_created': False, 'appearance_features_used': False, 'reason': reason})
    return audits

def prepare_identity_frames(frames, geometry, policy, relationships=None):
    """Decorate measured frames and return inspectable episode/role/hold evidence.

    ``frames`` is the full B v3 adapter iterator. Its source link and reviews
    must already have been verified by that adapter. ``relationships`` is kept
    as an explicit provenance input; it is never a license to extend a relation
    outside actual supported observation frames. ``geometry`` is accepted to
    keep the integration boundary explicit; geometry is not modified here.
    """
    controls = _validate_policy(policy)
    result, by_track = _validated_frames(frames)
    _parent_contexts(by_track, controls)
    candidates = _role_candidates(result, controls)
    _apply_roles(candidates, result)
    _bind_components(result, candidates)
    episode_audit = _split_contexts(by_track, controls)
    alias_audit = _align_local_alias_contexts(result, episode_audit)
    component_links = _bind_components(result, candidates)
    handoff_audit = _possible_handoffs(by_track, controls)
    assignments = []
    for row in result:
        for item in row['observations']:
            assignments.append({'frame_index': row['frame_index'], 'track_id': item['track_id'], 'physical_unit_id': item['unit_id'], 'episode_id': item['episode_id'], 'role': item['role'], 'category': item.get('resolved_class_name', item['class_name']), 'raw_category': item['class_name'], 'review_refs': deepcopy(item.get('review_refs', [])), 'association_verified': item.get('association_verified', False), 'association_id': item.get('association_id'), **{key: deepcopy(item[key]) for key in ('component_of_unit_id', 'component_of_track_id', 'component_of_episode_id', 'component_parent_context_id', 'physical_episode_id', 'identity_episode_boundary', 'identity_episode_reason', 'previous_physical_episode_id', 'identity_timeout_hold', 'source_bound_episode_id_before_context_split', 'role_scope_id', 'role_source') if key in item}})
            item.pop('_source_unit_id', None)
            item.pop('_source_episode_id', None)
            item.pop('_identity_parent_context', None)
            item.pop('_source_association_verified', None)
            item.pop('_source_association_id', None)
    for candidate in candidates:
        candidate['applied_support_frames'] = [person['frame_index'] for person, _ in candidate['_pairs'] if person.get('role_scope_id') == candidate['relation_id'] and person['role'] == 'rider']
        candidate['application_status'] = 'APPLIED_EXACT_OBSERVED_FRAME_SCOPE' if candidate['applied_support_frames'] else 'NOT_APPLIED_REJECTED_OR_COMPETING_PARENT'
        candidate.pop('_pairs', None)
    return {'frames': result, 'episode_audit': episode_audit, 'role_audit': candidates, 'handoff_audit': handoff_audit, 'policy': controls, 'assignments': assignments, 'component_links': component_links, 'alias_context_audit': alias_audit, 'provenance': {'source_measurements_unchanged': True, 'geometry_unchanged': True, 'source_adapter_relations_present': relationships is not None, 'no_observations_synthesized': True, 'no_detector_tracker_reid_or_enhancement': True, 'appearance_features_used': False, 'replay_is_causal': False, 'offline_role_runs_may_use_later_observations_within_exact_scope': True}}


### src/traffic_location/traffic/_kernel/identity_lifecycle.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/identity_lifecycle.py
"""P0 identity isolation over immutable detector-linked measurements.

Research candidate, not verified physical identity. Whole-ID retained group names
are hypotheses only. The B v4 bounded-role/context machinery is reused without
editing its source; positive relations retain exact episode/frame scope. No
appearance features, pixels, detector or tracker are run by this helper.
"""
from __future__ import annotations
from collections import defaultdict
from copy import deepcopy
from statistics import median
from . import identity_core as previous
VEHICLES = previous.VEHICLES

def experimental_identity_policy(*, component_profile='median_temporal_v1'):
    controls = previous.experimental_identity_policy(component_profile=component_profile)
    controls.update(schema_version='phase8_b_v5_identity_policy_v1', historical_unit_policy='HYPOTHESIS_ONLY_ISOLATE_RAW_CONTEXT_UNTIL_SCOPED_POSITIVE_PROOF', same_unit_scope='EXACT_FRAME_PROOF_OR_SOURCE_BOUND_REVIEW_NO_RETROACTIVE_ASSIGNMENT', negative_relation_policy='DISJOINT_COPRESENCE_IS_A_CONSTRAINT_NOT_GLOBAL_PHYSICAL_TRUTH', category_is_identity_gate=False, cross_category_candidate_policy='EXACT_FRAME_HIGH_IOU_HOLDS_WEAK_COVARIATION_AUDIT_ONLY', cross_category_candidate_profile='p0_v2_high_iou_hold_weak_covariation_audit')
    return controls

def _controls(policy):
    value = deepcopy(policy.get('identity', policy))
    previous._require(isinstance(value, dict) and value.get('schema_version') == 'phase8_b_v5_identity_policy_v1', 'Explicit B v5 identity policy required')
    previous._require(value.get('historical_unit_policy') == 'HYPOTHESIS_ONLY_ISOLATE_RAW_CONTEXT_UNTIL_SCOPED_POSITIVE_PROOF', 'Historical unit hypotheses cannot silently become identity proof')
    previous._require(value.get('category_is_identity_gate') is False, 'Category cannot hide cross-category identity candidates')
    previous._require(value.get('cross_category_candidate_policy') == 'EXACT_FRAME_HIGH_IOU_HOLDS_WEAK_COVARIATION_AUDIT_ONLY', 'Explicit supported cross-category conflict policy required')
    previous._require(value.get('cross_category_candidate_profile') == 'p0_v2_high_iou_hold_weak_covariation_audit', 'Unknown cross-category candidate profile')
    previous._require(value.get('same_unit_scope') == 'EXACT_FRAME_PROOF_OR_SOURCE_BOUND_REVIEW_NO_RETROACTIVE_ASSIGNMENT', 'Unknown same-unit scope policy')
    previous._require(value.get('negative_relation_policy') == 'DISJOINT_COPRESENCE_IS_A_CONSTRAINT_NOT_GLOBAL_PHYSICAL_TRUTH', 'Unknown negative relation policy')
    check = deepcopy(value)
    check['schema_version'] = 'phase8_b_v4_identity_policy_v1'
    previous._validate_policy(check)
    return (value, check)

def _review_identity(item):
    """An explicit scoped source assignment is distinct from a group label."""
    refs = list(item.get('review_refs', []))
    if isinstance(item.get('review_ref'), dict):
        refs.append(item['review_ref'])
    unit = str(item['unit_id'])
    generated = unit.startswith(('retained-clique:', 'track:', 'person-track:'))
    return bool(refs) and (not generated)

def _retained_positive_groups(row, controls):
    groups = defaultdict(list)
    for item in row['observations']:
        proof = item.get('source_ref', {}).get('retained_same_vehicle_association', {})
        if item['role'] == 'vehicle' and item.get('association_verified') is True and (item.get('association_id') is not None) and (proof.get('frame_index') == row['frame_index']) and (proof.get('whole_id_support_implied') is False):
            groups[item['association_id']].append(item)
    accepted = set()
    for association, items in groups.items():
        tracks = {o['track_id'] for o in items}
        if len(items) < 2 or any((o.get('identity_conflict') for o in items)):
            continue
        if any((set(o['source_ref']['retained_same_vehicle_association'].get('track_ids', [])) != tracks for o in items)):
            continue
        if any((previous._iou(a['raw_bbox'], b['raw_bbox']) < controls['possible_handoff_min_iou'] for index, a in enumerate(items) for b in items[index + 1:])):
            continue
        accepted.add(association)
    return accepted

def _isolate_hypotheses(frames, controls):
    audit = []
    for row in frames:
        positive = _retained_positive_groups(row, controls)
        for item in row['observations']:
            if item['role'] != 'vehicle':
                continue
            source_unit, source_episode = (item['unit_id'], item['episode_id'])
            item['p0_historical_unit_hypothesis'] = source_unit
            item['p0_historical_episode_hypothesis'] = source_episode
            if _review_identity(item):
                item['p0_identity_basis'] = 'PRESERVED_SOURCE_BOUND_REVIEW_ASSIGNMENT'
                continue
            item['unit_id'] = f"track:{item['track_id']}"
            item['episode_id'] = 'unreviewed-vehicle'
            source_positive = item.get('association_verified') is True and item.get('association_id') in positive
            if not source_positive:
                item['association_verified'] = False
                item.pop('association_id', None)
            item['p0_identity_basis'] = 'EXACT_FRAME_RETAINED_POSITIVE_RELATION' if source_positive else 'ISOLATED_RAW_CONTEXT_PHYSICAL_CONTINUITY_UNVERIFIED'
            if source_unit != item['unit_id']:
                audit.append({'frame_index': row['frame_index'], 'track_id': item['track_id'], 'historical_unit_hypothesis': source_unit, 'independent_raw_context': item['unit_id'], 'exact_frame_positive_proof_present': source_positive, 'whole_id_unification_permitted': False})
    return audit

def _remap_retained_component_parents(frames, relationships):
    """Retain a source-supported rider's exact observed parent after isolation.

    A historical group label is not enough to pick the parent. The old adapter
    has already validated the source measurements; we additionally require its
    retained relation ID, child ID, support frame and actually present raw parent.
    No frame between support observations is filled or relabelled here.
    """
    supported = defaultdict(list)
    for link in (relationships or {}).get('component_links', []):
        for frame in link.get('support_frames', []):
            supported[frame, link.get('component_track_id'), link.get('relation_id')].append(link)
    audit = []
    for row in frames:
        present = {o['track_id']: o for o in row['observations']}
        for child in present.values():
            if child['role'] != 'rider' or child['class_name'] != 'person':
                continue
            key = (row['frame_index'], child['track_id'], child.get('retained_component_relation_id'))
            links = supported.get(key, [])
            parents = {link['vehicle_track_id'] for link in links if link.get('vehicle_track_id') in present and present[link['vehicle_track_id']]['role'] == 'vehicle' and (present[link['vehicle_track_id']]['class_name'] == 'motorcycle')}
            if len(parents) != 1:
                continue
            parent = present[next(iter(parents))]
            child.update(component_of_track_id=parent['track_id'], component_of_unit_id=parent['unit_id'], p0_component_parent_remap_source='EXACT_FRAME_RETAINED_COMPONENT_RELATION')
            audit.append({'frame_index': row['frame_index'], 'component_track_id': child['track_id'], 'vehicle_track_id': parent['track_id'], 'relation_id': key[2], 'physical_unit_id': parent['unit_id'], 'scope_extended': False})
    return audit

def _refine_episode_scopes(frames):
    """Alias adoption starts a new bounded physical-context reference.

    A local association later in the raw-ID history must never give finalizers
    an edge into an earlier independently observed object/context.
    """
    state = {}
    for row in frames:
        for item in row['observations']:
            track = item['track_id']
            base_context = item['physical_episode_id']
            counting_key = (item['unit_id'], item['episode_id'])
            prior = state.get(track)
            vehicle_context = counting_key if item['role'] == 'vehicle' else prior[1] if prior is not None and prior[0] == base_context else None
            changed = prior is None or base_context != prior[0] or (item['role'] == 'vehicle' and prior[1] is not None and (counting_key != prior[1]))
            if changed:
                context = previous._hash('b5-physical-context:', [track, base_context, *counting_key, row['frame_index']])
            else:
                context = prior[2]
            if prior is not None and context != prior[2]:
                item['previous_physical_episode_id'] = prior[2]
            elif 'previous_physical_episode_id' in item:
                item.pop('previous_physical_episode_id')
            item['physical_episode_id'] = context
            state[track] = (base_context, vehicle_context, context)

def _rebind_diagnostic_episodes(result):
    """Keep inherited diagnostics source-addressable after ID refinement."""
    index = {(row['frame_index'], item['track_id']): item['physical_episode_id'] for row in result['frames'] for item in row['observations']}
    for audit in result['handoff_audit']:
        for prefix, frame_key in (('first', 'last_first_frame'), ('second', 'first_second_frame')):
            key = (audit[frame_key], audit[prefix + '_track_id'])
            old = audit[prefix + '_episode_id']
            audit['inherited_' + prefix + '_episode_id'] = old
            audit[prefix + '_episode_id'] = index[key]
        audit['episode_refs_rebound_to_exact_observed_endpoints'] = True
    for audit in result['alias_context_audit']:
        key = (audit['frame_index'], audit['track_id'])
        audit['inherited_physical_episode_id'] = audit['physical_episode_id']
        audit['physical_episode_id'] = index[key]
        audit['episode_ref_rebound_to_exact_observed_frame'] = True

def _relation_rows(frames, controls):
    groups = {}

    def add(kind, first, second, frame, verified, basis):
        if kind != 'COMPONENT_OF' and first['track_id'] > second['track_id']:
            first, second = (second, first)
        key = (kind, first['track_id'], second['track_id'], first['physical_episode_id'], second['physical_episode_id'], verified, basis)
        relation = groups.setdefault(key, {'relation_type': kind, 'type': kind, 'relation_id': previous._hash('b5-relation:', key), 'first_track_id': first['track_id'], 'second_track_id': second['track_id'], 'first_physical_episode_id': first['physical_episode_id'], 'second_physical_episode_id': second['physical_episode_id'], 'first_unit_id': first['unit_id'], 'second_unit_id': second['unit_id'], 'first_episode_id': first['episode_id'], 'second_episode_id': second['episode_id'], 'support_frames': [], 'verified': verified, 'verification_basis': basis, 'claim_type': 'SOURCE_SCOPED_RELATION_NOT_GLOBAL_RAW_ID_OR_GROUND_TRUTH', 'earlier_frames_reassigned': False})
        relation['support_frames'].append(frame)
    for row in frames:
        present = {o['track_id']: o for o in row['observations']}
        vehicles = [o for o in present.values() if o['role'] == 'vehicle']
        for index, first in enumerate(vehicles):
            for second in vehicles[index + 1:]:
                if first.get('association_verified') is True and second.get('association_verified') is True and (first.get('association_id') == second.get('association_id')) and (first.get('association_id') is not None) and ((first['unit_id'], first['episode_id']) == (second['unit_id'], second['episode_id'])):
                    add('SAME_UNIT', first, second, row['frame_index'], True, 'EXACT_FRAME_POSITIVE_ASSOCIATION_OR_BOUNDED_REVIEW')
                elif first.get('p0_historical_unit_hypothesis') == second.get('p0_historical_unit_hypothesis') and first.get('p0_historical_unit_hypothesis', '').startswith('retained-clique:') and (previous._iou(first['raw_bbox'], second['raw_bbox']) == 0):
                    add('DIFFERENT_UNIT', first, second, row['frame_index'], False, 'DISJOINT_COPRESENCE_CONTRADICTS_UNSUPPORTED_GROUP_NAME_ONLY')
        for child in present.values():
            parent = present.get(child.get('component_of_track_id'))
            if child['role'] == 'rider' and parent is not None and (parent['role'] == 'vehicle'):
                add('COMPONENT_OF', child, parent, row['frame_index'], True, 'EXACT_OBSERVED_BOUNDED_ROLE_SUPPORT_NOT_TRAINED_ROLE_TRUTH')
    reviewed = defaultdict(lambda: defaultdict(list))
    for row in frames:
        for item in row['observations']:
            if item['role'] != 'vehicle' or not _review_identity(item):
                continue
            association = item.get('association_id')
            refs = item.get('review_refs', [])
            if item.get('association_verified') is True and association is not None and any((ref.get('review_id') == association for ref in refs)):
                key = (association, item['unit_id'], item['episode_id'])
                reviewed[key][item['track_id'], item['physical_episode_id']].append(item)
    for episodes in reviewed.values():
        members = list(episodes.values())
        for index, first in enumerate(members):
            for second in members[index + 1:]:
                if first[0]['track_id'] == second[0]['track_id']:
                    continue
                for item in first + second:
                    add('SAME_UNIT', first[0], second[0], item['frame_index'], True, 'SOURCE_BOUND_REVIEWED_HANDOFF_SCOPE_NOT_MEASURED_GAP_CROSSING')
    rows = []
    for relation in groups.values():
        relation['support_frames'] = sorted(set(relation['support_frames']))
        relation['start_frame'], relation['end_frame'] = (relation['support_frames'][0], relation['support_frames'][-1])
        rows.append(relation)
    return sorted(rows, key=lambda r: (r['start_frame'], r['relation_type'], r['first_track_id'], r['second_track_id']))

def _cross_category_candidates(frames, controls):
    """Find hypotheses hidden by a class-equality gate; never merge them.

    The existing component controls are reused as explicit research inputs.
    Overlap and covariation do not prove same unit (two adjacent vehicles can
    satisfy them). Weak covariation is audit-only in this versioned ablation.
    Only high-overlap co-observation holds a conflict, never creates an alias.
    """
    pairs = defaultdict(list)
    for row in frames:
        vehicles = [o for o in row['observations'] if o['role'] == 'vehicle']
        for index, first in enumerate(vehicles):
            for second in vehicles[index + 1:]:
                if first['class_name'] == second['class_name']:
                    continue
                if first['track_id'] > second['track_id']:
                    a, b = (second, first)
                else:
                    a, b = (first, second)
                if a.get('association_verified') and b.get('association_verified') and (a.get('association_id') == b.get('association_id')) and ((a['unit_id'], a['episode_id']) == (b['unit_id'], b['episode_id'])):
                    continue
                scale = (previous._diag(a['raw_bbox']) + previous._diag(b['raw_bbox'])) / 2
                distance = previous._distance(previous._center(a['raw_bbox']), previous._center(b['raw_bbox'])) / scale
                if previous._iou(a['raw_bbox'], b['raw_bbox']) <= 0 or distance > controls['component_center_limit']:
                    continue
                key = (a['track_id'], b['track_id'], a['physical_episode_id'], b['physical_episode_id'])
                pairs[key].append((a, b))
    relations, audit = ([], [])
    for key, observations in sorted(pairs.items()):
        runs, current = ([], [])
        for pair in observations:
            if current and pair[0]['frame_index'] != current[-1][0]['frame_index'] + 1:
                runs.append(current)
                current = []
            current.append(pair)
        if current:
            runs.append(current)
        for run in runs:
            offsets = []
            scales = []
            for first, second in run:
                a, b = (previous._center(first['raw_bbox']), previous._center(second['raw_bbox']))
                offsets.append((a[0] - b[0], a[1] - b[1]))
                scales.append((previous._diag(first['raw_bbox']) + previous._diag(second['raw_bbox'])) / 2)
            residuals = [previous._distance(offsets[i], offsets[i - 1]) / ((scales[i] + scales[i - 1]) / 2) for i in range(1, len(run))]
            score = median(residuals) if residuals else None
            temporal_support = len(run) >= controls['min_component_run'] and score is not None and (score <= controls['component_motion_limit'])
            high_overlap = [pair for pair in run if previous._iou(pair[0]['raw_bbox'], pair[1]['raw_bbox']) >= controls['possible_handoff_min_iou']]
            paths = [[((pair[k]['raw_bbox'][0] + pair[k]['raw_bbox'][2]) / 2, pair[k]['raw_bbox'][3]) for pair in run] for k in (0, 1)]
            vectors = [(path[-1][0] - path[0][0], path[-1][1] - path[0][1]) for path in paths]
            monotone = [all(((b[0] - a[0]) * vector[0] + (b[1] - a[1]) * vector[1] > 0 for a, b in zip(path, path[1:]))) for path, vector in zip(paths, vectors)]
            opposite = len(run) >= controls['min_component_run'] and all(monotone) and (vectors[0][0] * vectors[1][0] + vectors[0][1] * vectors[1][1] < 0)
            hold_supported = bool(high_overlap) and (not opposite)
            selected = high_overlap if hold_supported else run if temporal_support or opposite else []
            supported = bool(selected)
            mode = 'EXACT_FRAME_HIGH_IOU_CONFLICT_ONLY' if hold_supported else 'OPPOSITE_OBSERVED_BC_MOTION_AUDIT_ONLY' if opposite else 'WEAK_COVARIATION_AUDIT_ONLY' if temporal_support else 'NO_SUPPORTED_CONFLICT'
            support = [a['frame_index'] for a, _ in selected]
            relation_id = previous._hash('b5-cross-category-hypothesis:', [*key, support, mode])
            audit.append({'relation_id': relation_id, 'first_track_id': key[0], 'second_track_id': key[1], 'first_physical_episode_id': key[2], 'second_physical_episode_id': key[3], 'support_frames': support, 'examined_source_frames': [a['frame_index'] for a, _ in run], 'common_observed_frames': len(run), 'support_mode': mode, 'opposite_observed_bc_motion': opposite, 'hold_supported': hold_supported, 'median_relative_motion_residual_norm': score, 'candidate_supported': supported, 'automatic_alias_created': False, 'claim_type': 'GEOMETRIC_COVARIATION_HYPOTHESIS_NOT_PHYSICAL_IDENTITY_TRUTH'})
            if not supported:
                continue
            first, second = selected[0]
            relations.append({'relation_type': 'SAME_UNIT', 'type': 'SAME_UNIT', 'relation_id': relation_id, 'first_track_id': key[0], 'second_track_id': key[1], 'first_physical_episode_id': key[2], 'second_physical_episode_id': key[3], 'first_unit_id': first['unit_id'], 'second_unit_id': second['unit_id'], 'first_episode_id': first['episode_id'], 'second_episode_id': second['episode_id'], 'support_frames': support, 'start_frame': support[0], 'end_frame': support[-1], 'verified': False, 'candidate_kind': 'POSSIBLE_CROSS_CATEGORY_UNIT_OR_COMPONENT' if hold_supported else 'WEAK_CROSS_CATEGORY_AUDIT_ONLY', 'support_mode': mode, 'verification_basis': 'OVERLAP_CENTER_OR_COVARIATION_REUSE_FROZEN_RESEARCH_CONTROLS', 'claim_type': 'UNRESOLVED_SCOPED_PHYSICAL_UNIT_HYPOTHESIS_NOT_MERGE_PROOF', 'earlier_frames_reassigned': False})
            if hold_supported:
                for a, b in selected:
                    for item in (a, b):
                        reason = 'CROSS_CATEGORY_UNIT_OR_COMPONENT_UNRESOLVED'
                        if reason not in item['identity_timeout_hold']:
                            item['identity_timeout_hold'].append(reason)
    return (relations, audit)

def _assignments(frames):
    rows = []
    fields = ('component_of_unit_id', 'component_of_track_id', 'component_of_episode_id', 'component_parent_context_id', 'physical_episode_id', 'identity_episode_boundary', 'identity_episode_reason', 'previous_physical_episode_id', 'identity_timeout_hold', 'source_bound_episode_id_before_context_split', 'role_scope_id', 'role_source', 'p0_historical_unit_hypothesis', 'p0_identity_basis', 'p0_component_parent_remap_source')
    for row in frames:
        for item in row['observations']:
            rows.append({'frame_index': row['frame_index'], 'track_id': item['track_id'], 'physical_unit_id': item['unit_id'], 'episode_id': item['episode_id'], 'role': item['role'], 'category': item.get('resolved_class_name', item['class_name']), 'raw_category': item['class_name'], 'review_refs': deepcopy(item.get('review_refs', [])), 'association_verified': item.get('association_verified', False), 'association_id': item.get('association_id'), **{key: deepcopy(item[key]) for key in fields if key in item}})
    return rows

def prepare_identity_frames(frames, geometry, policy, relationships=None):
    controls, inherited_controls = _controls(policy)
    copied, _ = previous._validated_frames(frames)
    hypothesis_audit = _isolate_hypotheses(copied, controls)
    parent_remap_audit = _remap_retained_component_parents(copied, relationships)
    result = previous.prepare_identity_frames(copied, geometry, {'identity': inherited_controls}, relationships=relationships)
    _refine_episode_scopes(result['frames'])
    _rebind_diagnostic_episodes(result)
    cross_category_relations, cross_category_audit = _cross_category_candidates(result['frames'], controls)
    result['assignments'] = _assignments(result['frames'])
    result['unit_relations'] = _relation_rows(result['frames'], controls) + cross_category_relations
    result['cross_category_audit'] = cross_category_audit
    result['historical_hypothesis_audit'] = hypothesis_audit
    result['retained_component_parent_remap_audit'] = parent_remap_audit
    result['policy'] = controls
    result['inherited_episode_audit'] = result['episode_audit']
    contexts = defaultdict(list)
    for row in result['frames']:
        for item in row['observations']:
            contexts[item['physical_episode_id']].append(item)
    result['episode_audit'] = [{'physical_episode_id': key, 'track_id': items[0]['track_id'], 'physical_unit_id': items[0]['unit_id'], 'episode_id': items[0]['episode_id'], 'start_frame': items[0]['frame_index'], 'end_frame': items[-1]['frame_index'], 'observed_frames': len(items), 'boundary_reason': items[0].get('identity_episode_reason'), 'claim_type': 'P0_BOUNDED_CONTEXT_NOT_VERIFIED_PHYSICAL_IDENTITY'} for key, items in sorted(contexts.items(), key=lambda pair: (pair[1][0]['frame_index'], pair[1][0]['track_id']))]
    result['provenance'].update(whole_id_historical_groups_used_as_identity=False, current_source_positive_aliases_only=True, disjoint_copresence_is_not_global_physical_truth=True, same_class_gradual_handoff_completely_solved=False, source_bound_reviews_are_preserved_not_new_model_predictions=True)
    return result


### src/traffic_location/traffic/_kernel/measurement_adapter.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/measurement_adapter.py
from collections import Counter, defaultdict
from copy import deepcopy
import math
from .support import require, records, bbox, iter_json_array
VEHICLES = {'motorcycle', 'car', 'truck', 'bus'}

def retained_relationships(path):
    """Reuse supported old relations; no new appearance/spatial association.

    Whole-ID grouping is explicitly experimental: supported raw-ID graph
    components must be cliques and must not have an unresolved internal pair.
    Ambiguous SAME_VEHICLE support is retained locally as a count conflict.
    """
    components, edges, ambiguous = ([], set(), set())
    conflicts = defaultdict(set)
    for relation in iter_json_array(path):
        kind, status = (relation.get('type'), relation.get('status'))
        if kind not in ('COMPONENT_OF', 'SAME_VEHICLE') or status not in ('SUPPORTED', 'UNRESOLVED'):
            continue
        first, second = (relation['first_track_id'], relation['second_track_id'])
        frames = relation.get('features', {}).get('common_frame_indices', [])
        require(type(first) is int and type(second) is int and (first > 0) and (second > 0) and (first != second) and isinstance(frames, list) and frames and all((type(f) is int and f >= 0 for f in frames)), 'Invalid retained relation support')
        if kind == 'COMPONENT_OF' and status == 'SUPPORTED':
            components.append({'type': 'COMPONENT_OF', 'relation_id': relation['relation_id'], 'vehicle_track_id': second, 'component_track_id': first, 'support_frames': sorted(set(frames))})
        elif kind == 'SAME_VEHICLE':
            pair = frozenset((first, second))
            if status == 'SUPPORTED':
                edges.add(pair)
            else:
                ambiguous.add(pair)
                for frame in frames:
                    conflicts[frame].update((first, second))
    graph = defaultdict(set)
    for pair in edges - ambiguous:
        first, second = sorted(pair)
        graph[first].add(second)
        graph[second].add(first)
    visited, units, groups = (set(), {}, [])
    for first in sorted(graph):
        if first in visited:
            continue
        todo, members = ([first], set())
        while todo:
            current = todo.pop()
            if current in members:
                continue
            members.add(current)
            todo.extend(graph[current] - members)
        visited.update(members)
        if any((frozenset((a, b)) not in edges - ambiguous for a in members for b in members if a < b)):
            continue
        group = sorted(members)
        unit = 'retained-clique:' + ','.join(map(str, group))
        units.update({track: unit for track in group})
        groups.append(group)
    return {'component_links': components, 'unit_by_track': units, 'unit_groups': groups, 'identity_conflicts_by_frame': dict(conflicts)}

def relationship_evidence(path):
    result = retained_relationships(path)
    pairs, supported, relation_ids = (defaultdict(list), defaultdict(list), defaultdict(dict))
    for row in iter_json_array(path):
        if row.get('type') != 'SAME_VEHICLE' or row.get('status') not in ('SUPPORTED', 'UNRESOLVED'):
            continue
        pair = tuple(sorted((row['first_track_id'], row['second_track_id'])))
        target = pairs if row['status'] == 'UNRESOLVED' else supported
        for frame in row.get('features', {}).get('common_frame_indices', []):
            target[frame].append(pair)
            if row['status'] == 'SUPPORTED':
                relation_ids[frame].setdefault(pair, []).append(row['relation_id'])
    result.update(identity_pairs_by_frame=dict(pairs), supported_pairs_by_frame=dict(supported), supported_relation_ids_by_frame=dict(relation_ids))
    return result

def _family(category):
    return 'vehicle' if category in VEHICLES else 'person' if category == 'person' else 'other'

def _scope_matches(scope, track, frame, category):
    return scope['track_id'] == track and scope['start_frame'] <= frame <= scope['end_frame'] and (scope['raw_family'] == _family(category))

def _review_at(reviews, track, frame, category):
    return [(review, scope) for review in reviews for scope in review['scopes'] if _scope_matches(scope, track, frame, category)]

def _same_reviewed_pair(reviews, pair, frame):
    for review in reviews:
        if review['type'] != 'SAME_VEHICLE':
            continue
        active = {s['track_id'] for s in review['scopes'] if s['raw_family'] == 'vehicle' and s['start_frame'] <= frame <= s['end_frame']}
        if set(pair) <= active:
            return True
    return False

def _local_retained_associations(links, present, frame, identity):
    """Verify only an exact-frame complete positive co-observation clique.

    The retained whole-ID unit name is an operational grouping assumption. It
    does not verify every observation, fill support gaps, or authorize a chain
    merge. A future/earlier positive relation cannot verify this source frame.
    """
    units = links.get('unit_by_track', {})
    groups = defaultdict(list)
    for track, observation in present.items():
        if observation.get('class_name') in VEHICLES and track in units:
            groups[units[track]].append(track)
    positive = {tuple(sorted(pair)) for pair in links.get('supported_pairs_by_frame', {}).get(frame, [])}
    unresolved = {tuple(sorted(pair)) for pair in links.get('identity_pairs_by_frame', {}).get(frame, [])}
    verified = {}
    for unit, members in groups.items():
        members = sorted(members)
        if len(members) < 2:
            continue
        required = {(first, second) for index, first in enumerate(members) for second in members[index + 1:]}
        if not required <= positive - unresolved:
            continue
        relation_ids = sorted({rid for pair in required for rid in links.get('supported_relation_ids_by_frame', {}).get(frame, {}).get(pair, [])})
        for track in members:
            verified[track] = {'association_id': 'retained-local:' + identity['records_sha256'] + ':' + unit, 'association_evidence': {'claim_type': 'RETAINED_SUPPORTED_SAME_VEHICLE_EXACT_SOURCE_FRAME', 'frame_index': frame, 'track_ids': members, 'relation_ids': relation_ids, 'records_sha256': identity['records_sha256'], 'whole_id_support_implied': False}}
    return verified

def adapted_frames(path, identity, frame_count, *, relationships=None, reviews=None, reviews_sha256=None, assignments=None, application_proof=None):
    """Validate every emitted/raw link; attach explicit roles without editing it."""
    links, reviews = (relationships or {}, reviews or [])
    units = links.get('unit_by_track', {})
    components = defaultdict(dict)
    for link in links.get('component_links', []):
        for frame in link['support_frames']:
            components[frame][link['component_track_id']] = link
    proof = application_proof if application_proof is not None else {}
    counts, anchors, coexistence, count = (Counter(), set(), [], 0)
    for record in records(path):
        frame = record.get('frame_index')
        require(type(frame) is int and frame == count, 'B v3 frames must start at zero and be continuous')
        require(all((record.get(key) == identity.get(key) for key in ('video_sha256', 'parent_run_id', 'variant', 'width', 'height'))), 'Source cache identity differs')
        raw, tracks = (record.get('raw_detections'), record.get('emitted_tracks'))
        require(isinstance(raw, list) and isinstance(tracks, list), 'Missing retained source links')
        seen, indices, observations = (set(), set(), [])
        present = {o['track_id']: o for o in tracks}
        local_associations = _local_retained_associations(links, present, frame, identity)
        for review in reviews:
            anchor = review.get('_pixel_anchor_proofs', {}).get(frame)
            if not anchor:
                continue
            require(record.get('detector_input_frame_pixel_sha256') == anchor['raw_bgr_pixel_sha256'], 'Reviewed pedestrian frame differs from actual retained detector input')
            targets = {s['track_id'] for s in review['scopes']}
            for source in anchor['observations']:
                if source['track_id'] not in targets:
                    continue
                actual = present.get(source['track_id'], {})
                require(all((actual.get(key) == source[key] for key in ('class_name', 'raw_bbox', 'original_detection_index'))), 'Reviewed pedestrian source measurement differs')
        for original in tracks:
            track, index = (original.get('track_id'), original.get('original_detection_index'))
            require(type(track) is int and track > 0 and (track not in seen) and (type(index) is int) and (0 <= index < len(raw)) and (index not in indices), 'Repeated or invalid emitted source link')
            seen.add(track)
            indices.add(index)
            require(original.get('frame_index') == frame and original.get('emitted') is True and (original.get('measurement_provenance') == 'retained_phase_8_detection'), 'Only raw measured emitted source observations may enter replay')
            box, detection = (bbox(original.get('raw_bbox')), raw[index])
            require(0 <= box[0] < box[2] <= identity['width'] and 0 <= box[1] < box[3] <= identity['height'], 'BBox outside source image')
            require(box == detection.get('bbox_xyxy_pixels') and all((original.get(key) == detection.get(key) for key in ('class_name', 'class_id', 'confidence'))), 'Measurement differs from original detection')
            confidence = original.get('confidence')
            require(isinstance(confidence, (int, float)) and (not isinstance(confidence, bool)) and math.isfinite(confidence), 'Missing finite source confidence')
            category = original['class_name']
            if _family(category) == 'other':
                continue
            item = deepcopy(original)
            item.update(observed=True, role='unknown_person' if category == 'person' else 'vehicle', unit_id=f'person-track:{track}' if category == 'person' else units.get(track, f'track:{track}'), episode_id='unreviewed-person' if category == 'person' else 'unreviewed-vehicle', association_verified=False)
            association = local_associations.get(track)
            if association is not None:
                item.update(association_verified=True, association_id=association['association_id'])
                original_ref = deepcopy(item.get('source_ref'))
                item['source_ref'] = {'retained_same_vehicle_association': association['association_evidence']}
                if original_ref is not None:
                    item['source_ref']['original_source_ref'] = original_ref
            pairs = [pair for pair in links.get('identity_pairs_by_frame', {}).get(frame, []) if track in pair]
            item['identity_conflict'] = any((not _same_reviewed_pair(reviews, pair, frame) for pair in pairs))
            if not links.get('identity_pairs_by_frame'):
                item['identity_conflict'] = track in links.get('identity_conflicts_by_frame', {}).get(frame, set())
            local = components.get(frame, {}).get(track)
            if category == 'person' and local and (local['vehicle_track_id'] in present):
                parent = present[local['vehicle_track_id']]
                if parent['class_name'] == 'motorcycle':
                    item.update(role='rider', component_of_unit_id=units.get(local['vehicle_track_id'], f"track:{local['vehicle_track_id']}"), retained_component_relation_id=local['relation_id'])
            applied = _review_at(reviews, track, frame, category)
            for review, scope in applied:
                rid, kind = (review['review_id'], review['type'])
                counts[rid] += 1
                if frame in review['confirmed_frames']:
                    anchors.add((rid, track, frame))
                item.setdefault('review_refs', []).append({'review_id': rid, 'reviews_sha256': reviews_sha256, 'source_frame': frame, 'track_id': track, 'claim_type': review.get('claim_type', 'USER_REVIEW'), 'scope_extent_is_engineering_assumption': True})
                if kind in ('SAME_VEHICLE', 'SPLIT_IDENTITY'):
                    item.update(unit_id=scope.get('physical_unit_id', review.get('physical_unit_id')), episode_id=scope.get('episode_id', review.get('episode_id')))
                    if kind == 'SAME_VEHICLE':
                        item.update(association_verified=True, association_id=rid)
                elif kind == 'COMPONENT_OF':
                    item.update(role='rider', component_of_unit_id=review['component_of_unit_id'])
                elif kind == 'PEDESTRIAN_ROLE':
                    item.update(role='pedestrian', resolved_class_name='pedestrian', unit_id=review.get('physical_unit_id', f'reviewed-pedestrian:{track}'), episode_id=review.get('episode_id', f'reviewed-role:{rid}'))
                resolved = scope.get('resolved_class_name', review.get('resolved_class_name'))
                if resolved:
                    item['resolved_class_name'] = resolved
            observations.append(item)
            if assignments is not None:
                assignments.append({'frame_index': frame, 'track_id': track, 'physical_unit_id': item['unit_id'], 'episode_id': item['episode_id'], 'role': item['role'], 'category': item.get('resolved_class_name', category), 'raw_category': category, 'review_refs': deepcopy(item.get('review_refs', [])), 'association_verified': item['association_verified'], 'association_id': item.get('association_id'), **({'component_of_unit_id': item['component_of_unit_id']} if 'component_of_unit_id' in item else {})})
        simultaneous = defaultdict(list)
        for item in observations:
            if item['role'] == 'vehicle' and item.get('association_verified'):
                simultaneous[item['unit_id'], item['episode_id']].append(item)
        for (unit, episode), items in simultaneous.items():
            if len(items) < 2:
                continue
            for index, first in enumerate(items):
                for second in items[index + 1:]:
                    a, b = (first['raw_bbox'], second['raw_bbox'])
                    overlap = max(a[0], b[0]) < min(a[2], b[2]) and max(a[1], b[1]) < min(a[3], b[3])
                    coexistence.append({'frame_index': frame, 'physical_unit_id': unit, 'episode_id': episode, 'track_ids': [first['track_id'], second['track_id']], 'raw_bboxes': [a, b], 'boxes_overlap_with_positive_area': overlap, 'association_basis': 'SOURCE_FRAME_BOUND_REVIEWED_CLAIM_NOT_AUTOMATIC_IDENTITY_PROOF' if first.get('review_refs') or second.get('review_refs') else 'RETAINED_SUPPORTED_SAME_VEHICLE_EXACT_SOURCE_FRAME'})
        yield {'frame_index': frame, 'observations': observations}
        count += 1
    require(count == frame_count, 'Full B v3 source cache frame count differs')
    require(all((counts[r['review_id']] > 0 for r in reviews)), 'Review did not match any measured source')
    proof.update(applied_measurements=dict(counts), matched_reviewed_anchors=[{'review_id': r, 'track_id': t, 'frame_index': f} for r, t, f in sorted(anchors)], source_links_checked=True, no_source_observations_modified=True, all_source_frames_retained=count, simultaneous_reviewed_vehicle_measurements=coexistence)


### src/traffic_location/traffic/_kernel/overlay.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/overlay.py
from collections import defaultdict
from copy import deepcopy
from pathlib import Path
from .support import require, read, bound, sha
KINDS = {'SAME_VEHICLE', 'SPLIT_IDENTITY', 'COMPONENT_OF', 'PEDESTRIAN_ROLE'}

def validate_overlay(value, jobs, root, guarded):
    """Bind adjudications to exact input bytes and retained native-pixel anchors.

    The additional claims are research review inputs, never automatic model
    predictions or newly confirmed user labels. The immutable old adapter checks
    anchor pixels again against each actual retained frame during replay.
    """
    require(value.get('schema_version') == 'phase8_b_v4_review_overlay_v1' and value.get('donor_manifest_sha256') == jobs[0]['source_manifest_sha256'], 'Unknown B v4 overlay schema or foreign donor')
    videos = {v['video_id']: v for v in value.get('videos', [])}
    require(len(videos) == len(value.get('videos', [])) and set(videos) <= {job['video_id'] for job in jobs}, 'Overlay has duplicate or unknown video IDs')
    seen = set()
    for job in jobs:
        vid = job['video_id']
        if vid not in videos:
            continue
        video, source = (videos[vid], job['b_v2_source_identity'])
        require(all((video.get(key) == source[key] for key in ('video_sha256', 'records_sha256', 'geometry_sha256'))), 'Overlay source binding differs')
        for review in video.get('reviews', []):
            rid = review.get('review_id')
            require(isinstance(rid, str) and rid and (rid not in seen), 'Missing or duplicate overlay review ID')
            seen.add(rid)
            require(review.get('claim_type') == 'ADJUDICATED_AI_PIXELS_REVIEW' and review.get('type') in KINDS and (review.get('human_confirmation_status') == 'NOT_NEWLY_ADJUDICATED_BY_USER'), 'Overlay must retain explicit AI adjudication provenance')
            scopes, anchors = (review.get('scopes', []), review.get('confirmed_frames', []))
            require(scopes and anchors and all((type(f) is int for f in anchors)), 'Overlay needs bounded scopes and confirmed frames')
            for scope in scopes:
                start, end = (scope.get('start_frame'), scope.get('end_frame'))
                require(type(scope.get('track_id')) is int and scope['track_id'] > 0 and (type(start) is int) and (type(end) is int) and (0 <= start <= end < job['metadata']['frame_count']) and (scope.get('raw_family') in ('person', 'vehicle')), 'Invalid overlay scope')
            if review['type'] in ('PEDESTRIAN_ROLE', 'COMPONENT_OF'):
                require(all((s['raw_family'] == 'person' for s in scopes)), 'Role review may only relabel raw person measurements')
            if review['type'] == 'COMPONENT_OF':
                require(bool(review.get('component_of_unit_id')), 'Rider review needs parent')
            if review['type'] in ('SAME_VEHICLE', 'SPLIT_IDENTITY'):
                require(all((s.get('physical_unit_id', review.get('physical_unit_id')) and s.get('episode_id', review.get('episode_id')) for s in scopes)), 'Identity overlay needs unit and episode')
            evidence = review.get('evidence', [])
            require(evidence, 'Adjudication must bind retained evidence')
            proofs = {}
            for entry in evidence:
                path = bound(root, entry['path'], entry['sha256'], guarded)
                proof = read(path)
                frame = entry.get('frame_index')
                require(frame in anchors, 'Overlay evidence is not a confirmed frame')
                if 'checks' in proof:
                    require(proof.get('source_sha256') == source['video_sha256'] and proof.get('retained_cache_file_sha256') == source['records_sha256'], 'Overlay pixel evidence belongs to another input')
                    found = [c for c in proof['checks'] if c['frame_index'] == frame]
                    require(len(found) == 1 and found[0].get('status') == 'PASS' and (found[0].get('actual_bgr_pixel_sha256') == entry.get('source_frame_pixel_sha256')) and (found[0].get('expected_retained_detector_input_pixel_sha256') == entry.get('source_frame_pixel_sha256')), 'Overlay native pixel evidence differs')
                    check = found[0]
                    png = check.get('retained_native_png_path')
                    if png:
                        image = path.parent.parent / png
                        require(image.is_file() and sha(image) == check.get('sequentially_decoded_native_png_sha256'), 'Retained adjudication image bytes differ')
                        guarded.append({'path': str(image), 'sha256': sha(image)})
                    proofs[frame] = {'raw_bgr_pixel_sha256': entry['source_frame_pixel_sha256'], 'observations': []}
                else:
                    require(proof.get('video_sha256') == source['video_sha256'] and proof.get('source_records_sha256') == source['records_sha256'], 'Overlay pixel evidence belongs to another input')
                    images = [image for image in proof.get('images', []) if image['frame_index'] == frame]
                    require(len(images) == 1 and images[0].get('verified_decode_pixel_sha256') is True and (images[0].get('raw_bgr_pixel_sha256') == entry.get('source_frame_pixel_sha256')), 'Overlay native pixel evidence differs')
                    proofs[frame] = images[0]
            require(set(anchors) <= set(proofs), 'Every overlay anchor needs pixel evidence')
            require(all((any((s['start_frame'] <= f <= s['end_frame'] for s in scopes)) for f in anchors)), 'Overlay anchor outside its bounded scope')
            review['_pixel_anchor_proofs'] = proofs
    return videos


### src/traffic_location/traffic/_kernel/passage.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/passage.py
"""B v5 P0 episode, anchor-quality and finalization experiment on retained measured tracks only.

This does not run detection/tracking/ReID and does not modify the frozen B v1/B v2
resolvers. Three explicitly different evidence routes share one passage ledger.
An estimated timeout is an operational count, never evidence of an ID change.
The source adapter must validate retained detection links before calling this
module; this module additionally rejects non-observed/non-emitted inputs.
"""
from __future__ import annotations
from collections import Counter, defaultdict
from copy import deepcopy
import hashlib
import json
import math
from numbers import Real
from .geometry import Geometry, segment_intersection, signed_distance
SUPPORTED_CATEGORIES = frozenset(('motorcycle', 'car', 'truck', 'pedestrian'))
VEHICLE_CLASSES = frozenset(('motorcycle', 'car', 'truck', 'bus'))
ROLES = frozenset(('vehicle', 'pedestrian', 'rider', 'unknown_person'))
DISPLAY_PREFIXES = {'pedestrian': 'P', 'motorcycle': 'M', 'car': 'C', 'truck': 'T'}

def experimental_counting_policy():
    """Return P0 controls; numerical suitability is not a production claim.

    Requiring whole-box translation is a deliberately conservative hypothesis:
    it rejects BC displacement made only by a shrinking/expanding measured box.
    Corner measurements are *quality evidence*, never a replacement crossing
    anchor. The crossing point stays the raw measured vehicle bottom-center.
    """
    return {'schema_version': 'phase8_b_v5_p0_counting_policy_v1', 'status': 'EXPERIMENTAL_RESEARCH_NOT_PRODUCTION_VALIDATED', 'anchor_quality': 'WHOLE_BOX_TOWARD_SUPPORT_OVER_CONTIGUOUS_RUN', 'minimum_corner_progress_px_for_measured': 0.0, 'timeout_corner_progress': 'REUSE_EXPLICIT_ELIGIBILITY_DISPLACEMENT', 'apply_measured_anchor_guard': True, 'apply_inferred_anchor_guard': True, 'finalization': 'VERIFIED_EPISODE_SCOPED_RELATIONS_ONLY', 'relation_support': 'WITHIN_BOTH_CANDIDATE_WINDOWS_NO_FUTURE_BACKPROPAGATION', 'category_conflict': 'PENDING_NOT_ARBITRARY_CATEGORY_WINNER', 'band_born_route': 'UNCHANGED_OPEN_P1'}

def _p0_controls(policy):
    controls = policy.get('p0_counting')
    _require(isinstance(controls, dict), 'Explicit B v5 p0_counting controls are required')
    expected = experimental_counting_policy()
    for key in ('schema_version', 'anchor_quality', 'timeout_corner_progress', 'finalization', 'relation_support', 'category_conflict', 'band_born_route'):
        _require(controls.get(key) == expected[key], f'Unsupported B v5 control {key}')
    for key in ('apply_measured_anchor_guard', 'apply_inferred_anchor_guard'):
        _require(type(controls.get(key)) is bool, f'Explicit boolean {key} is required')
    value = controls.get('minimum_corner_progress_px_for_measured')
    _require(type(value) in (int, float) and math.isfinite(value) and (value >= 0), 'Measured corner-progress control must be finite and nonnegative')
    return deepcopy(controls)

def _whole_box_progress(first, last, geometry, origin_sign):
    """Measure translation support of the four raw bbox corners.

    A stationary edge gives zero support, so changing only the opposing edge
    cannot establish a physical approach. This diagnostic cannot distinguish
    every real motion from camera shake; it is not an object-motion classifier.
    """
    corners = lambda box: ((box[0], box[1]), (box[2], box[1]), (box[0], box[3]), (box[2], box[3]))
    progress = [origin_sign * (signed_distance(a, geometry.p1, geometry.p2) - signed_distance(b, geometry.p1, geometry.p2)) for a, b in zip(corners(first['raw_bbox']), corners(last['raw_bbox']))]
    return {'first': _ref(first), 'last': _ref(last), 'corner_toward_progress_px': progress, 'minimum_corner_toward_progress_px': min(progress), 'bc_toward_progress_px': origin_sign * (signed_distance(first['point'], geometry.p1, geometry.p2) - signed_distance(last['point'], geometry.p1, geometry.p2)), 'anchor': 'RAW_VEHICLE_BC_UNCHANGED_CORNERS_QUALITY_ONLY'}

def _event_window(event):
    frames = event['evidence'].get('source_frames', [])
    _require(bool(frames), 'Finalization requires actual source frames')
    return (min(frames), max(frames))

def _endpoint_matches(event, relation, prefix):
    track = relation.get(prefix + '_track_id')
    episode = relation.get(prefix + '_physical_episode_id')
    if type(track) is not int or not isinstance(episode, str) or (not episode):
        return False
    return any((ref.get('track_id') == track and ref.get('physical_episode_id') == episode for ref in event['evidence'].get('source_refs', [])))

def _within(event, support):
    first, last = _event_window(event)
    return [frame for frame in support if first <= frame <= last]

def finalize_passages(result, relations, *, controls=None):
    """Reconcile only positively scoped relations before final aggregation.

    Relations never create a crossing or category. A positive SAME_UNIT claim
    can remove a duplicate representation of the *same-direction overlapping
    passage*. COMPONENT_OF can exclude a measured component, but cannot replace
    the parent's raw BC with the child's BC. Negative relation evidence blocks
    transitive merges. Unsupported/late relations remain audit-only.
    """
    controls = experimental_counting_policy() if controls is None else controls
    _require(isinstance(relations, list), 'episode_relations must be an explicit list')
    signature = hashlib.sha256(_canonical({'relations': relations, 'controls': controls}).encode()).hexdigest()
    prior = result.get('report', {}).get('offline_finalization')
    if isinstance(prior, dict) and prior.get('finalization_signature') is not None:
        _require(prior['finalization_signature'] == signature, 'A finalized ledger needs a fresh replay to apply different relation evidence')
        _require(prior.get('finalized_active_event_ids') == sorted((event['event_id'] for event in result['events'])), 'Finalized active ledger changed; replay source before finalizing again')
        return deepcopy(result)
    output = deepcopy(result)
    events = {event['event_id']: event for event in output['events']}
    cancelled = {event['event_id']: event for event in output.get('cancelled_events', [])}
    audits, positive, negative, components, uncertain = ([], [], [], [], [])
    for index, relation in enumerate(relations):
        _require(isinstance(relation, dict), 'Each episode relation must be a dictionary')
        kind = relation.get('relation_type', relation.get('type'))
        _require(kind in ('SAME_UNIT', 'COMPONENT_OF', 'DIFFERENT_UNIT'), 'Unknown episode relation')
        support = relation.get('support_frames', [])
        _require(isinstance(support, list) and all((type(f) is int and f >= 0 for f in support)), 'Relation support must contain source-frame integers')
        rid = relation.get('relation_id', f'relation:{index}')
        audit = {'relation_id': rid, 'relation_type': kind, 'support_frames': sorted(set(support)), 'status': 'NOT_APPLIED', 'reason': None, 'event_ids': [], 'count_delta': 0}
        audits.append(audit)
        if support and relation.get('verified') is False and (relation.get('candidate_kind') == 'POSSIBLE_CROSS_CATEGORY_UNIT_OR_COMPONENT'):
            first = [event for event in events.values() if _endpoint_matches(event, relation, 'first') and _within(event, support)]
            second = [event for event in events.values() if _endpoint_matches(event, relation, 'second') and _within(event, support)]
            matches = list({event['event_id']: event for event in first + second}.values())
            if first and second and all((event['evidence']['measured_crossing'] for event in matches)) and all((a['direction'] != b['direction'] for a in first for b in second)):
                audit['reason'] = 'OPPOSITE_MEASURED_DIRECTIONS_NOT_ONE_PHYSICAL_PASSAGE'
                audit['event_ids'] = sorted((event['event_id'] for event in matches))
                continue
            if matches:
                uncertain.append(({event['event_id'] for event in matches}, relation, audit))
                audit.update(status='UNRESOLVED_CROSS_CATEGORY_COMPONENT', reason='GEOMETRIC_CANDIDATE_IS_NOT_POSITIVE_PHYSICAL_IDENTITY', event_ids=sorted((event['event_id'] for event in matches)))
            else:
                audit['reason'] = 'NO_ACCEPTED_EVENT_WITHIN_AMBIGUOUS_CANDIDATE_SCOPE'
            continue
        if not support or (kind != 'DIFFERENT_UNIT' and relation.get('verified') is not True):
            audit['reason'] = 'NO_VERIFIED_POSITIVE_EPISODE_SCOPE'
            continue
        first = [event for event in events.values() if _endpoint_matches(event, relation, 'first')]
        second = [event for event in events.values() if _endpoint_matches(event, relation, 'second')]
        if kind == 'COMPONENT_OF':
            matched = [event for event in first if _within(event, support)]
            if not matched:
                audit['reason'] = 'NO_COMPONENT_EVENT_IN_SUPPORTED_SCOPE'
                continue
            for event in matched:
                components.append((event['event_id'], relation, audit))
            continue
        applied = False
        for a in first:
            for b in second:
                if a['event_id'] == b['event_id']:
                    continue
                overlap = sorted(set(_within(a, support)) & set(_within(b, support)))
                if not overlap:
                    continue
                pair = frozenset((a['event_id'], b['event_id']))
                if kind == 'DIFFERENT_UNIT':
                    negative.append(pair)
                    applied = True
                    audit.update(status='NEGATIVE_CONSTRAINT_RETAINED', reason='SCOPED_NO_MERGE_CONSTRAINT_NOT_PHYSICAL_GROUND_TRUTH', event_ids=sorted(pair))
                elif a['direction'] == b['direction']:
                    positive.append((pair, relation, audit))
                    applied = True
                else:
                    audit.update(reason='DIFFERENT_DIRECTION_MAY_BE_GENUINE_RETURN', event_ids=sorted(pair))
        if not applied and audit['reason'] is None:
            audit['reason'] = 'NO_TWO_SAME_PASSAGE_EVENTS_IN_EXACT_SUPPORTED_EPISODES'

    def cancel_event(event_id, reason, audit, canonical=None):
        if event_id not in events:
            return
        event = events.pop(event_id)
        event.update(status='cancelled', accepted_count_event=False, display_id=None, cancellation_reason=reason)
        event['evidence']['offline_finalization'] = {'relation_id': audit['relation_id'], 'reason': reason, 'canonical_event_id': canonical, 'crossing_source_unchanged': True}
        cancelled[event_id] = event
        audit['count_delta'] -= 1
        output['passage_log'].append({'frame_index': output['report']['last_frame_index'], 'physical_unit_id': event['physical_unit_id'], 'track_id': event['track_id'], 'passage_id': event['passage_id'], 'episode_id': event.get('episode_id'), 'category': event['category'], 'direction': event['direction'], 'route': event['route'], 'status': 'cancelled', 'event_id': event_id, 'count_delta': -1, 'measured_crossing': event['evidence']['measured_crossing'], 'message': reason, 'canonical_event_id': canonical, 'relation_id': audit['relation_id'], 'role': event.get('role')})
        for decision in output.get('decisions', []):
            if decision.get('event_id') == event_id:
                decision.update(status='resolved_not_eligible', resolution_reason=reason, canonical_event_id=canonical)
                decision.pop('event_id', None)
                decision.pop('display_id', None)
    for event_id, relation, audit in components:
        audit.update(status='COMPONENT_EXCLUDED_PLUS_ZERO', reason='POSITIVE_SCOPED_COMPONENT_OF', event_ids=[event_id], parent_track_id=relation.get('second_track_id'), parent_physical_episode_id=relation.get('second_physical_episode_id'))
        cancel_event(event_id, 'POSITIVE_SCOPED_COMPONENT_NOT_INDEPENDENT_PASSAGE', audit)
    groups = [{event_id} for event_id in events]
    for pair, relation, audit in positive:
        if not pair <= set(events):
            audit['reason'] = 'COMPONENT_EVENT_ALREADY_EXCLUDED'
            continue
        a, b = tuple(pair)
        ga = next((group for group in groups if a in group))
        gb = next((group for group in groups if b in group))
        union = ga | gb
        if any((conflict <= union for conflict in negative)):
            audit.update(status='MERGE_BLOCKED', reason='DIFFERENT_UNIT_CONTRADICTION', event_ids=sorted(pair))
            continue
        if ga is not gb:
            groups.remove(ga)
            groups.remove(gb)
            groups.append(union)
        audit.update(status='SAME_PASSAGE_GROUP_SUPPORTED', reason='VERIFIED_LOCAL_SAME_UNIT', event_ids=sorted(pair))
    for group in groups:
        if len(group) < 2:
            continue
        members = [events[event_id] for event_id in group]
        related = [audit for pair, relation, audit in positive if pair <= group and audit['status'] == 'SAME_PASSAGE_GROUP_SUPPORTED']
        audit = {'relation_id': '+'.join(sorted((a['relation_id'] for a in related))), 'relation_type': 'SAME_PASSAGE_FINALIZATION', 'event_ids': sorted(group), 'support_frames': sorted({f for a in related for f in a['support_frames']}), 'count_delta': 0}
        audits.append(audit)
        if len({event['category'] for event in members}) != 1:
            audit.update(status='CATEGORY_CONFLICT_PENDING', reason='NO_ARBITRARY_CATEGORY_WINNER')
            for event in members:
                cancel_event(event['event_id'], 'SAME_PASSAGE_CATEGORY_CONFLICT', audit)
            canonical = min(members, key=lambda event: event['event_id'])
            output['decisions'].append({'frame_index': output['report']['last_frame_index'], 'track_id': canonical['track_id'], 'physical_unit_id': canonical['physical_unit_id'], 'passage_id': 'p0-conflict:' + hashlib.sha256(_canonical(sorted(group)).encode()).hexdigest()[:24], 'episode_id': canonical.get('episode_id'), 'role': canonical.get('role'), 'status': 'pending', 'reasons': ['SAME_PASSAGE_CATEGORY_CONFLICT'], 'source_ref': deepcopy(canonical['evidence']['source_refs'][-1]), 'reason_evidence': [{'reason': 'SAME_PASSAGE_CATEGORY_CONFLICT', 'relation_ids': [a['relation_id'] for a in related], 'event_ids': sorted(group)}]})
            continue
        primary = min(members, key=lambda event: (not event['evidence']['measured_crossing'], event['route'] == 'ESTIMATED_TIMEOUT', event['activation_frame'], event['event_id']))
        audit.update(status='ONE_ACTIVE_PASSAGE', reason='DUPLICATE_REPRESENTATIONS_PLUS_ZERO', canonical_event_id=primary['event_id'])
        primary['evidence']['reconciled_event_ids'] = sorted(group)
        primary['evidence']['reconciliation_relation_ids'] = [a['relation_id'] for a in related]
        for event in members:
            if event['event_id'] != primary['event_id']:
                cancel_event(event['event_id'], 'OFFLINE_SAME_PHYSICAL_PASSAGE_DUPLICATE', audit, canonical=primary['event_id'])
    uncertain_groups = []
    for event_ids, relation, audit in uncertain:
        live = event_ids & set(events)
        if not live:
            continue
        overlapping = [group for group in uncertain_groups if group[0] & live]
        merged_ids = set(live)
        merged_audits = [audit]
        for group in overlapping:
            merged_ids |= group[0]
            merged_audits.extend(group[1])
            uncertain_groups.remove(group)
        uncertain_groups.append((merged_ids, merged_audits))
    for group, related in uncertain_groups:
        members = [events[event_id] for event_id in group if event_id in events]
        if not members:
            continue
        audit = {'relation_id': '+'.join(sorted((a['relation_id'] for a in related))), 'relation_type': 'UNRESOLVED_COMPONENT_OR_SAME_UNIT', 'status': 'HELD_FOR_PHYSICAL_UNIT_REVIEW', 'count_delta': 0, 'reason': 'CROSS_CATEGORY_UNIT_OR_COMPONENT_UNRESOLVED', 'event_ids': sorted(group), 'support_frames': sorted({f for a in related for f in a['support_frames']}), 'physical_episodes_merged': False, 'pending_review_scope': 'PER_ORIGINAL_PASSAGE_NOT_CONNECTED_COMPONENT'}
        audits.append(audit)
        for event in members:
            cancel_event(event['event_id'], 'CROSS_CATEGORY_UNIT_OR_COMPONENT_UNRESOLVED', audit)
            decision = next((decision for decision in output['decisions'] if decision.get('physical_unit_id') == event['physical_unit_id'] and decision.get('passage_id') == event['passage_id']), None)
            if decision is None:
                decision = {'frame_index': event['decision_frame'], 'track_id': event['track_id'], 'physical_unit_id': event['physical_unit_id'], 'passage_id': event['passage_id'], 'episode_id': event.get('episode_id'), 'role': event.get('role'), 'source_ref': deepcopy(event['evidence']['source_refs'][-1]), 'reasons': [], 'reason_evidence': []}
                output['decisions'].append(decision)
            decision.update(status='pending', cancelled_candidate_event_id=event['event_id'], candidate_event_ids=[event['event_id']], investigation_event_ids=sorted(group), physical_episodes_merged=False, pending_review_scope='PER_ORIGINAL_PASSAGE_NOT_CONNECTED_COMPONENT')
            decision.pop('resolution_reason', None)
            decision.pop('canonical_event_id', None)
            decision.pop('event_id', None)
            decision.pop('display_id', None)
            reason = 'CROSS_CATEGORY_UNIT_OR_COMPONENT_UNRESOLVED'
            if reason not in decision['reasons']:
                decision['reasons'].append(reason)
                decision['reason_evidence'].append({'reason': reason, 'relation_ids': [a['relation_id'] for a in related], 'event_id': event['event_id'], 'investigation_event_ids': sorted(group)})
    active = sorted(events.values(), key=lambda event: (event['activation_frame'], event['event_id']))
    ordinals = Counter()
    prefixes = output['report']['display_category_prefixes']
    for event in active:
        key = (event['category'], event['direction'])
        ordinals[key] += 1
        event['display_id'] = passage_display_id(*key, ordinals[key], category_prefixes=prefixes)
    displays = {event['event_id']: event['display_id'] for event in active}
    for row in output['passage_log']:
        row['display_id'] = displays.get(row.get('event_id'))
        row['display_status'] = 'final_active_passage' if row['display_id'] else 'no_final_active_passage_id'
    for decision in output['decisions']:
        decision['display_id'] = displays.get(decision.get('event_id'))
    output['events'] = active
    output['cancelled_events'] = sorted(cancelled.values(), key=lambda event: event['event_id'])
    output['display_assignments'] = [{key: deepcopy(event[key]) for key in ('event_id', 'physical_unit_id', 'episode_id', 'passage_id', 'display_id', 'category', 'direction', 'activation_frame', 'decision_frame')} | {'source_frames': deepcopy(event['evidence']['source_frames'])} for event in active]
    categories = Counter((event['category'] for event in active))
    measured = sum((event['evidence']['measured_crossing'] for event in active))
    output['aggregate'] = {'total': len(active), 'by_category': dict(sorted(categories.items())), 'by_direction': dict(sorted(Counter((event['direction'] for event in active)).items())), 'by_route': dict(sorted(Counter((event['route'] for event in active)).items())), 'measured_total': measured, 'inferred_total': len(active) - measured, 'pending': sum((decision['status'] == 'pending' for decision in output['decisions'])), 'cancelled': len(cancelled), 'vehicle_total': len(active) - categories['pedestrian'], 'pedestrian_total': categories['pedestrian']}
    output['finalization_audit'] = audits
    output['report']['offline_finalization'] = {'relations_received': len(relations), 'audit_records': len(audits), 'additional_cancellations': len(cancelled) - len(result.get('cancelled_events', [])), 'finalization_signature': signature, 'finalized_active_event_ids': sorted((event['event_id'] for event in active)), 'positive_relations_do_not_create_crossing': True, 'newly_created_accepted_events': 0, 'full_physical_identity_correctness_proven': False}
    return output

def passage_display_id(category, direction, ordinal, *, category_prefixes=None):
    """Deterministic one-based ordinal in a per-video category/direction scope.

    Four numeric digits: even first digit A_TO_B, odd first digit B_TO_A;
    zero is never a leading digit. Exhaustion is explicit, never a wrap.
    """
    prefixes = DISPLAY_PREFIXES if category_prefixes is None else category_prefixes
    _require(category in prefixes, 'Display category has no agreed prefix')
    _require(direction in ('A_TO_B', 'B_TO_A'), 'Display direction must be explicit')
    _require(isinstance(ordinal, int) and (not isinstance(ordinal, bool)) and (ordinal > 0), 'Display ordinal must be a positive integer')
    first_digits = (2, 4, 6, 8) if direction == 'A_TO_B' else (1, 3, 5, 7, 9)
    _require(ordinal <= len(first_digits) * 1000, 'Four-digit passage display ID space exhausted; no IDs may be reused')
    offset = ordinal - 1
    return f'{prefixes[category]}{first_digits[offset // 1000]}{offset % 1000:03d}'

def _effective_category(observation):
    return observation['effective_class_name']

def _episode_key(observation):
    return (observation['unit_id'], observation.get('episode_id'))

def _verified_link(first, second):
    return first.get('association_verified') is True and second.get('association_verified') is True and (first.get('association_id') is not None) and (first.get('association_id') == second.get('association_id'))

def _unresolved_handoffs(state):
    return any((not change['association_verified'] for change in state['raw_track_changes']))
ROUTE_LABELS = {'LINE_MARGIN': 'CROSSING_LINE', 'OCCLUSION_DIFF_POLYGON': 'OCCLUSION_DIFF_POLYGON', 'ESTIMATED_TIMEOUT': 'CHANGE_ID'}

def _require(condition, message):
    if not condition:
        raise ValueError(message)

def _integer(value, name):
    _require(isinstance(value, int) and (not isinstance(value, bool)) and (value >= 0), f'{name} must be a nonnegative source-frame integer')
    return value

def _number(value, name):
    _require(isinstance(value, Real) and (not isinstance(value, bool)) and math.isfinite(float(value)), f'{name} must be finite numeric')
    return float(value)

def _canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)

def _identity(value):
    _require(isinstance(value, (str, int)) and (not isinstance(value, bool)) and (str(value) != ''), 'Track/unit identities must be nonempty strings or integers')
    return str(value)

def _bbox(value):
    _require(isinstance(value, (list, tuple)) and len(value) == 4, 'raw_bbox requires four measured coordinates')
    result = [_number(v, 'raw_bbox') for v in value]
    _require(result[0] < result[2] and result[1] < result[3], 'raw_bbox has no positive area')
    return result

def finite_band(point, geometry, half_width):
    """Flat-capped finite rectangular band; no extension beyond P1/P2."""
    p1, p2 = (geometry.p1, geometry.p2)
    dx, dy = (p2[0] - p1[0], p2[1] - p1[1])
    projection = ((point[0] - p1[0]) * dx + (point[1] - p1[1]) * dy) / (dx * dx + dy * dy)
    distance = signed_distance(point, p1, p2)
    return {'inside': 0 <= projection <= 1 and abs(distance) <= half_width, 'projection': projection, 'signed_distance_px': distance}

def _side_signs(geometry):
    means = {side: sum((signed_distance(p, geometry.p1, geometry.p2) for p in polygon)) / len(polygon) for side, polygon in (('A', geometry.side_a), ('B', geometry.side_b))}
    _require(means['A'] * means['B'] < 0, 'A/B polygons must have opposite mean line sides for direction mapping')
    return {side: 1 if distance > 0 else -1 for side, distance in means.items()}

def _observation(raw, frame, geometry, margin):
    _require(isinstance(raw, dict), 'Each observation must be a dictionary')
    _require(raw.get('observed') is True and raw.get('emitted') is True, 'Only observed, emitted, raw detector-linked tracks can be replayed')
    track = raw.get('track_id')
    _identity(track)
    index = _integer(raw.get('original_detection_index'), 'original_detection_index')
    if 'frame_index' in raw:
        _require(raw['frame_index'] == frame, 'Observation frame differs from source frame')
    box = _bbox(raw.get('raw_bbox'))
    category = raw.get('class_name')
    _require(isinstance(category, str) and category, 'Raw class_name must be explicit')
    _require(isinstance(raw.get('identity_conflict', False), bool), 'identity_conflict must be boolean')
    unit = _identity(raw.get('unit_id', f'track:{track}'))
    role = raw.get('role', 'vehicle' if category in VEHICLE_CLASSES else 'unknown_person' if category == 'person' else None)
    _require(role in ROLES or role is None, 'Unknown physical observation role')
    association_verified = raw.get('association_verified', False)
    _require(isinstance(association_verified, bool), 'association_verified must be boolean')
    association_id = raw.get('association_id')
    if association_id is not None:
        association_id = _identity(association_id)
    _require(not association_verified or association_id is not None, 'A verified association requires association_id')
    episode_id = raw.get('episode_id')
    if episode_id is not None:
        episode_id = _identity(episode_id)
    resolved = raw.get('resolved_class_name')
    _require(resolved is None or (isinstance(resolved, str) and bool(resolved)), 'resolved_class_name must be explicit nonempty category')
    if role == 'pedestrian':
        _require(category == 'person', 'Pedestrian BC must come from a raw person measurement')
        _require(resolved in (None, 'pedestrian'), 'Pedestrian role/category conflict')
        effective = 'pedestrian'
    elif role == 'vehicle':
        _require(category in VEHICLE_CLASSES, 'Vehicle BC requires a raw vehicle-class measurement')
        _require(resolved != 'pedestrian', 'Pedestrian requires a positive pedestrian role')
        effective = resolved or category
    else:
        effective = resolved or category
    point = ((box[0] + box[2]) / 2, box[3])
    band = finite_band(point, geometry, margin)
    result = {'frame_index': frame, 'track_id': track, 'unit_id': unit, 'original_detection_index': index, 'raw_bbox': box, 'point': list(point), 'class_name': category, 'effective_class_name': effective, 'resolved_class_name': resolved, 'role': role, 'episode_id': episode_id, 'association_verified': association_verified, 'association_id': association_id, 'observed': True, 'emitted': True, 'identity_conflict': raw.get('identity_conflict', False), 'polygon': geometry.side(point), 'band': band['inside'], 'signed_distance_px': band['signed_distance_px'], 'line_projection': band['projection']}
    for key in ('class_id', 'confidence', 'measurement_provenance', 'source_ref', 'review_ref', 'review_refs', 'physical_episode_id', 'identity_episode_boundary', 'identity_episode_reason', 'previous_physical_episode_id', 'identity_timeout_hold', 'role_scope_id', 'component_of_unit_id', 'component_of_track_id', 'association_provenance'):
        if key in raw:
            result[key] = deepcopy(raw[key])
    return result

def _ref(observation):
    return {key: deepcopy(observation[key]) for key in ('frame_index', 'track_id', 'original_detection_index', 'raw_bbox', 'point', 'class_name', 'effective_class_name', 'resolved_class_name', 'role', 'episode_id', 'association_verified', 'association_id', 'observed', 'emitted') if key in observation} | {key: deepcopy(observation[key]) for key in ('review_ref', 'review_refs', 'class_id', 'confidence', 'measurement_provenance', 'source_ref', 'physical_episode_id', 'identity_episode_boundary', 'identity_episode_reason', 'identity_timeout_hold', 'role_scope_id', 'component_of_unit_id', 'component_of_track_id', 'association_provenance') if key in observation}

def _direction(origin):
    return 'A_TO_B' if origin == 'A' else 'B_TO_A'

def _new_passage(observation, origin, ordinal):
    return {'origin': origin, 'ordinal': ordinal, 'origin_ref': _ref(observation), 'last_origin_ref': _ref(observation) if observation['polygon'] == origin else None, 'origin_is_roi': observation['polygon'] == origin, 'origin_track': observation['track_id'], 'origin_category': _effective_category(observation), 'band_refs': [_ref(observation)] if observation['band'] else [], 'band_votes': Counter({_effective_category(observation): 1}) if observation['band'] else Counter(), 'crossing_pairs': [], 'zero_contact': None, 'gaps': [], 'raw_track_changes': [], 'approaching': False, 'identity_conflicts': [_ref(observation)] if observation['identity_conflict'] else [], 'destination_observed_ref': None, 'turned_away': False, 'event': None, 'pending_reasons': set(), 'pending_decision': None, 'closed': False, 'closure_reason': None, 'approach_run': [_ref(observation)], 'approach_progress_px': 0.0, 'p0_motion_supported': False, 'p0_anchor_quality': None, 'closest_origin_distance': None, 'identity_timeout_hold': list(observation.get('identity_timeout_hold', [])), 'last': observation}

def _category(state, destination=None, *, timeout=False, supported_categories=SUPPORTED_CATEGORIES):
    if state['last']['role'] == 'unknown_person':
        return (None, {'policy': 'positive_role_required', 'raw_class_name': state['last']['class_name']}, 'PERSON_ROLE_UNRESOLVED')
    if state['band_votes']:
        votes = state['band_votes']
        maximum = max(votes.values())
        winners = sorted((category for category, count in votes.items() if count == maximum))
        basis = {'policy': 'band_majority_ties_pending', 'votes': dict(sorted(votes.items())), 'source': 'observed_object_BC_inside_finite_band', 'reviewed_category_attribution': any((ref.get('resolved_class_name') is not None for ref in state['band_refs']))}
        if len(winners) != 1:
            return (None, basis, 'CATEGORY_TIE_IN_BAND')
        category = winners[0]
    else:
        origin = state['last_origin_ref'] or state['origin_ref']
        latest = destination or state['last']
        basis = {'policy': 'matching_observed_endpoints', 'origin_class': _effective_category(origin), 'destination_class': _effective_category(latest), 'origin_raw_class': origin['class_name'], 'destination_raw_class': latest['class_name'], 'source': 'timeout_origin_and_latest' if timeout else 'measured_endpoints_no_sample_in_band'}
        if _effective_category(origin) != _effective_category(latest):
            return (None, basis, 'CATEGORY_ENDPOINT_CONFLICT')
        category = _effective_category(origin)
    if category not in supported_categories:
        return (None, basis, 'UNSUPPORTED_CATEGORY_TAXONOMY_OPEN')
    return (category, basis, None)

def replay_frames(frames, geometry, *, source_identity, policy):
    """Return a deterministic offline B v5 ledger from contiguous source frames.

    Required policy values are line_margin_half_width_px, timeout_frames,
    timeout_clock='last_observed_origin', and
    category_policy='band_majority_ties_pending'. Frames include empty source
    frames; no synthetic post-EOF frames are generated. Known unit_id groups
    may deduplicate supported components/duplicate tracks. Otherwise raw track
    identity is an explicitly unverified physical-unit assumption.
    """
    _require(isinstance(geometry, Geometry), 'geometry must be a validated Phase 5 Geometry')
    _require(isinstance(policy, dict), 'Explicit experimental policy is required')
    p0 = _p0_controls(policy)
    margin = _number(policy.get('line_margin_half_width_px'), 'line_margin_half_width_px')
    _require(margin >= 0, 'Margin half-width cannot be negative')
    timeout = _integer(policy.get('timeout_frames'), 'timeout_frames')
    _require(timeout > 0, 'timeout_frames must be positive')
    _require(policy.get('timeout_clock') == 'last_observed_origin', 'Unknown timeout clock')
    _require(policy.get('category_policy') == 'band_majority_ties_pending', 'Unknown category policy')
    supported_categories = policy.get('supported_categories', sorted(SUPPORTED_CATEGORIES))
    _require(isinstance(supported_categories, list) and bool(supported_categories) and all((isinstance(item, str) for item in supported_categories)) and (len(set(supported_categories)) == len(supported_categories)) and (set(supported_categories) <= VEHICLE_CLASSES | {'pedestrian'}), 'supported_categories must be an explicit unique list of known raw product categories')
    supported_categories = frozenset(supported_categories)
    display_prefixes = policy.get('display_category_prefixes', DISPLAY_PREFIXES)
    _require(isinstance(display_prefixes, dict) and display_prefixes in (DISPLAY_PREFIXES, {**DISPLAY_PREFIXES, 'bus': 'B'}), 'Display prefix mapping must retain agreed P/M/C/T; optional bus B requires explicit policy')
    _require(supported_categories <= set(display_prefixes), 'Every supported category requires an explicit agreed display prefix')
    pedestrian_routes = policy.get('pedestrian_routes', 'measured_only')
    _require(pedestrian_routes in ('measured_only', 'all_operational'), 'pedestrian_routes must be measured_only or all_operational')
    return_timeout_policy = policy.get('return_timeout_policy', 'allow_operational_estimate')
    _require(return_timeout_policy in ('allow_operational_estimate', 'require_new_passage_evidence'), 'return_timeout_policy must be allow_operational_estimate or require_new_passage_evidence')
    eligibility = policy.get('eligibility')
    _require(isinstance(eligibility, dict) and eligibility.get('schema_version') == 'phase8_b_v4_eligibility_policy_v1', 'Explicit experimental B v4 eligibility policy is required')
    progress_required = _number(eligibility.get('min_toward_progress_px'), 'min_toward_progress_px')
    minimum_observations = eligibility.get('min_contiguous_observations')
    _require(progress_required > 0 and type(minimum_observations) is int and (minimum_observations >= 2), 'Approach evidence requires positive displacement and >=2 contiguous observations')
    _require(eligibility.get('finite_projection_required') is True, 'This experiment requires current approach projection within the finite gate')
    _require(isinstance(source_identity, dict), 'source_identity must be explicit')
    for key in ('video_sha256', 'records_sha256', 'geometry_sha256'):
        value = source_identity.get(key)
        _require(isinstance(value, str) and len(value) == 64 and all((c in '0123456789abcdef' for c in value)), f'Invalid source identity {key}')
    for key in ('width', 'height'):
        _require(isinstance(source_identity.get(key), int) and (not isinstance(source_identity[key], bool)) and (source_identity[key] > 0), f'Invalid source {key}')
    if geometry.width is not None:
        _require((geometry.width, geometry.height) == (source_identity['width'], source_identity['height']), 'Geometry and source dimensions differ')
    namespace = hashlib.sha256(_canonical({'source_identity': source_identity, 'policy': policy, 'geometry': geometry.as_dict(), 'version': 'b_v5_p0_eligibility_finalization_v1'}).encode()).hexdigest()
    signs = _side_signs(geometry)
    states, accepted, cancelled, decisions, logs = ({}, {}, {}, [], [])
    decision_by_passage = {}
    frame_count = 0
    first_frame = last_frame = None
    counters = Counter()
    eligibility_audit, raw_active_episode = ([], {})

    def audit(state, frame, status, reason):
        observation = state['last']
        last_origin = (state['last_origin_ref'] or {}).get('frame_index')
        signature = (status, reason, last_origin, tuple(state['identity_timeout_hold']))
        if signature == state.get('last_audit_signature'):
            return
        row = {'frame_index': frame, 'track_id': observation['track_id'], 'physical_unit_id': observation['unit_id'], 'episode_id': observation.get('episode_id'), 'passage_id': passage_id(state), 'status': status, 'reason': reason, 'source_ref': _ref(observation), 'last_observed_origin_frame': last_origin, 'elapsed_source_frames': frame - last_origin if last_origin is not None else None, 'toward_progress_px': state['approach_progress_px'], 'geometric_approach_established': state['approaching'], 'eligible': bool(state['approaching'] and state['origin_is_roi'] and (not state['closed']) and (not state['turned_away']) and (not state['identity_conflicts']) and (not state['identity_timeout_hold']) and (not _unresolved_handoffs(state)) and (not p0['apply_inferred_anchor_guard'] or state['p0_motion_supported'])), 'identity_timeout_hold': deepcopy(state['identity_timeout_hold'])}
        row['p0_motion_supported'] = state['p0_motion_supported']
        row['p0_anchor_quality'] = deepcopy(state['p0_anchor_quality'])
        eligibility_audit.append(row)
        state['last_audit_signature'] = signature

    def update_approach(state, previous, observation, same_identity):
        if not same_identity or observation['frame_index'] != previous['frame_index'] + 1:
            state['approach_run'] = [_ref(observation)]
            state['approach_progress_px'] = 0.0
            state['p0_motion_supported'] = False
            state['p0_anchor_quality'] = None
            return
        run = state['approach_run']
        run.append(_ref(observation))
        sign = signs[state['origin']]
        distance = observation['signed_distance_px'] * sign
        start_distance = signed_distance(run[0]['point'], geometry.p1, geometry.p2) * sign
        progress = start_distance - distance
        state['approach_progress_px'] = max(0.0, progress)
        quality = _whole_box_progress(run[0], observation, geometry, sign)
        state['p0_anchor_quality'] = quality
        state['p0_motion_supported'] = bool(len(run) >= minimum_observations and quality['minimum_corner_toward_progress_px'] >= progress_required)
        closest = state['closest_origin_distance']
        state['closest_origin_distance'] = distance if closest is None else min(closest, distance)
        if state['approaching'] and (not state['band_refs']) and (closest is not None) and (distance - closest >= progress_required):
            state['approaching'], state['turned_away'] = (False, True)
            state['approach_run'] = [_ref(observation)]
            state['approach_progress_px'] = 0.0
            state['closest_origin_distance'] = distance
            cancel(state, observation['frame_index'], 'MEASURED_U_TURN_BEFORE_BAND')
        elif len(run) >= minimum_observations and progress >= progress_required and state['origin_is_roi'] and (0 <= observation['line_projection'] <= 1) and (not p0['apply_inferred_anchor_guard'] or state['p0_motion_supported']):
            state['approaching'], state['turned_away'] = (True, False)
        elif not state['approaching'] and distance > start_distance:
            state['approach_run'] = [_ref(observation)]

    def log(state, frame, status, *, event=None, reason=None, count_delta=0):
        observation = state['last']
        row = {'frame_index': frame, 'physical_unit_id': observation['unit_id'], 'track_id': observation['track_id'], 'passage_id': passage_id(state), 'category': event['category'] if event else _effective_category(observation), 'episode_id': observation.get('episode_id'), 'role': observation['role'], 'direction': _direction(state['origin']), 'route': event['route'] if event else None, 'status': status, 'event_id': event['event_id'] if event else None, 'requested_label': ROUTE_LABELS[event['route']] if event else None, 'count_delta': count_delta, 'measured_crossing': event['evidence']['measured_crossing'] if event else False, 'message': reason or f"ID {observation['track_id']} {event['category']}; {_direction(state['origin'])}; {ROUTE_LABELS[event['route']]}"}
        logs.append(row)

    def passage_id(state):
        observation = state['last']
        episode = observation.get('episode_id')
        scope = observation['unit_id'] if episode is None else f"{observation['unit_id']}:episode:{episode}"
        return f"{scope}:passage:{state['ordinal']}"

    def pending(state, frame, reason):
        key = (state['last']['unit_id'], passage_id(state))
        decision = decision_by_passage.get(key)
        if decision is None:
            decision = {'frame_index': frame, 'track_id': state['last']['track_id'], 'physical_unit_id': state['last']['unit_id'], 'passage_id': passage_id(state), 'episode_id': state['last'].get('episode_id'), 'role': state['last']['role'], 'status': 'pending', 'reasons': [], 'source_ref': _ref(state['last']), 'reason_evidence': []}
            decisions.append(decision)
            decision_by_passage[key] = decision
        decision['status'] = 'pending'
        decision.pop('event_id', None)
        decision.pop('display_id', None)
        state['pending_decision'] = decision
        state['pending_reasons'].add(reason)
        decision['latest_frame_index'] = frame
        if reason in decision['reasons']:
            return
        decision['reasons'].append(reason)
        decision['reason_evidence'].append({'reason': reason, 'frame_index': frame, 'source_ref': _ref(state['last'])})
        log(state, frame, 'pending', reason=reason)

    def cancel(state, frame, reason):
        previous = state['event']
        if previous is None:
            return
        previous = deepcopy(previous)
        previous.update(status='cancelled', accepted_count_event=False, decision_frame=frame, cancellation_reason=reason)
        previous['evidence']['cancellation_source_ref'] = _ref(state['last'])
        accepted.pop(previous['event_id'], None)
        cancelled[previous['event_id']] = previous
        state['event'] = None
        log(state, frame, 'cancelled', event=previous, reason=reason, count_delta=-1)

    def emit(state, frame, route, category, basis, measured, destination=None):
        previous = state['event']
        material = {'namespace': namespace, 'unit_id': state['last']['unit_id'], 'passage_id': passage_id(state)}
        event_id = 'b_v5_' + hashlib.sha256(_canonical(material).encode()).hexdigest()[:24]
        refs = [state['origin_ref']]
        if state['last_origin_ref']:
            refs.append(state['last_origin_ref'])
        refs.extend(state['band_refs'])
        refs.append(_ref(state['last']))
        refs = list({_canonical(ref): ref for ref in refs}.values())
        event = {'event_id': event_id, 'physical_unit_id': state['last']['unit_id'], 'passage_id': passage_id(state), 'track_id': state['last']['track_id'], 'episode_id': state['last'].get('episode_id'), 'role': state['last']['role'], 'category': category, 'direction': _direction(state['origin']), 'route': route, 'status': 'accepted', 'accepted_count_event': True, 'activation_frame': previous['activation_frame'] if previous else frame, 'decision_frame': frame, 'source_identity': deepcopy(source_identity), 'namespace': namespace, 'evidence': {'measured_crossing': measured, 'band_observed': bool(state['band_refs']), 'category_basis': basis, 'source_refs': refs, 'source_frames': sorted({ref['frame_index'] for ref in refs}), 'crossing_pairs': deepcopy(state['crossing_pairs']), 'gaps': deepcopy(state['gaps']), 'raw_track_changes': deepcopy(state['raw_track_changes']), 'identity_conflicts': deepcopy(state['identity_conflicts']), 'origin_polygon': state['origin'], 'destination_polygon': destination['polygon'] if destination else None, 'destination_observed_ref': deepcopy(state['destination_observed_ref']), 'last_observed_origin_frame': (state['last_origin_ref'] or {}).get('frame_index'), 'timeout_frames': timeout if route == 'ESTIMATED_TIMEOUT' else None, 'direction_assumed': route == 'ESTIMATED_TIMEOUT', 'id_change_proven': False, 'physical_identity_basis': 'explicit_verified_association' if all((_verified_link(refs[0], ref) for ref in refs)) else 'raw_track_assumption', 'association_id': state['last'].get('association_id'), 'reviewed_category_attribution': any((ref.get('resolved_class_name') is not None for ref in refs)), 'finite_segment_crossing_unobserved': not measured, 'anchor': 'raw_person_bbox_bottom_center' if category == 'pedestrian' else 'raw_vehicle_bbox_bottom_center'}}
        event['evidence']['p0_anchor_quality'] = deepcopy(state['p0_anchor_quality'])
        event['evidence']['p0_motion_supported'] = state['p0_motion_supported']
        if previous:
            event['previous_evidence_route'] = previous['route']
        state['event'] = event
        accepted[event_id] = event
        cancelled.pop(event_id, None)
        for decision in decisions:
            if decision['physical_unit_id'] == event['physical_unit_id'] and decision['passage_id'] == event['passage_id']:
                decision['status'] = 'superseded_by_accepted'
                decision['event_id'] = event_id
        log(state, frame, 'upgraded' if previous else 'accepted', event=event, count_delta=0 if previous else 1)

    def normal_side(observation):
        distance = observation['signed_distance_px']
        if abs(distance) <= margin:
            return None
        return 'A' if distance * signs['A'] > 0 else 'B'
    for record in frames:
        _require(isinstance(record, dict), 'Each source frame must be a dictionary')
        frame = _integer(record.get('frame_index'), 'frame_index')
        _require(last_frame is None or frame == last_frame + 1, 'Source frames must be sorted and contiguous; retain empty source frames')
        if first_frame is None:
            first_frame = frame
        last_frame = frame
        frame_count += 1
        raw_observations = record.get('observations')
        _require(isinstance(raw_observations, list), 'Frame observations must be a list')
        observations = [_observation(raw, frame, geometry, margin) for raw in raw_observations]
        active_aliases = defaultdict(set)
        for observation in observations:
            if observation['role'] not in (None, 'rider'):
                active_aliases[_episode_key(observation)].add(_identity(observation['track_id']))
        grouped = defaultdict(list)
        seen_tracks, seen_detections = (set(), set())
        for observation in observations:
            track_key = _identity(observation['track_id'])
            _require(track_key not in seen_tracks, 'Duplicate track measurement in one source frame')
            _require(observation['original_detection_index'] not in seen_detections, 'Repeated original detection link in one source frame')
            seen_tracks.add(track_key)
            seen_detections.add(observation['original_detection_index'])
            current_key = _episode_key(observation)
            prior_key = raw_active_episode.get(track_key)
            if prior_key is not None and prior_key != current_key:
                previous_state = states.get(prior_key)
                other_active_aliases = active_aliases.get(prior_key, set()) - {track_key}
                if previous_state is not None and other_active_aliases:
                    counters['episode_closure_deferred_current_alias_remains'] += 1
                    audit(previous_state, frame, 'continued_by_current_alias', 'RAW_CONTEXT_ENDED_OTHER_UNIT_OBSERVATION_REMAINS')
                elif previous_state is not None and (not previous_state['closed']) and (observation.get('identity_episode_boundary') is not True):
                    reason = 'RAW_CONTEXT_CHANGED_WITHOUT_PHYSICAL_CONTINUITY_PROOF'
                    previous_state['p0_origin_context_suspended'] = True
                    if reason not in previous_state['identity_timeout_hold']:
                        previous_state['identity_timeout_hold'].append(reason)
                    previous_state.setdefault('p0_context_change_source_refs', []).append(_ref(observation))
                    counters['raw_context_changes_suspend_unverified_origin_timer'] += 1
                    if previous_state['event'] is not None and previous_state['event']['route'] == 'ESTIMATED_TIMEOUT':
                        cancel(previous_state, frame, reason)
                    if previous_state['approaching'] and previous_state['event'] is None:
                        pending(previous_state, frame, reason)
                    audit(previous_state, frame, 'suspended_unresolved', reason)
                    eligibility_audit[-1]['current_context_source_ref'] = _ref(observation)
                elif previous_state is not None and (not previous_state['closed']):
                    previous_state['closed'] = True
                    previous_state['closure_reason'] = 'PHYSICAL_EPISODE_ENDED_NO_TIMER_INHERITANCE'
                    previous_state['approaching'] = False
                    counters['closed_physical_episode_candidates'] += 1
                    audit(previous_state, frame, 'closed', previous_state['closure_reason'])
                    decision = previous_state.get('pending_decision')
                    if decision is not None and previous_state['event'] is None:
                        decision['status'] = 'resolved_not_eligible'
                        decision['resolution_reason'] = previous_state['closure_reason']
                        log(previous_state, frame, 'resolved_not_eligible', reason=previous_state['closure_reason'])
            raw_active_episode[track_key] = current_key
            if observation['role'] == 'rider':
                counters['rider_observations_excluded_plus_zero'] += 1
                continue
            if observation['role'] is None:
                counters['ignored_nontraffic_observations'] += 1
                continue
            grouped[_episode_key(observation)].append(observation)
            counters[f"{observation['role']}_observations"] += 1
        for unit in sorted(grouped, key=lambda key: (key[0], key[1] or '')):
            options = sorted(grouped[unit], key=lambda item: ((0, item['track_id']) if isinstance(item['track_id'], int) else (1, _identity(item['track_id'])), item['original_detection_index']))
            old = states.get(unit)
            if old:
                preferred = [item for item in options if item['track_id'] == old['last']['track_id']]
                observation = preferred[0] if preferred else options[0]
            else:
                observation = options[0]
            _require(len({item['role'] for item in options}) == 1, 'Non-rider measurements require separate role-specific physical episodes')
            if len(options) > 1 and (not all((_verified_link(options[0], item) for item in options[1:]))):
                for item in options:
                    item['identity_conflict'] = True
            counters['known_unit_duplicate_observations_suppressed'] += len(options) - 1
            combined_holds = sorted({reason for item in options for reason in item.get('identity_timeout_hold', [])})
            observation['identity_timeout_hold'] = combined_holds
            side = observation['polygon'] or normal_side(observation)
            if old is None:
                if side is not None:
                    states[unit] = _new_passage(observation, side, 1)
                    states[unit]['identity_conflicts'] = [_ref(item) for item in options if item['identity_conflict']]
                    audit(states[unit], frame, 'not_eligible', 'APPROACH_NOT_ESTABLISHED')
                else:
                    counters['unarmed_band_only_observations'] += 1
                continue
            state = old
            _require(not state['closed'], f"Closed counting episode key reused; context requires a fresh episode_id: frame={frame}, track={observation['track_id']}, key={unit}")
            previous = state['last']
            _require(previous['role'] == observation['role'], f"Cross-frame role changes require a separate physical episode: frame={frame}, track={observation['track_id']}, key={unit}, roles={previous['role']}->{observation['role']}")
            same_track = previous['track_id'] == observation['track_id']
            verified_handoff = not same_track and _verified_link(previous, observation)
            same_identity = same_track or verified_handoff
            delta_frames = frame - previous['frame_index']
            if delta_frames > 1:
                state['gaps'].append({'from_frame': previous['frame_index'], 'to_frame': frame, 'missing_source_frames': delta_frames - 1})
            if not same_track:
                state['raw_track_changes'].append({'from_track_id': previous['track_id'], 'to_track_id': observation['track_id'], 'frame_index': frame, 'association_verified': verified_handoff, 'association_id': observation.get('association_id'), 'first': _ref(previous), 'second': _ref(observation)})
            toward = -(observation['signed_distance_px'] - previous['signed_distance_px']) * signs[state['origin']]
            state['last'] = observation
            opposite = 'B' if state['origin'] == 'A' else 'A'
            if observation['polygon'] == opposite:
                state['destination_observed_ref'] = _ref(observation)
            conflicts = [item for item in options if item['identity_conflict']]
            state['identity_conflicts'].extend((_ref(item) for item in conflicts))
            state['identity_timeout_hold'] = sorted(set(state['identity_timeout_hold']) | set(observation.get('identity_timeout_hold', [])))
            update_approach(state, previous, observation, same_identity)
            if observation['band']:
                state['band_refs'].append(_ref(observation))
                state['band_votes'][_effective_category(observation)] += 1
            if state['event'] is not None and state['event']['route'] == 'ESTIMATED_TIMEOUT':
                if state['identity_conflicts'] or _unresolved_handoffs(state) or state['identity_timeout_hold']:
                    reason = 'UNRESOLVED_DUPLICATE_IDENTITY_IN_PASSAGE' if state['identity_conflicts'] else 'IDENTITY_CONTINUITY_UNRESOLVED_FOR_TIMEOUT' if state['identity_timeout_hold'] else 'TIMEOUT_RAW_TRACK_HANDOFF_UNRESOLVED'
                    cancel(state, frame, reason)
                    pending(state, frame, reason)
                else:
                    revised_category, revised_basis, category_error = _category(state, timeout=True, supported_categories=supported_categories)
                    if category_error:
                        cancel(state, frame, category_error)
                        pending(state, frame, category_error)
                    elif revised_category != state['event']['category']:
                        emit(state, frame, 'ESTIMATED_TIMEOUT', revised_category, revised_basis, False)
            if same_track and delta_frames == 1:
                intersection = segment_intersection(previous['point'], observation['point'], geometry.p1, geometry.p2)
                distances = (previous['signed_distance_px'], observation['signed_distance_px'])
                if intersection['kind'] == 'proper' and distances[0] * distances[1] < 0:
                    state['crossing_pairs'].append({'first': _ref(previous), 'second': _ref(observation), 'intersection': intersection, 'proof': 'opposite_signed_measured_endpoints'})
                    state['zero_contact'] = None
                elif distances[0] != 0 and distances[1] == 0 and (intersection['kind'] == 'touch'):
                    state['zero_contact'] = {'entry': _ref(previous), 'contacts': [_ref(observation)]}
                elif distances[0] == 0 and distances[1] == 0:
                    if state['zero_contact'] is not None and 0 < observation['line_projection'] < 1:
                        state['zero_contact']['contacts'].append(_ref(observation))
                    else:
                        state['zero_contact'] = None
                elif distances[0] == 0 and distances[1] != 0:
                    contact = state['zero_contact']
                    if contact is not None and intersection['kind'] == 'touch':
                        entry_distance = signed_distance(contact['entry']['point'], geometry.p1, geometry.p2)
                        if entry_distance * distances[1] < 0:
                            state['crossing_pairs'].append({'first': contact['entry'], 'second': _ref(observation), 'contact_refs': contact['contacts'], 'intersection': intersection, 'proof': 'opposite_sides_bracket_contiguous_zero_contact'})
                    state['zero_contact'] = None
                else:
                    state['zero_contact'] = None
            else:
                state['zero_contact'] = None
            outside_side = normal_side(observation)
            arrived = observation['polygon'] == opposite and outside_side == opposite
            crossed_outside = outside_side == opposite
            if crossed_outside:
                category, basis, error = _category(state, observation, supported_categories=supported_categories)
                same_origin_identity = state['origin_track'] == observation['track_id'] or _verified_link(state['origin_ref'], observation)
                latest_pair = state['crossing_pairs'][-1] if state['crossing_pairs'] else None
                measured = bool(latest_pair is not None and signed_distance(latest_pair['first']['point'], geometry.p1, geometry.p2) * signs[state['origin']] > 0 and (signed_distance(latest_pair['second']['point'], geometry.p1, geometry.p2) * signs[state['origin']] < 0) and (not any((gap['from_frame'] >= latest_pair['first']['frame_index'] and gap['to_frame'] <= frame for gap in state['gaps']))) and (not any((change['frame_index'] >= latest_pair['first']['frame_index'] for change in state['raw_track_changes']))))
                route = None
                if measured:
                    route = 'LINE_MARGIN'
                elif state['band_refs'] and (state['gaps'] or state['raw_track_changes']) and arrived and same_origin_identity and (not _unresolved_handoffs(state)):
                    route = 'LINE_MARGIN'
                elif not state['band_refs'] and state['gaps'] and arrived and state['origin_is_roi'] and same_origin_identity and (not _unresolved_handoffs(state)) and (state['origin_category'] == _effective_category(observation)) and state['approaching'] and (not state['turned_away']) and (toward >= 0):
                    route = 'OCCLUSION_DIFF_POLYGON'
                if state['identity_conflicts']:
                    error = 'UNRESOLVED_DUPLICATE_IDENTITY_IN_PASSAGE'
                elif _unresolved_handoffs(state):
                    error = 'RAW_TRACK_CHANGED_DURING_PASSAGE'
                if route and (not error) and (category == 'pedestrian') and (not measured) and (pedestrian_routes != 'all_operational'):
                    error = 'PEDESTRIAN_INFERRED_ROUTE_REQUIRES_POLICY_APPROVAL'
                if route and (not error) and state.get('p0_origin_context_suspended'):
                    error = 'ORIGIN_TIMER_CONTEXT_CONTINUITY_UNRESOLVED'
                if route and (not error) and (observation['role'] == 'vehicle'):
                    quality = state['p0_anchor_quality']
                    if measured and p0['apply_measured_anchor_guard'] and (quality is None or quality['minimum_corner_toward_progress_px'] <= p0['minimum_corner_progress_px_for_measured']):
                        error = 'ANCHOR_DEFORMATION_OR_PHYSICAL_MOTION_UNRESOLVED'
                    elif not measured and p0['apply_inferred_anchor_guard'] and (not state['p0_motion_supported']):
                        error = 'INFERRED_ROUTE_PHYSICAL_MOTION_UNRESOLVED'
                if route and (not error):
                    emit(state, frame, route, category, basis, measured, destination=observation)
                    states[unit] = _new_passage(observation, opposite, state['ordinal'] + 1)
                    continue
                if arrived:
                    reason = error or ('RAW_TRACK_CHANGED_DURING_PASSAGE' if _unresolved_handoffs(state) else 'NO_VALID_FINITE_LINE_OR_OCCLUSION_EVIDENCE')
                    if state['event'] is not None and state['event']['route'] == 'ESTIMATED_TIMEOUT':
                        cancel(state, frame, 'DESTINATION_ARRIVAL_DOES_NOT_SUPPORT_VALID_PASSAGE')
                    pending(state, frame, reason)
            if observation['polygon'] == state['origin']:
                if state['event'] is not None and state['event']['route'] == 'ESTIMATED_TIMEOUT' and (toward < 0):
                    cancel(state, frame, 'MEASURED_RETURN_TO_ORIGIN_CONTRADICTS_TIMEOUT')
                state['last_origin_ref'] = _ref(observation)
                state['origin_is_roi'] = True
                if outside_side == state['origin'] and state['band_refs']:
                    if state['event'] is not None:
                        cancel(state, frame, 'RETURNED_TO_ORIGIN_WITHOUT_CONFIRMED_PASSAGE')
                    states[unit] = _new_passage(observation, state['origin'], state['ordinal'])
            elif outside_side == state['origin'] and state['band_refs']:
                if state['event'] is not None:
                    cancel(state, frame, 'RETURNED_TO_ORIGIN_SIDE_WITHOUT_CONFIRMED_PASSAGE')
                states[unit] = _new_passage(observation, state['origin'], state['ordinal'])
            audit(state, frame, 'eligible' if state['approaching'] else 'not_eligible', 'MEASURED_APPROACH_ESTABLISHED' if state['approaching'] else 'APPROACH_NOT_ESTABLISHED')
        for unit in sorted(states, key=lambda key: (key[0], key[1] or '')):
            state = states[unit]
            if state['closed']:
                continue
            origin_ref = state['last_origin_ref']
            if origin_ref is None or state['event'] is not None or state['destination_observed_ref'] is not None or (frame - origin_ref['frame_index'] < timeout):
                continue
            if not state['approaching'] or state['turned_away']:
                audit(state, frame, 'not_eligible', 'NO_POSITIVE_MEASURED_APPROACH')
                continue
            if p0['apply_inferred_anchor_guard'] and (not state['p0_motion_supported']):
                audit(state, frame, 'not_eligible', 'NO_POSITIVE_WHOLE_BOX_MOTION_SUPPORT')
                continue
            if not state['origin_is_roi'] or state['last_origin_ref'] is None:
                audit(state, frame, 'not_eligible', 'NO_ORIGIN_ROI_FOR_TIMEOUT')
                continue
            if state['last']['frame_index'] == frame:
                audit(state, frame, 'not_eligible', 'CURRENT_MEASUREMENT_CONTRADICTS_DISAPPEARANCE')
                continue
            if state['identity_timeout_hold']:
                pending(state, frame, 'IDENTITY_CONTINUITY_UNRESOLVED_FOR_TIMEOUT')
                audit(state, frame, 'pending', 'IDENTITY_CONTINUITY_UNRESOLVED_FOR_TIMEOUT')
                continue
            if _unresolved_handoffs(state):
                pending(state, frame, 'TIMEOUT_RAW_TRACK_HANDOFF_UNRESOLVED')
                continue
            if state['identity_conflicts']:
                pending(state, frame, 'UNRESOLVED_DUPLICATE_IDENTITY_IN_PASSAGE')
                continue
            category, basis, error = _category(state, timeout=True, supported_categories=supported_categories)
            if error:
                pending(state, frame, error)
                continue
            if category == 'pedestrian' and pedestrian_routes != 'all_operational':
                pending(state, frame, 'PEDESTRIAN_INFERRED_ROUTE_REQUIRES_POLICY_APPROVAL')
                continue
            if state['ordinal'] > 1 and return_timeout_policy == 'require_new_passage_evidence':
                pending(state, frame, 'RETURN_TIMEOUT_REQUIRES_NEW_PASSAGE_EVIDENCE')
                continue
            emit(state, frame, 'ESTIMATED_TIMEOUT', category, basis, False)
            audit(state, frame, 'accepted_estimate', 'ELIGIBLE_TIMEOUT_60_SOURCE_FRAMES')
    events = sorted(accepted.values(), key=lambda event: (event['activation_frame'], event['event_id']))
    display_ordinals = Counter()
    for event in events:
        key = (event['category'], event['direction'])
        display_ordinals[key] += 1
        event['display_id'] = passage_display_id(*key, display_ordinals[key], category_prefixes=display_prefixes)
    displays = {event['event_id']: event['display_id'] for event in events}
    for event in cancelled.values():
        event['display_id'] = None
    for row in logs:
        row['display_id'] = displays.get(row.get('event_id'))
        row['display_status'] = 'final_active_passage' if row['display_id'] else 'no_final_active_passage_id'
    for decision in decisions:
        decision['display_id'] = displays.get(decision.get('event_id'))
    display_assignments = [{key: deepcopy(event[key]) for key in ('event_id', 'physical_unit_id', 'episode_id', 'passage_id', 'display_id', 'category', 'direction', 'activation_frame', 'decision_frame')} | {'source_frames': deepcopy(event['evidence']['source_frames'])} for event in events]
    final_pending = [decision for decision in decisions if decision['status'] == 'pending']
    by_category = Counter((event['category'] for event in events))
    by_direction = Counter((event['direction'] for event in events))
    by_route = Counter((event['route'] for event in events))
    measured_total = sum((event['evidence']['measured_crossing'] for event in events))
    open_passages = []
    for unit, state in sorted(states.items(), key=lambda item: (item[0][0], item[0][1] or '')):
        if state['closed']:
            disposition = 'PHYSICAL_EPISODE_ENDED_NO_TIMER_INHERITANCE'
        elif state['event'] is not None:
            disposition = 'ACCEPTED_ESTIMATE_AWAITING_STRONGER_EVIDENCE'
        elif decision_by_passage.get((state['last']['unit_id'], passage_id(state)), {}).get('status') == 'pending':
            disposition = 'PENDING'
        elif state['turned_away']:
            disposition = 'TURNED_AWAY_WITHOUT_CONFIRMED_PASSAGE'
        elif not state['approaching']:
            disposition = 'NO_NEW_PASSAGE_EVIDENCE'
        elif state['last_origin_ref'] is None:
            disposition = 'NO_ORIGIN_ROI_FOR_TIMEOUT'
        else:
            disposition = 'EOF_WITHOUT_DESTINATION_OR_ELIGIBLE_TIMEOUT'
        open_passages.append({'physical_unit_id': state['last']['unit_id'], 'episode_id': state['last'].get('episode_id'), 'passage_id': passage_id(state), 'track_id': state['last']['track_id'], 'disposition': disposition, 'last_observed_frame': state['last']['frame_index'], 'last_origin_frame': (state['last_origin_ref'] or {}).get('frame_index'), 'destination_observed': state['destination_observed_ref'] is not None, 'band_observed': bool(state['band_refs']), 'approach_progress_px': state['approach_progress_px'], 'timeout_eligible': bool(state['approaching'] and state['origin_is_roi'] and (not state['closed']) and (not state['turned_away']) and (not state['identity_conflicts']) and (not state['identity_timeout_hold']) and (not _unresolved_handoffs(state)) and (not p0['apply_inferred_anchor_guard'] or state['p0_motion_supported'])), 'identity_timeout_hold': deepcopy(state['identity_timeout_hold']), 'closure_reason': state['closure_reason']})
    aggregate = {'total': len(events), 'by_category': dict(sorted(by_category.items())), 'by_direction': dict(sorted(by_direction.items())), 'by_route': dict(sorted(by_route.items())), 'measured_total': measured_total, 'inferred_total': len(events) - measured_total, 'pending': len(final_pending), 'cancelled': len(cancelled), 'vehicle_total': sum((event['category'] != 'pedestrian' for event in events)), 'pedestrian_total': by_category.get('pedestrian', 0)}
    report = {'version': 'b_v5_p0_eligibility_finalization_v1', 'namespace': namespace, 'frame_count': frame_count, 'first_frame_index': first_frame, 'last_frame_index': last_frame, 'observed_units': len(states), 'observed_physical_episodes': len(states), 'counters': dict(sorted(counters.items())), 'open_passage_diagnostic_counts': dict(sorted(Counter((item['disposition'] for item in open_passages)).items())), 'policy': deepcopy(policy), 'source_identity': deepcopy(source_identity), 'no_model_inference': True, 'post_eof_frames_generated': 0, 'physical_identity_assumption': 'unreviewed retained raw track/episode denotes one unit; only explicit scoped associations permit handoff', 'pedestrian_policy': 'positive pedestrian role only; rider plus zero; unknown person role unresolved', 'pedestrian_routes': pedestrian_routes, 'return_timeout_policy': return_timeout_policy, 'return_timeout_policy_basis': 'explicit_policy' if 'return_timeout_policy' in policy else 'legacy_operational_estimate_default', 'pedestrian_route_policy_basis': 'explicit_policy' if 'pedestrian_routes' in policy else 'conservative_research_default_pending_user_decision', 'supported_categories': sorted(supported_categories), 'display_category_prefixes': deepcopy(display_prefixes), 'display_id_semantics': 'per-video final active passage; raw tracker IDs remain unchanged', 'timeout_semantics': 'automatic operational estimate, not proof of crossing or ID change', 'unsupported_taxonomy': ['bus_angkot', 'pickup_business_mapping'], 'evaluation_limit': 'Software ledger; no ground-truth counting performance claim'}
    result = {'source_identity': deepcopy(source_identity), 'policy': deepcopy(policy), 'events': events, 'cancelled_events': sorted(cancelled.values(), key=lambda event: event['event_id']), 'decisions': decisions, 'passage_log': logs, 'open_passage_diagnostics': open_passages, 'display_assignments': display_assignments, 'eligibility_audit': eligibility_audit, 'aggregate': aggregate, 'report': report}
    return finalize_passages(result, policy.get('episode_relations', []), controls=p0)


### src/traffic_location/traffic/_kernel/publication.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/publication.py
"""Offline B v7 joint publication gate over the immutable B v6 engine.

Trajectory compatibility is an unresolved identity conflict, never an alias or
crossing measurement. The private B v6 result is not published until this gate
has finalized the exact implicated candidate passages. No model executes here.
"""
from __future__ import annotations
from collections import Counter
from copy import deepcopy
from . import eligibility as previous
INVALID_REASON = 'INVALID_AMBIGUOUS_UNIT'
SCOPE = 'THIS_CANDIDATE_PASSAGE_NOT_GLOBAL_RAW_ID'

def _require(condition, message):
    if not condition:
        raise ValueError(message)

def _source_index(frames):
    index = {}
    for row in frames:
        frame = row['frame_index']
        for obs in row.get('observations', []):
            if obs.get('observed') is not True or obs.get('emitted') is not True or obs.get('role', 'vehicle') != 'vehicle' or (obs.get('class_name') == 'person'):
                continue
            key = (obs.get('track_id'), obs.get('physical_episode_id'), frame)
            index.setdefault(key, []).append(obs)
    return index

def _validate_endpoint(event, relation, prefix, source_index):
    """Use each endpoint's own window; another endpoint cannot extend it."""
    _require(event['track_id'] == relation.get(prefix + '_track_id'), 'Handoff event and raw track do not match')
    _require(event['physical_unit_id'] == relation.get(prefix + '_unit_id') and event.get('episode_id') == relation.get(prefix + '_episode_id'), 'Handoff belongs to another unit/episode')
    physical = relation.get(prefix + '_physical_episode_id')
    _require(isinstance(physical, str) and bool(physical), 'Physical context required')
    refs = [ref for ref in event['evidence'].get('source_refs', []) if ref.get('track_id') == event['track_id'] and ref.get('physical_episode_id') == physical]
    _require(bool(refs), 'Handoff has no source reference in this physical context')
    low, high = (min((r['frame_index'] for r in refs)), max((r['frame_index'] for r in refs)))
    support = relation.get(prefix + '_support_frames', [])
    _require(isinstance(support, list) and len(support) == 5 and all((type(f) is int for f in support)) and (support == list(range(support[0], support[0] + 5))), 'Handoff needs five consecutive actual observations at each endpoint')
    _require(all((low <= frame <= high for frame in support)), "Handoff support is outside the candidate's own measured window")
    _require(all(((event['track_id'], physical, frame) in source_index for frame in support)), 'Handoff support is absent from actual vehicle observations')
    return support

def _rename_event_ids(result):
    mapping = {e['event_id']: e['event_id'].replace('b_v6_', 'b_v7_', 1) for e in result['events'] + result.get('cancelled_events', [])}

    def walk(value):
        if isinstance(value, dict):
            return {mapping.get(key, key): walk(item) for key, item in value.items()}
        if isinstance(value, list):
            return [walk(item) for item in value]
        if isinstance(value, str):
            return mapping.get(value, value)
        return value
    return walk(result)

def finalize_handoff_candidates(provisional, proposal, *, frames, finalization_frame=None):
    """Finalize only exact active pairs. Never revisit old pending decisions.

    The input is a private candidate snapshot, not an already published ledger.
    Rejected candidates retain their source evidence and candidate timestamps;
    the actual offline invalid decision is recorded at the last supplied frame.
    """
    frames = list(frames)
    last_frame = max((row['frame_index'] for row in frames))
    decision_frame = last_frame if finalization_frame is None else finalization_frame
    _require(decision_frame == last_frame == provisional['report']['last_frame_index'], 'Offline publication decision must use the actual source EOF')
    output = deepcopy(provisional)
    candidates = {e['event_id']: e for e in output['events']}
    _require(len(candidates) == len(output['events']), 'Duplicate candidate event IDs')
    source_index = _source_index(frames)
    implicated, audits, seen_relations = ({}, [], set())
    for relation in proposal.get('relations', []):
        rid = relation.get('relation_id')
        _require(isinstance(rid, str) and rid and (rid not in seen_relations), 'Handoff relation must have a unique ID')
        seen_relations.add(rid)
        _require(relation.get('verified') is False and relation.get('candidate_kind') == 'UNVERIFIED_ORIGIN_HANDOFF' and (relation.get('relation_type', relation.get('type')) == 'SAME_UNIT'), 'Only explicit unverified handoff conflicts are supported')
        ids = [relation.get('first_event_id'), relation.get('second_event_id')]
        _require(len(set(ids)) == 2 and all((eid in candidates for eid in ids)) and (sorted(relation.get('event_ids', [])) == sorted(ids)), 'Handoff must name exactly two distinct active candidate events')
        first, second = [candidates[eid] for eid in ids]
        _require(first.get('role') == second.get('role') == 'vehicle' and first['category'] == second['category'] and (first['category'] != 'pedestrian') and (first['direction'] == second['direction']) and (first['direction'] in ('A_TO_B', 'B_TO_A')), 'Handoff must concern same-category same-direction vehicle candidates')
        _require(not all((e['evidence']['measured_crossing'] for e in (first, second))), 'This experiment cannot suppress two measured passages')
        first_support = _validate_endpoint(first, relation, 'first', source_index)
        second_support = _validate_endpoint(second, relation, 'second', source_index)
        _require(0 < second_support[0] - first_support[-1] <= 60, 'Handoff must be ordered and bounded by the existing source-frame horizon')
        _require(relation.get('support_frames') == sorted(set(first_support + second_support)), 'Combined support must preserve actual separate endpoint observations')
        for event in (first, second):
            implicated.setdefault(event['event_id'], []).append(rid)
        audits.append({'relation_id': rid, 'event_ids': ids, 'status': 'INVALID_PLUS_ZERO_SCOPED_AMBIGUITY', 'physical_episodes_merged': False, 'identity_verified': False, 'decision_frame': decision_frame, 'support_frames': deepcopy(relation['support_frames'])})
    original_pending = [deepcopy(d) for d in output['decisions'] if d['status'] == 'pending']
    invalid_keys = {previous._key(d) for d in output.get('invalid_decisions', [])}
    for event_id, relation_ids in sorted(implicated.items()):
        event = candidates.pop(event_id)
        key = previous._key(event)
        _require(key not in invalid_keys, 'Candidate already has an invalid passage decision')
        invalid_keys.add(key)
        candidate_frame = event['decision_frame']
        event.update(status='cancelled', accepted_count_event=False, display_id=None, cancellation_reason=INVALID_REASON, never_published=True, candidate_decision_frame=candidate_frame, decision_frame=decision_frame, finalization_mode='OFFLINE_BEFORE_PUBLICATION')
        output['cancelled_events'].append(event)
        for log in output['passage_log']:
            if log.get('event_id') == event_id:
                log.update(candidate_count_delta=log.get('count_delta', 0), count_delta=0, candidate_status=log['status'], status='provisional_not_published', accepted_count_event=False, never_published=True, final_decision_frame=decision_frame)
        for old in output['decisions']:
            if old.get('event_id') == event_id and old.get('status') == 'superseded_by_accepted':
                old.update(status='superseded_by_invalid', superseded_at_frame=decision_frame)
        decision = {'frame_index': decision_frame, 'decision_frame': decision_frame, 'candidate_decision_frame': candidate_frame, **{field: deepcopy(event.get(field)) for field in ('track_id', 'physical_unit_id', 'episode_id', 'passage_id', 'role', 'category', 'direction')}, 'status': 'invalid', 'invalid_reason': INVALID_REASON, 'reasons': [INVALID_REASON], 'count_delta': 0, 'accepted_count_event': False, 'display_id': None, 'source_event_id': event_id, 'cancelled_candidate_event_id': event_id, 'source_ref': deepcopy(event['evidence']['source_refs'][-1]), 'physical_episodes_merged': False, 'invalid_scope': SCOPE, 'relation_ids': sorted(relation_ids), 'never_published': True, 'finalization_mode': 'OFFLINE_BEFORE_PUBLICATION', 'reason_evidence': [{'reason': INVALID_REASON, 'relation_ids': sorted(relation_ids), 'intentional_exclusion_not_zero_traffic': True, 'trajectory_compatibility_is_not_identity_proof': True}]}
        output['decisions'].append(decision)
        output['passage_log'].append({'frame_index': decision_frame, 'candidate_decision_frame': candidate_frame, **{field: decision.get(field) for field in ('track_id', 'physical_unit_id', 'episode_id', 'passage_id', 'category', 'direction')}, 'event_id': None, 'source_event_id': event_id, 'status': 'invalid', 'count_delta': 0, 'message': INVALID_REASON, 'invalid_scope': SCOPE})
    output['events'] = list(candidates.values())
    scoped_keys = {previous._key(e) for e in output['cancelled_events'] if e.get('never_published') is True}
    for row in output.get('open_passage_diagnostics', []):
        if previous._key(row) in scoped_keys:
            row.update(disposition=INVALID_REASON, timeout_eligible=False)
    previous._refresh(output)
    _require([d for d in output['decisions'] if d['status'] == 'pending'] == original_pending, 'The handoff gate must not modify previously pending candidates')
    _require(sum((log.get('count_delta', 0) for log in output['passage_log'])) == output['aggregate']['total'], 'Final passage log does not reconcile')
    output['v7_handoff_relations'] = deepcopy(proposal.get('relations', []))
    output['v7_handoff_audit'] = deepcopy(proposal.get('audit', []))
    output['v7_publication_audit'] = audits
    output['report'].update(version='b_v7_scoped_handoff_publication_v1', previous_engine_version='b_v6_user_invalid_policy_v1', v7_handoff_controls=deepcopy(proposal.get('controls', {})), open_passage_diagnostic_counts=dict(sorted(Counter((r['disposition'] for r in output.get('open_passage_diagnostics', []))).items())), v7_publication={'mode': 'OFFLINE_BEFORE_PUBLICATION', 'decision_frame': decision_frame, 'private_candidate_count': len(provisional['events']), 'additional_invalid_candidates': len(implicated), 'previous_pending_changed': 0, 'physical_episodes_merged': 0, 'newly_created_accepted_events': 0, 'inference_promoted_to_measured': 0, 'unpublished_candidate_logs_have_zero_delta': True, 'finalized_active_event_ids': sorted(candidates)})
    return _rename_event_ids(output)

def replay_frames(frames, geometry, *, source_identity, policy):
    """One isolated offline replay; only the finalized B v7 result is returned."""
    from .handoff import propose_handoff_conflicts
    copied = deepcopy(list(frames))
    provisional = previous.replay_frames(copied, geometry, source_identity=source_identity, policy=deepcopy(policy))
    proposal = propose_handoff_conflicts(provisional, copied, geometry, policy)
    return finalize_handoff_candidates(provisional, proposal, frames=copied)


### src/traffic_location/traffic/_kernel/render_base.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/render_base.py
"""B v3 audit rendering from measured records and a completed passage ledger.

No detector, tracker, encoder, association or counting inference occurs here.
The sidebar replays the explicit event log, including upgrades and reversals;
its total is operational and contains inferred passages separately identified.
"""
from __future__ import annotations
from bisect import bisect_right
from collections import Counter, defaultdict
from copy import deepcopy
import json
import math
import re
from pathlib import Path
import shutil
import subprocess
import time
from . import render_sources as frozen
VERSION = 'phase8_b_v3_annotation_v1'
SEMANTICS = 'OFFLINE_LEDGER_LOG_REPLAY_NOT_AN_ONLINE_DECISION_OR_ACCURACY_MEASUREMENT'
VEHICLE_CATEGORIES = ('motorcycle', 'car', 'truck')
CATEGORIES = VEHICLE_CATEGORIES + ('pedestrian',)
CATEGORY_LETTERS = {'motorcycle': 'M', 'car': 'C', 'truck': 'T', 'pedestrian': 'P'}
CATEGORY_COLORS = dict(frozen.CATEGORY_COLORS, pedestrian=(220, 95, 220))
DISPLAY_CATEGORY = dict(frozen.DISPLAY_CATEGORY, pedestrian='Pedestrian')
DIRECTIONS = ('A_TO_B', 'B_TO_A')
ROUTE_LABELS = {'LINE_MARGIN': 'CROSSING_LINE', 'OCCLUSION_DIFF_POLYGON': 'OCCLUSION_DIFF_POLYGON', 'ESTIMATED_TIMEOUT': 'CHANGE_ID (ESTIMASI)'}
_require = frozen._require

def _identity(identity):
    _require(isinstance(identity, dict), 'Result source_identity is required')
    for key in ('video_sha256', 'records_sha256', 'parent_run_id', 'variant'):
        frozen._identifier(identity.get(key), 'source identity ' + key)
    for key in ('width', 'height'):
        frozen._integer(identity.get(key), key, 1)
    _require(identity['variant'] == 'raw', 'B v2 renderer needs raw source measurements')

def _display_id(value, category, direction):
    """Validate final, per-passage display IDs; never turn them into track IDs."""
    _require(isinstance(value, str) and re.fullmatch('[PMCT][1-9][0-9]{3}', value), 'Final passage needs one category letter and four digits, without leading zero')
    _require(value[0] == CATEGORY_LETTERS[category], 'Display ID category differs')
    _require((int(value[1]) % 2 == 0) == (direction == 'A_TO_B'), 'Display ID direction parity differs')
    return value

def _compile_assignments(assignments, frame_count):
    """Index only exact retained observation assignments; never fill a gap."""
    indexed = defaultdict(dict)
    _require(isinstance(assignments, list), 'unit_assignments must be an array')
    for item in assignments:
        _require(isinstance(item, dict), 'Observation assignment must be an object')
        frame = frozen._integer(item.get('frame_index'), 'assignment frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Assignment outside source frames')
        track = frozen._integer(item.get('track_id'), 'assignment raw track', 1)
        _require(track not in indexed[frame], 'Duplicate frame/track assignment')
        frozen._identifier(item.get('physical_unit_id'), 'assignment physical unit')
        frozen._identifier(item.get('episode_id'), 'assignment episode')
        _require(item.get('role') in ('vehicle', 'pedestrian', 'rider', 'unknown', 'unknown_person'), 'Unknown observation role')
        indexed[frame][track] = deepcopy(item)
    return dict(indexed)

def build_timeline(result, *, frame_count=None, component_relations=None, unit_assignments=None):
    """Validate one source ledger and compile constant-size counter snapshots.

    Changes in route/category revise the same event; count_delta does not
    accumulate duplicate passage counts. Cancelled events can appear during
    replay before their explicit -1 cancellation even though final totals omit
    them. Input objects are not modified.
    """
    identity = result.get('source_identity')
    _identity(identity)
    if frame_count is not None:
        frozen._integer(frame_count, 'frame_count', 1)
    events, cancelled = (result.get('events', []), result.get('cancelled_events', []))
    _require(isinstance(events, list) and isinstance(cancelled, list), 'Event arrays required')
    by_id, final_ids, final_display_ids = ({}, set(), set())
    for event in events + cancelled:
        _require(isinstance(event, dict), 'Event must be an object')
        event_id = frozen._identifier(event.get('event_id'), 'event ID')
        _require(event_id not in by_id, 'Duplicate event ID')
        _require(event.get('source_identity') == identity, 'Event belongs to another source')
        frozen._identifier(event.get('physical_unit_id'), 'physical unit')
        frozen._identifier(event.get('passage_id'), 'passage')
        frozen._integer(event.get('track_id'), 'vehicle track', 1)
        _require(event.get('category') in CATEGORIES, 'Unsupported product category')
        _require(event.get('direction') in DIRECTIONS, 'Invalid event direction')
        _require(event.get('route') in ROUTE_LABELS, 'Unknown event route')
        measured = event.get('evidence', {}).get('measured_crossing')
        _require(type(measured) is bool, 'Event requires explicit measured_crossing evidence')
        _require(event['route'] == 'LINE_MARGIN' or measured is False, 'Polygon/timeout cannot claim a measured line crossing')
        by_id[event_id] = deepcopy(event)
        if event in events:
            _require(event.get('accepted_count_event') is True, 'Final event must explicitly contribute to the operational count')
            _require(event.get('status') == 'accepted', 'Pending cannot enter accepted ledger')
            final_ids.add(event_id)
            display_id = _display_id(event.get('display_id'), event['category'], event['direction'])
            _require(display_id not in final_display_ids, 'Duplicate final display ID')
            final_display_ids.add(display_id)
    final_passages = [(by_id[key]['physical_unit_id'], by_id[key]['passage_id']) for key in final_ids]
    _require(len(final_passages) == len(set(final_passages)), 'Duplicate final physical passage')
    logs = result.get('passage_log', [])
    _require(isinstance(logs, list), 'passage_log must be an array')
    logs = sorted(enumerate(deepcopy(logs)), key=lambda pair: (pair[1].get('frame_index', -1), pair[0]))
    active, active_passages, snapshots, operations, displayed = ({}, {}, [], [], [])
    pivot = {category: {direction: 0 for direction in DIRECTIONS} for category in CATEGORIES}
    route_counts = {route: 0 for route in ROUTE_LABELS}

    def adjust(event, increment):
        pivot[event['category']][event['direction']] += increment
        route_counts[event['route']] += increment
    for _, log in logs:
        frame = frozen._integer(log.get('frame_index'), 'log frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Log outside source frames')
        delta = log.get('count_delta', 0)
        _require(type(delta) is int and delta in (-1, 0, 1), 'Invalid passage count_delta')
        event_id = log.get('event_id')
        status = log.get('status')
        operational = delta != 0 or status == 'upgraded'
        if not operational:
            continue
        _require(event_id in by_id, 'Count log has no retained event')
        payload = dict(by_id[event_id])
        for key in ('track_id', 'category', 'direction', 'route'):
            if key in log:
                payload[key] = log[key]
        _require(payload['category'] in CATEGORIES and payload['direction'] in DIRECTIONS and (payload['route'] in ROUTE_LABELS), 'Invalid logged passage semantics')
        measured = log.get('measured_crossing')
        _require(type(measured) is bool, 'Operational log requires explicit measured_crossing')
        _require(payload['route'] == 'LINE_MARGIN' or measured is False, 'Polygon/timeout log cannot claim a measured line crossing')
        payload['_measured_crossing'] = measured
        frozen._integer(payload['track_id'], 'log vehicle track', 1)
        passage = (payload['physical_unit_id'], payload['passage_id'])
        if delta == 1:
            _require(event_id not in active and passage not in active_passages, 'Log attempts to increment one passage twice')
            active[event_id] = payload
            active_passages[passage] = event_id
            adjust(payload, 1)
        elif delta == -1:
            _require(event_id in active, 'Cancellation has no active event')
            previous = active.pop(event_id)
            adjust(previous, -1)
            active_passages.pop((previous['physical_unit_id'], previous['passage_id']))
        else:
            _require(event_id in active, 'Upgrade has no previously accepted event')
            previous = active[event_id]
            _require((previous['physical_unit_id'], previous['passage_id']) == passage, 'Upgrade changed physical passage identity')
            adjust(previous, -1)
            active[event_id] = payload
            adjust(payload, 1)
        display = dict(log, track_id=payload['track_id'], category=payload['category'], direction=payload['direction'], route=payload['route'], display_id=by_id[event_id].get('display_id'), display_category=by_id[event_id]['category'] if event_id in final_ids else payload['category'], display_direction=by_id[event_id]['direction'] if event_id in final_ids else payload['direction'], requested_label=ROUTE_LABELS[payload['route']])
        displayed.append(display)
        operations.append(frame)
        measured = sum((item['_measured_crossing'] for item in active.values()))
        snapshots.append({'pivot': deepcopy(pivot), 'route_counts': dict(route_counts), 'accepted_so_far': len(active), 'measured_so_far': measured, 'inferred_so_far': len(active) - measured, 'vehicle_so_far': sum((sum(pivot[key].values()) for key in VEHICLE_CATEGORIES)), 'pedestrian_so_far': sum(pivot['pedestrian'].values())})
    _require(set(active) == final_ids, 'Event log does not reconcile with final accepted ledger')
    for event_id in final_ids:
        for key in ('category', 'direction', 'route', 'track_id'):
            _require(active[event_id][key] == by_id[event_id][key], 'Final log/ledger differs: ' + key)
        _require(active[event_id]['_measured_crossing'] == by_id[event_id]['evidence']['measured_crossing'], 'Final log/ledger measured evidence differs')
    pending = []
    decisions = result.get('decisions', result.get('machine_decisions', []))
    _require(isinstance(decisions, list), 'Decisions must be an array')
    for decision in decisions:
        if decision.get('status') not in ('pending', 'pending_review'):
            continue
        _require(decision.get('accepted_count_event') is not True, 'Pending decision cannot contribute')
        frame = decision.get('frame_index', decision.get('diagnostic', {}).get('frame_index'))
        frozen._integer(frame, 'pending frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Pending outside source frames')
        pending.append({'frame_index': frame, 'track_id': decision.get('track_id'), 'reasons': decision.get('reasons', decision.get('hard_block_reasons', []))})
    pending.sort(key=lambda item: item['frame_index'])
    relations = component_relations if component_relations is not None else result.get('component_links', [])
    links = defaultdict(list)
    for relation in relations:
        _require(relation.get('type') == 'COMPONENT_OF', 'Only explicit component relations may merge display boxes')
        vehicle = frozen._integer(relation.get('vehicle_track_id'), 'component parent', 1)
        component = frozen._integer(relation.get('component_track_id'), 'component track', 1)
        _require(vehicle != component, 'Component cannot be its own vehicle')
        frozen._identifier(relation.get('relation_id'), 'component relation ID')
        support = relation.get('support_frames')
        _require(isinstance(support, list) and bool(support), 'Component needs retained measured support frames')
        _require(len(support) == len(set(support)), 'Repeated component support frame')
        for frame in support:
            frozen._integer(frame, 'component support frame')
            if frame_count is not None:
                _require(frame < frame_count, 'Component support outside source frames')
            links[frame].append(deepcopy(relation))
    assignments = _compile_assignments(unit_assignments if unit_assignments is not None else result.get('frame_assignments', []), frame_count)
    display_by_unit = defaultdict(list)
    for event_id in sorted(final_ids):
        event = by_id[event_id]
        refs = event.get('evidence', {}).get('source_refs', [])
        source_frames = [ref.get('frame_index') for ref in refs if isinstance(ref, dict)]
        supplied = next((entry for entry in result.get('display_assignments', []) if entry.get('event_id') == event_id), {})
        source_frames += list(supplied.get('source_frames', []))
        source_frames = [frame for frame in source_frames if type(frame) is int and frame >= 0]
        if not source_frames:
            source_frames = [event.get('activation_frame')]
        start = min((frozen._integer(frame, 'display source frame') for frame in source_frames))
        episode = event.get('episode_id', supplied.get('episode_id'))
        if episode is None:
            continue
        frozen._identifier(episode, 'event episode')
        display_by_unit[event['physical_unit_id'], episode].append({'start_frame': start, 'display_id': event['display_id'], 'category': event['category'], 'passage_id': event['passage_id'], 'event_id': event_id})
    for candidates in display_by_unit.values():
        candidates.sort(key=lambda item: (item['start_frame'], item['event_id']))
        _require(len({entry['start_frame'] for entry in candidates}) == len(candidates), 'Ambiguous display passage start in one physical episode')
    return {'source_identity': deepcopy(identity), 'event_by_id': by_id, 'final_accepted': len(final_ids), 'final_pending': len(pending), 'operations': operations, 'snapshots': snapshots, 'display_logs': displayed, 'pending': pending, 'pending_frames': [item['frame_index'] for item in pending], 'component_links_by_frame': dict(links), 'frame_count': frame_count, 'unit_assignments_by_frame': assignments, 'display_by_unit': dict(display_by_unit), 'final_vehicle': sum((by_id[key]['category'] in VEHICLE_CATEGORIES for key in final_ids)), 'final_pedestrian': sum((by_id[key]['category'] == 'pedestrian' for key in final_ids)), 'semantics': SEMANTICS}

def timeline_at(timeline, frame_index):
    frozen._integer(frame_index, 'frame index')
    if timeline['frame_count'] is not None:
        _require(frame_index < timeline['frame_count'], 'Frame outside source')
    position = bisect_right(timeline['operations'], frame_index)
    pending_n = bisect_right(timeline['pending_frames'], frame_index)
    state = deepcopy(timeline['snapshots'][position - 1]) if position else {'pivot': {category: {direction: 0 for direction in DIRECTIONS} for category in CATEGORIES}, 'route_counts': {route: 0 for route in ROUTE_LABELS}, 'accepted_so_far': 0, 'measured_so_far': 0, 'inferred_so_far': 0, 'vehicle_so_far': 0, 'pedestrian_so_far': 0}
    state.update(frame_index=frame_index, accepted_final=timeline['final_accepted'], pending_so_far=pending_n, pending_final=timeline['final_pending'], vehicle_final=timeline['final_vehicle'], pedestrian_final=timeline['final_pedestrian'], recent_logs=list(reversed(timeline['display_logs'][max(0, position - 4):position])), recent_pending=list(reversed(timeline['pending'][max(0, pending_n - 2):pending_n])))
    _require(sum((sum(row.values()) for row in state['pivot'].values())) == state['accepted_so_far'], 'Pivot/event mismatch')
    return state

def _geometry_native(geometry, width, height):
    native = geometry.get('rendered_native_pixels', {})
    _require((native.get('width'), native.get('height')) == (width, height), 'Geometry source dimensions differ')
    for key in ('side_a_polygon', 'side_b_polygon'):
        _require(isinstance(native.get(key), list) and len(native[key]) >= 3, 'Polygon missing')
        for point in native[key]:
            _require(isinstance(point, (list, tuple)) and len(point) == 2, 'Invalid polygon point')
            for value in point:
                frozen._finite(value, 'polygon coordinate')
    line = native.get('line', {})
    for key in ('P1', 'P2'):
        _require(isinstance(line.get(key), (list, tuple)) and len(line[key]) == 2, 'Line endpoint missing')
        for value in line[key]:
            frozen._finite(value, 'line coordinate')
    half_width = geometry.get('line_margin_half_width_px', native.get('line_margin_half_width_px'))
    frozen._finite(half_width, 'line margin half width')
    _require(half_width >= 0 and line['P1'] != line['P2'], 'Invalid finite line band')
    return (native, half_width)

def _display_units(by_track, relations, assignments, display_by_unit, frame_index):
    """Union only positively supported person->motorcycle boxes at this frame.

    Ambiguous parent assignment is displayed as separate boxes. Bbox unions
    never supply geometry: vehicle_bc is retained from raw vehicle measurement.
    """
    parents = defaultdict(set)
    _require(set(assignments).issubset(by_track), 'Assignment has no measured emitted track at this frame')
    for relation in relations:
        parent, child = (relation['vehicle_track_id'], relation['component_track_id'])
        if parent in by_track and child in by_track:
            if by_track[parent]['class_name'] == 'motorcycle' and by_track[child]['class_name'] == 'person':
                _require(assignments.get(child, {}).get('role') != 'pedestrian', 'Pedestrian assignment conflicts with supported rider relation')
                parents[child].add(parent)
    for child, assignment in assignments.items():
        if assignment['role'] != 'rider' or not assignment.get('component_of_unit_id'):
            continue
        reviews = assignment.get('review_refs', [])
        if not reviews:
            continue
        _require(isinstance(reviews, list) and all((isinstance(ref, dict) and ref.get('review_id') for ref in reviews)), 'Reviewed rider assignment needs explicit review references')
        _require(by_track[child]['class_name'] == 'person', 'Reviewed rider needs a raw person box')
        for parent, parent_assignment in assignments.items():
            if parent != child and parent_assignment['role'] == 'vehicle' and (parent_assignment['physical_unit_id'] == assignment['component_of_unit_id']) and (by_track[parent]['class_name'] == 'motorcycle'):
                parents[child].add(parent)
    children = defaultdict(list)
    for child, choices in parents.items():
        if len(choices) == 1:
            children[next(iter(choices))].append(child)
    suppressed = {child for members in children.values() for child in members}
    units = []
    for track, observation in by_track.items():
        if track in suppressed:
            continue
        box = observation['raw_bbox']
        components = sorted(children.get(track, []))
        boxes = [box] + [by_track[child]['raw_bbox'] for child in components]
        display_box = [min((item[0] for item in boxes)), min((item[1] for item in boxes)), max((item[2] for item in boxes)), max((item[3] for item in boxes))]
        assignment = assignments.get(track, {})
        role = assignment.get('role', 'unknown_person' if observation['class_name'] == 'person' else 'vehicle')
        if role in ('pedestrian', 'rider', 'unknown', 'unknown_person'):
            _require(observation['class_name'] == 'person', 'Person role requires a raw person measurement')
        if role == 'vehicle':
            _require(observation['class_name'] != 'person', 'Raw person cannot silently become a vehicle')
        category = 'pedestrian' if role == 'pedestrian' else observation['class_name']
        candidates = display_by_unit.get((assignment.get('physical_unit_id'), assignment.get('episode_id')), [])
        applicable = [entry for entry in candidates if entry['start_frame'] <= frame_index]
        passage = applicable[-1] if applicable and role in ('vehicle', 'pedestrian') else {}
        if passage:
            _require((passage['category'] == 'pedestrian') == (role == 'pedestrian'), 'Passage display category conflicts with observation role')
            category = passage['category']
        bc = frozen._bc(box) if role in ('vehicle', 'pedestrian') else None
        units.append({'track_id': track, 'class_name': category, 'raw_class_name': observation['class_name'], 'role': role, 'physical_unit_id': assignment.get('physical_unit_id'), 'episode_id': assignment.get('episode_id'), 'display_id': passage.get('display_id'), 'passage_id': passage.get('passage_id'), 'display_bbox': display_box, 'raw_vehicle_bbox': list(box), 'vehicle_bc': bc if role == 'vehicle' else None, 'counting_bc': bc, 'person_bc': bc if role == 'pedestrian' else None, 'component_track_ids': components, 'component_additional_increment': 0})
    return units

def _render(source_frame, record, timeline, geometry, *, fps, video_id, frame_index=None):
    import cv2
    import numpy as np
    frozen._finite(fps, 'FPS')
    _require(fps > 0, 'FPS must be positive')
    identity = timeline['source_identity']
    height, width = source_frame.shape[:2]
    _require((width, height) == (identity['width'], identity['height']), 'Native source dimensions differ')
    index = record.get('frame_index') if frame_index is None else frame_index
    by_track = frozen._validate_record(record, index, identity, source_frame)
    native, margin = _geometry_native(geometry, width, height)
    state = timeline_at(timeline, index)
    image = source_frame.copy()
    line = native['line']
    p1, p2 = (np.asarray(line['P1'], dtype=float), np.asarray(line['P2'], dtype=float))
    unit = (p2 - p1) / np.linalg.norm(p2 - p1)
    normal = np.asarray([-unit[1], unit[0]])
    band = np.asarray([p1 + normal * margin, p2 + normal * margin, p2 - normal * margin, p1 - normal * margin])
    overlay = image.copy()
    cv2.fillPoly(overlay, [np.rint(band).astype(np.int32)], (50, 205, 235))
    cv2.addWeighted(overlay, 0.18, image, 0.82, 0, image)
    cv2.polylines(image, [np.rint(band).astype(np.int32)], True, (90, 225, 250), 1, cv2.LINE_AA)
    cv2.line(image, tuple(np.rint(p1).astype(int)), tuple(np.rint(p2).astype(int)), (90, 225, 250), 2, cv2.LINE_AA)
    for key, color, letter in (('side_a_polygon', (235, 155, 65), 'A'), ('side_b_polygon', (65, 155, 245), 'B')):
        points = np.asarray(native[key], dtype=float)
        cv2.polylines(image, [np.rint(points).astype(np.int32)], True, color, 1, cv2.LINE_AA)
        midpoint = points.mean(axis=0)
        frozen._label(image, letter, midpoint[0], midpoint[1], color, width)
    units = _display_units(by_track, timeline['component_links_by_frame'].get(index, []), timeline['unit_assignments_by_frame'].get(index, {}), timeline['display_by_unit'], index)
    state['display_units'] = units
    state['band_vertices_native'] = band.tolist()
    for unit_info in units:
        box, name, track = (unit_info['display_bbox'], unit_info['class_name'], unit_info['track_id'])
        color = CATEGORY_COLORS.get(name, CATEGORY_COLORS['other'])
        cv2.rectangle(image, (round(box[0]), round(box[1])), (round(box[2]), round(box[3])), color, 2, cv2.LINE_AA)
        text = f"{unit_info['display_id'] or 'passage unresolved'} | {name} | raw {track}"
        if unit_info['component_track_ids']:
            text += ' | riders ' + ','.join(map(str, unit_info['component_track_ids'])) + ' (+0)'
        elif unit_info['role'] in ('unknown', 'unknown_person'):
            text = f'person | role unknown | not counted | raw {track}'
        elif unit_info['role'] == 'rider':
            text = f'rider | component +0 | raw {track}'
        unit_info['label'] = text
        frozen._label(image, text, box[0], box[1] - 4, color, width)
        if unit_info['counting_bc'] is not None:
            point = tuple((round(value) for value in unit_info['counting_bc']))
            cv2.circle(image, point, 4, (10, 15, 20), -1, cv2.LINE_AA)
            cv2.circle(image, point, 3, color, -1, cv2.LINE_AA)
    canvas_height = max(height + 240, 960)
    if canvas_height % 2:
        canvas_height += 1
    canvas = np.full((canvas_height, width + 520, 3), (20, 25, 33), dtype=np.uint8)
    canvas[:height, :width] = image
    cv2.rectangle(canvas, (width, 0), (width + 519, canvas_height - 1), (27, 34, 44), -1)
    x, right = (width + 18, width + 501)
    panel_text = []

    def text(value, y, *, offset=0, scale=0.42, color=(225, 232, 239), thickness=1):
        panel_text.append({'text': str(value), 'x': x + offset, 'y': y})
        frozen._text(canvas, str(value), (x + offset, y), scale=scale, color=color, thickness=thickness)
    text('REVIEW COUNTING B v3', 28, scale=0.6, thickness=2)
    text(video_id, 52, scale=0.45, color=(170, 192, 211))
    text('OFFLINE FINAL LEDGER / LOG REPLAY', 77, scale=0.4, color=(100, 205, 250))
    text('Bukan keputusan online atau ukuran akurasi', 97, scale=0.36)
    cv2.line(canvas, (x, 109), (right, 109), (65, 76, 91), 1)
    text('PASSAGE ACCEPTED (OPERASIONAL)', 135, scale=0.45)
    text(f"{state['accepted_so_far']} / {state['accepted_final']}", 166, scale=0.76, color=(150, 235, 185), thickness=2)
    text(f"Measured {state['measured_so_far']} | inferred {state['inferred_so_far']}", 190, scale=0.39)
    text(f"Vehicle {state['vehicle_so_far']} | pedestrian {state['pedestrian_so_far']}", 207, scale=0.36)
    text('Kategori', 221, color=(170, 192, 211))
    text('A > B', 221, offset=208, color=(170, 192, 211))
    text('B > A', 221, offset=309, color=(170, 192, 211))
    text('Total', 221, offset=411, color=(170, 192, 211))
    for position, category in enumerate(CATEGORIES):
        y = 245 + position * 24
        color = CATEGORY_COLORS[category]
        cv2.rectangle(canvas, (x, y - 11), (x + 8, y - 3), color, -1)
        text(DISPLAY_CATEGORY[category], y, offset=17)
        row = state['pivot'][category]
        for offset, value in zip((228, 330, 424), [row[key] for key in DIRECTIONS] + [sum(row.values())]):
            text(value, y, offset=offset, scale=0.48)
    text('A>B mendekati | B>A menjauhi kamera', 335, scale=0.36)
    text('LINE / POLYGON / TIMEOUT : ' + ' / '.join((str(state['route_counts'][key]) for key in ROUTE_LABELS)), 359, scale=0.38)
    text('Timeout = estimasi; CHANGE_ID belum terbukti', 380, scale=0.35, color=(100, 205, 250))
    cv2.line(canvas, (x, 394), (right, 394), (65, 76, 91), 1)
    text('LOG KENDARAAN LEWAT', 420, scale=0.46, color=(150, 235, 185))
    text('+1 event baru | +0 revisi | -1 pembatalan', 441, scale=0.35)
    if not state['recent_logs']:
        text('Belum ada passage pada posisi replay ini', 469, scale=0.39)
    for position, log in enumerate(state['recent_logs']):
        y = 469 + position * 65
        direction = log['display_direction'].replace('_TO_', '>')
        display_id = log.get('display_id') or 'NO FINAL PASSAGE ID'
        text(f"{display_id} - {log['display_category']}; {direction}", y, scale=0.42)
        color = (100, 205, 250) if log['route'] != 'LINE_MARGIN' else (150, 235, 185)
        text(log['requested_label'], y + 19, scale=0.36, color=color)
        evidence_label = 'terukur' if log['measured_crossing'] else 'inferensi'
        text(f"raw {log['track_id']} | f{log['frame_index']} | delta {log.get('count_delta', 0):+d} | {evidence_label}", y + 37, scale=0.34)
    cv2.line(canvas, (x, 724), (right, 724), (65, 76, 91), 1)
    text(f"PENDING {state['pending_so_far']} / {state['pending_final']}", 750, scale=0.46, color=(100, 205, 250))
    text('Di luar subtotal; kandidat != kendaraan unik', 773, scale=0.35)
    for position, pending in enumerate(state['recent_pending']):
        y = 802 + position * 53
        text(f"ID {pending['track_id']} | f{pending['frame_index']}", y, scale=0.38)
        reasons = pending['reasons']
        reason = ', '.join(map(str, reasons)) if isinstance(reasons, list) else str(reasons)
        lines = frozen._wrap(reason or 'Inspect retained decision evidence', 66)
        text(lines[0], y + 18, scale=0.3, color=(170, 192, 211))
    text('Rider +0. Vehicle dan pedestrian terpisah.', 934, scale=0.35)
    frozen._text(canvas, f'FRAME {index:06d} | t ~= {index / fps:.3f}s', (16, height + 30), scale=0.58, thickness=2)
    frozen._text(canvas, 'Indeks frame nol / FPS; source PTS tidak ditetapkan', (16, height + 55), scale=0.37)
    frozen._text(canvas, 'BC = raw vehicle/person; bbox gabungan hanya tampilan', (16, height + 82), scale=0.36)
    frozen._text(canvas, f'Margin half-width {margin:g}px | pita total {2 * margin:g}px | finite flat caps', (16, height + 106), scale=0.35)
    frozen._text(canvas, 'ID tampilan per passage; raw ID dan peran tetap diaudit', (16, height + 132), scale=0.34)
    for position, label in enumerate(('motorcycle', 'car', 'truck', 'bus', 'person', 'other')):
        lx = 16 + position * 102
        cv2.rectangle(canvas, (lx, height + 153), (lx + 9, height + 162), frozen.CATEGORY_COLORS[label], -1)
        frozen._text(canvas, label, (lx + 13, height + 163), scale=0.3)
    frozen._text(canvas, 'Tanpa asosiasi positif, komponen tidak digabung berdasarkan kedekatan.', (16, height + 192), scale=0.33)
    state['panel_text'] = panel_text
    state['panel_sections_y'] = {'accepted': 135, 'passage_log': 420, 'pending': 750}
    return (canvas, state)

def render_frame(source_frame, record, result, geometry, *, fps, video_id, frame_index=None, unit_assignments=None, component_relations=None):
    """Render a single frame; renderer never computes new passage decisions."""
    timeline = build_timeline(result, unit_assignments=unit_assignments, component_relations=component_relations)
    return _render(source_frame, record, timeline, geometry, fps=fps, video_id=video_id, frame_index=frame_index)

def render_video(video_path, records_path, result, geometry, output_path, *, fps, expected_frame_count, source_identity, component_relations=None, unit_assignments=None, progress=None):
    """Write H.264/yuv420p directly with ffmpeg, then decode-check every frame.

    Requires an explicit, installed libx264 encoder. No silent codec fallback,
    source resizing, skipped frames, model dependencies, or count recomputation.
    Source/cache SHA and every source pixel/raw-detection link are verified.
    """
    import cv2
    frozen._finite(fps, 'FPS')
    _require(fps > 0, 'FPS must be positive')
    frozen._integer(expected_frame_count, 'expected frame count', 1)
    _identity(source_identity)
    _require(result.get('source_identity') == source_identity, 'Renderer result/source identity differs')
    source = Path(video_path)
    records = Path(records_path)
    target = Path(output_path)
    _require(not target.exists(), 'Output already exists; preserve previous evidence')
    _require(target.resolve() not in (source.resolve(), records.resolve()), 'Output cannot replace an input')
    _require(target.suffix.lower() == '.mp4', 'Output must be an MP4')
    input_hashes = {'video_sha256': frozen.sha256_file(source), 'records_sha256': frozen.sha256_file(records)}
    for key, actual in input_hashes.items():
        _require(actual == source_identity[key], 'Input source SHA differs: ' + key)
    if geometry.get('source_video', {}).get('sha256') is not None:
        _require(geometry['source_video']['sha256'] == source_identity['video_sha256'], 'Geometry belongs to another video')
    timeline = build_timeline(result, frame_count=expected_frame_count, component_relations=component_relations, unit_assignments=unit_assignments)
    ffmpeg = shutil.which('ffmpeg')
    _require(ffmpeg is not None, 'ffmpeg with libx264 must be installed')
    target.parent.mkdir(parents=True, exist_ok=True)
    log_path = target.with_suffix('.ffmpeg.log')
    _require(not log_path.exists(), 'Codec log already exists; choose a fresh output')
    capture = cv2.VideoCapture(str(source))
    _require(capture.isOpened(), 'Source video decoder unavailable')
    started = time.perf_counter()
    process, count, dimension, final_state, command = (None, 0, None, None, None)
    try:
        with log_path.open('wb') as codec_log:
            for record in frozen._records(records):
                ok, image = capture.read()
                _require(ok and record.get('frame_index') == count, 'Source/cache frame alignment differs')
                canvas, final_state = _render(image, record, timeline, geometry, fps=fps, video_id=result.get('video_id', source.stem), frame_index=count)
                if process is None:
                    dimension = (canvas.shape[1], canvas.shape[0])
                    _require(all((value % 2 == 0 for value in dimension)), 'H264 output dimensions must be even')
                    command = [ffmpeg, '-v', 'error', '-nostdin', '-n', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s:v', f'{dimension[0]}x{dimension[1]}', '-r', format(fps, '.17g'), '-i', 'pipe:0', '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-threads', '2', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(target)]
                    process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=codec_log)
                _require(process.stdin is not None, 'Encoder pipe unavailable')
                process.stdin.write(canvas.tobytes())
                count += 1
                if progress and (count % 500 == 0 or count == expected_frame_count):
                    progress({'frames_rendered': count, 'expected_frame_count': expected_frame_count})
            more, _ = capture.read()
            _require(not more and count == expected_frame_count, 'Full source/cache frame count differs')
            _require(process is not None, 'No frames rendered')
            process.stdin.close()
            _require(process.wait() == 0, 'H264 encoding failed; inspect codec log')
    finally:
        capture.release()
        if process is not None and process.poll() is None:
            if process.stdin is not None and (not process.stdin.closed):
                process.stdin.close()
            process.terminate()
            process.wait()
    verify, decoded = (cv2.VideoCapture(str(target)), 0)
    try:
        _require(verify.isOpened(), 'H264 output cannot be decoded')
        while True:
            ok, image = verify.read()
            if not ok:
                break
            _require((image.shape[1], image.shape[0]) == dimension, 'Encoded dimensions changed')
            decoded += 1
    finally:
        verify.release()
    _require(decoded == count, 'Encoded full frame count differs')
    for key, path in (('video_sha256', source), ('records_sha256', records)):
        _require(frozen.sha256_file(path) == input_hashes[key], 'Source changed during rendering')
    return {'schema_version': VERSION, 'claim_type': 'ACTUAL_B_V3_LEDGER_RENDER_NOT_MODEL_INFERENCE', 'semantics': SEMANTICS, 'source_identity': deepcopy(source_identity), 'output_video': {'path': str(target), 'sha256': frozen.sha256_file(target), 'size_bytes': target.stat().st_size, 'codec': 'h264', 'pixel_format': 'yuv420p', 'frames_written': count, 'frames_redecoded': decoded, 'fps': fps, 'width': dimension[0], 'height': dimension[1], 'audio_preserved': False}, 'source_image_region': {'x': 0, 'y': 0, 'width': source_identity['width'], 'height': source_identity['height'], 'resized': False}, 'final_accepted': final_state['accepted_so_far'], 'final_pending': final_state['pending_final'], 'final_vehicle': final_state['vehicle_so_far'], 'final_pedestrian': final_state['pedestrian_so_far'], 'final_pivot': final_state['pivot'], 'route_counts': final_state['route_counts'], 'final_measured': final_state['measured_so_far'], 'final_inferred': final_state['inferred_so_far'], 'frame_pixel_and_raw_link_verified_every_frame': True, 'inputs_unchanged_after_render': True, 'new_detector_tracker_encoder_inference': False, 'counting_rerun_by_renderer': False, 'component_policy': 'SUPPORT_FRAME_PERSON_MOTORCYCLE_UNION_DISPLAY_ONLY_RAW_VEHICLE_BC_PRESERVED', 'display_id_policy': 'FINAL_OFFLINE_PER_PASSAGE_CATEGORY_AND_DIRECTION_WITH_EXACT_FRAME_UNIT_EPISODE_ROLE_ASSIGNMENTS', 'pending_excluded_from_pivot': True, 'ffmpeg_command': command, 'codec_log': {'path': str(log_path), 'sha256': frozen.sha256_file(log_path)}, 'opencv_version': cv2.__version__, 'software_source_sha256': frozen.sha256_file(__file__), 'render_and_decode_wall_seconds': time.perf_counter() - started}


### src/traffic_location/traffic/_kernel/render_kernel.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/render_kernel.py
"""B v6 render-only audit: exact observation roles and readable raw-ID callouts.

The B v3 log compiler is extended locally for BUS and invalid audit rows.
Old modules remain unchanged. Layout does not resolve
identity, propagate roles, create observations, or recompute counting. Native
source pixels are not resized; a fixed, separately reserved audit area carries
labels which cannot be placed without collision inside the image.
"""
from __future__ import annotations
from bisect import bisect_right
from collections import defaultdict
from copy import deepcopy
import gzip
import json
from pathlib import Path
import csv
import hashlib
import shutil
import subprocess
import time
from . import render_base as previous
frozen = previous.frozen
VERSION = 'phase8_b_v6_annotation_v1'
SEMANTICS = 'Offline final accepted ledger replay; cancelled/invalid source candidates are audit-only +0 at every frame. Measured/inferred remains source-time scoped.'
VEHICLE_CATEGORIES = ('motorcycle', 'car', 'truck', 'bus')
CATEGORIES = VEHICLE_CATEGORIES + ('pedestrian',)
CATEGORY_LETTERS = {'motorcycle': 'M', 'car': 'C', 'truck': 'T', 'bus': 'B', 'pedestrian': 'P'}
DIRECTIONS = previous.DIRECTIONS
CATEGORY_COLORS = dict(previous.CATEGORY_COLORS, bus=(200, 180, 45))
ROUTE_LABELS = dict(previous.ROUTE_LABELS)
_require = frozen._require
LABEL_SCALE = 0.45
LABEL_LINE_HEIGHT = 20
PANEL_WIDTH = 640

def invalid_decisions(result, frame_count=None):
    """One invalid row per passage, with offline source attribution, never +1.

    frame_index/decision_frame retain when the decision was available. The
    separately named offline_attribution_frame is where its retained source
    candidate became active, and must not be read as a causal online decision.
    """
    rows = result.get('invalid_decisions', [])
    _require(isinstance(rows, list), 'invalid_decisions must be an array')
    decisions = result.get('decisions', [])
    _require(isinstance(decisions, list), 'decisions must be an array')
    candidates = rows + [entry for entry in decisions if entry.get('status') == 'invalid']
    indexed = {}
    active = {(entry['physical_unit_id'], entry.get('episode_id'), entry['passage_id']) for entry in result.get('events', [])}
    for entry in candidates:
        _require(isinstance(entry, dict) and entry.get('status') == 'invalid', 'Invalid audit needs explicit invalid status')
        _require(entry.get('accepted_count_event') is not True and entry.get('count_delta', 0) == 0, 'Invalid passage must contribute +0')
        key = (entry.get('physical_unit_id'), entry.get('episode_id'), entry.get('passage_id'))
        _require(all((isinstance(value, str) and value for value in key)), 'Invalid passage needs unit/episode/passage scope')
        _require(key not in active, 'A final accepted passage cannot also be invalid')
        frame = frozen._integer(entry.get('frame_index'), 'invalid frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Invalid outside source frames')
        if key in indexed:
            _require(indexed[key] == entry, 'Conflicting invalid decisions for one passage')
        indexed[key] = deepcopy(entry)
    cancelled = {entry['event_id']: entry for entry in result.get('cancelled_events', [])}
    for entry in indexed.values():
        source_id = entry.get('cancelled_candidate_event_id', entry.get('source_event_id'))
        source = cancelled.get(source_id, {})
        attribution = source.get('activation_frame')
        if attribution is None:
            attribution = (entry.get('source_ref') or {}).get('frame_index')
        if attribution is None:
            attribution = entry['frame_index']
        frozen._integer(attribution, 'invalid offline attribution frame')
        if frame_count is not None:
            _require(attribution < frame_count, 'Invalid attribution outside source frames')
        entry['decision_frame'] = entry.get('decision_frame', entry['frame_index'])
        entry['offline_attribution_frame'] = attribution
        entry['attribution_semantics'] = 'OFFLINE_FINAL_REVIEW_NOT_CAUSAL'
    return sorted(indexed.values(), key=lambda entry: (entry['offline_attribution_frame'], entry['passage_id']))
_identity = previous._identity
_compile_assignments = previous._compile_assignments

def _display_id(value, category, direction):
    import re
    _require(isinstance(value, str) and re.fullmatch('[PMCTB][1-9][0-9]{3}', value), 'Final passage needs one category letter and four digits, without leading zero')
    _require(value[0] == CATEGORY_LETTERS[category], 'Display ID category differs')
    _require((int(value[1]) % 2 == 0) == (direction == 'A_TO_B'), 'Display ID direction parity differs')
    return value

def _compile_ledger_timeline(result, *, frame_count=None, component_relations=None, unit_assignments=None):
    """Validate one source ledger and compile constant-size counter snapshots.

    Changes in route/category revise the same final event. Candidates cancelled
    by the offline result never enter the pivot, even before their cancellation
    decision; their original operations remain explicitly void audit rows.
    Input objects are not modified.
    """
    identity = result.get('source_identity')
    _identity(identity)
    if frame_count is not None:
        frozen._integer(frame_count, 'frame_count', 1)
    events, cancelled = (result.get('events', []), result.get('cancelled_events', []))
    _require(isinstance(events, list) and isinstance(cancelled, list), 'Event arrays required')
    by_id, final_ids, final_display_ids = ({}, set(), set())
    for event in events + cancelled:
        _require(isinstance(event, dict), 'Event must be an object')
        event_id = frozen._identifier(event.get('event_id'), 'event ID')
        _require(event_id not in by_id, 'Duplicate event ID')
        _require(event.get('source_identity') == identity, 'Event belongs to another source')
        frozen._identifier(event.get('physical_unit_id'), 'physical unit')
        frozen._identifier(event.get('passage_id'), 'passage')
        frozen._integer(event.get('track_id'), 'vehicle track', 1)
        _require(event.get('category') in CATEGORIES, 'Unsupported product category')
        _require(event.get('direction') in DIRECTIONS, 'Invalid event direction')
        _require(event.get('route') in ROUTE_LABELS, 'Unknown event route')
        measured = event.get('evidence', {}).get('measured_crossing')
        _require(type(measured) is bool, 'Event requires explicit measured_crossing evidence')
        _require(event['route'] == 'LINE_MARGIN' or measured is False, 'Polygon/timeout cannot claim a measured line crossing')
        by_id[event_id] = deepcopy(event)
        if event in events:
            _require(event.get('accepted_count_event') is True, 'Final event must explicitly contribute to the operational count')
            _require(event.get('status') == 'accepted', 'Pending cannot enter accepted ledger')
            final_ids.add(event_id)
            display_id = _display_id(event.get('display_id'), event['category'], event['direction'])
            _require(display_id not in final_display_ids, 'Duplicate final display ID')
            final_display_ids.add(display_id)
    final_passages = [(by_id[key]['physical_unit_id'], by_id[key]['passage_id']) for key in final_ids]
    _require(len(final_passages) == len(set(final_passages)), 'Duplicate final physical passage')
    logs = result.get('passage_log', [])
    _require(isinstance(logs, list), 'passage_log must be an array')
    logs = sorted(enumerate(deepcopy(logs)), key=lambda pair: (pair[1].get('frame_index', -1), pair[0]))
    active, active_passages, snapshots, operations, displayed = ({}, {}, [], [], [])
    pivot = {category: {direction: 0 for direction in DIRECTIONS} for category in CATEGORIES}
    route_counts = {route: 0 for route in ROUTE_LABELS}

    def adjust(event, increment):
        pivot[event['category']][event['direction']] += increment
        route_counts[event['route']] += increment
    for _, log in logs:
        frame = frozen._integer(log.get('frame_index'), 'log frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Log outside source frames')
        delta = log.get('count_delta', 0)
        _require(type(delta) is int and delta in (-1, 0, 1), 'Invalid passage count_delta')
        event_id = log.get('event_id')
        status = log.get('status')
        operational = delta != 0 or status == 'upgraded'
        if not operational:
            continue
        _require(event_id in by_id, 'Count log has no retained event')
        payload = dict(by_id[event_id])
        for key in ('track_id', 'category', 'direction', 'route'):
            if key in log:
                payload[key] = log[key]
        _require(payload['category'] in CATEGORIES and payload['direction'] in DIRECTIONS and (payload['route'] in ROUTE_LABELS), 'Invalid logged passage semantics')
        measured = log.get('measured_crossing')
        _require(type(measured) is bool, 'Operational log requires explicit measured_crossing')
        _require(payload['route'] == 'LINE_MARGIN' or measured is False, 'Polygon/timeout log cannot claim a measured line crossing')
        payload['_measured_crossing'] = measured
        frozen._integer(payload['track_id'], 'log vehicle track', 1)
        passage = (payload['physical_unit_id'], payload['passage_id'])
        if event_id not in final_ids:
            pass
        elif delta == 1:
            _require(event_id not in active and passage not in active_passages, 'Log attempts to increment one passage twice')
            active[event_id] = payload
            active_passages[passage] = event_id
            adjust(payload, 1)
        elif delta == -1:
            _require(event_id in active, 'Cancellation has no active event')
            previous = active.pop(event_id)
            adjust(previous, -1)
            active_passages.pop((previous['physical_unit_id'], previous['passage_id']))
        else:
            _require(event_id in active, 'Upgrade has no previously accepted event')
            previous = active[event_id]
            _require((previous['physical_unit_id'], previous['passage_id']) == passage, 'Upgrade changed physical passage identity')
            adjust(previous, -1)
            active[event_id] = payload
            adjust(payload, 1)
        display = dict(log, track_id=payload['track_id'], category=payload['category'], direction=payload['direction'], route=payload['route'], display_id=by_id[event_id].get('display_id'), display_category=by_id[event_id]['category'] if event_id in final_ids else payload['category'], display_direction=by_id[event_id]['direction'] if event_id in final_ids else payload['direction'], requested_label=ROUTE_LABELS[payload['route']])
        if event_id not in final_ids:
            display.update(status='offline_void', original_status=status, original_count_delta=delta, count_delta=0, display_id=None, requested_label='VOID +0 / OFFLINE FINAL')
        displayed.append(display)
        operations.append(frame)
        measured = sum((item['_measured_crossing'] for item in active.values()))
        snapshots.append({'pivot': deepcopy(pivot), 'route_counts': dict(route_counts), 'accepted_so_far': len(active), 'measured_so_far': measured, 'inferred_so_far': len(active) - measured, 'vehicle_so_far': sum((sum(pivot[key].values()) for key in VEHICLE_CATEGORIES)), 'pedestrian_so_far': sum(pivot['pedestrian'].values())})
    _require(set(active) == final_ids, 'Event log does not reconcile with final accepted ledger')
    for event_id in final_ids:
        for key in ('category', 'direction', 'route', 'track_id'):
            _require(active[event_id][key] == by_id[event_id][key], 'Final log/ledger differs: ' + key)
        _require(active[event_id]['_measured_crossing'] == by_id[event_id]['evidence']['measured_crossing'], 'Final log/ledger measured evidence differs')
    pending = []
    decisions = result.get('decisions', result.get('machine_decisions', []))
    _require(isinstance(decisions, list), 'Decisions must be an array')
    for decision in decisions:
        if decision.get('status') not in ('pending', 'pending_review'):
            continue
        _require(decision.get('accepted_count_event') is not True, 'Pending decision cannot contribute')
        frame = decision.get('frame_index', decision.get('diagnostic', {}).get('frame_index'))
        frozen._integer(frame, 'pending frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Pending outside source frames')
        pending.append({'frame_index': frame, 'track_id': decision.get('track_id'), 'reasons': decision.get('reasons', decision.get('hard_block_reasons', []))})
    pending.sort(key=lambda item: item['frame_index'])
    relations = component_relations if component_relations is not None else result.get('component_links', [])
    links = defaultdict(list)
    for relation in relations:
        _require(relation.get('type') == 'COMPONENT_OF', 'Only explicit component relations may merge display boxes')
        vehicle = frozen._integer(relation.get('vehicle_track_id'), 'component parent', 1)
        component = frozen._integer(relation.get('component_track_id'), 'component track', 1)
        _require(vehicle != component, 'Component cannot be its own vehicle')
        frozen._identifier(relation.get('relation_id'), 'component relation ID')
        support = relation.get('support_frames')
        _require(isinstance(support, list) and bool(support), 'Component needs retained measured support frames')
        _require(len(support) == len(set(support)), 'Repeated component support frame')
        for frame in support:
            frozen._integer(frame, 'component support frame')
            if frame_count is not None:
                _require(frame < frame_count, 'Component support outside source frames')
            links[frame].append(deepcopy(relation))
    assignments = _compile_assignments(unit_assignments if unit_assignments is not None else result.get('frame_assignments', []), frame_count)
    display_by_unit = defaultdict(list)
    for event_id in sorted(final_ids):
        event = by_id[event_id]
        refs = event.get('evidence', {}).get('source_refs', [])
        source_frames = [ref.get('frame_index') for ref in refs if isinstance(ref, dict)]
        supplied = next((entry for entry in result.get('display_assignments', []) if entry.get('event_id') == event_id), {})
        source_frames += list(supplied.get('source_frames', []))
        source_frames = [frame for frame in source_frames if type(frame) is int and frame >= 0]
        if not source_frames:
            source_frames = [event.get('activation_frame')]
        start = min((frozen._integer(frame, 'display source frame') for frame in source_frames))
        episode = event.get('episode_id', supplied.get('episode_id'))
        if episode is None:
            continue
        frozen._identifier(episode, 'event episode')
        display_by_unit[event['physical_unit_id'], episode].append({'start_frame': start, 'display_id': event['display_id'], 'category': event['category'], 'passage_id': event['passage_id'], 'event_id': event_id})
    for candidates in display_by_unit.values():
        candidates.sort(key=lambda item: (item['start_frame'], item['event_id']))
        _require(len({entry['start_frame'] for entry in candidates}) == len(candidates), 'Ambiguous display passage start in one physical episode')
    return {'source_identity': deepcopy(identity), 'event_by_id': by_id, 'final_accepted': len(final_ids), 'final_pending': len(pending), 'operations': operations, 'snapshots': snapshots, 'display_logs': displayed, 'pending': pending, 'pending_frames': [item['frame_index'] for item in pending], 'component_links_by_frame': dict(links), 'frame_count': frame_count, 'unit_assignments_by_frame': assignments, 'display_by_unit': dict(display_by_unit), 'final_vehicle': sum((by_id[key]['category'] in VEHICLE_CATEGORIES for key in final_ids)), 'final_pedestrian': sum((by_id[key]['category'] == 'pedestrian' for key in final_ids)), 'semantics': SEMANTICS}

def _ledger_timeline_at(timeline, frame_index):
    frozen._integer(frame_index, 'frame index')
    if timeline['frame_count'] is not None:
        _require(frame_index < timeline['frame_count'], 'Frame outside source')
    position = bisect_right(timeline['operations'], frame_index)
    pending_n = bisect_right(timeline['pending_frames'], frame_index)
    state = deepcopy(timeline['snapshots'][position - 1]) if position else {'pivot': {category: {direction: 0 for direction in DIRECTIONS} for category in CATEGORIES}, 'route_counts': {route: 0 for route in ROUTE_LABELS}, 'accepted_so_far': 0, 'measured_so_far': 0, 'inferred_so_far': 0, 'vehicle_so_far': 0, 'pedestrian_so_far': 0}
    state.update(frame_index=frame_index, accepted_final=timeline['final_accepted'], pending_so_far=pending_n, pending_final=timeline['final_pending'], vehicle_final=timeline['final_vehicle'], pedestrian_final=timeline['final_pedestrian'], recent_logs=list(reversed(timeline['display_logs'][max(0, position - 4):position])), recent_pending=list(reversed(timeline['pending'][max(0, pending_n - 2):pending_n])))
    _require(sum((sum(row.values()) for row in state['pivot'].values())) == state['accepted_so_far'], 'Pivot/event mismatch')
    return state

def build_b_v6_timeline(result, geometry=None, frame_count=None, review_source_sha256=None, component_relations=None, unit_assignments=None):
    """Compile ledger counters and exact-frame assignments, without mutation."""
    timeline = _compile_ledger_timeline(result, frame_count=frame_count, component_relations=component_relations, unit_assignments=unit_assignments)
    if geometry is not None:
        identity = timeline['source_identity']
        previous._geometry_native(geometry, identity['width'], identity['height'])
    if review_source_sha256 is not None:
        _require(isinstance(review_source_sha256, str) and len(review_source_sha256) == 64 and all((c in '0123456789abcdef' for c in review_source_sha256)), 'review_source_sha256 must be a lowercase SHA256')
    audit = result.get('eligibility_audit', [])
    _require(isinstance(audit, list), 'eligibility_audit must be an array')
    indexed = []
    for position, entry in enumerate(audit):
        _require(isinstance(entry, dict), 'eligibility audit entry must be an object')
        frame = entry.get('frame_index', entry.get('decision_frame'))
        frozen._integer(frame, 'eligibility audit frame')
        if frame_count is not None:
            _require(frame < frame_count, 'Eligibility audit outside source')
        indexed.append((frame, position, deepcopy(entry)))
    indexed.sort(key=lambda item: (item[0], item[1]))
    timeline['eligibility_audit'] = [item[2] for item in indexed]
    timeline['eligibility_audit_frames'] = [item[0] for item in indexed]
    timeline['review_source_sha256'] = review_source_sha256
    event_logs = defaultdict(list)
    for entry in timeline['display_logs']:
        event_logs[entry.get('event_id')].append(entry)
    timeline['event_logs'] = dict(event_logs)
    timeline['layout_max_units'] = max((len(items) for items in timeline['unit_assignments_by_frame'].values()), default=0)
    invalid = invalid_decisions(result, frame_count)
    timeline['invalid_decisions'] = invalid
    timeline['invalid_frames'] = [entry['offline_attribution_frame'] for entry in invalid]
    invalid_by_unit = defaultdict(list)
    for entry in invalid:
        invalid_by_unit[entry['physical_unit_id'], entry['episode_id']].append(entry)
    timeline['invalid_by_unit'] = dict(invalid_by_unit)
    timeline['final_invalid'] = len(invalid)
    not_eligible = [deepcopy(entry) for entry in result.get('decisions', []) if entry.get('status') in ('resolved_not_eligible', 'not_eligible')]
    not_eligible.sort(key=lambda entry: entry.get('frame_index', -1))
    timeline['not_eligible_decisions'] = not_eligible
    timeline['not_eligible_frames'] = [entry['frame_index'] for entry in not_eligible]
    aggregate = result.get('aggregate', {})
    if aggregate:
        _require(aggregate.get('total') == timeline['final_accepted'], 'Aggregate accepted differs from event ledger')
        _require(aggregate.get('invalid', len(invalid)) == len(invalid), 'Aggregate invalid differs from audit ledger')
        for category in CATEGORIES:
            actual = sum((entry['category'] == category for entry in result.get('events', [])))
            _require(aggregate.get('by_category', {}).get(category, 0) == actual, 'Aggregate category differs from accepted ledger')
    timeline['b_v6_timeline'] = True
    return timeline

def timeline_at(timeline, frame_index):
    state = _ledger_timeline_at(timeline, frame_index)
    upto = bisect_right(timeline.get('eligibility_audit_frames', []), frame_index)
    state['recent_eligibility_audit'] = list(reversed(timeline.get('eligibility_audit', [])[max(0, upto - 3):upto]))
    invalid_upto = bisect_right(timeline.get('invalid_frames', []), frame_index)
    state.update(invalid_so_far=invalid_upto, invalid_final=timeline.get('final_invalid', 0), recent_invalid=list(reversed(timeline.get('invalid_decisions', [])[max(0, invalid_upto - 2):invalid_upto])), not_eligible_so_far=bisect_right(timeline.get('not_eligible_frames', []), frame_index), not_eligible_final=len(timeline.get('not_eligible_decisions', [])))
    return state

def _display_units(by_track, timeline, frame_index):
    """Union boxes only when the child is positively assigned at this frame.

    Even a previously supported component link cannot override an unknown or
    pedestrian assignment now. A supported exact-frame link or bounded reviewed
    assignment must identify the same physical parent. Parent absent => no union.
    """
    assignments = timeline['unit_assignments_by_frame'].get(frame_index, {})
    _require(set(assignments).issubset(by_track), 'Assignment has no measured emitted track at this frame')
    local_relations = timeline['component_links_by_frame'].get(frame_index, [])
    parents = defaultdict(set)
    for child, assignment in assignments.items():
        if assignment['role'] != 'rider':
            continue
        _require(by_track[child]['class_name'] == 'person', 'Rider requires a raw person box')
        parent_unit = assignment.get('component_of_unit_id')
        parent_episode = assignment.get('component_of_episode_id')
        direct_parent = assignment.get('component_of_track_id', assignment.get('parent_vehicle_track_id', assignment.get('vehicle_track_id')))
        supported = {relation['vehicle_track_id'] for relation in local_relations if relation['component_track_id'] == child}
        has_provenance = bool(assignment.get('review_refs') or assignment.get('role_provenance') or assignment.get('association_provenance') or assignment.get('evidence_refs'))
        for parent, parent_assignment in assignments.items():
            if parent == child or parent_assignment['role'] != 'vehicle':
                continue
            if by_track[parent]['class_name'] != 'motorcycle':
                continue
            same_parent = parent_unit is not None and parent_assignment['physical_unit_id'] == parent_unit
            if parent_episode is not None and parent_assignment['episode_id'] != parent_episode:
                continue
            direct = direct_parent == parent
            if parent_unit is not None and (not same_parent):
                continue
            if parent in supported and (parent_unit is None or same_parent):
                parents[child].add(parent)
            elif (same_parent or direct) and has_provenance:
                parents[child].add(parent)
    children = defaultdict(list)
    for child, options in parents.items():
        if len(options) == 1:
            children[next(iter(options))].append(child)
    suppressed = {child for members in children.values() for child in members}
    units = []
    for track, observation in sorted(by_track.items()):
        if track in suppressed:
            continue
        assignment = assignments.get(track, {})
        role = assignment.get('role', 'unknown_person' if observation['class_name'] == 'person' else 'vehicle')
        if role != 'vehicle':
            _require(observation['class_name'] == 'person', 'Person role requires raw person measurement')
        else:
            _require(observation['class_name'] != 'person', 'Raw person cannot silently become a vehicle')
        components = sorted(children.get(track, []))
        boxes = [observation['raw_bbox']] + [by_track[child]['raw_bbox'] for child in components]
        union = [min((box[0] for box in boxes)), min((box[1] for box in boxes)), max((box[2] for box in boxes)), max((box[3] for box in boxes))]
        candidates = timeline['display_by_unit'].get((assignment.get('physical_unit_id'), assignment.get('episode_id')), [])
        applicable = [candidate for candidate in candidates if candidate['start_frame'] <= frame_index]
        passage = applicable[-1] if applicable and role in ('vehicle', 'pedestrian') else {}
        event = timeline['event_by_id'].get(passage.get('event_id'), {})
        if passage:
            _require((passage['category'] == 'pedestrian') == (role == 'pedestrian'), 'Passage display category conflicts with observation role')
        active_logs = [entry for entry in timeline['event_logs'].get(passage.get('event_id'), []) if entry['frame_index'] <= frame_index]
        counted = sum((log.get('count_delta', 0) for log in active_logs)) > 0
        invalid_candidates = [entry for entry in timeline['invalid_by_unit'].get((assignment.get('physical_unit_id'), assignment.get('episode_id')), []) if entry['offline_attribution_frame'] <= frame_index]
        invalid = invalid_candidates[-1] if invalid_candidates else None
        if invalid is not None and (not passage or invalid['offline_attribution_frame'] >= passage['start_frame']):
            passage, event, active_logs, counted = ({}, {}, [], False)
        else:
            invalid = None
        category = passage.get('category') or ('pedestrian' if role == 'pedestrian' else observation['class_name'])
        bc = frozen._bc(observation['raw_bbox']) if role in ('vehicle', 'pedestrian') else None
        parent_tracks = sorted({parent for parent, entry in assignments.items() if entry.get('role') == 'vehicle' and entry.get('physical_unit_id') == assignment.get('component_of_unit_id') and (assignment.get('component_of_episode_id') is None or entry.get('episode_id') == assignment['component_of_episode_id'])})
        if not parent_tracks and assignment.get('component_of_track_id') in by_track:
            supplied_parent = assignment['component_of_track_id']
            if assignment.get('component_of_episode_id') is None or assignments.get(supplied_parent, {}).get('episode_id') == assignment['component_of_episode_id']:
                parent_tracks = [supplied_parent]
        if role == 'rider':
            status = 'RIDER +0'
        elif role in ('unknown', 'unknown_person'):
            status = 'ROLE UNKNOWN / NO COUNT'
        elif invalid:
            status = 'INVALID +0 / OFFLINE FINAL'
        elif passage and (not counted):
            status = 'OFFLINE ID / NOT COUNTED YET'
        elif counted:
            measured = active_logs[-1]['measured_crossing']
            status = 'COUNTED MEASURED' if measured else 'COUNTED ESTIMATE'
        else:
            status = 'NO ACCEPTED PASSAGE'
        units.append({'track_id': track, 'raw_track_ids': [track] + components, 'class_name': category, 'raw_class_name': observation['class_name'], 'role': role, 'physical_unit_id': assignment.get('physical_unit_id'), 'episode_id': assignment.get('episode_id'), 'display_id': passage.get('display_id'), 'passage_id': passage.get('passage_id'), 'event_id': passage.get('event_id'), 'event_activation_frame': event.get('activation_frame'), 'counted_at_frame': counted, 'display_bbox': union, 'raw_vehicle_bbox': list(observation['raw_bbox']), 'vehicle_bc': bc if role == 'vehicle' else None, 'counting_bc': bc, 'person_bc': bc if role == 'pedestrian' else None, 'component_track_ids': components, 'component_additional_increment': 0, 'parent_vehicle_track_ids': parent_tracks, 'status_label': status, 'invalid_decision': deepcopy(invalid), 'assignment_provenance': deepcopy(assignment)})
    return units

def _overlaps(a, b):
    return a[0] < b[2] and b[0] < a[2] and (a[1] < b[3]) and (b[1] < a[3])

def _label_lines(unit):
    raw = 'raw' + '+'.join(map(str, unit['raw_track_ids']))
    first = raw + ' | ' + unit['class_name'].upper()
    if unit['display_id']:
        first += ' | ' + unit['display_id']
    second = unit['status_label']
    if unit['component_track_ids']:
        second += ' | rider ' + ','.join(map(str, unit['component_track_ids'])) + ' +0'
    if unit['role'] == 'rider':
        second += ' -> raw' + ','.join(map(str, unit['parent_vehicle_track_ids'])) if unit['parent_vehicle_track_ids'] else ' | parent not observed'
    return [first, second]

def _text_size(lines):
    import cv2
    return (max((cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, LABEL_SCALE, 1)[0][0] for line in lines)) + 12, len(lines) * LABEL_LINE_HEIGHT + 6)

def _layout_labels(units, width, height, max_units):
    """Deterministic, fixed-font placement; collision => outside-source callout."""
    occupied, layouts, overflow = ([], [], 0)
    for item in units:
        lines = _label_lines(item)
        label_width, label_height = _text_size(lines)
        _require(label_width <= width - 8, 'Label exceeds source width; use a wider explicit audit canvas')
        x1, y1, x2, y2 = item['display_bbox']
        clamp_x = lambda x: max(4, min(int(x), width - label_width - 4))
        candidates = [(clamp_x(x1), int(y1) - label_height - 4), (clamp_x(x1), int(y2) + 4), (clamp_x(x2 + 4), int(y1)), (clamp_x(x1 - label_width - 4), int(y1))]
        candidates += [(4, y) for y in range(4, height - label_height - 3, label_height + 4)]
        rectangle = None
        for x, y in candidates:
            candidate = [x, y, x + label_width, y + label_height]
            if 0 <= y and candidate[2] <= width and (candidate[3] <= height) and (not any((_overlaps(candidate, prior) for prior in occupied))):
                rectangle = candidate
                break
        outside = rectangle is None
        if outside:
            y = height + 166 + overflow * (2 * LABEL_LINE_HEIGHT + 10)
            rectangle = [4, y, 4 + label_width, y + label_height]
            overflow += 1
            _require(overflow <= max_units, 'Reserved callout capacity exceeded')
        occupied.append(rectangle)
        layouts.append({'track_id': item['track_id'], 'raw_track_ids': item['raw_track_ids'], 'lines': lines, 'text': ' / '.join(lines), 'rectangle': rectangle, 'outside_source': outside, 'font_scale': LABEL_SCALE, 'connector_from': [min(width - 1, max(0, round((x1 + x2) / 2))), min(height - 1, max(0, round(y1)))], 'connector_to': [rectangle[0], rectangle[1] + label_height // 2]})
        layout = layouts[-1]
        layout['connector_polyline'] = [layout['connector_from'], [width - 2, layout['connector_from'][1]], [width - 2, layout['connector_to'][1]], layout['connector_to']] if outside else [layout['connector_from'], layout['connector_to']]
    return layouts

def _audit_lines(entry):
    track = entry.get('track_id', entry.get('raw_track_id', '?'))
    status = entry.get('status', entry.get('decision', entry.get('eligibility', 'AUDIT')))
    reason = entry.get('reasons', entry.get('reason', entry.get('reason_codes', [])))
    if isinstance(reason, list):
        reason = '; '.join(map(str, reason))
    frame = entry.get('frame_index', entry.get('decision_frame'))
    lines = [f'raw{track} | f{frame} | {status}', str(reason or 'Inspect retained eligibility evidence')]
    details = []
    for key in ('last_origin_frame', 'last_observed_origin_frame', 'timeout_frame', 'elapsed_source_frames', 'identity_conflict', 'identity_timeout_hold', 'episode_id'):
        if key in entry:
            details.append(f'{key}={entry[key]}')
    if details:
        lines.append(' | '.join(details))
    return lines

def _render(source_frame, record, timeline, geometry, *, fps, video_id, frame_index=None):
    import cv2
    import numpy as np
    frozen._finite(fps, 'FPS')
    _require(fps > 0, 'FPS must be positive')
    height, width = source_frame.shape[:2]
    identity = timeline['source_identity']
    _require((width, height) == (identity['width'], identity['height']), 'Native source dimensions differ')
    index = record.get('frame_index') if frame_index is None else frame_index
    by_track = frozen._validate_record(record, index, identity, source_frame)
    native, margin = previous._geometry_native(geometry, width, height)
    state = timeline_at(timeline, index)
    image = source_frame.copy()
    p1, p2 = (np.asarray(native['line']['P1'], float), np.asarray(native['line']['P2'], float))
    direction = (p2 - p1) / np.linalg.norm(p2 - p1)
    normal = np.asarray([-direction[1], direction[0]])
    band = np.asarray([p1 + normal * margin, p2 + normal * margin, p2 - normal * margin, p1 - normal * margin])
    overlay = image.copy()
    cv2.fillPoly(overlay, [np.rint(band).astype(np.int32)], (50, 205, 235))
    cv2.addWeighted(overlay, 0.18, image, 0.82, 0, image)
    cv2.polylines(image, [np.rint(band).astype(np.int32)], True, (90, 225, 250), 1, cv2.LINE_AA)
    cv2.line(image, tuple(np.rint(p1).astype(int)), tuple(np.rint(p2).astype(int)), (90, 225, 250), 2, cv2.LINE_AA)
    for key, color in (('side_a_polygon', (235, 155, 65)), ('side_b_polygon', (65, 155, 245))):
        cv2.polylines(image, [np.rint(native[key]).astype(np.int32)], True, color, 1, cv2.LINE_AA)
    units = _display_units(by_track, timeline, index)
    maximum = max(len(units), timeline.get('layout_max_units', 0), 1)
    layouts = _layout_labels(units, width, height, maximum)
    overflow_capacity = timeline.get('layout_callout_capacity', sum((item['outside_source'] for item in layouts)))
    _require(sum((item['outside_source'] for item in layouts)) <= overflow_capacity, 'Reserved callout capacity exceeded')
    canvas_height = max(1450, height + 174 + overflow_capacity * (2 * LABEL_LINE_HEIGHT + 10))
    canvas_height += canvas_height % 2
    canvas = np.full((canvas_height, width + PANEL_WIDTH, 3), (20, 25, 33), np.uint8)
    canvas[:height, :width] = image
    source_canvas = canvas[:height, :width]
    for item in units:
        box = item['display_bbox']
        color = CATEGORY_COLORS.get(item['class_name'], CATEGORY_COLORS['other'])
        cv2.rectangle(source_canvas, (round(box[0]), round(box[1])), (round(box[2]), round(box[3])), color, 2, cv2.LINE_AA)
        if item['counting_bc'] is not None:
            point = tuple((round(value) for value in item['counting_bc']))
            cv2.circle(source_canvas, point, 4, (10, 15, 20), -1, cv2.LINE_AA)
            cv2.circle(source_canvas, point, 3, color, -1, cv2.LINE_AA)
    for item, layout in zip(units, layouts):
        item['label'] = layout['text']
        color = CATEGORY_COLORS.get(item['class_name'], CATEGORY_COLORS['other'])
        cv2.polylines(canvas, [np.asarray(layout['connector_polyline'], dtype=np.int32)], False, color, 1, cv2.LINE_AA)
    for item, layout in zip(units, layouts):
        color = CATEGORY_COLORS.get(item['class_name'], CATEGORY_COLORS['other'])
        x1, y1, x2, y2 = layout['rectangle']
        cv2.rectangle(canvas, (x1, y1), (x2 - 1, y2 - 1), color, -1)
        for offset, line in enumerate(layout['lines']):
            frozen._text(canvas, line, (x1 + 6, y1 + 17 + offset * LABEL_LINE_HEIGHT), scale=LABEL_SCALE, color=(8, 12, 18))
    x, right = (width + 18, width + PANEL_WIDTH - 18)
    cv2.rectangle(canvas, (width, 0), (width + PANEL_WIDTH - 1, canvas_height - 1), (27, 34, 44), -1)
    panel_text = []

    def text(value, y, *, offset=0, scale=0.45, color=(225, 232, 239), thickness=1):
        value = str(value)
        _require(cv2.getTextSize(value, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)[0][0] <= right - x - offset, 'Panel text exceeds reserved width')
        panel_text.append({'text': value, 'x': x + offset, 'y': y})
        frozen._text(canvas, value, (x + offset, y), scale=scale, color=color, thickness=thickness)

    def wrapped(value, y, *, max_chars=76, scale=0.4, max_lines=None):
        lines = frozen._wrap(str(value), max_chars)
        if max_lines is not None:
            lines = lines[:max_lines]
        for line in lines:
            text(line, y, scale=scale)
            y += 18
        return y
    text('REVIEW COUNTING B v6', 28, scale=0.62, thickness=2)
    wrapped(video_id, 52)
    text('OFFLINE FINAL ID; COUNTERS FOLLOW LOG FRAME', 77, scale=0.42, color=(100, 205, 250))
    text('Event ledger only. Reviewed overlay not added.', 99, scale=0.4)
    cv2.line(canvas, (x, 111), (right, 111), (65, 76, 91), 1)
    text('PASSAGE ACCEPTED (OPERASIONAL)', 137, scale=0.49)
    text(f"{state['accepted_so_far']} / {state['accepted_final']}", 169, scale=0.8, thickness=2, color=(150, 235, 185))
    text(f"Measured {state['measured_so_far']} | inferred {state['inferred_so_far']}", 193)
    text(f"Vehicle {state['vehicle_so_far']} | pedestrian {state['pedestrian_so_far']}", 217)
    text('Kategori', 246)
    text('A > B', 246, offset=290)
    text('B > A', 246, offset=400)
    text('Total', 246, offset=515)
    for position, category in enumerate(CATEGORIES):
        y = 272 + position * 25
        text(category, y, color=CATEGORY_COLORS[category])
        row = state['pivot'][category]
        for offset, number in zip((310, 420, 533), [row[key] for key in DIRECTIONS] + [sum(row.values())]):
            text(number, y, offset=offset)
    text('A>B mendekati | B>A menjauhi kamera', 409, scale=0.42)
    text('LINE / POLYGON / TIMEOUT : ' + ' / '.join((str(state['route_counts'][key]) for key in ROUTE_LABELS)), 432, scale=0.42)
    text('Timeout = ESTIMATE, not proof of ID change.', 455, scale=0.4, color=(100, 205, 250))
    text('LOG KENDARAAN LEWAT', 491, scale=0.49, color=(150, 235, 185))
    text('Final ledger only; VOID candidates always +0 (offline).', 512, scale=0.4)
    if not state['recent_logs']:
        text('No accepted passage at this replay frame', 537)
    for position, log in enumerate(state['recent_logs']):
        y = 538 + position * 60
        name = log.get('display_id') or 'NO FINAL ID'
        text(f"raw{log['track_id']} | {name} | {log['display_category']} | {log['display_direction'].replace('_TO_', '>')}", y)
        text(f"{log['requested_label']} | {('MEASURED' if log['measured_crossing'] else 'INFERRED')}", y + 18, scale=0.4)
        text(f"f{log['frame_index']} | delta {log.get('count_delta', 0):+d}", y + 36, scale=0.4)
    cv2.line(canvas, (x, 776), (right, 776), (65, 76, 91), 1)
    text(f"INVALID AMBIGUITY +0 : {state['invalid_so_far']} / {state['invalid_final']}", 802, scale=0.49, color=(100, 160, 255))
    text('OFFLINE final exclusions; actual traffic is not zero.', 824, scale=0.4)
    for position, entry in enumerate(state['recent_invalid']):
        y = 850 + position * 47
        text(f"raw{entry['track_id']} | source f{entry['offline_attribution_frame']} | decision f{entry['decision_frame']} | +0", y, scale=0.4)
        wrapped(', '.join(map(str, entry.get('reasons', [entry.get('invalid_reason', 'INVALID_AMBIGUOUS_UNIT')]))), y + 18, max_lines=1)
    text(f"PENDING {state['pending_so_far']} / {state['pending_final']}", 972, scale=0.49, color=(100, 205, 250))
    text('Outside subtotal; pending keys != physical vehicles', 993, scale=0.4)
    for position, entry in enumerate(state['recent_pending']):
        y = 1019 + position * 55
        text(f"raw{entry['track_id']} | f{entry['frame_index']}", y)
        wrapped(', '.join(map(str, entry['reasons'])), y + 18, max_lines=1)
    text(f"NOT ELIGIBLE: {state['not_eligible_so_far']} / {state['not_eligible_final']}", 1140, scale=0.42)
    text('ELIGIBILITY / CLOCK / IDENTITY AUDIT', 1177, scale=0.46, color=(170, 205, 240))
    if not state['recent_eligibility_audit']:
        text('No eligibility audit entry at this frame', 1202, scale=0.4)
    for position, entry in enumerate(state['recent_eligibility_audit'][:2]):
        y = 1202 + position * 55
        lines = _audit_lines(entry)
        text(lines[0], y, scale=0.4)
        wrapped(lines[1], y + 18, max_lines=1)
        if len(lines) > 2:
            wrapped(lines[2], y + 36, max_lines=1)
    parent_links = []
    for item in units:
        parent_links += [{'component_track_id': child, 'vehicle_track_id': item['track_id'], 'frame_index': index, 'additional_increment': 0} for child in item['component_track_ids']]
        if item['role'] == 'rider' and len(item['parent_vehicle_track_ids']) == 1:
            parent_links.append({'component_track_id': item['track_id'], 'vehicle_track_id': item['parent_vehicle_track_ids'][0], 'frame_index': index, 'additional_increment': 0})
    text('RIDER PARENT / EXACT OBSERVATION FRAME', 1342, scale=0.42)
    for position, link in enumerate(parent_links[:2]):
        text(f"raw{link['component_track_id']} -> motorcycle raw{link['vehicle_track_id']} | +0 | f{index}", 1364 + position * 19, scale=0.4)
    if not parent_links:
        text('No positive parent link at this measured frame', 1364, scale=0.4)
    frozen._text(canvas, f'FRAME {index:06d} | t ~= {index / fps:.3f}s', (12, height + 28), scale=0.6, thickness=2)
    frozen._text(canvas, 'Native image, zero-based frame index / recorded FPS.', (12, height + 53), scale=0.43)
    frozen._text(canvas, 'BC from raw parent; union bbox is display-only. Rider +0.', (12, height + 78), scale=0.41)
    frozen._text(canvas, f'Finite band half-width {margin:g}px. A blue / B orange.', (12, height + 103), scale=0.42)
    frozen._text(canvas, 'OVERFLOW RAW-ID CALLOUTS (fixed font, no hidden labels)', (12, height + 140), scale=0.41)
    state.update(display_units=units, band_vertices_native=band.tolist(), label_layouts=layouts, panel_text=panel_text, panel_sections_y={'accepted': 137, 'passage_log': 491, 'invalid': 802, 'pending': 972, 'not_eligible': 1140, 'eligibility_audit': 1177}, label_font_scale=LABEL_SCALE, maximum_visible_units=maximum, role_parent_links=parent_links, source_image_region={'x': 0, 'y': 0, 'width': width, 'height': height, 'resized': False}, raw_ids_visible=sorted((track for item in layouts for track in item['raw_track_ids'])))
    _require(state['raw_ids_visible'] == sorted(by_track), 'A raw ID is missing or duplicated in visible labels')
    for layout in layouts:
        _require(0 <= layout['rectangle'][0] < layout['rectangle'][2] <= width and 0 <= layout['rectangle'][1] < layout['rectangle'][3] <= canvas_height, 'Label outside reserved canvas')
    return (canvas, state)

def render_b_v6_frame(source_frame, record, result_or_timeline, geometry, *, fps, video_id, frame_index=None, unit_assignments=None, component_relations=None, review_source_sha256=None):
    timeline = result_or_timeline if result_or_timeline.get('b_v6_timeline') else build_b_v6_timeline(result_or_timeline, geometry, review_source_sha256=review_source_sha256, unit_assignments=unit_assignments, component_relations=component_relations)
    return _render(source_frame, record, timeline, geometry, fps=fps, video_id=video_id, frame_index=frame_index)

def render_b_v6_video(video_path, records_path, result, geometry, output_path, *, fps, expected_frame_count, source_identity, component_relations=None, unit_assignments=None, progress=None, review_source_sha256=None):
    """Sequential source verification, H264 encode and full decode; no inference."""
    import cv2
    frozen._finite(fps, 'FPS')
    _require(fps > 0, 'FPS must be positive')
    frozen._integer(expected_frame_count, 'expected frame count', 1)
    previous._identity(source_identity)
    _require(result.get('source_identity') == source_identity, 'Renderer result/source identity differs')
    source, records, target = (Path(video_path), Path(records_path), Path(output_path))
    _require(not target.exists(), 'Output already exists; preserve previous evidence')
    _require(target.resolve() not in (source.resolve(), records.resolve()), 'Output cannot replace an input')
    _require(target.suffix.lower() == '.mp4', 'Output must be an MP4')
    hashes = {'video_sha256': frozen.sha256_file(source), 'records_sha256': frozen.sha256_file(records)}
    for key, actual in hashes.items():
        _require(actual == source_identity[key], 'Input source SHA differs: ' + key)
    if geometry.get('source_video', {}).get('sha256') is not None:
        _require(geometry['source_video']['sha256'] == hashes['video_sha256'], 'Geometry belongs to another video')
    timeline = build_b_v6_timeline(result, geometry, frame_count=expected_frame_count, component_relations=component_relations, unit_assignments=unit_assignments, review_source_sha256=review_source_sha256)
    maximum, overflow_capacity = (0, 0)
    for record in frozen._records(records):
        by_track = {item['track_id']: item for item in record.get('emitted_tracks', [])}
        units = _display_units(by_track, timeline, record['frame_index'])
        maximum = max(maximum, len(by_track))
        layouts = _layout_labels(units, source_identity['width'], source_identity['height'], max(len(by_track), 1))
        overflow_capacity = max(overflow_capacity, sum((item['outside_source'] for item in layouts)))
    timeline['layout_max_units'] = maximum
    timeline['layout_callout_capacity'] = overflow_capacity
    ffmpeg = shutil.which('ffmpeg')
    _require(ffmpeg is not None, 'ffmpeg with libx264 must be installed')
    target.parent.mkdir(parents=True, exist_ok=True)
    log_path, labels_path = (target.with_suffix('.ffmpeg.log'), target.with_suffix('.labels.jsonl.gz'))
    _require(not log_path.exists() and (not labels_path.exists()), 'Render evidence exists; choose a fresh output')
    capture = cv2.VideoCapture(str(source))
    _require(capture.isOpened(), 'Source video decoder unavailable')
    started = time.perf_counter()
    process, count, dimension, final_state, command = (None, 0, None, None, None)
    try:
        with log_path.open('wb') as codec_log, gzip.open(labels_path, 'wt', encoding='utf-8') as labels:
            for record in frozen._records(records):
                ok, image = capture.read()
                _require(ok and record.get('frame_index') == count, 'Source/cache frame alignment differs')
                canvas, final_state = _render(image, record, timeline, geometry, fps=fps, video_id=result.get('video_id', source.stem), frame_index=count)
                if process is None:
                    dimension = (canvas.shape[1], canvas.shape[0])
                    _require(all((value % 2 == 0 for value in dimension)), 'H264 dimensions must be even')
                    command = [ffmpeg, '-v', 'error', '-nostdin', '-n', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s:v', f'{dimension[0]}x{dimension[1]}', '-r', format(fps, '.17g'), '-i', 'pipe:0', '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-threads', '2', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(target)]
                    process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=codec_log)
                _require((canvas.shape[1], canvas.shape[0]) == dimension, 'Layout dimensions changed during video')
                process.stdin.write(canvas.tobytes())
                labels.write(json.dumps({'frame_index': count, 'raw_ids_visible': final_state['raw_ids_visible'], 'label_layouts': final_state['label_layouts'], 'accepted_so_far': final_state['accepted_so_far'], 'measured_so_far': final_state['measured_so_far'], 'inferred_so_far': final_state['inferred_so_far'], 'pending_so_far': final_state['pending_so_far'], 'invalid_so_far': final_state['invalid_so_far'], 'not_eligible_so_far': final_state['not_eligible_so_far'], 'role_parent_links': final_state['role_parent_links'], 'display_units': [{key: item.get(key) for key in ('track_id', 'raw_track_ids', 'role', 'physical_unit_id', 'episode_id', 'display_id', 'passage_id', 'event_id', 'event_activation_frame', 'counted_at_frame', 'component_track_ids', 'parent_vehicle_track_ids', 'status_label', 'raw_vehicle_bbox', 'counting_bc')} for item in final_state['display_units']]}, sort_keys=True) + '\n')
                count += 1
                if progress and (count % 500 == 0 or count == expected_frame_count):
                    progress({'frames_rendered': count, 'expected_frame_count': expected_frame_count})
            more, _ = capture.read()
            _require(not more and count == expected_frame_count, 'Full source/cache frame count differs')
            _require(process is not None, 'No frames rendered')
            process.stdin.close()
            _require(process.wait() == 0, 'H264 encoding failed; inspect codec log')
    finally:
        capture.release()
        if process is not None and process.poll() is None:
            if process.stdin is not None and (not process.stdin.closed):
                process.stdin.close()
            process.terminate()
            process.wait()
    decoder, decoded = (cv2.VideoCapture(str(target)), 0)
    try:
        _require(decoder.isOpened(), 'H264 output cannot be decoded')
        while True:
            ok, image = decoder.read()
            if not ok:
                break
            _require((image.shape[1], image.shape[0]) == dimension, 'Encoded dimensions changed')
            decoded += 1
    finally:
        decoder.release()
    _require(decoded == count, 'Encoded full frame count differs')
    for key, path in (('video_sha256', source), ('records_sha256', records)):
        _require(frozen.sha256_file(path) == hashes[key], 'Source changed during rendering')
    return {'schema_version': VERSION, 'claim_type': 'ACTUAL_B_V6_LEDGER_RENDER_NOT_MODEL_INFERENCE', 'semantics': SEMANTICS, 'source_identity': deepcopy(source_identity), 'output_video': {'path': str(target), 'sha256': frozen.sha256_file(target), 'size_bytes': target.stat().st_size, 'codec': 'h264', 'pixel_format': 'yuv420p', 'frames_written': count, 'frames_redecoded': decoded, 'fps': fps, 'width': dimension[0], 'height': dimension[1], 'audio_preserved': False}, 'source_image_region': final_state['source_image_region'], 'final_accepted': final_state['accepted_so_far'], 'final_pending': final_state['pending_final'], 'final_invalid': final_state['invalid_final'], 'final_not_eligible': final_state['not_eligible_final'], 'final_vehicle': final_state['vehicle_so_far'], 'final_pedestrian': final_state['pedestrian_so_far'], 'final_pivot': final_state['pivot'], 'route_counts': final_state['route_counts'], 'final_measured': final_state['measured_so_far'], 'final_inferred': final_state['inferred_so_far'], 'frame_pixel_and_raw_link_verified_every_frame': True, 'inputs_unchanged_after_render': True, 'new_detector_tracker_encoder_inference': False, 'counting_rerun_by_renderer': False, 'component_policy': 'EXACT_FRAME_POSITIVE_RIDER_PARENT_ASSIGNMENT_UNION_RAW_PARENT_BC_PRESERVED', 'display_id_policy': 'RAW_ID_FIRST_WITH_OFFLINE_PREACTIVATION_VS_COUNTED_STATUS', 'label_layout_policy': 'FIXED_FONT_COLLISION_FREE_OR_RESERVED_OUTSIDE_SOURCE_CALLOUT', 'maximum_visible_units': timeline['layout_max_units'], 'pending_excluded_from_pivot': True, 'invalid_excluded_from_pivot': True, 'categories': list(CATEGORIES), 'maximum_overflow_callouts': timeline['layout_callout_capacity'], 'layout_audit': {'path': str(labels_path), 'sha256': frozen.sha256_file(labels_path)}, 'review_source_sha256': review_source_sha256, 'ffmpeg_command': command, 'codec_log': {'path': str(log_path), 'sha256': frozen.sha256_file(log_path)}, 'opencv_version': cv2.__version__, 'software_source_sha256': frozen.sha256_file(__file__), 'render_and_decode_wall_seconds': time.perf_counter() - started}
build_timeline = build_b_v6_timeline
render_frame = render_b_v6_frame
render_video = render_b_v6_video

def export_audit(result, output_directory):
    """Export one immutable snapshot; ambiguity loss is visible and never +1.

    Classification/identity/counting are not performed here. BUS zero rows are
    schema coverage, not invented bus observations. Machine and reviewed input
    snapshots are exported independently, never added together.
    """
    output = Path(output_directory)
    _require(not output.exists(), 'Export output exists; choose a new snapshot directory')
    timeline = build_timeline(result)
    last = max(timeline['operations'] + timeline.get('invalid_frames', []) + timeline['pending_frames'] + [0])
    state = timeline_at(timeline, last)
    _require(state['accepted_so_far'] == len(result.get('events', [])), 'Accepted CSV/pivot do not reconcile')
    output.mkdir(parents=True)

    def write_csv(name, rows, fieldnames):
        path = output / name
        with path.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow({field: json.dumps(row.get(field), sort_keys=True, ensure_ascii=False) if isinstance(row.get(field), (dict, list, tuple)) else row.get(field) for field in fieldnames})
    identity = result['source_identity']
    shared = {'video_id': result.get('video_id'), 'branch': result.get('branch'), 'video_sha256': identity['video_sha256'], 'records_sha256': identity['records_sha256'], 'geometry_sha256': identity.get('geometry_sha256')}
    common = ['video_id', 'branch', 'video_sha256', 'records_sha256', 'geometry_sha256', 'status', 'count_delta', 'track_id', 'physical_unit_id', 'episode_id', 'passage_id']
    events = [{**shared, **entry, 'count_delta': 1, 'measured_crossing': entry['evidence']['measured_crossing'], 'source_refs': entry['evidence'].get('source_refs', [])} for entry in result.get('events', [])]
    write_csv('accepted_events.csv', events, common + ['event_id', 'display_id', 'category', 'direction', 'route', 'measured_crossing', 'activation_frame', 'decision_frame', 'source_refs'])
    invalid = [{**shared, **entry, 'count_delta': 0} for entry in timeline['invalid_decisions']]
    decision_fields = common + ['frame_index', 'decision_frame', 'offline_attribution_frame', 'attribution_semantics', 'invalid_reason', 'reasons', 'source_event_id', 'cancelled_candidate_event_id', 'source_ref', 'reason_evidence']
    write_csv('invalid_passages.csv', invalid, decision_fields)
    pending = [{**shared, **entry, 'count_delta': 0} for entry in result.get('decisions', []) if entry.get('status') in ('pending', 'pending_review')]
    write_csv('pending_passages.csv', pending, decision_fields)
    not_eligible = [{**shared, **entry, 'count_delta': 0} for entry in timeline['not_eligible_decisions']]
    write_csv('not_eligible_passages.csv', not_eligible, decision_fields + ['resolution_reason'])
    pivot_rows = []
    for category in CATEGORIES:
        for direction in DIRECTIONS:
            members = [entry for entry in result.get('events', []) if entry['category'] == category and entry['direction'] == direction]
            measured = sum((entry['evidence']['measured_crossing'] for entry in members))
            pivot_rows.append(dict(shared, category=category, direction=direction, accepted=len(members), measured=measured, inferred=len(members) - measured))
    write_csv('summary_pivot.csv', pivot_rows, list(shared) + ['category', 'direction', 'accepted', 'measured', 'inferred'])
    files = [{'path': path.name, 'bytes': path.stat().st_size, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()} for path in sorted(output.glob('*.csv'))]
    report = {'schema_version': 'phase8_b_v6_audit_export_v1', 'source_identity': deepcopy(identity), 'branch': result.get('branch'), 'accepted': len(events), 'invalid': len(invalid), 'pending': len(pending), 'not_eligible': len(not_eligible), 'measured': sum((entry['measured_crossing'] for entry in events)), 'inferred': sum((not entry['measured_crossing'] for entry in events)), 'categories': list(CATEGORIES), 'invalid_contribution': 0, 'pending_contribution': 0, 'deliberate_ambiguity_exclusions_are_not_zero_actual_traffic': True, 'new_counting_or_model_execution': False, 'machine_and_reviewed_not_added': True, 'files': files, 'manifest_self_excluded': True}
    (output / 'export_manifest.json').write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
    return report


### src/traffic_location/traffic/_kernel/render_sources.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/render_sources.py
"""Render-only Phase 8 audit video from immutable, selected result evidence.

This module does not detect, track, associate, classify, or count again. The
pivot replays the FINAL OFFLINE accepted-event ledger at retained geometry
confirmation frames. Category/identity evidence can occur later; this display
is explicitly not an online decision or latency measurement.
"""
from __future__ import annotations
from bisect import bisect_right
from collections import Counter, defaultdict
from copy import deepcopy
import gzip
import hashlib
import json
import math
from pathlib import Path
import time
VERSION = 'phase8_review_annotation_v1'
TIMELINE_SEMANTICS = 'OFFLINE_FINAL_SNAPSHOT_REPLAY_AT_GEOMETRY_CONFIRMATION_NOT_CAUSAL_DECISION_TIME'
DIRECTIONS = ('A_TO_B', 'B_TO_A')
CATEGORIES = ('motorcycle', 'car', 'truck', 'bus_angkot', 'pedestrian')
CATEGORY_COLORS = {'motorcycle': (95, 210, 250), 'car': (245, 175, 90), 'truck': (65, 125, 245), 'bus': (80, 190, 115), 'person': (210, 125, 245), 'other': (175, 180, 185)}
DISPLAY_CATEGORY = {'motorcycle': 'Motor', 'car': 'Mobil', 'truck': 'Truk', 'bus_angkot': 'Bus / angkot', 'pedestrian': 'Pejalan kaki'}

def _require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _integer(value, name, minimum=0):
    _require(isinstance(value, int) and (not isinstance(value, bool)) and (value >= minimum), name + ' must be an integer >= ' + str(minimum))
    return value

def _finite(value, name):
    _require(isinstance(value, (int, float)) and (not isinstance(value, bool)) and math.isfinite(value), name + ' must be finite')
    return value

def _identifier(value, name):
    _require(isinstance(value, str) and bool(value.strip()), name + ' must be nonempty')
    return value

def _frame(value, frame_count, name):
    _integer(value, name)
    _require(value < frame_count, name + ' outside source frames')
    return value

def _bbox(value):
    _require(isinstance(value, (list, tuple)) and len(value) == 4, 'raw_bbox needs four coordinates')
    box = [_finite(number, 'raw_bbox coordinate') for number in value]
    _require(box[0] < box[2] and box[1] < box[3], 'raw_bbox has no positive area')
    return box

def _bc(box):
    return [(box[0] + box[2]) / 2.0, box[3]]

def load_json(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as handle:
        return json.load(handle)

def iter_json_array(path):
    """Stream large retained relation arrays without loading rejected graphs."""
    opener = gzip.open if str(path).endswith('.gz') else open
    decoder = json.JSONDecoder()
    with opener(path, 'rt', encoding='utf-8') as handle:
        buffer, position, started, finished, need_value = ('', 0, False, False, True)
        while not finished:
            if position >= len(buffer) or not buffer[position:].strip():
                buffer = buffer[position:] + handle.read(65536)
                position = 0
                if not buffer.strip():
                    raise ValueError('Truncated JSON array')
            while position < len(buffer) and buffer[position].isspace():
                position += 1
            if not started:
                _require(buffer[position] == '[', 'Expected a retained JSON array')
                started = True
                position += 1
                continue
            if position >= len(buffer):
                continue
            char = buffer[position]
            if char == ']':
                position += 1
                finished = True
                break
            if not need_value:
                _require(char == ',', 'Missing JSON array separator')
                position += 1
                need_value = True
                while True:
                    while position < len(buffer) and buffer[position].isspace():
                        position += 1
                    if position < len(buffer):
                        _require(buffer[position] != ']', 'Trailing JSON array comma')
                        break
                    more = handle.read(65536)
                    _require(bool(more), 'Truncated JSON array')
                    buffer = buffer[position:] + more
                    position = 0
            while True:
                try:
                    value, end = decoder.raw_decode(buffer, position)
                    break
                except json.JSONDecodeError:
                    more = handle.read(65536)
                    _require(bool(more), 'Invalid or truncated JSON array item')
                    buffer = buffer[position:] + more
                    position = 0
            yield value
            position = end
            need_value = False
        _require(not (buffer[position:] + handle.read()).strip(), 'Trailing data after JSON array')

def _validated_geometry(event, identity, frame_count):
    diagnostic = event.get('geometry_evidence')
    _require(isinstance(diagnostic, dict) and diagnostic.get('type') == 'GEOMETRY_TRANSITION' and (diagnostic.get('accepted_count_event') is False), 'Accepted event requires retained measured geometry transition')
    frames = [diagnostic.get(key) for key in ('origin_frame', 'crossing_frame_start', 'crossing_frame_end', 'confirmation_frame')]
    for frame in frames:
        _frame(frame, frame_count, 'geometry frame')
    _require(frames == sorted(frames), 'Geometry frame order invalid')
    _require(event.get('direction') in DIRECTIONS and diagnostic.get('direction') == event['direction'], 'Geometry/event direction mismatch')
    _require(diagnostic.get('track_id') == event.get('track_id'), 'Geometry source track differs')
    support = event.get('measurement_support')
    _require(isinstance(support, list) and bool(support), 'Event lacks measured vehicle BC support')
    _require([item.get('frame_index') for item in support] == list(range(frames[0], frames[3] + 1)), 'Measured passage support must be contiguous origin through confirmation')
    for item in support:
        _require(item.get('track_id') == event['track_id'] and item.get('emitted') is True, 'Accepted passage must use emitted source vehicle track')
        _require(item.get('measurement_provenance') == 'retained_phase_8_detection', 'Expected actual retained Phase 8 measurement provenance')
        _integer(item.get('original_detection_index'), 'source detection index')
        _require(item.get('class_name') in ('motorcycle', 'car', 'truck', 'bus'), 'Person/other component cannot supply vehicle BC')
        _require(item.get('point') == _bc(_bbox(item.get('raw_bbox'))), 'Vehicle BC differs from raw measurement')
    for endpoint in ('origin_observation',):
        observation = diagnostic.get(endpoint)
        _require(isinstance(observation, dict), 'Geometry origin source observation missing')
        metadata = observation.get('metadata', {})
        for key in ('video_sha256', 'records_sha256', 'parent_run_id', 'variant'):
            _require(metadata.get(key) == identity.get(key), 'Geometry source mismatch: ' + key)
        _require(metadata.get('track_id') == event['track_id'] and observation.get('point') == _bc(_bbox(metadata.get('raw_bbox'))), 'Geometry origin BC/source mismatch')
    return frames

def build_review_timeline(events, decisions, relations, *, source_identity, frame_count, ledger_entries=None, video_id=None):
    """Validate a selected result snapshot and create an immutable display index.

    ``relations`` may be a generator: only IDs already used by accepted events
    are retained. Pending/exploratory component fields never create connectors.
    This is reconciliation and presentation, not a new identity/count resolver.
    """
    _integer(frame_count, 'frame_count', 1)
    _require(isinstance(events, list) and isinstance(decisions, list), 'Events/decisions must be arrays')
    for key in ('video_sha256', 'records_sha256', 'geometry_sha256', 'parent_run_id', 'variant'):
        _identifier(source_identity.get(key), 'source identity ' + key)
    _require(source_identity['variant'] == 'raw', 'This review renderer preserves raw Phase 8 source pixels')
    by_candidate, pending = ({}, [])
    for decision in decisions:
        candidate_id = _identifier(decision.get('candidate_id'), 'candidate ID')
        _require(candidate_id not in by_candidate, 'Duplicate candidate decision')
        by_candidate[candidate_id] = decision
        _integer(decision.get('track_id'), 'decision track', 1)
        diagnostic = decision.get('diagnostic')
        _require(isinstance(diagnostic, dict), 'Candidate diagnostic missing')
        frame = _frame(diagnostic.get('frame_index'), frame_count, 'candidate frame')
        if decision.get('status') == 'pending_review':
            _require(decision.get('accepted_count_event') is False, 'Pending candidate cannot contribute count')
            pending.append({'candidate_id': candidate_id, 'frame': frame, 'track_id': decision['track_id'], 'category': decision.get('category'), 'diagnostic_type': diagnostic.get('type'), 'reasons': list(dict.fromkeys(decision.get('hard_block_reasons', []) + decision.get('reasons', [])))})
    selected, event_ids, physical_passages, representative_candidates = ([], set(), set(), set())
    for event in events:
        _require(event.get('type') == 'CountEvent' and event.get('status') == 'accepted' and (event.get('accepted_count_event') is True), 'Only selected accepted CountEvents enter pivot')
        event_id = _identifier(event.get('event_id'), 'event ID')
        _require(event_id not in event_ids, 'Duplicate event ID')
        event_ids.add(event_id)
        unit = _identifier(event.get('physical_unit_id'), 'physical unit ID')
        passage = _identifier(event.get('passage_id'), 'passage ID')
        _require((unit, passage) not in physical_passages, 'Duplicate physical unit/passage')
        physical_passages.add((unit, passage))
        _require(event.get('source_identity') == source_identity, 'Event belongs to another source snapshot')
        _require(event.get('category') in ('motorcycle', 'car', 'truck'), 'Unsupported accepted category for frozen B v1 coverage')
        _require(event.get('is_causal_online_decision') is False, 'Offline result semantics must be explicit')
        decision = by_candidate.get(event.get('candidate_id'))
        _require(decision is not None and decision.get('status') == 'accepted' and (decision.get('accepted_count_event') is True), 'Event lacks accepted representative decision')
        for key in ('event_id', 'category', 'track_id', 'physical_unit_id', 'passage_id'):
            _require(decision.get(key) == event.get(key), 'Accepted event/decision mismatch: ' + key)
        _require(not decision.get('reasons') and (not decision.get('hard_block_reasons')), 'Accepted representative retains unresolved conflicts')
        _require(decision.get('diagnostic') == event.get('geometry_evidence') and decision.get('measurement_support') == event.get('measurement_support'), 'Event changed retained geometry/raw BC support')
        _require(decision.get('episode_id') == event.get('vehicle_measurement_episode_id'), 'Event/decision measurement episode mismatch')
        for key in ('component_episode_ids', 'vehicle_episode_ids', 'identity_relation_ids', 'passage_identity_relation_evidence'):
            _require(decision.get(key) == event.get(key), 'Event/decision association mismatch: ' + key)
        representative_candidates.add(event['candidate_id'])
        origin, crossing_start, crossing_end, confirmation = _validated_geometry(event, source_identity, frame_count)
        latest = _frame(event.get('category_decision_latest_evidence_frame'), frame_count, 'category evidence frame')
        _require(latest >= confirmation, 'Full-episode category evidence ends before confirmation')
        selected.append({'event': deepcopy(event), 'activation_frame': confirmation, 'origin_frame': origin, 'crossing_start': crossing_start, 'crossing_end': crossing_end, 'category_latest_evidence_frame': latest})
    _require({cid for cid, decision in by_candidate.items() if decision.get('accepted_count_event') is True} == representative_candidates, 'Representative decisions and accepted ledger differ')
    if ledger_entries is not None:
        _identifier(video_id, 'video ID')
        ledger = [entry for entry in ledger_entries if entry.get('video_id') == video_id]
        _require(len(ledger) == len(events), 'Selected video ledger event count differs')
        _require(len({item.get('source_event_id') for item in ledger}) == len(ledger), 'Repeated ledger event')
        by_event = {event['event_id']: event for event in events}
        kinds, runs = (set(), set())
        for entry in ledger:
            _require(entry.get('source_identity') == source_identity, 'Ledger source snapshot differs')
            _require(entry.get('source_event_id') in by_event and entry.get('event') == by_event[entry['source_event_id']], 'Ledger and selected event payload differ')
            _require(entry.get('category') == entry['event']['category'] and entry.get('direction') == entry['event']['direction'], 'Ledger category/direction mismatch')
            kinds.add(entry.get('snapshot_kind'))
            runs.add(entry.get('run_id'))
        _require(len(kinds) <= 1 and len(runs) <= 1 and (kinds <= {'machine', 'reviewed'}), 'Ledger mixes machine/reviewed or rerun snapshots')
    selected.sort(key=lambda item: (item['activation_frame'], item['event']['event_id']))
    units = {unit: f'U{index + 1:03d}' for index, unit in enumerate(sorted({item['event']['physical_unit_id'] for item in selected}))}
    wanted_relations = {rid for item in selected for rid in item['event'].get('identity_relation_ids', [])}
    relation_index = {}
    for relation in relations:
        rid = relation.get('relation_id')
        if rid in wanted_relations:
            _require(rid not in relation_index, 'Duplicate retained identity relation')
            relation_index[rid] = relation
    _require(set(relation_index) == wanted_relations, 'Accepted identity relation evidence missing')
    connections, measurements, crossing_frames, event_cards = (defaultdict(list), defaultdict(dict), defaultdict(list), [])
    for index, item in enumerate(selected, 1):
        event = item['event']
        item['event_alias'] = f'E{index:03d}'
        item['unit_alias'] = units[event['physical_unit_id']]
        local = {entry['relation_id']: entry for entry in event.get('passage_identity_relation_evidence', [])}
        component_track_ids, supporting_vehicle_ids = (set(), {event['track_id']})
        for rid in event.get('identity_relation_ids', []):
            relation, evidence = (relation_index[rid], local.get(rid))
            _require(relation.get('status') == 'SUPPORTED' and evidence is not None and (evidence.get('status') == 'SUPPORTED'), 'Accepted identity lacks supported local relation')
            kind = relation.get('type')
            _require(kind in ('COMPONENT_OF', 'SAME_VEHICLE') and evidence.get('type') == kind, 'Identity relation type mismatch')
            first, second = (relation.get('first_episode_id'), relation.get('second_episode_id'))
            vehicle_ids, component_ids = (event.get('vehicle_episode_ids', []), event.get('component_episode_ids', []))
            _require(second in vehicle_ids and first in (component_ids if kind == 'COMPONENT_OF' else vehicle_ids), 'Accepted relation points outside event member episodes')
            first_track = _integer(relation.get('first_track_id'), 'relation first track', 1)
            second_track = _integer(relation.get('second_track_id'), 'relation second track', 1)
            (component_track_ids if kind == 'COMPONENT_OF' else supporting_vehicle_ids).add(first_track)
            supporting_vehicle_ids.add(second_track)
            frames = evidence.get('features', {}).get('common_frame_indices')
            _require(isinstance(frames, list) and bool(frames) and (frames == sorted(set(frames))), 'Accepted local relation needs unique measured common frames')
            global_frames = set(relation.get('features', {}).get('common_frame_indices', []))
            _require(evidence.get('passage_support_window') == [item['origin_frame'], item['activation_frame']], 'Relation passage support window differs')
            for frame in frames:
                _frame(frame, frame_count, 'relation support frame')
                _require(item['origin_frame'] <= frame <= item['activation_frame'] and frame in global_frames, 'Relation support lies outside measured passage/global support')
                connections[frame].append({'first_track_id': first_track, 'second_track_id': second_track, 'type': kind, 'event_alias': item['event_alias'], 'unit_alias': item['unit_alias'], 'event_id': event['event_id'], 'relation_id': rid})
        for observation in event['measurement_support']:
            frame = observation['frame_index']
            key = (event['track_id'], observation['original_detection_index'])
            existing = measurements[frame].get(key)
            _require(existing is None or existing['observation'] == observation, 'Conflicting accepted raw source observation')
            measurements[frame][key] = {'observation': deepcopy(observation), 'event_alias': item['event_alias'], 'unit_alias': item['unit_alias']}
        for frame in range(item['crossing_start'], item['crossing_end'] + 1):
            crossing_frames[frame].append(item)
        event_cards.append({'event_alias': item['event_alias'], 'unit_alias': item['unit_alias'], 'event_id': event['event_id'], 'candidate_id': event['candidate_id'], 'physical_unit_id': event['physical_unit_id'], 'passage_id': event['passage_id'], 'category': event['category'], 'direction': event['direction'], 'source_vehicle_track_id': event['track_id'], 'supporting_vehicle_track_ids': sorted(supporting_vehicle_ids), 'component_track_ids': sorted(component_track_ids), 'activation_frame': item['activation_frame'], 'origin_frame': item['origin_frame'], 'crossing_start': item['crossing_start'], 'crossing_end': item['crossing_end'], 'category_latest_evidence_frame': item['category_latest_evidence_frame'], 'identity_latest_evidence_frame': event.get('identity_decision_latest_evidence_frame'), 'replay_is_causal': False, 'accepted_increment': 1, 'component_additional_increment': 0})
    pending.sort(key=lambda item: (item['frame'], item['candidate_id']))
    return {'semantics': TIMELINE_SEMANTICS, 'frame_count': frame_count, 'source_identity': deepcopy(source_identity), 'events': selected, 'event_cards': event_cards, 'activation_frames': [item['activation_frame'] for item in selected], 'pending': pending, 'pending_frames': [item['frame'] for item in pending], 'connections_by_frame': dict(connections), 'source_measurements_by_frame': dict(measurements), 'crossings_by_frame': dict(crossing_frames), 'final_accepted': len(selected), 'final_pending': len(pending), 'decision_status_counts': dict(Counter((decision.get('status') for decision in decisions)))}

def timeline_at(timeline, frame_index):
    """Pure display pivot: unique selected events only, never bbox/ID totals."""
    _frame(frame_index, timeline['frame_count'], 'replay frame')
    n = bisect_right(timeline['activation_frames'], frame_index)
    pending_n = bisect_right(timeline['pending_frames'], frame_index)
    pivot = {category: {direction: 0 if category in ('motorcycle', 'car', 'truck') else None for direction in DIRECTIONS} for category in CATEGORIES}
    for item in timeline['events'][:n]:
        event = item['event']
        pivot[event['category']][event['direction']] += 1
    _require(sum((value for row in pivot.values() for value in row.values() if value is not None)) == n, 'Pivot reconciliation failed')
    return {'frame_index': frame_index, 'accepted_so_far': n, 'accepted_final': timeline['final_accepted'], 'pending_so_far': pending_n, 'pending_final': timeline['final_pending'], 'pivot': pivot, 'recent_events': list(reversed(timeline['event_cards'][:n][-3:])), 'recent_pending': list(reversed(timeline['pending'][:pending_n][-2:])), 'connections': deepcopy(timeline['connections_by_frame'].get(frame_index, [])), 'crossings': timeline['crossings_by_frame'].get(frame_index, [])}

def _validate_record(record, frame, identity, image):
    _require(record.get('frame_index') == frame, 'Record/source frame alignment failed')
    for key in ('video_sha256', 'parent_run_id', 'variant', 'width', 'height'):
        _require(record.get(key) == identity.get(key), 'Source record identity mismatch: ' + key)
    donor = record.get('detector_input_frame_pixel_sha256')
    _require(isinstance(donor, str) and hashlib.sha256(image.tobytes()).hexdigest() == donor, 'Decoded source pixels differ from retained detector input; use compatible decoder')
    raw, observations = (record.get('raw_detections'), record.get('emitted_tracks'))
    _require(isinstance(raw, list) and isinstance(observations, list), 'Missing retained measured source arrays')
    by_track, seen_detection = ({}, set())
    for observation in observations:
        track = _integer(observation.get('track_id'), 'source track', 1)
        _require(track not in by_track, 'Duplicate emitted source track/frame')
        index = _integer(observation.get('original_detection_index'), 'source detection index')
        _require(index < len(raw) and index not in seen_detection, 'Broken or repeated emitted detection link')
        _require(observation.get('frame_index') == frame and observation.get('emitted') is True and (observation.get('measurement_provenance') == 'retained_phase_8_detection'), 'Only retained raw measured emitted observations may be drawn')
        box = _bbox(observation.get('raw_bbox'))
        detection = raw[index]
        _require(box == detection.get('bbox_xyxy_pixels') and all((observation.get(key) == detection.get(key) for key in ('class_name', 'class_id', 'confidence'))), 'Emitted bbox/class differs from linked raw detector measurement')
        by_track[track] = observation
        seen_detection.add(index)
    return by_track

def _text(image, text, position, *, scale=0.46, color=(225, 232, 239), thickness=1):
    import cv2
    cv2.putText(image, str(text), position, cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

def _wrap(text, max_chars):
    words, lines, line = (str(text).split(), [], '')
    for word in words:
        if len(line) + len(word) + 1 > max_chars and line:
            lines.append(line)
            line = ''
        line += (' ' if line else '') + word
    if line:
        lines.append(line)
    return lines

def _label(image, text, x, y, color, max_width):
    import cv2
    scale, thickness = (0.38, 1)
    (width, height), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    width = min(width + 8, max_width)
    x = max(0, min(int(x), image.shape[1] - width))
    y = max(height + 7, min(int(y), image.shape[0] - baseline - 2))
    cv2.rectangle(image, (x, y - height - 5), (x + width, y + baseline), color, -1)
    _text(image, text, (x + 4, y - 2), scale=scale, color=(15, 20, 25))

def render_review_frame(source_frame, record, timeline, geometry, *, fps, video_id, panel_width=520, footer_height=240):
    """Return an expanded audit canvas. Native road image is never resized."""
    import cv2
    import numpy as np
    _require(panel_width >= 500 and footer_height >= 240, 'Review layout needs readable panel/footer space')
    image = source_frame.copy()
    h, w = image.shape[:2]
    identity = timeline['source_identity']
    _require((w, h) == (identity['width'], identity['height']), 'Native source dimensions differ')
    frame_index = record.get('frame_index')
    by_track = _validate_record(record, frame_index, identity, source_frame)
    state = timeline_at(timeline, frame_index)
    linked_indices = {observation['original_detection_index'] for observation in record['emitted_tracks']}
    raw_only = [(index, detection) for index, detection in enumerate(record['raw_detections']) if index not in linked_indices]
    state['raw_only_detection_count'] = len(raw_only)
    for index, detection in raw_only:
        box = _bbox(detection.get('bbox_xyxy_pixels'))
        raw_class = _identifier(detection.get('class_name'), 'raw-only class')
        color = CATEGORY_COLORS.get(raw_class, CATEGORY_COLORS['other'])
        cv2.rectangle(image, (round(box[0]), round(box[1])), (round(box[2]), round(box[3])), color, 1, cv2.LINE_AA)
        _label(image, f'DET {index} | {raw_class} | no track +0', box[0], box[1] - 5, color, w)
    native_geometry = geometry.get('rendered_native_pixels', {})
    _require((native_geometry.get('width'), native_geometry.get('height')) == (w, h), 'Geometry native frame differs')
    for key, color, letter in (('side_a_polygon', (235, 155, 65), 'A'), ('side_b_polygon', (65, 155, 245), 'B')):
        polygon = np.array(native_geometry[key], dtype=float)
        cv2.polylines(image, [np.rint(polygon).astype(np.int32)], True, color, 1, cv2.LINE_AA)
        point = tuple(np.rint(polygon.mean(axis=0)).astype(int))
        _label(image, letter, point[0], point[1], color, w)
    line = native_geometry['line']
    p1, p2 = (tuple((round(v) for v in line[key])) for key in ('P1', 'P2'))
    cv2.line(image, p1, p2, (70, 235, 250), 2, cv2.LINE_AA)
    for connection in state['connections']:
        first = by_track.get(connection['first_track_id'])
        second = by_track.get(connection['second_track_id'])
        _require(first is not None and second is not None, 'Retained local relation lacks simultaneous measured source observations')
        if connection['type'] == 'COMPONENT_OF':
            _require(first['class_name'] == 'person' and second['class_name'] in ('motorcycle', 'car', 'truck', 'bus'), 'Component relation disagrees with current raw unit families')
        first_point = (round((first['raw_bbox'][0] + first['raw_bbox'][2]) / 2), round((first['raw_bbox'][1] + first['raw_bbox'][3]) / 2))
        second_point = (round((second['raw_bbox'][0] + second['raw_bbox'][2]) / 2), round((second['raw_bbox'][1] + second['raw_bbox'][3]) / 2))
        cv2.line(image, first_point, second_point, (160, 225, 180), 1, cv2.LINE_AA)
    measured = timeline['source_measurements_by_frame'].get(frame_index, {})
    for observation in record['emitted_tracks']:
        track, box, raw_class = (observation['track_id'], observation['raw_bbox'], observation['class_name'])
        color = CATEGORY_COLORS.get(raw_class, CATEGORY_COLORS['other'])
        cv2.rectangle(image, (round(box[0]), round(box[1])), (round(box[2]), round(box[3])), color, 2, cv2.LINE_AA)
        text = f'ID {track} | {raw_class}'
        _label(image, text, box[0], box[1] - 5, color, w)
        if raw_class in ('motorcycle', 'car', 'truck', 'bus'):
            point = tuple((round(v) for v in _bc(box)))
            cv2.circle(image, point, 4, (15, 20, 25), -1, cv2.LINE_AA)
            cv2.circle(image, point, 3, color, -1, cv2.LINE_AA)
        source = measured.get((track, observation['original_detection_index']))
        if source:
            _require(source['observation']['raw_bbox'] == observation['raw_bbox'], 'Accepted BC source differs from record')
            _label(image, source['unit_alias'] + ' / ' + source['event_alias'] + ' vehicle BC', box[0], box[3] + 15, (150, 225, 180), w)
        for connection in state['connections']:
            if connection['type'] == 'COMPONENT_OF' and track == connection['first_track_id']:
                _label(image, f"{connection['unit_alias']} component +0", box[0], box[3] + 15, (150, 225, 180), w)
    for crossing in state['crossings']:
        point = crossing['event']['geometry_evidence'].get('crossing_point')
        if point is not None:
            point = tuple((round(v) for v in point))
            cv2.drawMarker(image, point, (255, 255, 255), cv2.MARKER_CROSS, 16, 2, cv2.LINE_AA)
    canvas_height = max(h + footer_height, 680)
    canvas = np.full((canvas_height, w + panel_width, 3), (20, 25, 33), dtype=np.uint8)
    canvas[:h, :w] = image
    cv2.rectangle(canvas, (w, 0), (w + panel_width - 1, canvas_height - 1), (27, 34, 44), -1)
    x, right = (w + 20, w + panel_width - 20)
    _text(canvas, 'REVIEW PASSAGE KENDARAAN', (x, 29), scale=0.6, color=(245, 247, 250), thickness=2)
    _text(canvas, video_id, (x, 53), scale=0.49, color=(170, 192, 211))
    _text(canvas, 'OFFLINE FINAL SNAPSHOT', (x, 78), scale=0.48, color=(100, 205, 250), thickness=1)
    _text(canvas, 'Replay konfirmasi geometri; bukan keputusan online', (x, 98), scale=0.37)
    cv2.line(canvas, (x, 112), (right, 112), (65, 76, 91), 1)
    _text(canvas, 'PASSAGE ACCEPTED', (x, 138), scale=0.48, thickness=1)
    _text(canvas, f"{state['accepted_so_far']} / {state['accepted_final']}", (right - 120, 138), scale=0.7, color=(150, 235, 185), thickness=2)
    _text(canvas, 'replay / akhir video', (right - 130, 158), scale=0.34)
    _text(canvas, 'Kategori', (x, 188), scale=0.43, color=(170, 190, 210))
    _text(canvas, 'A > B', (x + 207, 188), scale=0.43, color=(170, 190, 210))
    _text(canvas, 'B > A', (x + 309, 188), scale=0.43, color=(170, 190, 210))
    _text(canvas, 'Total', (x + 411, 188), scale=0.43, color=(170, 190, 210))
    for index, category in enumerate(CATEGORIES):
        y = 218 + index * 29
        color = CATEGORY_COLORS.get(category, (150, 160, 175))
        cv2.rectangle(canvas, (x, y - 11), (x + 8, y - 3), color, -1)
        _text(canvas, DISPLAY_CATEGORY[category], (x + 17, y), scale=0.43)
        row = state['pivot'][category]
        supported = category in ('motorcycle', 'car', 'truck')
        values = [str(row[direction]) for direction in DIRECTIONS] + [str(sum(row.values()))] if supported else ['--'] * 3
        for offset, value in zip((228, 330, 424), values):
            _text(canvas, value, (x + offset, y), scale=0.49, color=(220, 234, 241), thickness=1)
    _text(canvas, 'A>B mendekati kamera | B>A menjauhi kamera', (x, 367), scale=0.36, color=(170, 192, 211))
    _text(canvas, '-- cakupan belum didukung; nol != tidak ada', (x, 387), scale=0.35, color=(170, 192, 211))
    cv2.line(canvas, (x, 400), (right, 400), (65, 76, 91), 1)
    _text(canvas, f"PENDING  {state['pending_so_far']} / {state['pending_final']}", (x, 425), scale=0.48, color=(100, 205, 250), thickness=1)
    _text(canvas, 'Kandidat, bukan kendaraan unik; di luar subtotal', (x, 446), scale=0.36)
    _text(canvas, 'ACCEPTED TERAKHIR / +1 PER EVENT', (x, 476), scale=0.4, color=(170, 192, 211))
    if not state['recent_events']:
        _text(canvas, 'Belum ada accepted pada posisi replay ini', (x, 504), scale=0.4)
    for index, event in enumerate(state['recent_events']):
        y = 500 + index * 55
        direction = event['direction'].replace('_TO_', '>')
        _text(canvas, f"{event['event_alias']}  {event['unit_alias']}  {event['category']}  {direction}  +1", (x, y), scale=0.42, color=(155, 230, 185))
        components = ','.join((str(track) for track in event['component_track_ids'])) or 'none'
        _text(canvas, f"BC ID {event['source_vehicle_track_id']} | component IDs {components}", (x, y + 17), scale=0.34)
        _text(canvas, f"confirm f{event['activation_frame']} | category evidence <= f{event['category_latest_evidence_frame']}", (x, y + 34), scale=0.33, color=(170, 192, 211))
    _text(canvas, f"FRAME {frame_index:06d} / {timeline['frame_count'] - 1:06d}", (16, h + 28), scale=0.62, color=(245, 247, 250), thickness=2)
    _text(canvas, f't ~= {frame_index / fps:8.3f}s  |  index / FPS; source PTS unknown', (16, h + 52), scale=0.43)
    _text(canvas, 'MEASURED BOX COLOR = CURRENT RAW CLASS', (16, h + 81), scale=0.43, color=(170, 192, 211))
    for index, label in enumerate(('motorcycle', 'car', 'truck', 'bus', 'person', 'other')):
        lx = 16 + index * 102
        cv2.rectangle(canvas, (lx, h + 95), (lx + 9, h + 104), CATEGORY_COLORS[label], -1)
        _text(canvas, label, (lx + 13, h + 105), scale=0.31)
    _text(canvas, 'BC dot = raw vehicle measurement  |  white cross = retained crossing', (16, h + 129), scale=0.37)
    _text(canvas, 'Komponen rider +0; kendaraan + komponen = SATU event accepted.', (16, h + 151), scale=0.36)
    _text(canvas, f'DET tanpa track: {len(raw_only)} (+0). ID pascaocclusion belum diperbaiki.', (16, h + 173), scale=0.35, color=(160, 180, 200))
    if state['recent_pending']:
        pending = state['recent_pending'][0]
        _text(canvas, f"LAST PENDING: ID {pending['track_id']} | f{pending['frame']} | {pending['diagnostic_type']}", (16, h + 198), scale=0.36, color=(100, 205, 250))
        reason = ', '.join(pending['reasons']) or 'Unresolved; inspect retained decision evidence'
        _text(canvas, _wrap(reason.replace('_', ' '), 85)[0], (16, h + 220), scale=0.34)
    return (canvas, state)

def _records(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as handle:
        for line in handle:
            _require(bool(line.strip()), 'Empty source record')
            yield json.loads(line)

def render_review_video(video_path, records_path, geometry_path, decisions_path, events_path, relations_path, output_dir, *, source_identity, fps, expected_frame_count, video_id, ledger_path, expected_input_sha256=None, preview_frames=(), progress=None):
    """Re-render all source frames, decode-check MP4, and retain an audit report.

    No model/runtime dependency is imported. Output must be a new empty folder;
    original videos, original annotated MP4s and scientific results stay intact.
    Optional expected_input_sha256 maps the input keys below to original donor
    retention hashes, adding explicit transport validation to source bindings.
    """
    import cv2
    start = time.perf_counter()
    _finite(fps, 'fps')
    _require(fps > 0, 'FPS must be positive')
    _integer(expected_frame_count, 'expected_frame_count', 1)
    _identifier(video_id, 'video ID')
    paths = {'video': Path(video_path), 'records': Path(records_path), 'geometry': Path(geometry_path), 'decisions': Path(decisions_path), 'events': Path(events_path), 'relations': Path(relations_path), 'ledger': Path(ledger_path)}
    inputs = {key: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size} for key, path in paths.items()}
    for key, field in (('video', 'video_sha256'), ('records', 'records_sha256'), ('geometry', 'geometry_sha256')):
        _require(inputs[key]['sha256'] == source_identity.get(field), 'Input/source SHA mismatch: ' + key)
    for key, expected in (expected_input_sha256 or {}).items():
        _require(key in inputs and inputs[key]['sha256'] == expected, 'Original donor retention SHA mismatch: ' + key)
    _require(expected_input_sha256 is not None and {'decisions', 'events', 'relations', 'ledger'} <= set(expected_input_sha256), 'Donor hashes required for decisions/events/relations/ledger')
    geometry = load_json(paths['geometry'])
    _require(geometry.get('source_video', {}).get('sha256') == source_identity['video_sha256'], 'Geometry belongs to another source video')
    _require(geometry.get('geometry_review_confirmed') is True and geometry.get('geometry_contract_accepted') is True, 'Geometry lacks initial trial review')
    events, decisions, ledger = (load_json(paths[key]) for key in ('events', 'decisions', 'ledger'))
    timeline = build_review_timeline(events, decisions, iter_json_array(paths['relations']), source_identity=source_identity, frame_count=expected_frame_count, ledger_entries=ledger, video_id=video_id)
    event_file_hash = inputs['events']['sha256']
    _require(all((entry.get('snapshot_events_sha256') == event_file_hash for entry in ledger if entry.get('video_id') == video_id)), 'Selected ledger snapshot event hash differs')
    target = Path(output_dir)
    _require(not target.exists() or not any(target.iterdir()), 'Choose a new empty output directory; original results are immutable')
    _require(all((target.resolve() != path.resolve().parent for path in paths.values())), 'Output cannot be an input directory')
    target.mkdir(parents=True, exist_ok=True)
    preview = {_frame(frame, expected_frame_count, 'preview frame') for frame in preview_frames}
    preview.update((0, expected_frame_count - 1))
    if timeline['event_cards']:
        first = timeline['event_cards'][0]
        preview.update((first['origin_frame'], first['crossing_start'], first['activation_frame']))
    preview.update(list(sorted(timeline['connections_by_frame']))[:1])
    capture = cv2.VideoCapture(str(video_path))
    _require(capture.isOpened(), 'Source video decoder unavailable')
    writer, count, dimension, preview_records = (None, 0, None, [])
    output_path = target / 'annotated_review.mp4'
    try:
        for record in _records(records_path):
            ok, frame = capture.read()
            _require(ok and record.get('frame_index') == count, 'Source/cache contiguous full-frame alignment failed')
            canvas, state = render_review_frame(frame, record, timeline, geometry, fps=fps, video_id=video_id)
            if writer is None:
                dimension = (canvas.shape[1], canvas.shape[0])
                _require(dimension[0] % 2 == 0 and dimension[1] % 2 == 0, 'MP4 canvas dimensions must be even')
                writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, dimension)
                _require(writer.isOpened(), 'MP4v writer unavailable; no silent codec substitution')
            writer.write(canvas)
            if count in preview:
                path = target / f'preview_frame_{count:09d}.png'
                _require(cv2.imwrite(str(path), canvas), 'Preview PNG writer failed')
                preview_records.append({'path': path.name, 'sha256': sha256_file(path), 'frame_index': count, 'accepted_so_far': state['accepted_so_far'], 'pending_so_far': state['pending_so_far']})
            count += 1
            if progress and (count % 500 == 0 or count == expected_frame_count):
                progress({'video_id': video_id, 'frames_rendered': count, 'expected_frame_count': expected_frame_count})
        more, _ = capture.read()
        _require(not more and count == expected_frame_count, 'Source/cache/manifest full-frame counts differ')
    finally:
        capture.release()
        if writer is not None:
            writer.release()
    _require(writer is not None, 'No source frames rendered')
    verify, decoded = (cv2.VideoCapture(str(output_path)), 0)
    try:
        _require(verify.isOpened(), 'Written review video cannot be reopened')
        while True:
            ok, image = verify.read()
            if not ok:
                break
            _require((image.shape[1], image.shape[0]) == dimension, 'Written canvas dimensions changed')
            decoded += 1
    finally:
        verify.release()
    _require(decoded == count, 'Review MP4 re-decoded frame count differs')
    for key, path in paths.items():
        _require(sha256_file(path) == inputs[key]['sha256'], 'Input changed during rendering: ' + key)
    final = timeline_at(timeline, expected_frame_count - 1)
    audit_path = target / 'passage_display_index.json'
    audit_path.write_text(json.dumps({'schema_version': VERSION, 'semantics': TIMELINE_SEMANTICS, 'events': timeline['event_cards'], 'pending_candidates': timeline['pending'], 'final_pivot': final['pivot'], 'final_accepted': timeline['final_accepted'], 'final_pending': timeline['final_pending']}, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    report = {'schema_version': VERSION, 'claim_type': 'ACTUAL_RENDER_ONLY_RESULT_REPLAY_NOT_NEW_MODEL_OR_COUNTING_RUN', 'video_id': video_id, 'timeline_semantics': TIMELINE_SEMANTICS, 'source_identity': deepcopy(source_identity), 'inputs': inputs, 'source_image_region': {'x': 0, 'y': 0, 'width': source_identity['width'], 'height': source_identity['height'], 'resized': False, 'coordinate_transform': 'IDENTITY_NATIVE_TOP_LEFT'}, 'canvas': {'width': dimension[0], 'height': dimension[1], 'sidebar_width': 520, 'footer_minimum_height': 240}, 'output_video': {'path': output_path.name, 'sha256': sha256_file(output_path), 'size_bytes': output_path.stat().st_size, 'codec': 'mp4v', 'frames_written': count, 'frames_redecoded': decoded, 'fps': fps, 'source_audio_preserved': False, 'timestamp_semantics': 'frame_index/FPS; source PTS unknown'}, 'final_accepted': timeline['final_accepted'], 'final_pending_candidates': timeline['final_pending'], 'final_pivot': final['pivot'], 'decision_status_counts': timeline['decision_status_counts'], 'category_coverage': {'motorcycle': 'experimental_vehicle_passage', 'car': 'experimental_vehicle_passage', 'truck': 'experimental_vehicle_passage', 'bus_angkot': 'BUS_ANGKOT_MAPPING_OPEN', 'pedestrian': 'NOT_IMPLEMENTED_BY_FROZEN_B_V1'}, 'source_pixel_hash_verified_every_frame': True, 'inputs_unchanged_after_render': True, 'new_detector_tracker_encoder_inference': False, 'counting_or_identity_resolver_rerun': False, 'occlusion_identity_recovery_modified': False, 'category_color_basis': 'CURRENT_RAW_MEASURED_CLASS', 'component_connector_policy': 'ACCEPTED_EVENT_SUPPORTED_PASSAGE_LOCAL_MEASURED_COMMON_FRAMES_ONLY', 'accepted_increment_policy': 'ONE_UNIQUE_EVENT_PER_PHYSICAL_UNIT_PASSAGE; COMPONENT_ADDITIONAL_INCREMENT_ZERO', 'pending_excluded_from_pivot': True, 'previews': preview_records, 'passage_display_index': {'path': audit_path.name, 'sha256': sha256_file(audit_path)}, 'opencv_version': cv2.__version__, 'render_and_decode_wall_seconds': time.perf_counter() - start, 'software_source_sha256': sha256_file(__file__)}
    (target / 'annotation_report.json').write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    return report


### src/traffic_location/traffic/_kernel/review_finalizer.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/review_finalizer.py
"""Exact reviewed pending closures; no passage creation or history relabeling.

Positive component review can close the recorded candidate with +0. It does
not label a whole tracker history, assign a parent identity, repair geometry,
or establish that later observations of the same raw ID are components.
"""
from __future__ import annotations
from collections import Counter
from copy import deepcopy
import hashlib
import json
from pathlib import Path
SCHEMA = 'phase9_b_v8_pending_adjudications_v1'
REASON = 'REVIEWED_NON_INDEPENDENT_COMPONENT'
STATUS = 'resolved_not_eligible'
ROLES = frozenset(('RIDER_PASSENGER', 'VEHICLE_OCCUPANT', 'CARGO'))
ORIGINS = frozenset(('AI_PIXELS_REVIEW', 'USER_REVIEW'))
SCOPE_FIELDS = ('track_id', 'physical_unit_id', 'episode_id', 'passage_id')
SOURCE_KEYS = ('video_sha256', 'records_sha256', 'geometry_sha256')

def _require(condition, message):
    if not condition:
        raise ValueError(message)

def _digest(value):
    data = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
    return hashlib.sha256(data.encode()).hexdigest()

def _sha(value):
    return isinstance(value, str) and len(value) == 64 and all((character in '0123456789abcdef' for character in value))

def _scope(row):
    return tuple((row.get(field) for field in SCOPE_FIELDS))

def _pending_index(result):
    rows = [row for row in result.get('decisions', []) if row.get('status') == 'pending']
    index = {_scope(row): row for row in rows}
    _require(len(index) == len(rows), 'Repeated pending candidate scope')
    return index

def _verify_artifacts(artifacts, root):
    _require(isinstance(artifacts, list) and bool(artifacts), 'Positive review needs retained proof artifacts')
    paths, has_pixels = (set(), False)
    for item in artifacts:
        relative = item.get('path')
        _require(isinstance(relative, str) and bool(relative) and (not Path(relative).is_absolute()), 'Proof artifact must use a repository-relative path')
        path = (root / relative).resolve()
        _require(path.is_relative_to(root) and path.is_file(), 'Proof artifact is missing or outside repository')
        _require(relative not in paths and _sha(item.get('sha256')), 'Proof artifacts need unique paths and SHA256')
        paths.add(relative)
        _require(hashlib.sha256(path.read_bytes()).hexdigest() == item['sha256'], 'Proof artifact SHA256 differs')
        has_pixels |= path.suffix.lower() in ('.jpg', '.jpeg', '.png')
    _require(has_pixels, 'A source pixel artifact is required for positive component review')

def _validate_review(review, baseline, source_identity, root):
    _require(isinstance(review, dict), 'Adjudication must be an object')
    _require(isinstance(review.get('adjudication_id'), str) and review['adjudication_id'], 'Stable adjudication ID required')
    scope = review.get('baseline_candidate_scope')
    _require(isinstance(scope, dict) and set(scope) == set(SCOPE_FIELDS), 'Exact baseline candidate scope required')
    key = _scope(scope)
    _require(key in baseline, 'Review does not target a baseline pending candidate')
    row = baseline[key]
    ref = review.get('baseline_source_ref')
    _require(ref == row.get('source_ref'), 'Review witness differs from baseline candidate measurement')
    _require(isinstance(ref, dict) and ref.get('observed') is True and (ref.get('emitted') is True) and (len(ref.get('raw_bbox', [])) == 4), 'Review needs an actual emitted measured source witness')
    _require(ref.get('track_id') == scope['track_id'], 'Review witness belongs to another raw track')
    frame = ref.get('frame_index')
    _require(type(frame) is int and type(ref.get('original_detection_index')) is int, 'Review needs exact source frame and detection index')
    point, box = (ref.get('point'), ref['raw_bbox'])
    _require(point == [(box[0] + box[2]) / 2, box[3]], 'Review source point differs from measured raw BC')
    role = review.get('reviewed_role')
    _require(role in ROLES, 'Only positive non-independent component roles may close +0')
    _require(review.get('reviewer_origin') in ORIGINS, 'Explicit reviewed decision origin required')
    _require(review.get('final_status', STATUS) == STATUS and review.get('reason', REASON) == REASON, 'This finalizer cannot promote accepted, invent invalid, or change disposition semantics')
    _require(review.get('previous_status', 'pending') == 'pending', 'Review previous state must be pending')
    if role in ('RIDER_PASSENGER', 'VEHICLE_OCCUPANT'):
        _require(ref.get('class_name') == 'person', 'Person-component review cannot relabel a vehicle measurement')
    else:
        parents = review.get('related_vehicle_raw_ids', [])
        _require(isinstance(parents, list) and bool(parents) and all((type(t) is int and t != ref['track_id'] for t in parents)), 'Cargo exclusion requires positively reviewed related vehicle context')
    scope = review.get('role_scope')
    _require(isinstance(scope, dict) and type(scope.get('start_frame')) is int and (type(scope.get('end_frame')) is int) and (scope['start_frame'] <= frame <= scope['end_frame']), "Role proof must contain the candidate's own source frame")
    positive = review.get('positive_source_frames', [])
    _require(isinstance(positive, list) and frame in positive and all((type(f) is int and scope['start_frame'] <= f <= scope['end_frame'] for f in positive)), 'Positive review source frames do not support the exact candidate witness')
    for source_key in SOURCE_KEYS[:2]:
        review_key = 'source_video_sha256' if source_key == 'video_sha256' else 'records_sha256'
        _require(review.get(review_key) == source_identity[source_key], 'Review proof belongs to another video/cache')
    artifacts = review.get('proof_artifacts', [])
    _verify_artifacts(artifacts, root)
    if review['reviewer_origin'] == 'USER_REVIEW':
        authority = review.get('reviewed_decision_authority', {})
        _require(authority.get('kind') == 'EXPLICIT_USER_INSTRUCTION' and any((item.get('path') == authority.get('path') and item.get('sha256') == authority.get('sha256') for item in artifacts)), 'USER_REVIEW must retain explicit user instruction authority')
        path = root / authority['path']
        try:
            authority_text = path.read_text()
        except UnicodeError as error:
            raise ValueError('User instruction authority must contain retained text') from error
        excerpt = authority.get('exact_supporting_excerpt')
        _require(isinstance(excerpt, str) and bool(excerpt) and (excerpt in authority_text), 'User review lacks its exact authority excerpt')
    return key

def finalize_reviewed_pending(result, adjudications, *, baseline_result, source_identity, finalization_frame=None, repo_root=None):
    """Close only exact source-reviewed pending candidates with existing +0 state.

    The caller must bind ``baseline_result_sha256`` to the actual retained
    baseline file before loading it; this function verifies source identity,
    candidate records, and the retained proof bytes. It never creates events.
    """
    root = Path(repo_root or Path(__file__).resolve().parent.parent).resolve()
    _require(isinstance(adjudications, dict) and adjudications.get('schema_version') == SCHEMA, 'Explicit supported P1 pending adjudication manifest required')
    _require(_sha(adjudications.get('baseline_result_sha256')), 'Baseline file SHA256 binding required')
    for key in SOURCE_KEYS:
        _require(_sha(source_identity.get(key)) and adjudications.get('source_identity', {}).get(key) == source_identity[key] and (baseline_result.get('source_identity', {}).get(key) == source_identity[key]) and (result.get('source_identity', result.get('report', {}).get('source_identity', {})).get(key) == source_identity[key]), 'Baseline, candidate, review and input source identities must match')
    last = result['report']['last_frame_index']
    frame = last if finalization_frame is None else finalization_frame
    _require(type(frame) is int and frame == last, 'Reviewed offline decision must use actual source EOF')
    reviews = adjudications.get('adjudications')
    _require(isinstance(reviews, list), 'Adjudications must be an explicit list')
    baseline = _pending_index(baseline_result)
    candidate = _pending_index(result)
    validated, ids, scopes = ([], set(), set())
    for review in reviews:
        key = _validate_review(review, baseline, source_identity, root)
        _require(review['adjudication_id'] not in ids and key not in scopes, 'Duplicate or conflicting pending adjudication')
        ids.add(review['adjudication_id'])
        scopes.add(key)
        _require(key in candidate, 'Baseline candidate is no longer pending in this replay')
        _require(candidate[key].get('source_ref') == baseline[key].get('source_ref') and candidate[key].get('role') == baseline[key].get('role'), 'Reviewed replay changed candidate source/context; explicit new review required')
        validated.append((key, review))
    output = deepcopy(result)
    pending = _pending_index(output)
    audits = []
    for key, review in validated:
        row = pending[key]
        original = deepcopy(row)
        source = deepcopy(row['source_ref'])
        origin = 'AI_ASSISTED_SOURCE_REVIEW' if review['reviewer_origin'] == 'AI_PIXELS_REVIEW' else 'EXPLICIT_USER_SOURCE_REVIEW'
        row.update(status=STATUS, count_delta=0, accepted_count_event=False, display_id=None, candidate_decision_frame=original.get('frame_index'), decision_frame=frame, resolved_at_frame=frame, resolution_reason=REASON, reasons=[REASON], previous_status='pending', previous_reasons=deepcopy(original.get('reasons', [])), previous_reason_evidence=deepcopy(original.get('reason_evidence', [])), reviewed_role=review['reviewed_role'], adjudication_id=review['adjudication_id'], decision_origin=origin, finalization_mode='OFFLINE_REVIEWED_PLUS_ZERO', role_assignment_effect='EXACT_PENDING_SOURCE_SAMPLE_ONLY_NO_HISTORY_RELABEL_OR_PARENT_ALIAS', reason_evidence=[{'reason': REASON, 'source_ref': source, 'adjudication_id': review['adjudication_id'], 'proof_artifacts': deepcopy(review['proof_artifacts']), 'role_scope': deepcopy(review['role_scope'])}])
        output['passage_log'].append({'frame_index': frame, 'candidate_decision_frame': original.get('frame_index'), **{name: deepcopy(row.get(name)) for name in SCOPE_FIELDS}, 'status': STATUS, 'count_delta': 0, 'event_id': None, 'source_ref': source, 'message': REASON, 'adjudication_id': review['adjudication_id'], 'decision_origin': origin, 'reviewed_role': review['reviewed_role']})
        for diagnostic in output.get('open_passage_diagnostics', []):
            if _scope(diagnostic) == key:
                diagnostic.update(previous_disposition=diagnostic.get('disposition'), disposition=REASON, timeout_eligible=False, reviewed_pending_adjudication_id=review['adjudication_id'])
        audits.append({'adjudication_id': review['adjudication_id'], 'baseline_candidate_scope': deepcopy(review['baseline_candidate_scope']), 'baseline_source_ref': source, 'previous_decision': original, 'final_status': STATUS, 'reason': REASON, 'reviewed_role': review['reviewed_role'], 'candidate_decision_frame': original.get('frame_index'), 'decision_frame': frame, 'count_delta': 0, 'decision_origin': origin, 'review': deepcopy(review), 'crossing_evidence_changed': False, 'physical_identity_merged': False, 'not_a_recovered_physical_passage': True})
    output['aggregate']['pending'] = len(_pending_index(output))
    _require(output['events'] == result['events'] and output.get('cancelled_events') == result.get('cancelled_events') and (output.get('invalid_decisions') == result.get('invalid_decisions')), 'Reviewed pending closure must not change accepted or invalid passage decisions')
    _require(sum((item.get('count_delta', 0) for item in output['passage_log'])) == output['aggregate']['total'], 'Published log no longer reconciles')
    output['v8_pending_finalization_audit'] = audits
    if 'open_passage_diagnostic_counts' in output['report']:
        output['report']['open_passage_diagnostic_counts'] = dict(sorted(Counter((row['disposition'] for row in output.get('open_passage_diagnostics', []))).items()))
    output['report']['v8_reviewed_pending_finalization'] = {'baseline_pending': len(baseline), 'candidate_pending_before': len(candidate), 'candidate_pending_after': output['aggregate']['pending'], 'resolved_not_eligible': len(audits), 'resolved_to_invalid': 0, 'accepted_events_added': 0, 'actual_offline_decision_frame': frame, 'roles': dict(sorted(Counter((a['reviewed_role'] for a in audits)).items())), 'review_manifest_sha256_semantic': _digest(adjudications), 'baseline_result_sha256': adjudications['baseline_result_sha256'], 'source_observations_or_histories_reassigned': False, 'qc_adjudications_used_as_machine_predictions': False}
    return output


### src/traffic_location/traffic/_kernel/review_metadata.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/review_metadata.py
from copy import deepcopy
def _rename_event_ids(result):
    mapping = {e["event_id"]: e["event_id"].replace("b_v7_", "b_v8_", 1)
        for e in result["events"] + result.get("cancelled_events", [])}
    def visit(value):
        if isinstance(value, dict):
            return {mapping.get(k, k): visit(v) for k, v in value.items()}
        if isinstance(value, list):
            return [visit(v) for v in value]
        return mapping.get(value, value) if isinstance(value, str) else value
    return visit(result)

def _decision_provenance(result, overlay, *, finalization_frame, overlay_sha256):
    """Keep review dependency per decision, including unchanged machine evidence."""
    reviews = {r["review_id"]: r for v in overlay["videos"] for r in v.get("reviews", [])}
    def dependencies(value):
        if isinstance(value, dict):
            found = {value["review_id"]} if value.get("review_id") in reviews else set()
            for child in value.values():
                found |= dependencies(child)
            return found
        if isinstance(value, list):
            return set().union(*(dependencies(child) for child in value)) if value else set()
        return set()
    rows = []
    for item in result.get("events", []) + result.get("decisions", []):
        if item.get("status") not in ("accepted", "pending", "invalid", "resolved_not_eligible"):
            continue
        ids = sorted(dependencies(item))
        origin = item.get("decision_origin") or (
            "SOURCE_REVIEW_ASSISTED_REPLAY" if ids else "REPLAYED_MACHINE_RULES")
        rows.append({"event_id": item.get("event_id"), "track_id": item.get("track_id"),
            "physical_unit_id": item.get("physical_unit_id"), "episode_id": item.get("episode_id"),
            "passage_id": item.get("passage_id"), "status": item.get("status"),
            "candidate_decision_frame": item.get("candidate_decision_frame", item.get("decision_frame")),
            "offline_finalization_frame": finalization_frame,
            "decision_origin": origin, "p1_review_ids": ids,
            "review_claim_types": sorted({reviews[rid].get("claim_type", "UNKNOWN") for rid in ids}),
            "overlay_sha256": overlay_sha256 if ids else None,
            "not_an_automatic_role_or_identity_prediction": bool(ids)})
    result["v8_decision_provenance"] = rows


### src/traffic_location/traffic/_kernel/reviews.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/reviews.py
"""Apply source-bound P1 adjudications to a separate reviewed branch.

The immutable B v4 overlay validator checks retained pixel receipts. This
adapter adds exact cache-anchor checks and typed identity provenance while
preserving every raw measurement. Reviews never create missing observations.
"""
from __future__ import annotations
from collections import Counter, defaultdict
from copy import deepcopy
import gzip
import hashlib
import json
from pathlib import Path
import re
from . import overlay as previous
from . import measurement_adapter
from . import identity as identity
IDENTITY_TYPES = {'SAME_VEHICLE', 'SPLIT_IDENTITY'}
RAW_KEYS = ('frame_index', 'track_id', 'original_detection_index', 'raw_bbox', 'class_name', 'class_id', 'confidence', 'measurement_provenance', 'observed', 'emitted')

def _require(value, message):
    if not value:
        raise ValueError(message)

def _digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()

def _file_hash(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

def _family(item):
    return measurement_adapter._family(item['class_name'])

def _matches(scope, item, frame):
    return scope['track_id'] == item['track_id'] and scope['raw_family'] == _family(item) and (scope['start_frame'] <= frame <= scope['end_frame'])

def _authority(review, root, guarded):
    authority = review.get('reviewed_decision_authority', {})
    if review.get('claim_type') != 'USER_REVIEW' and (not authority):
        return
    _require(authority.get('kind') == 'EXPLICIT_USER_INSTRUCTION', 'USER_REVIEW requires an explicit retained user instruction')
    path = (root / authority.get('path', '')).resolve()
    _require(path.is_relative_to(root) and path.is_file() and (_file_hash(path) == authority.get('sha256')), 'User decision authority is missing or differs')
    if review.get('claim_type') == 'USER_REVIEW':
        excerpt = authority.get('exact_supporting_excerpt')
        _require(isinstance(excerpt, str) and bool(excerpt) and (excerpt in path.read_text()), 'User review lacks its exact authority excerpt')
    guarded.append({'path': str(path), 'sha256': _file_hash(path)})

def _validated_reviews(overlay, source_identity, video_id, frame_count, root, guarded):
    normalized = deepcopy(overlay)
    originals = {r['review_id']: r for v in overlay.get('videos', []) for r in v.get('reviews', [])}
    _require(len(originals) == sum((len(v.get('reviews', [])) for v in overlay.get('videos', []))), 'Duplicate review IDs')
    occupied = defaultdict(list)
    for video in normalized.get('videos', []):
        for review in video.get('reviews', []):
            _authority(review, root, guarded)
            if review.get('claim_type') == 'USER_REVIEW':
                review['claim_type'] = 'ADJUDICATED_AI_PIXELS_REVIEW'
                review['human_confirmation_status'] = 'NOT_NEWLY_ADJUDICATED_BY_USER'
            if review.get('type') in IDENTITY_TYPES:
                for scope in review.get('scopes', []):
                    key = (scope.get('track_id'), scope.get('raw_family'))
                    start, end = (scope.get('start_frame'), scope.get('end_frame'))
                    _require(type(start) is int and type(end) is int, 'Explicit identity scope required')
                    _require(all((end < old[0] or start > old[1] for old in occupied[key])), 'Overlapping identity adjudications are forbidden')
                    occupied[key].append((start, end))
    jobs = [{'video_id': video_id, 'b_v2_source_identity': source_identity, 'metadata': {'frame_count': frame_count}, 'source_manifest_sha256': overlay['donor_manifest_sha256']}]
    validated = previous.validate_overlay(normalized, jobs, root, guarded)
    _require(set(validated) == {video_id}, 'P1 overlay must name exactly the supplied source video')
    reviews = validated[video_id].get('reviews', [])
    for review in reviews:
        original = originals[review['review_id']]
        review['claim_type'] = original['claim_type']
        review['human_confirmation_status'] = original.get('human_confirmation_status')
        if 'reviewed_decision_authority' in original:
            review['reviewed_decision_authority'] = deepcopy(original['reviewed_decision_authority'])
        _require(all((any((scope['start_frame'] <= frame <= scope['end_frame'] for frame in review['confirmed_frames'])) for scope in review['scopes'])), 'Every bounded scope needs its own reviewed source anchor')
    return reviews

def _anchor_records(path, source_identity, anchors):
    _require(_file_hash(path) == source_identity['records_sha256'], 'Review must use the exact frozen source cache')
    found = {}
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt') as handle:
        for line in handle:
            row = json.loads(line)
            frame = row.get('frame_index')
            if frame not in anchors:
                continue
            _require(frame not in found, 'Repeated cache anchor frame')
            _require(all((row.get(k) == source_identity[k] for k in ('video_sha256', 'width', 'height'))), 'Anchor cache belongs to a different video')
            found[frame] = row
    _require(set(found) == anchors, 'Reviewed source frame is missing from frozen cache')
    return found

def prepare_reviewed_frames(frames, overlay, *, source_identity, repo_root, source_records_path, video_id='weekday_siang', frame_count=None, overlay_sha256=None, branch='review_assisted'):
    """Return copied reviewed frames plus application proof; never machine data.

    ``frames`` must be the unchanged source adapter output before B v6 identity
    preparation. The caller retains/guards the manifest file if passing its
    file SHA; the semantic input digest is recorded independently here.
    """
    _require(branch == 'review_assisted', 'Machine branch cannot apply P1 adjudications')
    root, copied = (Path(repo_root).resolve(), deepcopy(list(frames)))
    _require(copied, 'Review needs actual source frames')
    frame_ids = [row['frame_index'] for row in copied]
    _require(all((type(f) is int and f >= 0 for f in frame_ids)) and frame_ids == sorted(set(frame_ids)), 'Source frames must be unique and ordered')
    frame_count = max(frame_ids) + 1 if frame_count is None else frame_count
    _require(type(frame_count) is int and max(frame_ids) < frame_count, 'Invalid source frame count')
    semantic = _digest(overlay)
    manifest_digest = semantic if overlay_sha256 is None else overlay_sha256
    _require(isinstance(manifest_digest, str) and re.fullmatch('[0-9a-f]{64}', manifest_digest), 'Review manifest SHA must be explicit SHA256')
    guarded = []
    reviews = _validated_reviews(overlay, source_identity, video_id, frame_count, root, guarded)
    anchors = {f for review in reviews for f in review['confirmed_frames']}
    records = _anchor_records(source_records_path, source_identity, anchors)
    guarded.append({'path': str(Path(source_records_path).resolve()), 'sha256': source_identity['records_sha256']})
    source = {(row['frame_index'], item['track_id']): item for row in copied for item in row.get('observations', [])}
    _require(len(source) == sum((len(row.get('observations', [])) for row in copied)), 'Repeated raw track within a source frame')
    counts, matched, assignments = (Counter(), set(), [])
    scope_counts = Counter()
    for review in reviews:
        rid = review['review_id']
        for frame in review['confirmed_frames']:
            proof = review['_pixel_anchor_proofs'][frame]
            record = records[frame]
            _require(record.get('detector_input_frame_pixel_sha256') == proof['raw_bgr_pixel_sha256'], 'Review pixel receipt differs from actual source cache frame')
            actual = {item['track_id']: item for item in record['emitted_tracks']}
            anchor_matches = 0
            for scope in review['scopes']:
                item = source.get((frame, scope['track_id']))
                if item is None or not _matches(scope, item, frame):
                    continue
                cached = actual.get(item['track_id'], {})
                _require(all((cached.get(k) == item.get(k) for k in ('raw_bbox', 'class_name', 'original_detection_index'))), 'Review anchor differs from actual source measurement')
                anchor_matches += 1
                matched.add((rid, item['track_id'], frame))
            _require(anchor_matches, 'Confirmed review frame has no measured target in its own scope')
    for row in copied:
        frame = row['frame_index']
        for item in row.get('observations', []):
            original = deepcopy(item)
            for review in reviews:
                for si, scope in enumerate(review['scopes']):
                    if not _matches(scope, item, frame):
                        continue
                    rid, kind = (review['review_id'], review['type'])
                    before = {k: deepcopy(item.get(k)) for k in ('unit_id', 'episode_id', 'role', 'resolved_class_name', 'association_verified', 'association_id', 'component_of_unit_id')}
                    if kind in IDENTITY_TYPES:
                        unit = scope.get('physical_unit_id', review.get('physical_unit_id'))
                        episode = scope.get('episode_id', review.get('episode_id'))
                        if identity._identity_review_proof(item):
                            _require((unit, episode) == (item['unit_id'], item.get('episode_id')), 'Conflicting positive identity review requires explicit replacement')
                        item.update(unit_id=unit, episode_id=episode, association_verified=kind == 'SAME_VEHICLE', association_id=rid if kind == 'SAME_VEHICLE' else None)
                    elif kind == 'COMPONENT_OF':
                        _require(item.get('role') != 'pedestrian', 'Contradictory positive role review')
                        item.update(role='rider', component_of_unit_id=review['component_of_unit_id'], role_scope_id=rid)
                    elif kind == 'PEDESTRIAN_ROLE':
                        _require(item.get('role') != 'rider', 'Contradictory positive role review')
                        item.update(role='pedestrian', resolved_class_name='pedestrian', unit_id=review.get('physical_unit_id', f'reviewed-pedestrian:{rid}'), episode_id=review.get('episode_id', f'reviewed-role:{rid}'), role_scope_id=rid)
                    resolved = scope.get('resolved_class_name', review.get('resolved_class_name'))
                    if resolved:
                        item['resolved_class_name'] = resolved
                    item.setdefault('review_refs', []).append({'review_id': rid, 'reviews_sha256': manifest_digest, 'source_frame': frame, 'track_id': item['track_id'], 'review_type': kind, 'claim_type': review['claim_type'], 'decision_origin': 'HUMAN_REVIEW' if review['claim_type'] == 'USER_REVIEW' else 'AI_ASSISTED_REVIEW_NOT_NEW_HUMAN_CONFIRMATION', 'scope_extent_is_engineering_assumption': True})
                    counts[rid] += 1
                    scope_counts[rid, si] += 1
                    assignments.append({'frame_index': frame, 'track_id': item['track_id'], 'original_detection_index': item['original_detection_index'], 'raw_bbox': deepcopy(item['raw_bbox']), 'review_id': rid, 'review_type': kind, 'before': before, 'after': {k: deepcopy(item.get(k)) for k in before}, 'claim_type': review['claim_type'], 'source_measurement_unchanged': True})
            _require(all((item.get(k) == original.get(k) for k in RAW_KEYS)), 'A review cannot modify raw measurements')
    _require(all((scope_counts[r['review_id'], i] > 0 for r in reviews for i, _ in enumerate(r['scopes']))), 'Review scope did not match a measured observation')
    audit = {'applied_measurements': dict(sorted(counts.items())), 'matched_reviewed_anchors': [{'review_id': rid, 'track_id': track, 'frame_index': frame} for rid, track, frame in sorted(matched)], 'source_links_checked': True, 'raw_measurements_modified': False, 'new_observations_created': 0, 'review_assignments': assignments}
    return {'frames': copied, 'application_audit': audit, 'guarded_inputs': guarded, 'provenance': {'schema_version': 'phase8_b_v8_review_application_v1', 'branch': branch, 'review_manifest_sha256': manifest_digest, 'review_semantic_sha256': semantic, 'source_identity': deepcopy(source_identity), 'qc_used_as_machine_predictions': False, 'source_records_sha256_verified': True, 'typed_identity_provenance': True, 'decision_origin': 'SOURCE_BOUND_REVIEW_ASSISTED_NOT_AUTOMATIC_MODEL_RESULT'}}


### src/traffic_location/traffic/_kernel/support.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/_kernel/support.py
"""Local read-only guards shared by extracted source adapters."""
import gzip,hashlib,json,math
from pathlib import Path

def require(condition,message):
 if not condition:raise ValueError(message)

def read(path):
 data=Path(path).read_bytes()
 return json.loads(gzip.decompress(data) if str(path).endswith('.gz') else data)

def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
 return h.hexdigest()

def bound(root,name,digest,guarded):
 relative=Path(name);p=(Path(root)/relative).resolve()
 require(not relative.is_absolute() and p.is_relative_to(Path(root).resolve()) and p.is_file() and not p.is_symlink(),'Missing/unsafe proof')
 require(sha(p)==digest,'Proof checksum differs')
 guarded.append({'path':str(p),'sha256':digest});return p

def bbox(value):
 require(isinstance(value,(list,tuple)) and len(value)==4,'raw_bbox needs four coordinates')
 require(all(isinstance(v,(int,float)) and not isinstance(v,bool) and math.isfinite(v) for v in value),'raw_bbox must be finite')
 box=list(value);require(box[0]<box[2] and box[1]<box[3],'Invalid bbox');return box

def records(path):
 opener=gzip.open if str(path).endswith('.gz') else open
 with opener(path,'rt',encoding='utf-8') as f:
  for line in f:
   require(bool(line.strip()),'Empty source record');yield json.loads(line)

# Retained relation arrays are bounded (~5.8MB gzip for the pilot). No network.
def iter_json_array(path):
 yield from read(path)


### src/traffic_location/traffic/batch.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/batch.py
"""Eight source-bound machine snapshots; category x direction x evidence pivot."""
from collections import Counter
from ..artifacts import require,semantic_hash
from .results import validate_native
CATEGORIES=('motorcycle','car','truck','bus','pedestrian')
DIRECTIONS=('A_TO_B','B_TO_A')

def aggregate(expected,results,*,experiment_sha256):
 require(len(expected)==8 and len(set(expected))==8,'Eight unique source slots required')
 ids=[v['video_id'] for v in results];require(len(set(ids))==len(ids) and set(ids)<=set(expected),'Duplicate rerun or foreign video')
 pivot={c:{d:{'measured':0,'estimated':0} for d in DIRECTIONS} for c in CATEGORIES}
 ledger=[];sources=set();versions={};pending=invalid=0;complete=True
 for slot in expected:
  row=next((r for r in results if r['video_id']==slot),None)
  if not row or row.get('status')!='SUCCEEDED':complete=False;continue
  require(row['experiment_sha256']==experiment_sha256 and row['branch']=='machine','Mixed policy/model/branch batch forbidden')
  require(row.get('scope')=='FULL_VIDEO','Smoke/prefix cannot enter full-video totals')
  native=validate_native(row['native']);require(native['source_identity']['video_sha256']==expected[slot],'Wrong video source')
  require(expected[slot] not in sources,'Same source repeated across slots');sources.add(expected[slot]);versions[slot]=row['result_version']
  for event in native['events']:
   c,d=event['category'],event['direction'];require(c in CATEGORIES and d in DIRECTIONS,'Unsupported event category/direction')
   evidence='measured' if event['evidence']['measured_crossing'] is True else 'estimated'
   pivot[c][d][evidence]+=1;ledger.append({'video_id':slot,'source_sha256':expected[slot],'event_id':event['event_id'],'category':c,'direction':d,'evidence':evidence})
  pending+=native['aggregate']['pending'];invalid+=native['aggregate']['invalid']
 require(len({(v['source_sha256'],v['event_id']) for v in ledger})==len(ledger),'Duplicate batch passage')
 vehicles=sum(1 for e in ledger if e['category']!='pedestrian')
 require(sum(x for c in pivot.values() for d in c.values() for x in d.values())==len(ledger),'Pivot/ledger mismatch')
 return {'schema_version':'traffic_batch_v1','execution_status':'SUCCEEDED' if complete else 'PARTIAL','result_quality':'PARTIAL',
  'experiment_sha256':experiment_sha256,'result_version':semantic_hash({'experiment':experiment_sha256,'versions':versions,'pivot':pivot}),
  'videos_expected':8,'videos_completed':len(versions),'complete_vehicle_total':vehicles if complete else None,
  'observed_vehicle_subtotal':vehicles,'pedestrian_total':len(ledger)-vehicles,'pivot':pivot,'events':ledger,
  'pending':pending,'invalid':invalid,'missing_or_failed':[v for v in expected if v not in versions],
  'semantics':'PASSAGES_IN_RECORDED_SAMPLES_NOT_UNIQUE_VEHICLES_OR_DAILY_TRAFFIC','production_ready':False}


### src/traffic_location/traffic/contracts.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/contracts.py
"""Strict explicit profiles; replay is an operation over one declared branch."""
from copy import deepcopy
from ..artifacts import require
PROFILES={'machine','review_assisted','replay'}
def valid_hash(value):return isinstance(value,str) and len(value)==64 and all(x in '0123456789abcdef' for x in value)
REQUIRED={'schema_version','profile','video_id','frame_count','source_identity','artifacts','source_manifest_sha256'}
OPTIONAL={'replay_branch','request_id','measurement_format'}
ALLOWED_ARTIFACTS={'measurements','geometry','policy','relationships','baseline_reviews','review_overlay','adjudications','baseline_result','historical_baseline'}

def validate_request(request):
 r=deepcopy(request)
 require(isinstance(r,dict) and REQUIRED<=set(r) and set(r)<=REQUIRED|OPTIONAL,'Missing/unknown traffic request fields')
 require(r['schema_version']=='traffic_request_v1' and isinstance(r['profile'],str) and r['profile'] in PROFILES,'Unsupported traffic profile/schema')
 require(isinstance(r['video_id'],str) and r['video_id'].strip() and '/' not in r['video_id'] and '\\' not in r['video_id'],'Explicit video ID required')
 require(type(r['frame_count']) is int and r['frame_count']>0,'Positive actual frame_count required')
 branch=r.get('replay_branch') if r['profile']=='replay' else r['profile']
 require(isinstance(branch,str) and branch in {'machine','review_assisted'},'Replay requires an explicit replay_branch')
 require(r['profile']=='replay' or 'replay_branch' not in r,'replay_branch only allowed on replay')
 a=r['artifacts'];require(isinstance(a,dict) and {'measurements','geometry','policy'}<=set(a) and set(a)<=ALLOWED_ARTIFACTS,'Invalid artifact set')
 require(r.get('measurement_format','notebook_native') in {'notebook_native','fresh_v1'},'Unsupported measurement format')
 if r.get('measurement_format')=='fresh_v1':
  require(branch=='machine' and not {'baseline_reviews','historical_baseline','review_overlay','adjudications','baseline_result','relationships'}&set(a),'Fresh machine input cannot inherit legacy review or relations')
 require(isinstance(r['source_identity'],dict),'Explicit source identity required')
 for ref in a.values():require(isinstance(ref,dict) and set(ref)=={'path','sha256'} and isinstance(ref['path'],str) and bool(ref['path']) and valid_hash(ref['sha256']),'Invalid artifact reference')
 require('baseline_reviews' not in a or 'historical_baseline' in a,'Historical review context needs an explicit frozen baseline')
 review_keys={'review_overlay','adjudications','baseline_result'}
 require(review_keys<=set(a) if branch=='review_assisted' else not review_keys&set(a),'Machine/review overlay mixing forbidden')
 for key in ['video_sha256','geometry_sha256','records_sha256']:
  require(valid_hash(r['source_identity'].get(key)),'Missing source hash '+key)
 for key in ['width','height']:require(type(r['source_identity'].get(key)) is int and r['source_identity'][key]>0,'Explicit native dimensions required')
 require(a['measurements']['sha256']==r['source_identity']['records_sha256'] and a['geometry']['sha256']==r['source_identity']['geometry_sha256'],'Source/artifact hash mismatch')
 require(valid_hash(r['source_manifest_sha256']),'Source manifest SHA required')
 return r,branch


### src/traffic_location/traffic/fresh_bridge.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/fresh_bridge.py
"""Source-linked fresh observations, without pretending they are notebook caches."""
from copy import deepcopy
import math
from ..artifacts import read,require,semantic_hash

def frames(path,source_identity,frame_count,*,audit=None):
 value=read(path)
 require(value['schema_version']=='fresh_inference_result_v1' and value['status']=='SUCCEEDED','Successful fresh measurement required')
 require(value['source_sha256']==source_identity['video_sha256'],'Fresh source video differs')
 require(value['records_semantic_sha256']==semantic_hash(value['records']),'Fresh measurement integrity differs')
 require(value['frames']==frame_count==len(value['records']),'Fresh source scope differs')
 meta=value['metadata'];require(all(meta[k]==source_identity[k] for k in ['width','height']),'Fresh dimensions differ')
 names={int(k):v for k,v in value['class_names'].items()}
 for index,row in enumerate(value['records']):
  require(row['frame_index']==index,'Fresh frames must be continuous from zero')
  seen=set();linked=set();observations=[]
  for obs in row['emitted_measurements']:
   track=obs['track_id'];di=obs['original_detection_index']
   require(type(track) is int and track>0 and track not in seen and type(di) is int and 0<=di<len(row['detections']) and di not in linked,'Repeated/invalid fresh source link')
   seen.add(track);linked.add(di)
   raw=row['detections'][di];require(obs['raw_detection']==raw and obs['observed'] is True and obs['emitted'] is True,'Fresh prediction or altered raw link')
   box=raw['bbox'];require(len(box)==4 and all(type(x) in (int,float) and math.isfinite(x) for x in box) and 0<=box[0]<box[2]<=meta['width'] and 0<=box[1]<box[3]<=meta['height'],'Invalid fresh bbox')
   require(type(raw['confidence']) in (int,float) and math.isfinite(raw['confidence']) and 0<=raw['confidence']<=1,'Invalid fresh confidence')
   category=names[raw['class_id']]
   if category not in {'person','motorcycle','car','truck','bus'}:continue
   observations.append({'frame_index':index,'track_id':track,'original_detection_index':di,
    'raw_bbox':deepcopy(box),'class_name':category,'class_id':raw['class_id'],'confidence':raw['confidence'],
    'observed':True,'emitted':True,'role':'unknown_person' if category=='person' else 'vehicle',
    'unit_id':f'person-track:{track}' if category=='person' else f'track:{track}',
    'episode_id':'unreviewed-person' if category=='person' else 'unreviewed-vehicle','association_verified':False,
    'measurement_provenance':'fresh_modular_detection_v1','source_ref':{'measurement_sha256':source_identity['records_sha256'],
     'source_pixel_sha256':row['source_pixel_sha256'],'frame_index':index,'original_detection_index':di}})
  yield {'frame_index':index,'observations':observations}
 if audit is not None:audit.update(format='fresh_v1',frames=frame_count,legacy_provenance_relabelled=False)


### src/traffic_location/traffic/inference_worker.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/inference_worker.py
"""Isolated fresh detector/tracker/ReID worker; no counting or historical replay.

Run a new OS process for each job: tracker global counters cannot leak between jobs.
External assets are mandatory hash-bound inputs; this module never downloads weights.
"""
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import json,sys,random,platform,time
from pathlib import Path
from ..artifacts import artifact,read,write,require,sha,semantic_hash


def execute(request,output):
 import cv2,numpy as np,torch,ultralytics,onnxruntime
 from ultralytics import YOLO
 from ultralytics.trackers.bot_sort import BOTSORT
 from ultralytics.utils import YAML,IterableSimpleNamespace
 r=request;require(r['schema_version']=='fresh_inference_request_v1','Unsupported fresh request')
 output=Path(output);require(not output.exists(),'Fresh output must not overwrite existing artifacts')
 root=Path(r['input_root']);paths={k:artifact(v,root) for k,v in r['artifacts'].items()}
 require(set(paths)=={'video','detector','reid'},'Video/detector/ReID refs required')
 require(ultralytics.__version__==r['ultralytics_version'],'Pinned Ultralytics differs')
 require(r['device'] in {'cpu','cuda:0'},'Explicit supported device required')
 if r['device'].startswith('cuda'):
  require(torch.cuda.is_available(),'Requested GPU unavailable; no CPU fallback')
  require('CUDAExecutionProvider' in onnxruntime.get_available_providers(),'CUDA ReID provider unavailable')
 require(r['start_frame']==0 and type(r['frame_limit']) is int and r['frame_limit']>0,'Bounded prefix only; no skipped tracking context')
 require(r['preprocessing']=='RAW' and r['half'] is False,'Only declared RAW FP32 experiment profile supported')
 random.seed(r['seed']);np.random.seed(r['seed']);torch.manual_seed(r['seed']);torch.set_num_threads(r['threads'])
 torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
 print(json.dumps({'stage':'loading_detector','device':r['device']}),flush=True)
 model=YOLO(str(paths['detector']))
 require(model.task=='detect','Unexpected model task')
 config=YAML.load(Path(ultralytics.__file__).parent/'cfg/trackers/botsort.yaml')
 config.update(r['tracker']);config.update(with_reid=True,model=str(paths['reid']),device=r['device'])
 print(json.dumps({'stage':'loading_reid','device':r['device']}),flush=True)
 tracker=BOTSORT(IterableSimpleNamespace(**config))
 require(tracker.encoder is not None,'ReID must remain enabled')
 actual_providers=tracker.encoder.model.session.get_providers()
 require(('CUDAExecutionProvider' in actual_providers) if r['device'].startswith('cuda') else actual_providers==['CPUExecutionProvider'],'Actual ReID provider differs')
 cap=cv2.VideoCapture(str(paths['video']));require(cap.isOpened(),'Video decode unavailable')
 metadata={'width':int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),'height':int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
   'reported_frame_count':int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),'reported_fps':cap.get(cv2.CAP_PROP_FPS)}
 require(metadata['reported_frame_count']>=r['frame_limit'],'Source shorter than requested prefix')
 records=[];started=time.monotonic()
 print(json.dumps({'stage':'inference_start','frames':r['frame_limit'],'providers':actual_providers}),flush=True)
 try:
  for index in range(r['frame_limit']):
   ok,image=cap.read();require(ok,'Decode failed within requested scope')
   pixel_sha=__import__('hashlib').sha256(image.tobytes()).hexdigest()
   prediction=model.predict(image,device=r['device'],imgsz=r['imgsz'],conf=r['conf'],iou=r['iou'],half=False,verbose=False)[0]
   raw=prediction.boxes.cpu().numpy();tracks=tracker.update(raw,img=image)
   detections=[{'bbox':box.xyxy[0].tolist(),'confidence':float(box.conf[0]),'class_id':int(box.cls[0])} for box in raw]
   observations=[]
   for row in tracks:
    di=int(row[-1]);require(0<=di<len(detections),'Tracker detection link invalid')
    obs={'track_id':int(row[4]),'original_detection_index':di,'raw_detection':detections[di],
      'tracker_box':row[:4].tolist(),'observed':True,'emitted':True}
    require(int(row[6])==detections[di]['class_id'],'Tracker/raw class differs')
    observations.append(obs)
   features=[{'track_id':int(t.track_id),'shape':list(t.curr_feat.shape),'sha256':__import__('hashlib').sha256(t.curr_feat.tobytes()).hexdigest()}
     for t in tracker.tracked_stracks if t.curr_feat is not None]
   require(__import__('hashlib').sha256(image.tobytes()).hexdigest()==pixel_sha,'Inference mutated source pixels')
   records.append({'frame_index':index,'source_pixel_sha256':pixel_sha,
     'detections':detections,'emitted_measurements':observations,'reid_features':features})
   if index%128==0:print(json.dumps({'stage':'inference','frame':index,'of':r['frame_limit'],'fps_actual':round((index+1)/max(time.monotonic()-started,1e-9),2),'detections':len(detections),'tracks':len(observations)}),flush=True)
 finally:cap.release()
 print(json.dumps({'stage':'inference_finished','frames':len(records),'wall_seconds':round(time.monotonic()-started,2),'fps_actual':round(len(records)/max(time.monotonic()-started,1e-9),2)}),flush=True)
 require(all(sha(paths[k])==v['sha256'] for k,v in r['artifacts'].items()),'Fresh input changed')
 environment={'python':platform.python_version(),'torch':torch.__version__,'ultralytics':ultralytics.__version__,
   'onnxruntime':onnxruntime.__version__,'opencv':cv2.__version__,'numpy':np.__version__,
   'device':r['device'],'reid_providers':actual_providers,'torch_threads':torch.get_num_threads()}
 result={'schema_version':'fresh_inference_result_v1','status':'SUCCEEDED','scope':'BOUNDED_SOURCE_PREFIX_NOT_COUNTING_BENCHMARK',
  'frames':len(records),'metadata':metadata,'class_names':model.names,'source_sha256':r['artifacts']['video']['sha256'],'request_fingerprint':semantic_hash(r),
  'environment':environment,'effective_tracker_config':config,'records':records,'records_semantic_sha256':semantic_hash(records),
  'counting_executed':False,'physical_identity_accuracy_established':False,'vehicle_reid_suitability':'NOT_ESTABLISHED_PERSON_DOMAIN_CONTROL',
  'worker_process_isolation':True}
 output.mkdir(parents=True);write(output/'result.json.gz',result)
 print(json.dumps({'status':'PASS','frames':len(records),'records_semantic_sha256':result['records_semantic_sha256'],'environment':environment}),flush=True)
 return result

if __name__=='__main__':
 try:execute(read(sys.argv[1]),sys.argv[2])
 except Exception as error:
  print(json.dumps({'status':'FAILED','reason':str(error)}),file=sys.stderr);sys.exit(1)


### src/traffic_location/traffic/measurements.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/measurements.py
"""Read and validate native detector-linked observations, with no inference."""
from ._kernel.measurement_adapter import adapted_frames,relationship_evidence
from ._kernel.support import require

def read_measurements(path,source_identity,frame_count,*,relationships=None,baseline_reviews=None,baseline_reviews_sha256=None,audit=None):
 # Legacy historical review context is explicit; it is never a new model prediction.
 yield from adapted_frames(path,source_identity,frame_count,relationships=relationships,
   reviews=baseline_reviews or [],reviews_sha256=baseline_reviews_sha256,application_proof=audit)


### src/traffic_location/traffic/orchestration.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/orchestration.py
"""Inference runs in one fresh process per job; no tracker globals are shared."""
import os,subprocess,sys,tempfile,shutil
from pathlib import Path
from ..artifacts import read,require,semantic_hash

def fresh_inference(request_path,output,*,python_executable):
 output=Path(output).resolve();require(not output.exists(),'Immutable inference output required')
 output.parent.mkdir(parents=True,exist_ok=True)
 stage=Path(tempfile.mkdtemp(prefix='.inference-',dir=output.parent))
 try:
  # Keep the worker output nonexistent until its processing has succeeded.
  with (stage/'worker.log').open('w') as log:
   completed=subprocess.run([str(python_executable),'-B','-m','traffic_location.traffic.inference_worker',str(Path(request_path).resolve()),str(stage/'worker')],
    env={**os.environ,'PYTHONPATH':str(Path(__file__).resolve().parents[2])},stdout=log,stderr=subprocess.STDOUT)
  require(completed.returncode==0,'Inference worker failed; no finalized result was published')
  value=read(stage/'worker/result.json.gz')
  require(value['status']=='SUCCEEDED' and value['request_fingerprint']==semantic_hash(read(request_path)),'Worker request/result mismatch')
  require(value['records_semantic_sha256']==semantic_hash(value['records']),'Worker records integrity mismatch')
  shutil.move(stage/'worker/result.json.gz',stage/'result.json.gz');(stage/'worker').rmdir()
  require(not output.exists(),'Output appeared while inference ran');stage.rename(output)
  return value
 finally:
  if stage.exists():shutil.rmtree(stage)


### src/traffic_location/traffic/pipeline.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/pipeline.py
"""Fresh per-call state over explicit source/config inputs. No pilot constants."""
from copy import deepcopy
from pathlib import Path
from .contracts import validate_request
from .measurements import read_measurements,relationship_evidence
from ._kernel import identity,publication,reviews,review_finalizer,review_metadata
from ._kernel.geometry import Geometry
from .results import wrap_native
from ..artifacts import artifact,read,require,sha,semantic_hash


def implementation_hash():
 root=Path(__file__).resolve().parent
 return semantic_hash({str(p.relative_to(root)):sha(p) for p in sorted(root.rglob('*.py'))
   if not p.name.startswith('render') and p.name!='inference_worker.py'})


def run(request,*,root):
 request,branch=validate_request(request);root=Path(root).resolve()
 refs=request['artifacts'];paths={k:artifact(v,root) for k,v in refs.items()}
 before={k:sha(p) for k,p in paths.items()}
 source=deepcopy(request['source_identity']);policy=read(paths['policy']);audit={}
 require(policy.get('execution_branch')==branch,'Effective policy branch differs from request')
 geometry_data=read(paths['geometry'])
 if geometry_data.get('source_video',{}).get('sha256'):
  require(geometry_data['source_video']['sha256']==source['video_sha256'],'Geometry belongs to a different source video')
 geometry=Geometry.from_draft(paths['geometry'],source['width'],source['height'])
 links=relationship_evidence(paths['relationships']) if 'relationships' in paths else {}
 base_reviews=[]
 if 'baseline_reviews' in paths:
  historical=read(paths['historical_baseline'])
  require(historical['source_identity']==source and historical['report']['policy']['review_manifest_sha256']==refs['baseline_reviews']['sha256'],'Historical baseline/review binding differs')
  manifest=read(paths['baseline_reviews'])
  require(manifest.get('donor_manifest_sha256')==request['source_manifest_sha256'],'Historical review source manifest differs')
  entries=[v for v in manifest['videos'] if v['video_id']==request['video_id']]
  require(len(entries)==1,'Historical review video missing/duplicate')
  require(all(entries[0][k]==source[k] for k in ['video_sha256','records_sha256','geometry_sha256']),'Historical review source differs')
  base_reviews=entries[0].get('reviews',[])
  # Baseline source-context reviews remain bounded, explicit, and source-linked.
  for review in base_reviews:
   require(review.get('confirmed_frames') and review.get('scopes'),'Unbounded historical review')
   for scope in review['scopes']:
    require(type(scope.get('start_frame')) is int and type(scope.get('end_frame')) is int and
      0<=scope['start_frame']<=scope['end_frame']<request['frame_count'],'Historical review scope outside video')
 if request.get('measurement_format')=='fresh_v1':
  from .fresh_bridge import frames as fresh_frames
  frames=fresh_frames(paths['measurements'],source,request['frame_count'],audit=audit)
 else:
  frames=read_measurements(paths['measurements'],source,request['frame_count'],relationships=links,
    baseline_reviews=base_reviews,baseline_reviews_sha256=refs.get('baseline_reviews',{}).get('sha256'),audit=audit)
 if branch=='review_assisted':
  overlay=read(paths['review_overlay'])
  require(overlay.get('donor_manifest_sha256')==request['source_manifest_sha256'],'Overlay parent manifest differs')
  reviewed=reviews.prepare_reviewed_frames(frames,overlay,source_identity=source,repo_root=root,
    source_records_path=paths['measurements'],video_id=request['video_id'],frame_count=request['frame_count'],
    overlay_sha256=refs['review_overlay']['sha256'],branch=branch)
  frames=reviewed['frames']
 resolved=identity.prepare_identity_frames(frames,geometry,policy,relationships=links)
 require([r['frame_index'] for r in resolved['frames']]==list(range(request['frame_count'])),'Missing/extra source frames')
 effective=deepcopy(policy);effective['episode_relations']=resolved.get('unit_relations',[])
 result=publication.replay_frames(resolved['frames'],geometry,source_identity=source,policy=effective)
 if branch=='review_assisted':
  adjudications=read(paths['adjudications'])
  require(adjudications['baseline_result_sha256']==refs['baseline_result']['sha256'],'Review baseline bytes mismatch')
  result=review_finalizer.finalize_reviewed_pending(result,adjudications,baseline_result=read(paths['baseline_result']),
    source_identity=source,finalization_frame=request['frame_count']-1,repo_root=root)
 result.update(video_id=request['video_id'],source_identity=source,branch=branch,
   frame_assignments=resolved['assignments'],component_links=resolved.get('component_links',[]))
 if branch=='review_assisted':
  review_metadata._decision_provenance(result,overlay,finalization_frame=request['frame_count']-1,overlay_sha256=refs['review_overlay']['sha256'])
  result=review_metadata._rename_event_ids(result)
  result['report'].update(version='b_v8_p1_review_assisted_v1',reviewed_branch_not_machine_model_improvement=True)
 require(all(sha(paths[k])==h for k,h in before.items()),'Input mutated during execution')
 return {'result':wrap_native(result,request=request,branch=branch,implementation_hash=implementation_hash()),
   'native':result,'adapter_audit':audit}


### src/traffic_location/traffic/render.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/render.py
"""Render-only entry points: no imports from the counting/identity pipeline."""
from copy import deepcopy
from ._kernel import render_kernel
from ..artifacts import semantic_hash,require

def frame(image,measurement,native_result,geometry,*,fps,video_id,frame_index=None):
 before=semantic_hash(native_result)
 canvas,state=render_kernel.render_b_v6_frame(image,measurement,deepcopy(native_result),geometry,
   fps=fps,video_id=video_id,frame_index=frame_index,
   unit_assignments=native_result.get('frame_assignments',[]),component_relations=native_result.get('component_links',[]))
 require(semantic_hash(native_result)==before,'Renderer modified counting result')
 return canvas,state

def timeline(native_result,geometry=None,frame_count=None):
 before=semantic_hash(native_result)
 value=render_kernel.build_b_v6_timeline(deepcopy(native_result),geometry,frame_count=frame_count,
   unit_assignments=native_result.get('frame_assignments',[]),component_relations=native_result.get('component_links',[]))
 require(semantic_hash(native_result)==before,'Timeline mutated result')
 return value

def video(video_path,measurement_path,native_result,geometry,output_path,*,fps,frame_count,source_identity,progress=None):
 before=semantic_hash(native_result)
 report=render_kernel.render_b_v6_video(video_path,measurement_path,deepcopy(native_result),geometry,output_path,
   fps=fps,expected_frame_count=frame_count,source_identity=source_identity,
   unit_assignments=native_result.get('frame_assignments',[]),component_relations=native_result.get('component_links',[]),progress=progress)
 require(semantic_hash(native_result)==before,'Video render mutated counting result')
 return report


### src/traffic_location/traffic/results.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/src/traffic_location/traffic/results.py
"""Domain reconciliation and stable result identities, separate from native ledgers."""
from collections import Counter
from copy import deepcopy
import platform
from ..artifacts import require,semantic_hash

def validate_native(result):
 a=result['aggregate'];events=result['events']
 require(len(events)==a['total'],'Accepted total/event mismatch')
 require(len({e['event_id'] for e in events})==len(events),'Duplicate accepted event ID')
 measured=sum(e['evidence']['measured_crossing'] is True for e in events)
 require(measured==a['measured_total'] and len(events)-measured==a['inferred_total'],'Measured/estimated mismatch')
 require(dict(sorted(Counter(e['category'] for e in events).items()))==a['by_category'],'Category aggregation mismatch')
 require(dict(sorted(Counter(e['direction'] for e in events).items()))==a['by_direction'],'Direction aggregation mismatch')
 require(all(e.get('accepted_count_event') is True and e.get('status')=='accepted' for e in events),'Non-accepted event in accepted ledger')
 require(sum(d.get('status')=='pending' for d in result['decisions'])==a['pending'],'Pending decision aggregate mismatch')
 require(len(result['invalid_decisions'])==a['invalid'],'Invalid decision aggregate mismatch')
 return result

def wrap_native(result,*,request,branch,implementation_hash):
 validate_native(result);a=result['aggregate']
 runtime={'python':platform.python_version(),'implementation':platform.python_implementation()}
 fingerprint=semantic_hash({'source':request['source_identity'],'artifacts':{k:v['sha256'] for k,v in request['artifacts'].items()},
  'branch':branch,'video_id':request['video_id'],'frame_count':request['frame_count'],'implementation':implementation_hash,'runtime':runtime})
 return {'schema_version':'traffic_result_v1','execution_status':'SUCCEEDED',
  'result_quality':'PARTIAL','validation_level':'TECHNICAL_PILOT',
  'result_version':fingerprint,'branch':branch,'operation_profile':request['profile'],'runtime':runtime,
  'scope':{'video_id':request['video_id'],'source_frames':request['frame_count'],'processed_frames':result['report']['frame_count']},
  'summary':{'accepted_total':a['total'],'accepted_measured':a['measured_total'],'accepted_estimated':a['inferred_total'],
   'pending':a['pending'],'invalid':a['invalid'],'native_aggregate':deepcopy(a)},
  'source_identity':deepcopy(request['source_identity']),'native_semantic_sha256':semantic_hash(result),
  'legacy_baseline_review_context_present':'baseline_reviews' in request['artifacts'],
  'fresh_inference_executed':request.get('measurement_format')=='fresh_v1','production_ready':False,
  'limitations':['Counting correctness and identity/role generalization are not established by modular parity',
    'Machine native gate remains unresolved; review-assisted is not automatic model performance']}


### tools/run_t7sl_job.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/tools/run_t7sl_job.py
"""One complete video/job. Invoke in a new process, one visible GPU per job."""
import argparse,sys,json,shutil
from pathlib import Path
from traffic_location.artifacts import read,write,sha,require,semantic_hash
from traffic_location.traffic.inference_worker import execute
from traffic_location.traffic.pipeline import run
from traffic_location.persistence import publish

def main():
 p=argparse.ArgumentParser();p.add_argument('config');p.add_argument('video_root');p.add_argument('output');p.add_argument('slot');p.add_argument('--smoke',action='store_true');p.add_argument('--measurement');a=p.parse_args()
 config=read(a.config);job=next(j for j in config['videos'] if Path(j['filename']).stem==a.slot)
 out=Path(a.output)/a.slot;out.mkdir(parents=True,exist_ok=False);video=Path(a.video_root)/job['filename']
 require(sha(video)==job['sha256'],'Source dataset changed')
 assets=Path(a.config).parent/'assets';r=read(Path(a.config).parent/'fresh_request.json')
 r.update(input_root=str(assets),device='cuda:0',frame_limit=min(96,job['decoded_frames']) if a.smoke else job['decoded_frames'])
 r['artifacts']['video']={'path':str(video),'sha256':job['sha256']}
 for k in ['detector','reid']:r['artifacts'][k]['path']=str(assets/Path(r['artifacts'][k]['path']).name)
 write(out/'inference_request.json',r)
 if a.measurement and not a.smoke:
  cached=Path(a.measurement);fresh=read(cached)
  prior_path=cached.parent.parent/'inference_request.json' if cached.name in {'result.json','result.json.gz'} else cached.with_name(cached.name.split('.result.json')[0]+'.inference_request.json')
  prior=read(prior_path)
  require(fresh['frames']==job['decoded_frames'] and fresh['source_sha256']==job['sha256'],'Resume source/scope differs')
  require(all(prior[k]==r[k] for k in ['conf','iou','imgsz','preprocessing','half','tracker','ultralytics_version','seed']),'Resume model profile differs')
  require(all(prior['artifacts'][k]['sha256']==r['artifacts'][k]['sha256'] for k in ['detector','reid','video']),'Resume assets differ')
  require(fresh['records_semantic_sha256']==semantic_hash(fresh['records']),'Corrupt cached measurement')
  (out/'measurement').mkdir();write(out/'measurement/result.json.gz',fresh)
  print(json.dumps({'stage':'measurement_reused','video':a.slot,'frames':fresh['frames']}),flush=True)
 else:fresh=execute(r,out/'measurement')
 geometry=read(Path(a.config).parent/'geometry'/f'{a.slot}.json');write(out/'geometry.json',geometry)
 policy=read(Path(a.config).parent/'policy.json');write(out/'policy.json',policy)
 measurement=out/'measurement/result.json.gz'
 source={'video_sha256':job['sha256'],'records_sha256':sha(measurement),'geometry_sha256':sha(out/'geometry.json'),
  'width':job['width'],'height':job['height'],'variant':'raw','parent_run_id':config['experiment_id']}
 q={'schema_version':'traffic_request_v1','profile':'machine','measurement_format':'fresh_v1','video_id':a.slot,
  'frame_count':r['frame_limit'],'source_identity':source,'source_manifest_sha256':sha(a.config),'artifacts':
  {k:{'path':str(v),'sha256':sha(v)} for k,v in [('measurements',measurement),('geometry',out/'geometry.json'),('policy',out/'policy.json')]}}
 print(json.dumps({'stage':'counting_start','video':a.slot,'frames':r['frame_limit']}),flush=True)
 result=run(q,root=Path.cwd());checks=[]
 for _ in range(2):
  again=run(q,root=Path.cwd())
  require(semantic_hash(again['native'])==semantic_hash(result['native']) and again['result']['result_version']==result['result']['result_version'],'Counting repeat mismatch')
  checks.append({'native_sha256':semantic_hash(again['native']),'result_version':again['result']['result_version']})
 result['result']['fresh_inference_executed']=not bool(a.measurement and not a.smoke)
 publish(out/'counting',{'result.json':result['result'],'native.json.gz':result['native'],'request.json':q,'adapter_audit.json':result['adapter_audit'],'repetitions.json':{'total_runs':3,'status':'PASS','checks':checks}},metadata={'engine':'traffic'})
 write(out/'job.json',{'status':'SUCCEEDED','video_id':a.slot,'experiment_sha256':sha(a.config),'branch':'machine',
   'result_version':result['result']['result_version'],'measurement_reused':bool(a.measurement and not a.smoke),'scope':'SMOKE' if a.smoke else 'FULL_VIDEO','frames':r['frame_limit'],'native_path':str(out/'counting/native.json.gz')})
 print(json.dumps(result['result']['summary']),flush=True)
if __name__=='__main__':main()


### tools/run_t7sl_batch.py

In [ ]:
%%writefile /kaggle/working/t7sl_readable_v2_source/tools/run_t7sl_batch.py
"""Visible progress, bounded jobs and resumable source-bound T7SL experiment."""
import os,sys,subprocess,json,concurrent.futures,platform,time,threading,signal
from pathlib import Path
from traffic_location.artifacts import read,write,sha,require
from traffic_location.traffic.batch import aggregate

def main(config_path,video_root,output):
 import torch,onnxruntime
 config=read(config_path);out=Path(output);out.mkdir(parents=True,exist_ok=False)
 gpu={'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,'devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],'ort_providers':onnxruntime.get_available_providers()}
 write(out/'environment.json',gpu);print('RUNTIME',json.dumps(gpu),flush=True)
 require(len(gpu['devices'])==2 and all('T4' in n for n in gpu['devices']),'T4x2 required')
 require('CUDAExecutionProvider' in gpu['ort_providers'],'CUDA ReID required')
 jobs=config['videos'];cache={}
 for j in jobs:require(sha(Path(video_root)/j['filename'])==j['sha256'],'Mounted source changed')
 by_sha={j['sha256']:Path(j['filename']).stem for j in jobs}
 for p in Path('/kaggle/input').rglob('*result.json*'):
  if '/smoke/' not in str(p):
   if not p.is_file():continue
   v=read(p)
   if v.get('source_sha256') in by_sha and v.get('frames')==v.get('metadata',{}).get('reported_frame_count'):cache[by_sha[v['source_sha256']]]=p
 print('RESUME_CACHE',list(cache),flush=True)
 require(len(cache)>=5,'Previous five completed measurements must be attached; refusing duplicate GPU inference')
 write(out/'locked_config.json',config)
 stop=threading.Event()
 def worker(job,gpu_id,dest,smoke=False):
  vid=Path(job['filename']).stem;dest.mkdir(parents=True,exist_ok=True)
  command=[sys.executable,'-u','tools/run_t7sl_job.py',str(Path(config_path).resolve()),str(video_root),str(dest),vid]+(['--smoke'] if smoke else ['--measurement',str(cache[vid])] if vid in cache else [])
  limit=180 if smoke else 1800
  with (dest/(vid+'.log')).open('w') as log:
   proc=subprocess.Popen(command,env={**os.environ,'CUDA_VISIBLE_DEVICES':str(gpu_id),'OMP_NUM_THREADS':'4','MKL_NUM_THREADS':'4'},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
   def relay():
    for line in proc.stdout:
     log.write(line);log.flush()
     if "'half' is deprecated" not in line:print(f'[GPU{gpu_id} {vid}] '+line.rstrip(),flush=True)
   thread=threading.Thread(target=relay,daemon=True);thread.start();start=time.monotonic()
   while proc.poll() is None:
    if stop.is_set() or time.monotonic()-start>limit:
     os.killpg(proc.pid,signal.SIGTERM);proc.wait(timeout=20);print('TIMEOUT',vid,flush=True);break
    print(f'HEARTBEAT {vid} elapsed={time.monotonic()-start:.0f}s',flush=True);time.sleep(20)
   thread.join(timeout=5)
   return proc.returncode
 smoke=next(j for j in jobs if 'WEEKDAY-PAGI' in j['filename'])
 rc=worker(smoke,0,out/'smoke',True);write(out/'smoke_status.json',{'status':'PASS' if rc==0 else 'FAILED','exit_code':rc})
 require(rc==0,'Smoke failed: full batch blocked')
 def queue(slot):
  rows=[]
  for j in jobs[slot::2]:
   if stop.is_set():raise RuntimeError('Other GPU queue failed; stopping batch')
   vid=Path(j['filename']).stem;code=worker(j,slot,out/'videos')
   row=read(out/'videos'/vid/'job.json') if code==0 else {'video_id':vid,'status':'FAILED','exit_code':code}
   rows.append(row);write(out/f'progress_gpu{slot}.json',rows)
   print('VIDEO_FINISHED',vid,row['status'],flush=True)
   if code:stop.set();raise RuntimeError('First job failed; queue stopped: '+vid)
  return rows
 with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
  futures=[pool.submit(queue,g) for g in [0,1]];receipts=[r for f in futures for r in f.result()]
 rows=[{**r,'native':read(r['native_path'])} for r in receipts]
 combined=aggregate({Path(j['filename']).stem:j['sha256'] for j in jobs},rows,experiment_sha256=sha(config_path))
 write(out/'combined.json',combined);write(out/'jobs.json',receipts)
 import csv
 with (out/'combined.csv').open('w',newline='') as f:
  w=csv.writer(f);w.writerow(['category','direction','measured','estimated','accepted_total'])
  for c,ds in combined['pivot'].items():
   for d,n in ds.items():w.writerow([c,d,n['measured'],n['estimated'],sum(n.values())])
 print('BATCH_RESULT',json.dumps({k:v for k,v in combined.items() if k not in ['events','pivot']}),flush=True)
if __name__=='__main__':main(*sys.argv[1:])


## 4. Verify mounted inputs, old cache and geometry
All8 source hashes must match. At least5 completed old measurements must be available; missing cache blocks the run rather than spending GPU again.

In [ ]:
sys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT));os.environ['PYTHONPATH']=str(ROOT/'src')
from traffic_location.artifacts import read,write,sha
lock=read(CONFIG/'experiment.json')
lock['files']={str(p.relative_to(ROOT)):sha(p) for p in sorted((ROOT/'src').rglob('*.py'))}
write(CONFIG/'experiment.json',lock)
hits=list(Path('/kaggle/input').rglob(lock['videos'][0]['filename']));assert len(hits)==1,hits
VIDEO_ROOT=hits[0].parent
for j in lock['videos']:
 p=VIDEO_ROOT/j['filename'];assert sha(p)==j['sha256'];print(j['filename'],j['decoded_frames'],j['fps'])
cache=[p for p in Path('/kaggle/input').rglob('*result.json*') if p.is_file() and '/smoke/' not in str(p)]
print('CACHE FILES',*[str(p) for p in cache],sep='\n')
assert len(cache)>=5,'Attach V1 kernel outputs; do not run duplicate inference'


In [ ]:
import cv2,matplotlib.pyplot as plt,numpy as np
fig,axes=plt.subplots(4,2,figsize=(16,16))
for j,ax in zip(lock['videos'],axes.flat):
 cap=cv2.VideoCapture(str(VIDEO_ROOT/j['filename']));ok,im=cap.read();cap.release();assert ok
 geo=read(CONFIG/'geometry'/(Path(j['filename']).stem+'.json'))['normalized']
 for k,c in [('side_a_polygon','blue'),('side_b_polygon','orange')]:
  pts=np.array(geo[k])*[j['width'],j['height']];ax.fill(pts[:,0],pts[:,1],color=c,alpha=.2)
 ax.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB));a,b=np.array([geo['line']['P1'],geo['line']['P2']])*[j['width'],j['height']];ax.plot([a[0],b[0]],[a[1],b[1]],color='yellow');ax.set_title(j['filename']);ax.axis('off')
plt.tight_layout();plt.show()


## 5. Verified model assets and T4x2
ReID remains enabled. Person-domain encoder is an experimental control, not a validated vehicle-ReID model.

In [ ]:
import torch,onnxruntime as ort,requests
if hasattr(ort,'preload_dlls'):ort.preload_dlls()
names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())];print('ACTUAL GPU',names);assert len(names)==2 and all('T4' in n for n in names)
assets=CONFIG/'assets';assets.mkdir()
request=read(CONFIG/'fresh_request.json')
for k in ['detector','reid']:
 ref=request['artifacts'][k];name=Path(ref['path']).name;p=assets/name
 response=requests.get('https://github.com/ultralytics/assets/releases/download/v8.4.0/'+name,timeout=180);response.raise_for_status();p.write_bytes(response.content);assert sha(p)==ref['sha256'];print(k,sha(p))


## 6. Smoke → resumed8-video batch
First96 frames must pass detection/tracking/ReID/counting within180 seconds. Full jobs are capped at1800seconds each. Per-video stages, frame progress, throughput, heartbeat and errors are printed live. Completed model measurements are reused with source/model/config checks.

In [ ]:
from tools.run_t7sl_batch import main
OUTPUT=Path('/kaggle/working/results/t7sl_readable_v2')
main(CONFIG/'experiment.json',VIDEO_ROOT,OUTPUT)


## 7. Results
Only accepted passages enter this pivot. Pedestrian is separate; measured/estimated remain explicit. Pending/invalid do not contribute. Counts describe recorded samples, not unique vehicles or daily traffic.

In [ ]:
import pandas as pd
summary=read(OUTPUT/'combined.json')
display(pd.read_csv(OUTPUT/'combined.csv'))
print(json.dumps({k:v for k,v in summary.items() if k not in ['events','pivot']},indent=2))
print('AUDIT:',OUTPUT/'jobs.json')
